# TAS v7 — CPU preparation / A100 experiments

Upload this notebook to Colab. Use **CPU → Run all** first, then select your
**A100 80 GB → Run all**. `PHASE="auto"` selects preparation on CPU and the
study on GPU. Models/data/caches/results live in the same persistent Drive
folder across sessions and future notebooks. CPU/GPU environments are
recreated locally; the notebook kernel is left intact.

Bring **only this notebook**. It clones the fixed public baseline, verifies
and restores its packaged experiment modules, then imports them normally.
The editable implementation remains in the experiment branch; the
compressed package is generated from its exact commit. No separate source
upload is required. The code snapshot is also saved to Drive with its hashes.

Automatic flow: blocking P0 audit → 30-image smoke → 240-image screen →
predeclared shortlist → confirmation (3 seeds + greedy) → controls → P0-D.
Final remainder and P3/P4/P5 remain disabled by default. A session
budget saves progress; the next Run all continues completed stages.
Shortlisting requires the predeclared coverage guard and a nondominated
observed hallucination improvement. If none qualifies, the study saves its
comparison and runs baseline perception diagnostics, then stops dependent
stages without forcing a finalist.

Full A100 weights have not been tested by the local CPU suite. P0 rejects
baseline mismatches before bulk generation. No scientific gains are assumed.


In [ ]:
# Edit these settings once. Reuse PERSISTENT_ROOT in future idea notebooks.
REPO_URL = 'https://github.com/RachalHys/Tag-And-Segment-AGLA.git'
BASELINE_COMMIT = '624f74501bdafe9b7d6a05ae739d12b422ce8737'
RESEARCH_COMMIT = '4e3d533f1bd05975fad175738059426ee6da255b'
PERSISTENT_ROOT = "/content/drive/MyDrive/TAS-Shared"
LOCAL_ROOT = "/content/tas-v7-work"
PHASE = "auto"             # auto / prepare / run / score / check
STAGE = "flow"             # flow / audit / smoke / screen / confirm / controls / diagnostics / final / external
MAX_SESSION_MINUTES = 180  # planned stop; run again to resume without changing the experiment
SEEDS = [0, 1, 2]
REQUIRE_A100_80GB = True
AUTO_SHUTDOWN = True       # flush Drive, then release runtime after this session's work
RUN_CONFIRMATION = True
RUN_CONTROLS = True
RUN_DIAGNOSTICS = True
RUN_FINAL_REMAINDER = False
RESERVE_POPE_IMAGES = True
DATASET = "coco"           # AMBER/POPE require STAGE=external and explicit verified manifests below
EXTERNAL_MANIFEST = None
EXTERNAL_IMAGE_ROOT = None
METHODS = []               # empty = exact initial 14; external runs should select frozen finalists explicitly
BATCH_SIZES = [1, 2, 4, 8]  # measured once, correctness checked, then policy locked
ATTENTION_BACKENDS = ["eager", "sdpa"]
LAST_POSITION_LOGITS = True
SPLIT_SEED = 20261021
COVERAGE_TOLERANCE_PP = 2.0  # approved absolute percentage-point loss vs harmonized TAG; freeze before outputs

def release_runtime():
    if AUTO_SHUTDOWN:
        from google.colab import drive, runtime
        print("Saving pending Drive writes, then releasing the Colab runtime.")
        drive.flush_and_unmount()
        runtime.unassign()


In [ ]:
# @title Versioned experiment source package — generated, do not edit
SOURCE_PACKAGE = {'schema': 'tas-private-source-v1', 'base_commit': '624f74501bdafe9b7d6a05ae739d12b422ce8737', 'source_commit': '4e3d533f1bd05975fad175738059426ee6da255b', 'sha256': 'b3b448550d6607d45a23f24b11a7ae90ec6b38ebf8c1224e22c6b77e40206782', 'files': {'AGENTS.md': '7888f139b584e98115332f45d2c93cb787b9b3919236fb02a3e32adab6e30be0', 'docs/TAS_V7_COLAB.md': '8634b3341ca02aacfa9e265d37cc66bd3e677771f13453da64c500a44a8d22f5', 'experiments/__init__.py': 'fe7a0225cf031bed0d2204dbb763b431d530abac99b01ea33faa4d0f6b244526', 'experiments/tas_v7/__init__.py': '5ff13022425f78d78413600e40c3941e750ee983755c9b17a5d564609400edf7', 'experiments/tas_v7/audit.py': 'f6ea192ba422b551601e3c558e16429b5b7670b57ef4cf42fc42c7428df42a3f', 'experiments/tas_v7/cli.py': 'b16c5fbd3f0893697b332af31203fa97b686537f83b4663e71ab3b40861455a0', 'experiments/tas_v7/diagnostics.py': '2a1a56e626656d6172bf0c9355a5db5b60519f0ac82350f5003cf3416cd1f77a', 'experiments/tas_v7/evaluation.py': 'f3f8cac6999331c5dc67c0708c628b0cc81a25bf6045224b0c124690cba7c2f0', 'experiments/tas_v7/external.py': '7e9d9eb0f91f1d424c1bf34bcdbd45587d16faa8d2148eace5dec068cd97bc56', 'experiments/tas_v7/methods.py': '2913cc58e41e1a726d7e1ac8c7b2711a5259afa5dff4fa7e6f0d3fbf4d7ad9f5', 'experiments/tas_v7/prepared.py': '81b5d30c97e731f8e05bab40b944835333ba937d041c4cc135e601540d93fecc', 'experiments/tas_v7/report.py': '8bd6aa025a3ef41255b24cfca247f5d9d1779a1a6b32d5b2f246bd39bf7a0994', 'experiments/tas_v7/study.py': 'b48e222fc337c42a8a1e29fef4efb2ec7f40af10cc9985915ab5d8bea563ad5a', 'experiments/tas_v7/study_diagnostics.py': 'fe252e3e95b583423d59157d01364560a557f70e2ca556868ea00bb174f9bca1', 'experiments/tas_v7/views.py': '9bffe95f8996c3462b7e82b3a51587b9629d5dfe6d45c7654a17989ffb160775', 'requirements/colab-amber.txt': '7ee7be8843a02ea16d1333e4271ca0b8174445aab6867efcd46e57739a95389a', 'requirements/colab-base.txt': '992b028d26a85bd15ccd29321a8192bbc81efb6696df38eb834820f06377a9a2', 'requirements/colab-perception.txt': '0e98ee13388aa50ad35045d981a278d347fb0c56f1634fca4b9e5567ca6aa03d', 'tas_runtime/__init__.py': 'ebfdfb5371eb85b8640ab449c246efc862a11ea70e142718bbef4c9f7b5deca5', 'tas_runtime/artifacts.py': '37e4bc9271a13ca334df828af92b58e425706d0164f9f30384c81e8dde7b2c37', 'tas_runtime/assets.py': '675315d0c9b6d195e7941829b0bc93d3101a34c8f5205d9f3aa6fb9e0eeb3538', 'tas_runtime/attention.py': '94585b4980179e09acba88e0b3492c2161d94904faa7783f6f5a8391c51854fc', 'tas_runtime/bootstrap.py': 'd2a46d4d5fbe54c431d6b22961cbc0783fb1be15983d39d7b15deed4392510ee', 'tas_runtime/config.py': 'a58903c262e490dc90a3b1cb9ded9b6bf1c18c018a1592f4c9c2bd19c6aedcf6', 'tas_runtime/data.py': '39cb6a954ddf713a622630c584087fc3f5e8e5cd4da0d9a2377b7a7aa1be2523', 'tas_runtime/decoding.py': 'a5a370f0345e5ed6fbdbb612f16b9b285ab287bbb6241dfeb39189663ebf88b5', 'tas_runtime/perception.py': 'bf7179deaa1bcc8421d6c6e989554399155c1e391fd16c0be57185e43871e025', 'tas_runtime/persistence.py': '9fef2d9912d19ae2f20964167e121f2874e92ae3e3e695f3c64e96cbf1956a1b', 'tas_runtime/profiling.py': 'a77bb6b706612bcc2ecdf45a12756146f6db0b67f5a9eabc58f17ad7b9d8c4d8', 'tas_runtime/runtime.py': '068acf5e27671c6b1131374ba167cc464c86f74816a2553beb1758bac11d497a', 'tests/conftest.py': '5b20fdc9ab1817a64ddada47c22d695378148741eada4f98b1a745a49f7c73d7', 'tests/test_artifacts.py': 'b21d651d6110626e3e99471230b5d7e9386b91f30ebaf035ef8e9c5427d9f506', 'tests/test_assets.py': '6b51775e20a03a2c9384bc9922285075bd960131ae9368bfde5f73fec98b6753', 'tests/test_audit.py': 'c6fde3cbbc8ad8394a0456390e4b4f350871eda9fc7efa1fd2f30eb50bbb446c', 'tests/test_bootstrap.py': '1570933a9e6300e7e70914c960a4c323ec37d8a731034b87ca3808aa0ed89d56', 'tests/test_data.py': 'c55b36a369a9da72cfe17fc8c88b478cde2ae3d466bbbca67b60013ba18e92e3', 'tests/test_diagnostics.py': 'e5cb5776067150a2b1b6aff62102f7e881d0d8d88ee6a53485265478b2e02874', 'tests/test_evaluation.py': '4a8e77ae0c9e4b2723fb1fd1a40f42e3d5c4e15075ea2cc2e2d54dfd350da98d', 'tests/test_notebook.py': '4c6ad80f1e0811eab4fee9b07069268d5bdfb479f124b4ed6b3467c686e00d1c', 'tests/test_notebook_delivery.py': '377d9a8761c98b6ba6bcb2752d6f7549bb89f49feb0b109678ffb3ef5fbe85e6', 'tests/test_persistence.py': 'ae9b71a14af81561373fb273e58b773f8ced8fb8e4189a1206742cdbc682ffa5', 'tests/test_prepared.py': '1cb769c861c2f06cff94d17bd5027354a94f0cc8baf631ed3b8cc1019b797a73', 'tests/test_report.py': '65f0088060033ed91f2649124f7c11c7554154eedd3312b16cd7f3992113f60f', 'tests/test_runtime.py': '3db34ecd2dc499a6d56257bf95be6587b5d877fbd5e4add76341b4a51c285321', 'tests/test_science.py': '7303f63cd2f15c550ce45d78c5b6e904fe148a3d58e59f103239d84bb9ef2ea0', 'tests/test_study.py': '6d6783c7f216802eda1b1768353a1f7f780718728f57dc5e3e18e068fb6e7c80', 'tools/build_colab_notebook.py': 'd898ac0f4c1dd75360a04cf95d723ec260cc775755a384268bff1fea543d648a', 'tools/notebook_delivery.py': 'de704ff351f24c0ab3cf43a892e2aeeaef3d5d5bf87cd047f14ebe985886e8ef'}, 'archive_b85': 'P)h>@6aWAK2mk;8Apq0QNZ(%r001cl000R9002QpMNU*xE^TCOR84Q=HW0o0R}9cYTOgJa^bnxCTfoh>X|X|Tv`#KXVNulBVoi|@Kb&}fecwoS>@9k+M42=5-kUe`6)mIxhjpfHF`aUnXR}q$v|9X0`&;7Iz|josGMPl3*G1mBBje~V98-;p>!;iG-R}>JAFHAJ`<ws!`fWA)H78mga_=3PAy}3?DjOWF*>jSFO;%|gi_0ijH>6xg%^5(d<?sD%-MpJ+AiVQdwd8!}*880#^$|4`ZOp3x{IZDl!(ehnpO^zlxQ}LQoU*fq_sXu3yKb&%I<zXWHSPpr@Eu#~cw?NAWQih&=)0m#Yq^@u=Jc3&%xV%)i9ScF-&Hr)gu5GK`M(q^;La&W6&l+3h<KA`bRl^~AH5JnV9Ltcv4Z8T(gz@TE+&oyMZ4@LJ-$w~<LKB{0QICSTAG<&Ozx@q?P2w4{c*8ce5ixUeVu(BKtx6EYG(t53G#f9H%^)4C(-<QQ~}7E^9{%uKuJK&pXq|s1^h(Wdz+>-O+~9QU4c>x$p(n2a)}Ho%F@GwdRiQn7R`_RK%X`RRdmmK7rc?Q(CU8qC=Bn}aTJE4%$pU2?giWtMco!-yR{aP)~B0QIM94PS%cnx>6qi^ADRlOWFof5THU!&P<p4fFmLWvK3WVNxEPwF4Zy2d+?#Za5W>eXj0f_2jwAkHoJ00dT5B|&Joru|NYW<ZS%USdlUa1k9d{zao28f6Z&6o(>j%#9a7C#E8I4g^eq~*zfe=9xwbtvMaodXSmXE-O;gP1utdG_MR0TY#AKNfLw*lm%$p>l1;^rDA=Li!IvOxsY5;-{TnKsdXW=HTLpye7iA=5z`OcJMR92^BQFQ>1EaTav1+?GWwkm!Avu3+*|rskhXBu2q(bb?AOI!1m0Dh6k*IVE5wU$O)WsGlSA=gBOoha})7Z9^csND(cfmQkKhLEn#eRlo%A(Hg7(RjjN+07J(IFH`d3Nwt{!g1S&ai-VBvWY!H?pAw-+Dv5r&ng4PEi%Ocug@ru?C~wUi#{iU%qTx*|D32#p1%S~=KqH$o9;hv@6L$f$l)J~C1HM1d02OtAG}N>PB@Sc;Kc28P28FGJ6>zo~8a4p3#!&931%j*9>|anz0|XQR000O8001EX*b#ipl?ng=RuljL6aWAKWN%}0FH}KOUsg9?Lr+XWLN0A&eOSSA+c*-v^DDY$4wJYfQg)Kb?07R(isLxhacos=&(>~g%Y;Zs!WaS=0+g*e?LX`v?w9QA1}Q0X*h?Zy1knBZ_3H<IPq)(>+PtP#2d&y_Mb1Y(_x>TvZdaOgZK9f6e0XYfYu4IDs(h`<1oDo`{ntfQVZ3=guDn(AGACX=?8Q&3%&N{+E8Wtgi7QgH)K0f5>UxAN*9{^Fy2i(5OMXFd<%6c#7XLe1`?|9_<n$58TkmwpBKBXHC#-0?xiuBdRnXQrP0H1rPW1SP__2f7`;AC-(XUlBm9lo5XV=bd<<VWlPU{-0J<T<BjrFR=N%-@O)628$`|s&<=g7G9O_m+!bSKYIIcyRNfP24reV-PYX6xpqgoj9V6@kIg(b??o=!n|RMMRiqCppa+3H-wQBRu@dOAztOx8@H`?^?6b^4P+f<tp;xfH)UOSdy@+UFwX8YX9gsBmnmbMYCblpNw2;6_sp!bva|WeAiYC4E6|WBgyg><)^Kj$El2OR`|It^Xx@VHz|dqqv`QvLO)ID;_cDVh|hQ5#=ca>{h1CfMEshnb1}Q)gS6Y2);q4!XkWgC;c8hgbsCW*1LA4Ayi~tU2iQ`{8HL#-&tB$I8abHMB7$)Wd52)YB<&cDGw{u7h5?6kHU`-NEU9!8`LRlYse%T_*6I$_%;{IH8;Ln69E!(-36W{ldLNk!i!OEm(9~M(vJGI%)LQ>YDw5SIFlZ;QfEO)V-ItR>qbd!!P((SQ_1$s>QXaql@4x?Z{1ZQoh^mLi8y7>?*e>i#*e8+GC0D*pP%wkR_f{wHYrRl-h(he@EzP@nsU!DgTp@^Th4Kf^h(b4SS`R9OQ|jEpI1^Ub=+U<ia?V=w)=zOjG^Z(n7B&6$348m7)@qxz`fFzZM63HIH*VQeGorNESejOL_FZf`wDqcU4}hwi-rZi`-+Z{ceRuu$t2ei8r%PHOU9LyMDy7U&_6@$HL8K%LPFbq7wWG|q%62s$KQN)O$~m0dL%=aJLV%*<{iIia0Lc~$QyFEmFXhmjol{78>7G3=|MhGYV-tQZiY3IXo99rlV(I-7O~D%!@^AFshGK60yjX*b{R8Q@V%W9WKF_lo9lJ&n(@(nUgoC<mJ&K}i@Ckr#(*N2qkUb1T_oH$Rc+YU>+4;Vs@WUV$3GNZ?f!SAlcO*_ghyaiUnQ0e*Wax#Vb2Wf0_kvK=MlGEWkke2c^l?!RarI#)-8|pR@d9m;WpB~n4;;yCLgQa3to?(gXD=pX)@rFCTE{PGts#GPcrFiApsb^3Cod->5rY7#%P)!*KEy0O`oseCXW$<y4=T5;_y87LPezIOvJ}MUc3R@^dMo?XdZVpxIK~W1V~aZS_p{0P9UWv#&tAXkqkNvzd*NNHxz-F3&&Vc@S^j1(irLFz_G(1$FUA+6Y-Q?N^Y{QKH#h}hCD>W1a3bibs)soz$BK;RMQI>GX)3D*1>;f#jD*NZq6Mrd(%M9x9X$}lK!~ilh#)Dc2ane^>H(>$k=_Okir=gc0Rn`-0>D6F*WO{=)NlwF^sGF){_Xtp^y2*f_WBYZuFlTyXR|ja`J^0?ng?&YsIx_@DuH;CPfni8xbCHZvm-wboNi5LAEqBa8}e-?p8_zHjD!@a3z(f)Rr@tiWu<OBro3Sjzz{@l5w@$ui7Z<GhoG7DYQPE%P~&zWU}Akg-asCNwtzxI=Ur`}qnf^UV9UZV-CErMR;fp}a}@`};gA5kd3pLR$2f_BfizGk6M<y=gLqi#D-ovMgDCG6mZQhdC&l?O3z7)h<@v?*>>u3S(L+3Py3&t3JmjeJ$@l|Sj6cYl57W<|uCG4+^Zeau#@331<Wn)BhCKiyn0QBX%finXEirPilV>X=(~vD8suhHa>DN4ylUqMmb7L7_q`Tr0yq~=|Ii*L<8WZ{+z?a8A=G@guL&&A%223F;OYPCwEk-*gJ$ivT-8Bq<?~__ca4}c0;!!czE44A0kbPB?nGp<Kwiix8W(yafe@|Y=H+_SN6ji>hxn7KH#V&JLfmxuoA-PJF;&V@fR!2@|p$xEMeh6n_T0xtoZ?`nG5d4XD%eaEJn6)S?%xo%9)+octiB^nZ3u1_^YI*^H#P`h)%2%yv1Sb82W*TGg)-cr_i@5C3SJVWVy;THnNdOun(5`hdV;!<#!Pa<^*^LfalY$$F($Yy|WZ-8K(kXe%l9kX2JB|)}^>V@e8T$SpgBoFII<)~Zd`ZZ&cL$xpMC7J6>=F<orml~vIo49oASKwM#n~$n#+edN0ARa-!3*1tQLR$xQ1oSoT!0rB6gmUFn=>cjug+9zOzGj6=&ROqv+O*i%+Hwyv>s#@;aQvPQ9uLXTTFT+wK4iJO{g=z;q;C$S-eVT*_cY)aqj!MU6eobZ9>~ZY#6#lpvDP2TOrrH2+|m1I(NjTv9XD)b#E|p-gJLG34Dpdg@O0xXu!3h%STL38HO5k>GmRXAV#uV?>RbS>w7U8_k$6I)4;y)nGY=YiZ5N57Tg~P7bFnUe{&P!O_T2^`QleV?n~GpjbQmBCodnIZRG!yn0#a#%BiHg1qG)X46KN@PnKFPmg4Y7-}vxee{;Hq?mo7LeVbTQwyuR7AmmO$A-y_<<YNZOpy=<6FY@C_e*EL1y%B?V>-m~{lE2K4slXh;Ux}j_TE~CEm)x$mU|N92tU&;JnZJ0IpX57h_@G;-Er($BLLft<@`cQ&>k7*Z1f%RnU#e_*nw?#KoRL%}H5fHUoM>S4Cv}tbF_z?^J|pqiR~|(I4rrw8y;Wf~9e5>oXOo`ekeJteSGmz?cI<uZY@1R?_qFC}?_is)fROukS-=sGK{Q6SXt*WNKx4K9+&RVbQTC<V3;-`XOxgW5wFd(g%bLx@s^~WGxsFF>;1jUmU?0Kwi{QxyoW*nS!?G<8_WjPoyNNvcXYtxXZ7js%ZLv^a^Hp4180qXilp^UpG-vwh?fK_oc0D^EZoX9Bz|Wk&`EG#oyHYN#a+yLQdiT+A*Tw8_H`iB}g(xH%C$7To-rHM_-j?zXEkv}Jve<#RKq$S#m_rFxYt`~i@bD(KyACJq;Je55*>#?R{vqtMz!+&k|8BVM_|`1hF=V|IAY+c2o!?IIO95yM71V7WpJK^c%=AnOcgZ*Eek|k8RhF2^tL*CX_SgG&)7$Brg0n56FZ$T(0{u;?jx=8LC%%60KQP8$v;8lLjvDuMd`}00d7v`<Vujf*%ZIi9SGZ_API5(m<wd$48@|EwX2Sgb*Izi5<dIWojgps2GX8oQ>fbHEP5kMV5Af4=K|IXali|5l|88mT0_Eisw`0F10K&B@_^wG|i(sBc-!!D$q>RjN4kXXsch-*gH~FL~g}g)A8yH?Cg&r8Nz^cTHfeguEWu0TU+rJ+XtF`VzIxP_pLZ2+c#q4gdP4mt^3~%4C|0yfZ?miG22ng<?Al1QiJoM)fnaU~@atEQD&%TM@et_lw15ir?1QY-O00;m803iVS)VpCv0000^0000N0001GcyMKMX>Db0baO9XUukY>bYEXCaCtS!!3}^g2nEo)QxFDl0he$EVt*>Jv<;BP@jdyjkC{)`B?-bi;cDnfQ?5rU;vls-JhfNKTjLMvSa51XN|23$7CSR<A5cpJ1QY-O00;m803iU4>Ph&n0002u0000U0001GcyMKMX>Db0baO9sVRK)0H!ojbX>MtBUtcb8c}0%94#F@H1bctQ$~B~=qM(Qn1rI?$bm)i=V#)R)+X>|FacHP#cC=dSW_7}_K;*;}+YZ*h9D5F#vXxLN)pP~Hi|HMwKSnpcBu^~O5G12@WPO4>;F(e%*-Ubj;ip%?Rt!7!X8Y`N0tJKh;S6W*k)kbWiVP=Y%20wEaUW6o;19R`^aYMaj)lRnFY`)it(B^;hx)kL*1P6*sgI}4eupJ=rZgWGT73afO9KQH000080000X08GL!jpPyl0HsI(02=@R0A+Y^WpZh4Wo~qHFLYsZUv@VyVRd9_bS`jt<y>2H<3@IV*RSZn4+d-ql455csNz<QvzoON$205Q@@~Rnp_*iqf&~ItT%@5@`tLcHZlD_%lF}qwsR|!7WTP*q&*eLpZj9%7-=+EbA<1^)<F#1dtMx-AWVR7f?3JvVQf-9#q}ENH<k@ABr^)&tiX_czk!M1#>qe&HNhQ1cx{5r{ySUhv`Ci2Fwy9xo9E)UM<Yf(WW_c}fmg?d{x7EpBasQ31<vNvBr7AO^R7EP+s_UGW>w7(YEvvX}G8`YtvQD-@NIOv0`94|4zgBrR?7dOze1p5yo&5^|pFd!$`cM#gAvIA)6|YK}0atY67HCkLxG2>&`4p%5E~)jC61tb0xKe5pUIfE+i!$FPskN}m9+NW9_A0C6z08uWs_IZw2Urc7li!ryYP47NeZFaTw{0qARu=p$ULE3O6AIv0Qza`{rar)&)q|b^$SouaQcs~rZIwh3!8AqkNu}|FI>1-%YaYfcStHjkHfn2Fk~5opy|qxR^K6^!3^k$nDipgC7;%`tmuUqZ_VQDlsi(NkA5>P&f66lz%!q9}m8p`nLd^9%QP-w93xs5@h*)a)=}qna6nSZ_5V;S-c8*%@3+w~?om<_lW~*;r#D&uf8{Fp){pSze^GzOCa$ls*N^K)q9@vTIUpGwU6_X8uuI7gI5jBCXl#$&b47!1Z+_dfU;V}C#a_hvz-Lz$1=j%M3v&<v&yX#Tm#(7|+{@$o;t$^^;lxx}pt#g`7iY0DTj(%vhqwiYv=sR$TzdR=DNq<GV)L*~ie;f^^mKNsbULn^HYUAm-huRA!KC_jx2Ns=rn(eM|!pve|OJt8w{G@Wf+DO4hKC>;}cMac?bGM{5LtZAkB$MfE2n}w3{Fu&W@@S=M2|r+CeJML*gNxy}WNRj=ngTJWQ)FPbU>3F};lheD`6R|@@o76b4g(#5mU?yy;mW@K=IY@(-xDrYnJlXNyw-GrBTyykwpK~7-voS_a#Rf*QaZnmuD{i;LjSzspRFl*-@p6$KR*2Q!++iVL2DjL0DAt5)?sGl{4ZtGDIjn<Rhi$?5r}`9Q%g&V?no=k1i1eWylja(yerGR^u71w)Gpbn39~`iG4Q?;n`8@9M;;6JDKD!Upc=^Mt(80E_4;0BJM<))M$>>jLvd|<C$PCt>l$37i5!wl{FW3Q`Jou28;53cudutBV&>?{HYv${f+_)m&f8uK9JY+MX)bF&Fw45F&O~N|uq&Uwhl0ELw1l|SYXp&8Repdz#Sa2^4vJL3eN;90`$!VI<8_)>%6I4tSu^lY$XbDa+__u^i?8Rv14B&25B<v<k~&XnH3ndPd%s5<C=9W3J{R7HpMUtHM?Oc(Z8kuKzCbkrgMLE{vOh!kaRn3#3v7%*1e4RjW`{G{qJ71Dnn{CafibTh2ToX%tCijd4XT$2@ea{uUcvKjNXegji@oSjn{h1idQhvq&vV=Cp{{^p=4n%FpAjph%bT%gcZlb?ngtztw%qkJM6s9d2K0!v=LS8-8EZQt@Jeni+twEiuCmzxp`eE`ZFd-1mibfF#-?C5n*`!*FtdXO2}nHd<W7}hB`XC{wh}u8VNu_!4Du9pDeFsliF0=NQr(J7J;InO0lb93TBa$w>F9!{d<P9x4UZpGnJGZ33Vas`a|Kaao}~v=4ZzC|IN3D|i7rVzCG|ak6|8S%2OwnFy22oVBR%$BKoYw?038UnF|UGgTJp&r_EkmVy#y*Zu9K-C$dU)a>tMbF!noDnHJ%EEY`FpzO#p2RS>MC6y-Yxv4C7eOIWwS~i8L1z2kf3EEA#Lp{AAvgd5#!~4Spwz_EEHdfL0%61s;U`6(Dj_)p7nnKe;bwlQB^>=Si=G*UvU1B#NoTOcm2F&O+xF;9#E!aC%eu)~r#fFuqlv0ObP|nEi|h0toLyk_+3iXv=fkwycZ|w0JCk^Q{2GGpZPKQVl~EOY1R-x8Jx^IRrT60oskSFRqsh4;m$=$e!I<t4*BB6>I^9c#y8F6*XR>ih_dYf}aYxVf7O~{wRNWhe5}_s6E@r*9mMT*+S-Q02AkYb^y)$@$QfB{`Eg`tqf9^M<wpAZvhCOE>{ZhOMxxq5a(9_Wgou#x6ALZj8So7!$V{V#sND@*5dBQB7qjLVj$d5I~^$fqXd(E|M8oD#)ug9cm+>alF{@^W6>>IJJ=;auE?9ZXlgsVvle@?w~o0NxN6Npwz(^gG_L@QOcIipAsE|66%ffIV&q%#YrYa_+2yUl3F9b%U?Eck5Kiqjt(iMquYoX<9jucf4F(zUZyZvdAYwF+u8;td^AQnAH6XPDeOZRX>^Xv##~58;VDciE2mVG)R>3g!oAR#%$i)1vIP#dRb#K#FDM04BM%~CW-y=NH!Jx?)Km#de>Loi7HiNNN39BjLBMtmQ`~^VD3<oVhh)EUN64wI2DS+iv!bpY?HxlpAPXNHlTI-UYjSvK|JQGg{_!4lj0bGTspdz=ban{s+ZXpY&EPD;w37RRItW+rg>PFA^{d+wGQdbGF94|oYKIKii5y>tCZ3n`?1j}l6NH~>OfH{n-OkT_~LDq0Fs{<@Md36gR^FiOPza?GQZo0ApX9rF=YLz#JA=*%Ftp0=E$@cgF@p~QZ+aChho5+1&L1mlrcgVZY_!76CEbbQcmM$>2n+H2`x9{1;KXauLohtPJB<2MU^IA<crhIpOb$4?M5jQ|I=ncC|RywU~+DB>Zw~hJ>VS+*mPg6=Q8}#O=jcndJn25x`D^>J7I%lGZ@zh`RAqjhPE)6T730@i##L0gJXN_O6$3FRuw|_BGsu;Ci;5!veXaM*rH$_V}>_HGhGvRx8*Ip<*!1%z;D;QZZ$+F6qemB|;#;G`2ITCLrL<vmKS8{(3zR}a=V3Y%Y!>0p03|rQ0>O-O2Aus*p%3=bhSg`wHQpF?!f#@Sio5xe9iG8sXj`ir#k;nJ=hJCvA9aBLeWNQoGli)(d7T5#_^0XVmN)S7Gu8w6^2kqA`rU~dwqH{fM+mIHg;x>>3smj!IfQs7J(h7#1aC>vE{P!RA>Qb5?6#EUz&|`T0I>d5|)FXr3TD4oV9`F9(b`L-{(4Hn$zi`rKB_IvAXggNV8$56}I>JY*zI&!nhwTfL(TI?(!Y?OE-scO_Wez8H^NvYTTg(TBB2cu7+f#9NhW%V(8OZ`6f5_nHQDgU*e*l%1t&R88f?)cHZWu-XP7LTgH1h#7e77n(9Ku@A4H)Ok#AuWvRv$(U?&XLSu=E8{A8lp`%BRaYU0LVjwb)WZ4hd4n(H98Az>VV?Ah<H$G=NRD9qF3*p@S|6DJ9ntXhV_svr=3Sp}f~B`AqJ~3BzYoCU?rOW8gGA6W}O;fPUJjj~ezF#(m7dCyo4=p&v8$a}EBj=pDRO09{};DuLKwNZWEIh8)F_#pMn7T9+bLQf>Rw!AXy@GZ>-muQzZuu1{xMBiALS;#^Z*MKf%RgCs#TwOWh`gAWC%WKk;2|8X$eq&Hqe*5`*<d=Yn_bPWdbPi(Ua#w3NPDG=#=PW|T|0|`XRRH&VAy=RQRd>9!-ag62cIDP?~frb~aW2$w_!8fI|4mV&rUM+CT)m+>|v`DSs{Y(vYfQBo9Yzr8)Q3VfTx0!IvgzJ|}t8;rBVcx~JGdi;BCJUB2w=o&NGQ}e!!wR|kq%eQ4=s6xJWEFP#W&RC@G&#_Hh{W9^#F2WoE=Fnaq6rSGB-Mn$9yXWy&I4!rdRl!>MY&yQP617w0IR$L!mvx)rLMJZn>4+=S8^kMe)m`L@rS?N{n%M%73#tRE-1_mjL2e-x$ZF+3nx87MF^aeFe|}=zwf{GG}kzVjJe#i7znv5Ae~t}THGw1G!I7N@}0Ek>C!%qe8xYFkW90_kC9{+buV;kb<;mk`9-FU8nJf|Yeu?;h9x1ckC*E2O#&<w@UKiSVci|dgchl8K`UxiqUZhPPw!ahy2q*kRR^ucvQIfUSON3U{NuAcu+{a0-;q99HRB6H4&U*(Y<TNoW!-wfix)RmUNJtxz9(UI)~Coz_c)sJ<(L1Vyk6()t<1HG&IakAt8g9cbs6GVLg*%g`rZKEXR=jv=uM_o(R`MZ>Ebg5k(G`j!6-fE1J~n)$3-0Tt1T`qmTq`SMUQNwK0yu(2^k$vXU(3&Bp2<1k)u`6;#7?mHaKY7s~OE}<^JtWE#F*(AYAwd76JO_@mgMVh2=yv?nq%_SU#Q0Ox?JfjsqsI+w?!$z!xMvF=Hu;uf@%BdJM~J`Z<$2{rudhGdLrZLSU{YLo?qfaHqWEg6!_;m@^+LTIo|Bihzcmp)I$L$JQ7+bp)KV8NwLY5UCSxY$7CVRR=pQ@hZacoaAI)GFFIxO=9xDrAZ6MN1AGRWFCD9ig<@|Gfe+%6?|**!xt*ryks1It3`|POVQ6-{+|WWG?Tt+3wxm>XO7ZTxsB-xmMV3n%|we`;OySg?XFF@t|xTeHcw_<G#)?x(;1^#fhUukAksynigLpdV#TGqC1<`?1G|%*{&9|`6a8(gD97Ru`%FB^il)}*xH>8?<&!QH+;W62+T+YyLBV>PH$LWTxoT2b9s~yPo^#l)Q-`=+LI?3W*|frWJQzy338x9W1qo3}So(m-S;E{C@J8;Hm;-Ke^HE3Cp~;I(Wl0CxbPmb~QvAd{bv6fE<|4noE3iuHCk(+crhb$u1xvdI#14X0Bm#&i6wJ(79mcI`o*_$nXWQ)e+xs#Q6)xL5fZ9G{6F_KSX?Yau4wV~(wu(ckkAp0W7R-dXxx2JZD!Y-0k+P|CkG<5Q-=Pbmdf^(yns|%3qI2GsXcA>yff0@PtWIUx3o8@VS-oSBh|Qdd@@GpeUP5s%tBy>J^s%(4E~~c?o)M{GD(Bp{Nw~C$H)S|7=<g42p4ubE-8itWPO@h7Y<Z%#*mC~&{W`&TWy;nTl(|-q(av1uF$-WqI*eFeyAjL<{x1ugx>Ljs{k{p09T{}Dm!4nF(HbXt91G3e`y@6Cf(b6eKnfG|b|y2Dv(nVb1H`fDWO5{^Ie2j5;H044Jnj*z8jBh2W_h%~Nv=X3)8$z03p;-I=8k?3AVQU^{OKFBFVpcW7r5rH%(PyWZP{_MX2J2!7JtK2%sQHL9lhIsUh>sV!#9>kYaFMK;D#!}rKPp3{gVU8%bnG0BFA1N%LN!uvE=*FXHP{H4phMs%jiDIE;wuS=`~Kzb9SBKJXJSSA|tvzp+{jg3NUS(rfPae`n5@{E15hR4^7?7dYUe-@wj-qbdw}*Jwbp;ijFd1;zgF@2BhEGIN6iI<aEE&ua8ZA=#?1z1*MZC%xg353rWYv1#iFyyNWSZfSDa#QJ;<VY8<nZyPFI`TtKLrv=-;oOU|yFoLfIRw~q2AKk+JG(St^=058v9=+)JhH#wlsEx(+Ht_&EYWB_{J$-~~EHg#1V=jfUB_N+iBIL9C+2t92WuRB0?uXjAR*!v=$acT1&PWz^YK32Z9juesFHt)wQ6G%<)hVfzI)c7<nz7T(v<({sz@A1Ad7-h(Zj*Q|GlWS0vWZ&#L^B1dTvr~06F&!SE@#Xl5U>TD<Jodyh)c4<Z$EX*d2`m4wGVDofJ_|110uve@hWD@E3<Ynlz>c=+svYGzWWW^Z1KZ4lugwGfLauJVm<m}=h8w0hTi`2@n<GG}SDNd8r#td19aUVYGQQ87a{kTluCK*cI!U||CnwOauwnN(zW2A61qOiO0MvqDdXhrP2QU~g$wRz4)T){rj`knCMn0xmW{$&<#lgs3MXHbcKeIs*tc5+83&neUWi`omAM+%usVx{3(x=2}q*f{}YreXu-a~Pn<}@1_B~>gRWs)Lp`9}gGYlzwx^&zHG>qvTQs>=K_)$%y&igNPJ)vfoD@cr4YN4#7;w?_JM_1*lrEkWR>DQN$anf2xJWkS2RX;a-#U_$$bDS8eg{Z}4zct7P<t;_uPEM#xu<MkC^bo%TA4`FMY-<I;}#ZrATzf$dM(`GB7cu<GA{;vp-UFI~qZ^dQ~GqLfax{TO6AAm_`wA=!|Eh+4VFC2B7+FEE=&ca@4EU-${V?|CZaLz_KX1zWbPE*_448P%pdoVzFF;Lx@EAnpq<w1H@*5V050FWknWD|be!f3jbgy$ReeVI4AdwkUmYnhL-Owhi}RT#D1CirR}?2VVCo2LBUN_y}{-n<z*xFXVqM^~>>r-=$6O0pH)(s{9&M_>FmP)h>@6aWAK2mk;8Apq7Ho1hE`001!*000>P003opaAk67ZDnqBb1!sZb6<8hFJo+JE^v93Sl@5sxDkH$U%{#`R?e1_y}g$`=pmP+o5P`d!R89I2m*nYXq$~B>PRYyvDp8;-|&Ye*-4A$A(pA(%y8zLZ-!2i<m&Dptka_7np>&b1^bKzWBY2)cv~@gjkii$*>0I=t=_Xv$=0&oh>ETE>`K*qoh3;!n{BjeSW#?x+iOu2Og6L;dRk?jb~BsBHNAyNBjWde8`Z{VW#W_R*PT|SFvH5;crzWhyIQWJjXQh|4p?pq-M3aYA}dw9k=rPAPsh4)OS9Rm5*yalyq$ZY^wW&tuT<E|a#oA@B5Q=*smi2GX?F3^Nf{w3GnH~nse2nX6z+{~Aommdfkn9;T&bn8g)`+gXn}~e?3?>Se~E0RmAo#3s^pV*Y%xffMm-6v|9H;MPG6Q<h_(n?Aobzne-T5BttwSb4qL5ibJ*}<BpgoT5?L7DbhRh|kzxxS)!unRTaFKZ#^yd+Kd_7RpiORKJdmnltok4#Y9rYB)Pd+-Lhi|iZ-rSf)e4soWV1KSyYn;K$X6`IJr2xPG@Ss>fljvOM%1#U{jvhCrQI)$F9T5}Bde9<+g2GXOESDkz6HOBb)U%Z2Y+tPS2q_c@@gYnPN`n7>*d|oo9or}^6!`Tzv1JT-~UNpe!agBo-R>@ct9<xwJ>7`)_t`VwrFJATVV>EpfIQigvz19LzKV9#V9z^poQrh!U{r2X~PM<vn!vh6-93qYDJ4uX{3uINLsim=cQK0uwY=~oUu?-Y-;sF>esa@pNJ^#&KJz?M9X+@73H<WDYdanjZjt0?V9R|RRJ|a8HBVyBWbV_Q%06<E|#~W{Z0@;kWJqdhYdBnrg)XeFMW5!MOnN!_toHHcM!L##zR|*BfT)Xm1|i8B4k>vjnK~s%hm1WuU{pxf_pMVGaAYom~(h*6?h4?tyray6$fvkOa!FdL4hwK;8xYLl#_rs!>ua3ZB$wbg|Y%T$pyooMWAJ8$XL~}N_3(nD4&M+`zX@j;^L4MD0`UNR8T@0bkK#3hD#{b$FAM|c(U^GWaX!m1Bn+DUiq{}aU8$<+vnxy(c0q}zn;zM<nW%_RV7OT@mg#YV9Hs<HM^wLI%@ox!XF^J4DTuaNKqqAH2*y`0t3*Cg5p|D`@u;|0iFzSDu9#NSG3?2xE5qesRwJDmUlq3oee6okgl-hMN2CRzviX1Qq*^-k;RsBFl7sRn`Yv*L%LCudY-bC3yewpcGcUix73p!9=%oI(g9v*CR=7;2xlWb*|t{ed2+ToJ6lCh%?d!=3caLkzh=~qlQe{Y{+@Z=3n+j0sPkZJp!gbKHFTtJ-w$i#lvXd~kkj~(1iDEcKTY7^H$Oo3A$bz}V#m!cd3+=f)XPCKaup8kBs;#Z6|a)Vu_3Q2bM_44hSuYRQ6;vAVV38B{ZsNdc*$jr9uf8aU&{V+tjuE65fp|HpbR|*p$i{FLkvv_f*~?-Wuy!F7Y_J2n@_`qIV8#>rs@T*q!Wp=Bbhi@oQR=SOm&~?cl{hY>GX}Ngo~gjv&6UUixUkLLnn(Mj=aE%86A-R28V_xl>D<)J=h%OfzyYeQVF;c@_;>v<8T~Y$xq@$Vi`fi%qJ={hifZd?L190sGcWLN3sBbr$NH8r@O@~$`o*zNV!0au%Di|m-QVn`qaf3oIE~Ed6FRC8moK4K|$1VD8c*>-5>R?$cs58FjPxQ={g|E_bGe8{NfFN$7D5i*5~2WLztN^M)moroiVWU_%MAP1D3~!>FYH}4I$wj`6#$Gc_7fq3-nlSj|~qJ1nSd$wE9*^0^Mf3qozBD@ceB|rs-tD5ASFQ2`Uye;h?u*g_0Cj)7Y4(|IVTLIsbyD85r4JoixxLuTj~h-uA@Eckay`RaI))QNzoNqEcm1q@$e~ud0HFE%Ri#^ydcDwfM7_=&ue-?{67;^K#jNo6$~Lxl<BXwS1oV`y{PVhoTSHUg-gIPx(X-bn{BmBZIRc$p6c#d6M0d=%XGFI=pm81|3s7N0p4njGd^vJh=vZma47y?C9!?fk9vr^cP#AE4m58|17BRe!&^)3&B12K@J+1qcmJ0-5{Nh@0*TvNXCRasTT#4D3giyS!xiI<DI*I{c-#Im)pzx+xwfF>m+p@<vekP>h`;MJIbZt4sBC&j5X}5A5(6$R(e{i$vx0H)t;Zn5eogwU-9qu=nII2Y@ZQwh3h`9_oBv>iz$#bJv#c@&7N9SeMzI~@%11}j_L)6JaXJ6|I<-1QB+6Q@tNn)H#P@!4g39o<$tsY^HmU<%?2i<8-_XQ2=nBU;P?yuyP|*B4z_OYMLd!d<{w>K-Hd$7`ERI9-=z>bwf8j~e8D`bb7Ez8bb@-}7Z>pydH8%kOjB^Y5h91q*{M5jJ!XN<Ie*`Xuj@B5`Nljwivb$M|H{57_9EnVhXJ$|_C@KZ10y&LVvsj<lY~M^?QnyeBvk^gQ_@}JJqKWFL5s?US{Mz}1@eRoRDrVyqv~fdPs8wK9zG1C$M9Q|JM$#14`z=leHCSIIS`MOE*nQ;dzuFEMEvh;g8ga95Va-CKa<C!e*9A9BzLq*q<GZ5-@wT?&z|o0kB%GARN|+LPGhp1yV<7do6dwLvS8BP2lEebW>fcOH)8eBkh~;%4RTu)EpO;|JIZWPPz4l4NUm!`>FhsHO9KQH000080000X0D?=fTdEHL0Oc_N03iSX0A+Y^WpZh4Wo~qHFLYsZUv@VyWNBe%Zf|pRX=8IPaCyC3ZFAc;68^4Vf$GDJl%p6sy-!Nixui*2_nJ5pH|^bMG#rS8BsLVO5Tq?N^?$#80YH$V6z8UMo{24yi`~UyU!Db!;c)nUkymArWhxOnRmN&prA02%yi(<@%JGX9GEc-R-K%7>D;0cz*Q=~nd8|Z|%5`4oDvfnG7(CDOqLOA%>Aw}a2KKk9hIfI;)vYR#iz<=Jq)N9+RApUlL{SPU*0NIJ;QNit*J-{6`9O=>6y@RZve=7KeX7$^Y4}jqd5$zmahL0=RC4Rh<Ef}NsRrFMHAo7e2lWKZp;9oTY9lML1r4$T@X6#-Wk#6Vr=T-@-b(!u*{zClt9bxP3<p=%0*`;Xp{W8@RhLVeJaCn|2K@pO7#C%!;wo2K<MdW_mQ7@?@4y6CKZJwfa5xyO%3>>`XjKCf6-6T5?uxPkWt>3`O(H89XHsjJjUjT60?=-k$yg1{o2{%iJghqGa6;z!8@a=;fw%!f;w%PsL|$)q2O+^yJ0mc>P1T)~wM=ta9wIc%U@-Wn6FmajKU6;ZrL5I>K##-~EQkc=uWr)>g-izmep2%^N|UL86{Ck_{pKsIakM||4^vUqyG+ejSs|-H^!!;6J*qPWmBT!dWhoEhPw}?M6}{V)1-K4MB>rM1hTs878lzQ*M3R6*#Nb?!hNS{B;%s;Ty<>%9zC_ckn4MsU$VgnlYVRO4qBKvdC>m*%t;TF2veHV7+!y>oM?#v=vtNq<JS$T<@d>T{#^Rfq_(zuvS;;ILp?8F-UZDnRv>yv_bAiDv+DzPdPe1%AWvZ3<RYD-VD$AlA4eg49`zS~#7`KVZ9w<W~J6zmh!*hQThy(oFU&7x@Fj7mlZ;$<@cn0J+g9Cm%TzpxbTbY3dRxAunR0)pn$p^+HkwQJ?41w#^<_%QfT~d>Gjog4SI9$3#BR^@=pEnOVk`~)qLqI9<<Nh~KA@ctw${Jz82^I84?jYs{19Yl7C(pC%FlvRIp1OdacY|{g2x4SI4k|!^WneKFi^+FhZ1F@%4y+bD|3j699?{@HB#>y-Y*`c;&nQhaP|d9dzW0*5+3)CePE8JKusJ$3a3D8k=dS3x=ZMEvehK=gF;tL-`PEBpEPPUCEA*VY)+>%Z;z;h%`mdVx{@geJdkPZmS1zsATPSl}M3xGaRcz^Di+kfNybOQNS{~EV-YhE*fy!5xIgc_9V=6|Z<yJYriInk*kx8MHFI95Ze>&+8oXwP(Im|8)Ck8+giZ`lD48P3#NBk!)p#J1w8<)g!a>IbVlU+80s@><;sQZFf4{Ghp$(!dl|9bp#^5Xj60*puX9z7<l00^);3BZ*lI9q^I*N9uNcG}g9i#o4DPULTpp5AI;1E48~%nAjxP9WbUT>lB@fC&ZFQ^~~+(-<O=>Ok2bByJyaE59jGjHG~4rnyQ&Az1S`Q?k@fU(P7H5I4`?P-5Xasu-`}JgbcYhr7c(sPt3g3aB~EDzho+uBfvFCYY~ajtPySsf5;gHZt9Mm>G)CW<kUd^ZWg|0By$F3`k)H25CxzbGe8!$ma)T5I5@H&mb*cE|<8cV_Oq3yBK#Ii)XX84(BHHh6nL<atI`hFo=_eLtiQ~+IdpLTV$$7<FV7y>T^!~bq)x+m>|Z$D4wKEnQ5HUK<2~<-=<pQo0;g9FDCTdi)N>@8F@yb)$ons-i-Ha&}R)uQY9n9X?$;VD(Y%iS6D6<y}7zcL%q4$cMRi{Ho3GB5QzyP-9Q)Ml4ZL9Lj*trP%HPa1?xaX!@0MxEqX@55YL$N7Q2N%ohCSDHpv}9k!JLn0(GJ|QuZ^IM}k+#%-53SjnA0tSOnLP;}<^g2zuY5uo+?+LNuL81z?HYf%}R6;Vm1^THpEyG@ZcA*kRg=mr288cv@p_0A^_LW1{hiI;}FTrvE3iTI}qZ#Np{P3(-HJzrfH?t^O}|HsFR08;H-2Wa#f^45z{@{_sS&(P9Z!+|zH^Mo1<B06WY{E>8!(R6|O5A!0ZO^zGtvdVpMk)`p%Z<rCX25Cm=!6?q_S&jzchH^N&uFzpUi8Cv+@vNIrs?C{jKh@x2j4xO<!55&5br37Qbh#DLh3pT(6Bjny8$}KE^bPJu<D#hYycz;bV(z{d80)&n|Av7=-*64;M1IVufhh8!9YDll3!PyLZb#vok0ouJv6vjCW2XJUP3d7M!ZGeHUx1-*|2J8Bu7K#AG<pcDrWVu!bi4}2NmjGjBjA|B*Mkci8gKh>ot-#6B{7q{H#BuSZXaz-wE#F5EYdj110HOEIqwOvq5MoVfyR1x-#$Q(w;vYh44f4O*XZ^v`qn@Yb!4f(QRM!fg3T&Hw?~XZI!4Z`SfUkCFirbDLi+7KNgdii`jn0A1R^ElBdHww7n^y)-*fz>{cPqRljN78$(d8@-{r>v<Yp>8W{jLiM-?A--!sg@a_7~1}C$CY?gyrD&Ra|5l7#Dh@wAw%p^fnLsb?b*@eF5kj4_mD?$7~A=WTbm&%)Od9Xy6MCIyIL#Pg`kcVwb-Y(~q240w?850ZHszwIxY<;*pt!{eju_%2r@{q3ii4kQ_a02eQ7CyNLR09bqN=>i9Lr#a);$?nxLq3e6KGDlQWE3+u4mRUFH(SmaIrW87i~=IcXE>0jyUAM%fRahJD7(xfJ6$5XlA`kzCgJZ9#r8m@7tu1C|LDGD@d+#tD=b9g|FXtbV~gDUu)Y2QW9tQLZ-({DW<km1OsBVr$|Rjx{Q#Td`3)PorrZyx@JIfXP9tTFAD=S6eg#f>ix=d&!kWg>xUQtZfTBnJHSsW*%j(&ELd*Vpe~nIjHND!7{@!P{1J_*S)!`oa#BbL=A81?ubunn3HKE}O=I0C!>6MVYE&qD<{m=(0I7ke}P2YYq`7gi{l5Rv<g3GsMU60RZY#A-BtPT^F@BmTz{->~57^G_%*yr=8go!Db9Rv^HVE^y9$akKnyKe>1{b!7Xl*;%T3l*(rvoeyJ731QCZ>ai_|W=ld}%dzyegxYyM#vY(mXJ!S>N+B>G;W(D=M(qc(RpHWprcSV_K7CD|@F2;l7_Q`BQHS)&@F)!P_DjO!OcSzyWp6PFm5p9FZN|>$O(in*Oiodxf@)M31?MxRrH-N~oTS)ZA?~9Dws3;@n8fJ-Ln0edZ6*d0Q2(X&@B23fNOK&FEA_s_bg4(&}L>$k=)2z-l^jkEhmya>zMD(faVkR<n>}dqa2aM|CvDtYfg?vxpN3BH{?f!hoVfjE2PFrN|Y|+QoHLM++C>m=#g~fHTEij)|@RZUWpZX&{VmMlL(%kPl!marzE7obHN2|2d71VpksyU>CM}er+c88h1Zcct7NWzP;OIq(#If2G$3l(1cq;e24x!IsPIRq_)|5}-<_V)QNAKpLzc>=O<m$(Ad_4?hnPh7S#=j`C-@#g43nCIcD&MA@O#+??g2R>qGpXhp8(;m0?9g)Pv*)46p!GOiQHO`iwU7^9gx2m}t;-O)*A0JC%x5sug-|_wfy{3cHXJ2ShQ*20<H6MeJXy5``FfVhk42HNZV!5m{2^B)_Q@g>Q?GS)LvRy~ISXEoO=cNs41xM(G-qa9*ZD*59GxpQbSTm1@4Bo<EUNB&Q*m%DiWjiQ(pT-pG6}=ceF=ritKxx@7M7lAO@h5uwk*2}i?lw@fM?3MIxB$d29tq$I46`i^VhTEpjk5G3kd4P7?j!h`eWZE$%sw%*PsHrgNoJpVn3+V)v~dQ`KN_L7e@8KBpeF(JclqqX9!`^=M0|%Ra(9Th+;Qwh%R_|sO&s)Y0d6}oIi_QX?l1-(TpO@{Dt3b&=<%*B5L#v6QFQ4AXc3=R&~RkHNJ82#CjbLt-n_x9PzGPE(yXGZ1?F7v7sP6&P7^Rb(D|Q&&a%#h!h;r0V1)PRI#kRzv9xGlB_F%JGncMNlj?wMw=hf+BB~0^f$@eKNFAq0gAEd%LZ;^a18xh-GSvXLXzLk)gdbUz{0OO$PAA(tRYns~-JlBHIS716Zn7s&c#1c?et-1?9ui)^_-XPZet1E@U;i-qFueCunH%+tVOZNg+?idM?AS%l9-X;sTv#9IwA5TILmyZbb!i=e4@1+^E_-u~#}5d&&8frd(P56i7JR`5Khhl=F5PwlV|VF6<_E(9v!`W0zgW<4cLT3^b2(i+e9Pv_ySjIjU27Qm1&0UN(Seareb<zQXPqFP^J*_lGQK#<sCMK-a;iHV{Cw}gI0^}x%ZP#NYPi%kE$gW9x0?v#OzkWD^3r|r0Mp3$<W0o;j7(@gx9G@giE^3MCDYL-L$zpDh1H)sT^BHu4(#Y$(_+&EU7M08wGhe7``m-@jpW8O3#v62km}p3(URXM_53n*y@JIKJ<1_U?ka2Ps13yn5xoDmX!SwE*6mXhLB^6#yLT)Dv4QBVV=y0OSd47BOfyKG!SVe|_U4`wBvZ3Y5JVgV(QZ>pt#HXAkFQ_Cuy{f`?VLwoK{^X12Mw0Iskg|>QyhUTWbIeKzIy*3!78v6s!)}7Jr!HX)F9Sp9)@911=9tGxanIPCFiIZoi$+vf-WCyhC8z6$_N3j#B8I%ONeu`444yrU9^}_t3`;lPYEnQz>!4CxT>ihVd*pWv@smhwgps3B`cPSw^a7kgs11{C|gdAOgtW>dV8uMeHD7{Vq;rhP9y4bWu*@>n=6oHH3Oc#iy*XA42tcA8bD9q-qW>Ceo3HrK6$dRcSGC~x@y>X-b2g5TRh8~8?WYN3<ymxkmr1f&?4x4jT6C5s2t`Bpag)%I#<>sdiL+Jtns2gWApNt?`EC(U2>h{w`tzY>8!=y`9T2ztf7o~ZNbpLjmXYW!CYXZV{h*+THKD;qZ4(T#%g91<ZnJF_#FaTK*dLoMsrXdQM9*~6Y*q>fWEa_fw&waz?c5P|D%=_reFpjJI+nbfOhgWqaRR6@h4n^TdQ(3xeRa)-U|)+O8ucmEoub*_JPIn%_fN14WgV_BmjDN>v7Zq2ln_aXr6E@9zZ_U&GDT5$}hd7c)eO>Y2NxT1wDYZd$jtP`53R%CNW;+kl%T00E>amC`UOLa@pHLtV=xPp74K9fU;ARTYFjNU~k!>&IH9bMWJzjmHJY``tE-@K-gz#42XRL4A&d;UlU9|YNs)~*e%n5Gx@f`PIA;WFFfnmcRctnP)h>@6aWAK2mk;8Apo&{AV+*5004Pq001BW003opaAk67ZDnqBb1!sZb6<8hFJ*RNY;|FDX>V>WaCy~TdvoK)k^di`0%58WQcF<}dtdS?ckA^!*^0fp<+W3ll6fo;7?Q9+42*{;uA-{@8u#_?liaVnXMn+j(mJWSs@o(YFpr*|o_^0@G8&CuRZ(09%S8FF|MKeX4=1nRzkYvydiH$ke|Z1ljUS|up9OKQB0t^+YjqqZRZ*(kFUp{di!u(2ndfm=f2ET|?+X9BxCmaQsXF;fkR(+Yr)a6uld6o9;w0Pv+AP~M<R~bElVYFhbiXZT<!yQ4ZOSq$E>2F?ak;6MvrunOhT-`$-~V&2D}X8Us@(W-;g=ib|Mc$d`**%xd2tFH(@^<gP^z`gV^#2=dijeAOMk0U0#o>L>I1HytCh-?w9mK(1H6{5doR>CD#!f&rNSKjNG0(S1*J;%epLYS+jk#-eD&@%Q7qMEAfR=c$V7qLyZ1l(ufG5K%@5w|_kVly0|5uATg`)90UJy?OH`>Y{J=Ma4U$4xgb7yD>Da5%qRKLzmnxe2C`PX=*LnqF1b9h(>Q?T#+p@q5O_&H3gk=>Zezwu2Ugtr!!9xCs@={-`H2#D&Cak~v@vr_)=h4Czf93zEZp*I|z5HY0e+Wu2LwX#kJiY;s6mhz)k|0Mr4_Ms9k&;#UurD{j_6wgd^%*<mV@=jr&BTInyXPR#hakKrnuCH~t_oe{SRJfG!PMG<5wRRtxDfbZ@a3MN<`sB_y1&zYP?dTMLWO?aU*}#!K0ngCG|@p+_?sZhRO;_GF=U|tNZ<mBuoPI%9$?N_I{pTX0sf|zCri-fWE;fkiMk1r3JgNLu5Ll>WbdtX?k_Rpe2+C!Q5;%6ds`NMl~x5WNRSsG{x--8UtzX;tZI6ksx>gaq3ISGIw`Nz{@>qxoD#<*5aM)=E(u21f)ut9!G7fz3NV2Rgf_>R=mFefw4$<YaIFfj++eYnY7^YV!0{$7Vp0h^%u`pT51~!hGy<>MY88hu5Dmhx%7bu!^6lBo8;wSux61X_Uo2KtS><Z6fXd+Ic6i+LOxY$VHjp9v`b(ix`<yEQ6oTq1G(sv>ky-^+QWE8)&8kYnQtPB{k>u4P48o0pkp(3UV{5@^8>uMw8BNHPzlH`NB{;=OK&<)cdoTo=O#P4Q*NPNFjeA;cNr4r9ni<`<`1t;(A0Ygf{^+w6*ob-S;hvwvT_c0StDnAk`{SE$48X<5zq~qs`V4T#o{xW{<>}Ma^V73XJyYT7+4<8bdVYGkS}k9ks_5y9<tkjAKYQ^kTD=HgE~97X)b`oSaQX85MWnubcK&?iO}Yd9{oVU_fB*htf2e0Ko<9BJ`SOcrD|LRheE#w*eDVD3>Dlt-v*0CO)bi<za}}*tXD?S@M8Wy#mr)o#51zj~efsqDe05GkdB4Y^fbruHoR;dyC~spd{G7y57SdHDqAz=9Wd>*;AQLKw0L75CsLGO<>;h>=vJ5AX`w~JEKtudm#rbvd75FdDE&uIwl|(3kO^M5@Q~($>a%EP7BuBK<lw@lIDTWeGNeM&bj7))tpcoZt^O?6W++!$7Tre7CfI%SEah^>5rQS~=q<G(H{<)5!=tHO}{!uG5fe7bQKT|p*#tkBf2rz*1%}q-fHfpxq6O>$92L-IbnjVXoOgb9GQlmi|#bbp!n~K0wMG{1ND%&wmOBw)J!q{vHLmGOe<ab@6L74|PDxr>BO~gQTAirx&K9Lc!xD2^0OM*Qka~<<fO2Y(sBxONa|JJ3JPKGW#CiI_**J(ZR)>vRAMh6p8sSrn7a+IVpi6l&#GDa_<L`ljHoeP~rj2UcY8X4fkQI=M_AYn)mEQdU3s3s<BxDON17^7@rUa*R%S}K(WvI1ol@rYrt%?ywmt^fv%>847B1RI@ki<=lK3!OayuVne`X!UEA+V^do#!?23zgAut016vOy^N{B4b6gD@k5~^{;6&-cz{J(!Wt8T%c=j1QW+7i$_QF;8*^zAZ?(L7Sa6-2abZg}qwaI5!AqVkYV;!Ib?dFpROG=~1Y@L%#w?RKHJm~PsIre+o$97E%!5_Q9o$~1SXKp*#k4F`z)v0*Gj-^p%*pKRJ#cS~`6n8BSd-;thTiUiJrhA|GQoL8G|A%dy2_|+p6`iT6|>n2ZZo&=z_KMRT}+Jxog#=6DVE9eBU$myp7?I3Oi__370F`=eW+MKR^XvMj2sHDgMGluQU+TnBa$ec!Xn}Urn^wm^+tiSp-gfXs*<O&6C@RIma-iQnIT?Tt<Cu12EbT51o>K-4mJLHMB)-fi5Vz~SLSI&g18Z$Emg3UQf6y;T*B!~ln{km&?AwZ)C~ZwX?pNjlq&6;tt=qoK|-}UYqzXlzz_83(IWVpeFV~=+&p;Pk)U|c;MeRs!Q+N3!Om|;v+N*NG&jvI6N>aMVU~e^iib5VKh{f@A9TM~#BFfYLstHfC|U_cMkY0sT`>sgI$oP<3yxdMAY1Z2=wTJh@bOwvBPgUCE5_qmbl^9V?oA+!QY2uZ7IGy8JFTNkg{*?qIw!oKt>8&V3Ml_tVc`fsWt!9MS8>5Qcv*e=#E9&mT&SbWw%E}msLL<J{i`G(+%;){e5-}Bj4Ckzv))=43Hwewa~UYxsG#7suoz-+@ET<?tFw?NN){s}?t@asNU-ag^aGh3nOl%A#i9i3ELKF~Ak?gv@R!tf!eSW9dSn&Y1@LZQ5aE$ht2ENwJq&i-TgWH9wI+ots{-{%(iI~xNr~x6Y)QE-mwPYLYl59+v=U$mLNOx+06K4~0RLsVNrUPW>vBkH0sS4(%*%48Vx9sxgJHq~E}6{a@A90gHQ9zdkHs46hz5Xvng@(LhF?>=NFxMg)3dG0Qp)$tH7V19pfajhT$S+?#%3)cwl81DGIcR)YQNEfr6Qk*?GKE6%EOx~xJfWSBuQ0qz4yR%1(P*lYnkq82}5Oy5r1Pw#1HVGd9E0oREQ?w&4NUOkMxuyxq!GOO!jc}B6cB`kdx5-Xpg5rMA$BAh^>8Gvno+Lm^0#^kaY?e=ayDY-?GhLHRT&b3Z){J*eGZ!X89GQRtvSJ{wKAp$VN~&0^cRukhZ`YvsaKtfrkcp;K4P6CW7gJ7-no2uT#Wpc=D?4GFOmi4_%{Wz%n4hlvPX>kW{V0fXoN2;*#*jA0KaCQNbMwn09#b<uR$&Ygl^3Epf3_XmlMHYzDmAN%9H`Oc22ZGzt3afprN#SOCgyK-o4g>jVy{*EmxcHeVTKo)G`bBSzTTN33XRo_A1o(hZ`KB5cHpU(@{L7sifg?d`DO7zr+w7giaCzSrJ$3ZfIEP($;AA3_Ffo}#M=V_F7eC4q5@|9A+*0Fac$j&<>}%EfD{mh1w<vA0DVa0Nelq?bT6@J|T#jLF@B^nqhb5)?0sq3R&XX~mLLtvCFH1yc#yk}G0B%M0cp4@?SEMvkPZ2Y1ke$N4SR*kXn>py`URF7zrO*$RyZfGAnBg8}0#tre7w;u(>8TtSE<F9S6Z(37W98k5e?V9LmYzrazs1gz!<5_IjAI%b}$@|Z0poS3a9cimPb!3_m0>}-@J+?^YCqhj`0OR)Y;%+9rWgcN&8kd>PKzHL^q{zh02R?uMoRF(3Os8ru**0834d)1xb8e$4wuHg144_aPWDTK&SDoVUL@<;8P`Jo2pv1GxlD?vD@mC}^NAL!TH#$-@nO+3vO1uJo*7<+y)KR#hb4M4AeXzms8-GLlLZB6MtA_kr#c6{nzr9KgJVtQT(|M1$oA!#ncxG13bAQn-y_eE*&Gd+657v$Cpm%_s2hygGzLZf9~6&p^V03HO!e@zX(Q}C;GdLc3OX!O1v<eD_5WQ!;Z6se4&nv~OPwAjfpVjoUYMihUd2J1cJaR{UdRVn6VraiV%^zl|QAH{L0ZcEBJ9A_racXMVsM{4CS7GS?}v5=^r{=vtDvOStMImC-Pwc`K7e}|R9aPN4b++5$GRt$J@QNxc{o$MsqG_~}PQvzvHUfax0D|hL&+BTprJI$Q&Tr?jT(o$-s(pj9YC~svkaC4%qjxErat#&R!XD*)gbGyl5P@RUv2VfYqxx24xC@l$Tp~P?}Au%i9MsoyJj3@1BQ_4dX0EC`1@{+UhM4FFBQ-3sRPt_TFR%H=*a(qWLzc6sFPUnmUl@T6NdOvAnAJ!S6OCtLxl|pBZTfn8`c+&LDHhEAmrdKVTMol<6nhw=)aN4b^qt0$cD{$#nx?#+rjuO=Ld)g4ETjNGHgS`k~x1ts9^m<vJqgT;*7u_n?!#G&S4o7d^uJ_QbY`lkVjWOWe;>LjV7qc-ZeH^T+?hR9`>mCUmi`<*B_;KChjtv}GEyou1C%jcOS~<jWw3kB^bgJfUnQ7SnP986eM6kS>M}RTy+FAjT1cFnl?#K>>NWCTKbHKntact0|0J--)czV|@vMza#?gJ~xo_e>?IPN{E=6UuIsFp5WFZwFY5Uo~es}Utx{X2noKe{j!gx4_Qv~0hstK&K#TVn!#owoJwZWO}$L=h<S$ZKYwY0*Z%kZ2iB8O@u;{VjvmTzvsR#N7B{FOt1nF6CFMsO6T59U*o{=EwiYa*e<ro~LyupppX0+_OG+j1AjYv8748oY`j9yc##AF|CMRt337ZIx8GO235W=&4C_I+~(^$?8sY%Xyb0~bXkTr&Q-LqUeiK6u|=^7&YwQ3`ws5PJO;|{T^t0~-g)VOOLY#ug6)q60GWNanH02Fn*NQuTM|$F(a-5<0>>tVPdRSe^~r3bZX@{0MLC{yiD<{R_{gr;1zW>7^HmUT*zG7cD)oQS;8)sj^uPPo+0tCG$NSy4UC|^LkNKXLZ!GVVo^}_f%xzegbsMkwEWv`$5)hI@#|PwWL48u-*qjtFzIf4BU~*_+QH0=i7wEA8y6+tqaA4dUNvXFq$J^kxz`9;Od;0W?XOmiT3>xrX7vw(`U5DJ%Ur>%zTx$RLPkvb;s=kuHsF?b**=#=b4-~`{eh#*9^dgT=j#kep9aGh$<>Iw{+p$4%mQd~+{##?lwh94#H1!)v(?u*4!%#rzs%aae=(NFyH}hlv-zY3)6pD@)YwRClAb&I?9+XyBv44SnPyF+Fo8kL7%=p|uG>+R15jr%s)zz^VJ&}|7SvYRnEBWo<AoF~3>7TcXRZ^(OfSbu30(e#z|K9I}n?<I0(?7JH!yifc+mjq(0RA|0BPVMWhya2G@8FKuXf%awl%c#S^%M2Hb=FOUJ|!%QKK_=oxSF7ULO^nua|Q^o63+bHr9rXPV(#^cN;q<tX6yxxZ09h4gmb7rAxqsCHQxQ+biidzLei(oA&UG<)UcGWjB7%x0SD8V+O`#8^)MbLjGahF`AQdeFedE$Hv$+U=7SxCB2Lw$$!@ITTRz1Cy&GPOPeNLgj-EKai8NyGx~=<PneA!bhA;fU?#m+BC{Tg|_{KBXgt6K0C-2F$p-1wZyC&+#ou*y|BD=DI(+WHVBD>w)khBZ!m2W~R*d0R0$t4PuI?@6*PUQd~c6uo!Aj}Fsg*Q&e&}vr|UZ{YSuz{m!38MS38mE!fMc_A)oxpHI!&cnaht-64f|!<*Z+*=rklkpoR$l|N6?fR6*NN>6Byp3aZU6*Pox7)eGNR!HOYipTJ3(0xCx+o1t4B<4?B~>1w@}rR^Q-xD2LE!~OX27xpNJu~V$v%eKf)ADuFmFN4N>=nzO*Cut&c_5+Miwn2{6n!pg=u~kp+rE`-oCIEK2R;eGllwHA5DKj-3f&*s;G+vgl2o>wLT#iJdd2?+oGmZCq^W@WTaSF9Yb_$!!hC8qD~FB6%Hl2`0U0+2pboje}{rv~yjZjz>paZNs3JyJ*wLI@Fv(eq+xi)k);<bPy3elQ@jaWbdZ|9T&8Reoye{%a%)873SDqoZADVZ@-}<zXb1{2AzN~U>sAfhmU8WEuFXI)54%nT5Xpq=c9H7sYE%T2v>9y(4k@)@PvWU_|9QeNPm*60P@H!>C7F#gmP$>Q1Jcl<D!r=d^8lw=xir}s-0__$Yv%$JB%FjnO*e$p?BVw&);romyZ+WmSm46yc=#a-u;58Qe4yr2gz6V|G|e3=S^!XZJ*Wn$QgyEyNDz7v4<7OJD+zF>~;IZ>?5>x6^0tO6YX@AxIH0|mrL7-KP*-~*)(F7(~dCdv8AS?+cwsE_3oR2@XBGxwpZ54Vwjm?QuYZLyR#m*I*69M?6@yX7R<jUv(5UmgYDme)MHyG0gZhlg<JPJm};qM+KBxHdleF56{;i-n9qTjbJ`U;q3kBZX=48x6qPP4I<%uq@3k<HO`xD&e%eDf3$l<@$$$e?1BOx~T*k!F<7_e`bs>bym|`s+oA<P&pz{cj7E8_t4JS1XnAf^;+8U8l5Hbq>OlAn&(=q2hphzx9Rk=ETF`CffU)tYK1|lv_PF?be<{2F=A4|iwkL@HCT3a!VH8&~@=n}xIkz*z1{XvNyfMmN}ji_KWABu9gsYrKAY^>{jF=(U*4CYTM=~rVu8jSB~fZyf!qkfFl%z@Z+++Z6=qxocR{g4hqR|Ih0Bax{6nm+tGNSj(dFa270&5%R%sWpR@`;Lg)`e8b?g=!W=QKKymbQo3&SGHS!=FayE_4HjG`8W-l)`oD4gXCyW-JRUUKRF*J%3$9C9&s_X{36KC2SH4{9*->ez;q*UO<dbUa@bEERJYE2OxC=qn+&R*2khqmk=8N9RMA4DrKh#Ui#f<2pT~VIi1%JkI_mGMhIShU(HxRWw#L1df2bQCa1{Rk$k5{qcTk2}TC^9{wM9xIuX!)V>yys-Y!WWIlQ_ISy~Wsy&+fV%w#e~3#+{s?8TYvBvOL(#!kc=7abXV=((_$heyex88rh)4;V(oc7q!n(T;3J@vEI0cz~t)U>~wx#=j;1TZG~ysb%tfTfMY;}hZ`L=$*f#&7mn8!#SZde;?M{+iauM|^KO9&L<Zv!KI<W^F8vWAO^V<~|9qTsbacH7@^vv3jPO|2X}Oc_Oc~A@*{{pLY|6hJ-9NI3qn}uj)KRfte@`in4Uc`Yr>u}83mUjwips7@*2@ckvTYu@;(jOKlX}$`cRaxTj8yli<Nr48?b1Kkye9c&{nSk!xHRr$Jn5rH$-+)+NW{2hdk1L>C~+iTl@6pthrF$p^W*A6*A4B(GXu-^SGBj^=UaCF?kuQ#e@7qW0UlXD&VH5J=g%7Jud|%>V@)`f56d`PEQxT+GhaTUK5bI0hP5v11lmEB?e%P%?elrAU;VXmcL$(vzejYMH{k7SPM!A~SU(CahyVjEcw4UCA3!@i4m5M)Kr1u`C@t%{oIc<SG`W37!riGjusV~0EeQEG8nmpoV;<MA6c%O0`3gaWK{fq(XV$DiIhIQJ4)x+ZoZxHtzZ0$2Zooz3c@D<2PNviAXGbX7CE1a={mDO_47G~UN?M$7*3is?Q#Z8bzy|nqcG~GHd({ieP$uOloqOktDmB~?HkOC(9!aAzx5rQ;L;HT((^O1)XLGEN*XvPN^^>b6Qm|X_ogoa#n*Kz#Ml_jpZR;fGxKz3>H=+j&#9Yl*aT*0lGR{XoFRy;y&7X|_{o?A6|MOq}eLUya$<fctQLRHKU0fY<*~ji|SgowfHUW%pur9j*YmIWC$3d#qin4o3#_Gb}X_@=Rde|?LyN3_t^L*C{FdtUW`J}K_uh(i{`w~`2nzIUR6T^n8zZoir4%mqGRFz=iI_lFyZ5)gS8XX<o4Z6vUKOa6FtR`K=pEPk3yEU0dHsBoW1fE2n#_g$*Z;`FP|8YNkB>0iB8!U>$d%K;io@q~v!HdvUE+lP9LWkSd>zuAEP}XUMTA)wqim`O;OM&cXpzGZL(OYcT-}@M{NE~Kc4(?|xJEp0<k3tN6cm_2~*ExI3(j8_sX>{A(JM4qL)0V5PqxY2`Y17+WqTs7-R_^J33rYmgUJ4Ys@o6OKom$kS&~X~bJX#s(T7da5PY4XxzFGhsHUD7$`r;H<S}mZynr`oD9_CX}_5mOigc9lMZwLfz%lC5TGiYHRoK;UaMoqj)4$#!CFquiQ7k7P4G(6C~BO#4j;G%5`4<QO0mv%w8K0|W>&Z>DHuzUj^L*Gtycmm#X86<rNpyyJ!^J&+8=Cp=gj!vT?cVLA$0j^&LZ4wyA02LGV1Zt)oB1a7j7(v%9_*0|KR0e#U2l<Kqoea32u2<r@2feyTbvzRnmsZthQuUL{^}?*X)qeCLkpb-unby8qxxDmGACY5s`mGh~9i8M@ygonaq*v;V2U(A=Sn~O%C7*9vGARkG*6sB*h;A~qXs+RbTU6}mQzSSctj6(MZ=1iN3w)ZJZ&q<$l*e?L5^4Vv`nWDQksEq|5Z|A%_s4ux4vi>!xlt4T@`}$MsauK|Wjy=I%nU~TY;tuvpXKy{U#P~TDP=K?ezzwpsjux~Ke92(Xq4)aFwA~z&u>XL_tYhs8iETV$<GKZ&m1+{gG&vO*cLH`#$L0=Ll#enYtlk8rIs%zMP?0>V5yQ;HJ6y$n;?&aw6uw*R+8wSHmrPqlW8YcMp4J55@R~DMyg6n&W4Z549s>)xun!QtZd1OOT^tWd;<~RQ;2GBRJx>wa{hFX?;A{c;$(-)+)8Rqo%zI`f^`*v^r)RdS}j??`$vzqyfx@CG1l&-X?F~@y;~<sMqhRbUd^h`dj|vxE0IQg_#TGIZ$1=YvZg1Nj^@KflBV7@UqNBd^vHc@@CczR$f}px=i7Fe;|{pMjlg4qkLZG4FIjFnDeHo{G@{dNpvWYi5Cvy^d<e1OfBsw?AuvSCf24ZC5uqg+TD|@AyZ1l5dHw3+n~BY!droLUIq^}+;@uuum_p=tHJbQOI=S;vHqeRDHJ#GYfG!q!xS|zrjs=(5)Gz5S1pKaudSkbl32;QkK!`+wM0`Yc%qMw3C5+97r)37UKgQoDfXbHFJlDw!t9|GzvUdB_X`go6cf7NsBSF=5%j)=D44_kG?=_<Xc?`cyZNgxnbv+--lG$gcUXVwnJh-mQ8A{LnBZnkMcCMIaQ~Jb|w$wqty6qseGC#E9)_}Kp`QW?^7u#NzsUK|*^FgPi%%>iyM*Dz2<*4)P81^L9h;QOxooc#6lD~YIHulxrk!Ox?`Blmn&?)zev*sHe=N2`m+87i1^h2K7UI0*Cry}+ug(gvNUWN>v$l7q#(ewe^C&r5|fVv{W&ug#ZTRN3SD$Zy>{4!opWf;NiRnyft5uH~36lcd!+9~`U-e96Yu=Xj)J}9fV;LiO{!a+fK@~D9{DxZ#|ZUefP0XkRw#sbT<TB*GDH46HQtv-D!TdAVPL>HM9uEE4o&HbLs)fAX#!QLfjHJYHPV?s<OBu-sSO^FQ4m&3u9N<Hf&rUI353n}XFw?1y2WMhx%bVvCX_fV)a9E_-cQy7bCOGe&9`>90B!K1NvM@I~X_F61?jjwbe6TUL9^J&dz7lt^l!R{VIGpbRBHnOmw3tnk^5gm+VU)b9Z4Z(T`{T;W(-Ta;oG?=XwoYg|9Nc<H#sr^91Hb-=7<m|_^NfAygsEL`3Tvod<*VHbST9+6h$OLsdrAlA3FCKGfL-12fo9v6vbbTc^RExfewx@=AnF>nY1SZirKRq?78LrEUwh}^R;!$!SXD^>hEs@OjIqkW?ko*sFF>j`z2oZGA9lmeP=`n*nU29eq^|XneMX=52ZYKZj_pko^=A${v`wBLP#*;($R{OAY``p7`01OCp0i|3b3xo}!={j4UvS_1%7miQFv%?nn?9K7b6y_RRW0C!oE*g?H6p2$4GZ64E<P@&mR0lfE&5eTA31NN0tifw9r+5`fav}bqx+NU>z)5|iw@xAP!ubE@pPioC<E^K7Z5D>=9!%URzHXJ_{OOXw5CNV3q~=biXA`49jpN*9#O7>mm+u}J`)fKoYi?&Yp@w^Bs`Z>n<eXrqVTOiH=nUf2o<pQn7wbxQ%H)teo<xnvxQqO>hUv)lxHlDNz)5eDza1{g@NH!+Zt#GznjA0{g8OG^{MG22nyU;sg}E3K>-^4PMgc9JuZ@$u`ut#7Q%FpZDjQ-vWQ*IL`}cf=gHC+iC}V{A_GS_em+OH<@fF{1oS<R-fyh$PU6}Rd-KjZL-F5eyW6qa$e~2b*@8d9}wDY-lJC%Atx6a%W_izb)GRddzrp@6@#RvCkU#K2w;_Mock^p-`6WGOUo%;~2$}x-0Uv@ti6mvG5n4TU^P8}HKKyvS)L5FgE&A`LC<;O;^^OUSj3@m9lAHIdgAVpq(9S5}su$hCw4d+MqPaoO|eQ?|O;Voik+vU+L#!t;n=YDP1?d<O4x1Gsouau$R)IS4##M{sFU|+Po*+zr1)*H~44vR7X^iQ)HrA&+Qn7r<W)2IG9{uh<|OkJW4+xTZ+PQ2c>V59n)@rMti@{m_;3A_@CopZF_7pXS7^~YHJCY43CWioKlj8eS8ZCqTQniHQqOLD%s|J<un)B7>$m#Z@;*GmPg!U6IjWk>7Du_9K}EZ+TvPgI0rNa<!k=g1DZgj+FG1Q!F6f;y>RfWWc5oVqoqb9<^WhxfqCM3?vK?aX`>u|WBlbgBKAJ}Co-AAawDO}R4KmD^P5HGGR39b0pcQn}T>-YMj(0O+$qUD0KA1-;aVT^rL1<H3^17u3Nwu<ipG^ZbRY|G;nZX$7#H!4FjHI!?9!PWi^Q7RT`o0b|f_3~)7Yq7Mx`>2{b5<bUW|JxcHxUK2d#4PTrM@<r3%CPT-|s|(C(&SzNsKfot&-kT$bO!b+%Er~19hPy+J(K^;|yu`~^-Lk2h%t}oWw!QetxLE@FAs>ny-$t24d9X7Ux2n0*Xc&hp3&WOlZ{#Kx#`?7i*asiWm{b31d<KjgPmpp*=fD+(LpAe5JBu<r@4C^0ljGo#7LA|9&i=w`ta^{j3Xhwu3FAegcXg9QcPcmWdQ;bK;#O^qH=SatUz`>-Jr{=gIQu0$?q+B>T|^W&xj3QRq`!G;!#Uf~VRyH!iT#>03e<MMH61lk(4tCXzQTq<e#QS!ASU`bk$A{{kKf8YDL&^$KTTT6jNUNQ<-uUo`(IE?0|XQR000O8001EX{*FwaR|o(Agcbk*9smFUWq5F9a%pX4Zgg`mbYXK}b~i6&cywiQZeeULaCwbbOLOG55x(nJAk2jmbE%abCytysY!p>_tJb?)UON|AVL>D~f<+QE07@FG^xyq89()W(_UfP^fkvaBUw1P}lFP@wk)^bZSG>1E8&)gLR9$0{H%vTYt;55wgV1}{@=n&mSi^W%u{YoU=kf<;N~LAD%abHIIjOa3Sy9x39keJ4Cfi<VFz-5Lxs|FjCnsU8RGkI;M&87gdgw~4RAb_T{{cllSlNuLDn2RWFYLL!1y#K5BVHqk7M{fJdtoBMC;T?zJ#U5Sc`1U}mYYHkot3S~xwf+Ar5%sF2@986E1fZJRV&Nl7o)n2nOpwl*Z(ML*@!i#id!C;G>TV6G^sn>Wb9sOS?`Nz+CsK`Ct#+NO4QD{qTua~&}pyLFQT-CR?22j3|p1SScOBfbH>fs-)O_AL6^d8->Xh+&QBQr;G7HIG|b)#_*Bw~9TWGw8K8+~57OQ;32ouZog{z1fA^6Y0imocaY@O`TT$gFj+yMFO*goRR|o{QYt#)+aJ_@II(J1QcD&p(FG$Wl4SnD2=}LRcEd=l}=a_ZT3#^upa1kB%=HfRE9=m&0I^~>@YQXVII`MBrSKhW<-yxJLkw-nfh~D7Dc<+TKtFf(%jfEvQIT@_E7wN{)&CD6eXG?!Tows+DgqOG$oi*DZ^&m2D>q6bRSAS&mPh>wWz!Of)Q<k{gI;G%A4)3IB$Jx1{IN)OBmF)V#<`h}M4i#N<rPae#BCF)uWjEf)Q6Yk^Jz))}bt&jK>@QpQY=m|18<LB6`_vKAA0-pEpr@>*&>JQVLxO{;C0cqsWB=Z=iG<$HUcdgkyMML}F>r-r93s`3h)W}uDK}m2-0Y!eCpw{FfqM~Ah;>3hWduG$aLbCU-2-(5>%|P#AgI`3;q8C(WMu5EJIn0pPBM8mzFjS3c_PD}v(dyEOX8(*xdgPrP0981;^fR12GJ_2bWG|yvEMduX$9=CqCI#2J^#(f<Jybl7kmuC*~<u$T`drh$ijZ_RW-dRLAJ!!oK_10F9%~q6Ot}X3Ia11Ttzfrl7-v+&8@12Myy%-b<dR8r}S?#%8(Np3Z08b-0W#n|Kn-?$}R|WkV1%oX77Lj1>%->J5f<)U+h^gd(lWZo5~U4SI?ye9JI(n2@jflCSl$+J)|6~Oy20BaKM(f{L#qY#P*xBe|@=0C-S+1IS9+U!t6WM?ORhkz*B$1?(D0x*Z%~=(PJ-jnt3u1IXvAS)~bmYohrC73N9g6ZH?74EIFkxY&<eKJlISDY<Ml)t!8~_+RN>uou6uj)hGt*BVpr^ek%IwBeRBJCEXaZ1;L*?OZhjKU$(M=hwZ*_1#<g3|7O|^WevOw0;+AYOQub$`r>Z;)$4ic^jh3YHAFzp%ZDmJEy@k>ltNqOpDY2)+Ye#O>iq;*%D|_D{_|2Xd5Ws*)b0SzK}z+<*a|nK{+1d3&7z>?vP)+X;knjSqnl#_L|vkgyfdYiJ?Y@D$#=q%fnqgDkJgcx>fWdM18gk{l0ThThG_N92mdFJZiH%pFG966x%lqQ+aG`z$%R6nLl56M{J{GznQ7c|TT)TeA~%BT5>>HI{{I9%&*wjzFVd@@tNi+l4Ngw?XRL1c&TMh!`=8$Z`1ak~_m`8=QP4b7)Fx_#me&2#3gfEGhgdIsxtaQ<3YCSb92$vg0P0d}r3HFq0~C)YnU$6))@CNVf_B%0AOJ#}I$`sjRzshjZN^q%XvlR><Jp*O1?pLeGKvxtR;CzFN;SA0s^X!f%(5pL^(cJj3d&gvn+9y39T-!0I`Fj6jheV6lX<q&?q*f_Xw?PD+U`r>%GLx2Aa#Udj!8$hSNM~?;H&4E8C*A6AdU?c^K;)_=0#j7iaBkmdaD1)*mIx`{w6wh*tvWgo8{1yPHk=GLhJ_htqS5juR4^Y$*Tp`;h<Ltq8Lz^%et@>Mhpxp<4lqoJ5>9OLh4o_XnfMRnI@lLb?mO)aMD9j-(myeYPmzWIxe|?jol^b;#cx<0*W(FY#I;u-6-U`^uylpe}rJi8pF#tuo{cAp&eXbNd3Z*YjSq+@h7SW)Wx6r$qbE$GBlku2<BR9V^=-nI3;h7jd<xxYGZaofkT_U4HV%u_)0wxs>V}S2`wbot6}D4qiwL*L({=vdOVyWj*p!wU6MKe%i36kE4^!xdUsa|iR;#bR4?p>o^)L`$8NWn#Z1#63FN%_o<b`cC}UpC5T+RwOU%0|aDkd7sxLbY&+n7Xfu~l5^JSSAnmc+2&IgrlIZI7^JuQlvMUHV~+ftv@%>`b58X4THfym=aL<rytX@|Z)V^`OcIyvuq(N)AK<~Ajz8}~e!Z!(nLkJjx8s8FIhOP?nQFk;la3V4|;Pk!!9OO{BzHkb{dhIa`>cCCi{E)Xn2gN#XMr61U^^e)?OS<*ZBT-NMZ4JX`L*A9Q*8u8IIJYa;OxoRB8vLmOp<el9fO{|zIn;a%=Un&@wM~*7wp!ttA@}GHDLIvN=zy5r-+EWaivO0P4E_#0TL_Wd~IqCU`9j0$cK@&d!j>|7AZ;m!vGq^rPXfyOaZhc`yAbtGs@iL!fZV2oc@CU<p$f0u<!_TMDc?iQ*9-X_$bVE9^r4ikE;rVmE1I*2tEXwRivTTf!5CXX70K%4J?(=Hltqa;1a|cvDg`CLFBnR6K#v<R;P~$N+uj5DC)2S&#LXV%}zD1QpHDGq2LE56;N#lXiyC|P&_q_Pz-~!Xp6hWe^rIyzYdN3BfWVg-9?@&tv1QY-O00;m803iVR#)2p83;+OBDF6T+0001GcyMKMX>Db0baO9sVRK)0H!p2vbZBp6b1ras&06bo+cpyauD=3hKUC^bitM!08K=tBN#!JMY-bXuGd+)o1Cfw~GhZP=+j1`d`|SgK>*2dk=MM=5EOr-*-5++BbR6eHktq?Y2QxOwt%}96$aMn$C7Cd5C6-z8sN!*&sil4pSErxE-7i8tq$<;qip^v&_~a*<%7p|rnKDHdt3bS(pIv^O3!@?+lrX4Vkr!#2WJWA?Xkc}`az&aYzRL3?%SDn6av7x|j8BA^OMfkNoEAnzMQ%i(azE1xMJjzdJ)h&!C^bUHfmo>+g@9Prax(akB@4OGp*9<_NQyWB_0lxlh_ymZ;=~A(5U(`Jjnx9yml}vb=LX~z223#-TmY?#K~14^5ha0MY9Q9JNJHr>Q1@MgJHv%$%_K#rM38`4P?n}?5SjVw$-r@(!2rw}3C~*=XdBNfNn{N45~b#rCXhz@p#=5viYO1XZ`^vy1;djn14|J}v*zVylcHL7{Oa!ybMN)r`RhO5UA%F{IT!_I2LruqPn`?|{591c3<ked@`mgbv#YF7qX7+xb7j^^@F`WkT^yVUupvF)X?35U2vek?x`shlOeT{XnqqIONv)Hdz_BGEjmR)v%M-B-6KTXu98ab+yilewI-UGXqbnIjtzjna(Edb&Nu*Y-;a`|Xpu#r!F%92>u^PiimbEX_<{+~Gk1~m$gtRXhcix_UJiokn_s{w74zEgL10l!r{}vZXtbh#vtERJZd3)NbxCG}x-}Q8?jpq$>6)s0?d9wbJ91lk)I)LD+f6`&#Nd4=2dLzCAEiXU5d*jqh@Gp}(SK==j7HXblNj7v&MMOspn4vJPVmN9FLtG8MDYVHU#xsaoWbTZbN`)#O)^bNEm4{4ocyvIA)-3)xf5AEt%Lsym@B)yci1n93y$}VgwdS7vfI9v_*cN~@Y9COeG`7>?+R~a-v$#m{9E-0lYA1%6y;F|c9?g9j_6ChyZ5i_7B#UIIA63wu1wc|V>yAeHfuJK<{7d;}$81j_6$0@VO<9I#?X*kQj_Hc)7J_dFj&kXIj&I}SzPfk5HjMpU_RZ-ZL>XMwQ8@4CZ%$wT?YONtJ0#hu_>>;QY%nw$Y-;)FHRxg}bLbE(p-2QG{GXr)AM9^}7knT*!3RPRd;rXFIzDa#7J+-B^Cd>L8m@qqF&Tx1OR~G6vBRcck@j1)F=p2`Wjlya_Eh{jV+-S!@W)Y$#)3#q_B1N8L!0hUe3MOGL~Q~V=1^`1tbaXqk8ZYj1{qlV3c^cx153Al(^=riAq|L?i|9@&@VZikIEiCWi2`YlQ5&AzGj<O`cEiX{C^;OtPl(=w5lm6s`h^B@E?@mIe(RKbslofnhYViYFwi$hVZIa^6G`U06M*~RyB8q;?;y|;2pb;ur-#PtB0@>pGx8Haz<F&KlhYDzNy@H5%w6rxOl*DwAcQR4Y^v0vsS{@wV*Ij^i`3#d;os)>mPdC>%{k1bacfoDQ~s$C_R>fQu8(iFB@ZvfF_R*L+FwR>j${%KQ{{RO_^YQq=hI9Cc`hzQz|=DIIMvCAv^a#K;JVIpZ<rCJ^H)$vs0ZmAz?lFli~#P6A~OoADcI^#uZoOwdOSrVb593KP9ivlUnW=$Fm7rnh~PGN#bWJ><q906HtUdcGk4_nhRRUfGX)Q?4iTKa8GkNG*p}8h*VU?e>qBQgg+;7$g<9fF!fCiw_B$&t=7!`ZkJsCHP))Ie{-~z6c(|AR*s%ufC0P5G&5xSPEbY@bHAks<f)y-T29zJnpitSh@cG__<2@^Xa?oM^2a!MTUARZ&*|aiJHxb=u+q0u`j}<!aB|k3pdm4FV!3N0hT~HZr$ATLR2Mhz=Vgr#ov_EWtOUTeIelb<FRg#YTVA877coeI&*-X^D+m|PYBF_$Ly#G^ex@oi(R*L^xapYv0_p_E+l*AYd-a^9O;q(t-++~Ug28?sKO2$?0Y>TJ-1(bt#Dk~ci)LY@07W3(|`4LR`0FHTucRh5mD~ByxBhHLms4!>nL_k%?VKYg053xC_*c=^XQ=Ynauk>ndz(`@T%8N)nq(gji1yAIoZG95TkjR$+!Rze}@9=IG|KJoK*bJG;XhhyZvn6A6nn&FghR$N=Uy7;Kv{5^-(8lf7O9{?7ib{@<4n)`odsJECv@pNEc$n$+^rl<I^TY_9wodHnHs&^0Z8H9y8|mNLRW97a(QG;@|F@S8TOHCog$CCqXEpd4PZIC_ld@~YjU<P;Ur~F`xpA%EpEN>pK#tuetlatx+XI=r7uO4E{Iz?R_;OK%GTYo(!Qx&6r^0Xt5u0hUYGp<;zn$=rU$C=ZoxTw=gB75M_OWiz3u;b=8!&DpwM_ybJQiS(1$K4<vq`}wI`+dNz<wO^i;B=_Zc$*?3kx7o2?9+WNf4**1%o!F<7uPei4I0xkFyWYk0<3u`<~hzn8Y#U*s*WpT;}48w-)I<$9-eQYHVPYIoO!S+D9}hz0xl1x?}^j&Mj+hN?{;;t?=;MiTDa<1@Skuj7DWU;BgAdw1r-CO4Eu64U53<Vt|9ksVkZuFv61XaXN{?1F;KrTo$e{rgM)PdLYqD6CIl-V)vu!J-bO&Y}=*VJc_%>x!ncDJvOd%3GLnNK7nJ;`dh_+Zg$t5v}%&CWvXy%up@fzmg!6us5|YKbo^rnA$n;^htE$h=TF0sHySof1v&u~k!FActOb?Y$lIxXhz(<lKR}x@RqUnlkJH_2I9=@5Ek~0nLNe1+OJ^j0#2wNp4)<u$DHm(jb$!=4HOr$}T-As8z4&NvPR^qQ7$i{OxdcDdTRp+o8@bBvs(vQsA1YqKJQ@y8sljepfYFyS(&6TZ92ThMQu|uPhEN~lLdifROEN`_{;D*2n_PsFUwib7wbxYjBxR2<H{4eDR^;QZnd{2qF=-V)_F%Uj1@-@W#rd{^{9F5$>Y)b}LN|^2XX~+JNw6(_s}iDFyW77tkK1N6vM3rf=^5pFxbgr=R1Qu7X(SYo&*(*!nxvZ(LV1Dy4Oao&b2jO+cM4q_aFslRtD`<pmNm10$e=WA>f!SG8558`&yUL}omD_FnwX@0$bD?+M=9*N-t1mWH*2<SMz{NziSf}P2)d{^(G^`ao0!FS$inIAI(*tejl{F9NHW+52a{EOmo_2cAmlg)x-VOz3jdzsXf3)biuVO>uj>+~aM6Y-2Jji?fL@*~O(Y*0rsS+|d;||2lB7qn+Zdg)vuBiT$@{R>4g>(2jZ3!FZLXi~uENys?V@9gww~b^px+@kLAgH4wQ{Jb?TUsLG%~8TT%YO|#B0{8d$A5L2D_fS<7Mg4nZ!nSIn}t&gCx>1CQubx_#r?~+vCtsV>=B-(oqrBmag|lJ#?_Mj%P3jgtJ=q7Q9GRYy)<NmL0dZ@nfx8wSw$lFB|tNk!r3yH&@&$mEtqM%6^*vd^zYoT*<(F4oAjWxre2~u<G=I=AkWePCBKD+kCa{;<p5<dW{@{hxi+3Z$)!y*^epXvXu%}7lDHP2bclCt-nt0;Rxc5SY%*Byj$*7hFwhM7qDqt)VwEoSW{<7Q20QFg$vy{Siz#AqsFD<OG<$uuT@m5(Dx90K9<6|2NW=Gs}=-6WRyw-V{mP%S%f&HqLeIw-(o;c!m_!#wq=l{_5+v(sw(pW)9Hl{V1t4_WZ^GTa-$&+f}P40NLp-Y18`Y}kkRfgjRrek$?=w(FS)9hW&3IbBBHc6)V)_&AgJRL{j#J>@L$#_o#js{RvIZdxFAFkI0&GB_0rY)BH&NmO=`Y$?JI#HK+G!f%meq>;8DpXqXE`Wd*IHV54Cgdh5A+vJKJlgFZd<NsO%(o%(RAv8p>MP5Wj)62~XGfJiIkY8Z{C~>gnVltZV(QO=opzb8^}h*u1DAK9l8=^9kAwCSfd!b1%$Q>~FlXwj$kx37UXswS76W5-f-<3LOSs+2$Me+bjFqhG+D#*BCXX;;#JaVqE4$woh3KprTj`f1Tv`EgBjDSX(%H1qIUWGXN~W{(6N(2^81W>#bBjD4YR_2m*`}g>UGqt@@KB$e&-}!i_lp{OJl?l!|g^kf>v*!Lp+B0!Ls?W&nJ^wk`Cil$q`9Xj1!e=`{Kq-o@qFyZ2Y~k52V*+|0g0vHWHZe=O&L{cK#^bW);b?$~^xR>utYR%ZP9Rr}_qMZ4M%D1^pm<)=ZSq@)oln6_Npr7b3KQNx8Z>{LK(limwVYoy+I&uG7Y{^0T3s~#<>J)?j7ljS!+&W^>E$txe-@v^sDt=jl0w>|g|P)h>@6aWAK2mk;8ApoAXkg}Bq003<g0015U003opaAk67ZDnqBb1!sZb6<8hFK}{YaA9&~WG--drB`il+cpsX?q9*FA11A~+6~w+q%g1~>(XrL8U)*b!7v0`I@w$#Q6;G)Zjk@JBaeFXOWI)u64>Ou@wvO_$RG%=71Ue-gNM2hsW7a7HBYy%QeNG0&F%!;YsRaLiLz`AUlqVgQHd2d=>~Uo!wZ%JH;n=v2SG5I<Vuz-N%F?7B?&6lQW>;SNyCkhm7Yxe+J@^*Ay(b*swwL&L$_5;X09)<TK&tCub~w*Ql%U3%5a^irZS?0ID=GXqFT2~3V*9C(E!=La1APj+D6ohUw11vpqs*wPD~(pnQ+4J^A|3i%PJS^sr#KO!~hZ}Nh`IIDQGQKvIbd#QQL$trEP*ak*j|oHBOJ#F&8{3ExM2;3W1j;{?xbANi>;EGRV=e%UB{TAQX*1(UiHhq~>NbW&QC^_6{~>DFx5InKJLif;4|UiB6o!dto+&c`R$F!rpRZ6SP5{0!Y+F!0*}Iuf}Nm4wMOln~Sp`8XOlz_Rk#-jSRz%-MLxLi(<vo+xc(@49~dXi@@7!g@UECMovB6jO?^&z#xw1=5w*MrQ^|P%?y0n6D0&LI_k?L)M8z7%qUu$(|$Zy>XkLB4XY^}pMq5912?2viw7vI?qkJEIO!s#bxPmVBQpO(H4w2}DrSX!*)UO20I@Jo>M(j57+wTdZ!b>I|4u%f{W^OWEYZ=92TBA5^MZZ2xx8Ryqp_!@VOkVWnF3oo{n!X(8_qJ3=b&&ELonhFTv-#Iwtw+xM)B0d&K`**Qi-*wkZ|^#W46e%$elyS4>5d1b_#ctr`?HcjKMPMix`wbNak!;h0S(#V!lFvuqYnzzlAG|29+Z;uv4wE1+a47TBChk*-sw4ml+f@&#?|G!b3C1I3AMotIO*hT*5~e<Wm#}oylmFr?acm^WP37h?=>=hI4({76Vq{Cy2(SK~TCH&%_!qi}RMNW4+<ueg8vP)iKvxDZX{!$40KU1`w1e-oOKCc~DekTLBe_Xpp3HjvSQW`k>!>rVa;$G#!O8WdZHY6s!^3+1<CEU%aQDFVW*MDh+)vSL>11v&}bT;-fv-Hk9q8FrZGskk8!zUoK{U27_X2^L2Cj`_CWVCjooubeD&UyN_?-WDaD@^WYaE&_}VuaNWjd()C|bMxo7)WX@%8I@vE|PdFZ*qrB4@IWSoqiSHvbDjC3QO2r;KGXdERPJAFeB!SBSZVaZpTc>0=)3`gvF=$5>?mUj*3kLP$jrw2MOo(2J{WBeFw9^X3+pjwkV-N=Nfh!3+rm^|i3b0l^!QjOW8Q@i_Wc|jn4e}OpPse@TWIgPu+dg4mHlKIQ*szk^Qa$QnlJ3z^UPx}jozg^`yh3_;R-ZD_nfRO@l(n(o*pac4vLFRsZIAhBiUP$4qD3(heWI^V?N!ZIP;v{~5CPdpYz>=bPXaI4xgBy6r#4c2!Bb#n1A-_7@7q<~7?umAP0Vfp7cKvDMwT!?s8z~4T=dq2(M&hByQJo#Fra7vOSG!)?M&*DYejLW7@5lAYu(jS+>TK45vLR%o6lLW!T@h=eHwT$fqN0yv)H|XH14_18(a*jzx-Hw#)0ooqKAQluW-*z-QH$zpgn}MK!h=l&9m^y4slUThZtrr@Bu$tqm%y|z>f2QSHI)DOeR0k9Y$2{nv_CP;++8}>hKBRA6|)jjfLGSz;9!@QIy(U^V3aJ-6poz=Rf)q#P{0U3SUh4{NwnXg9mJ*_EBe`!kaCV==u%Ir0bCsrGRaO#@M#A4Dex)*(+Qj*H}{S`R8T_P64doIYR>mheLPv$0H5lCZ{iKoSi;ux9V{0*61nrG~n;15+55pb0JfDT-qz}(-i;qMlyxfXTt6YJir4^EWE;55x^HH;DsE*7Q<*<I_?1I!YVA<QL&_{JQ6&1Z*nTJd+3JG-kJu*?rqr&V2pI)aacVP3lg@2(zzOmbs#N5fAP1+kE}4MQFUh)!~(M%l1+cR@vnRba0jk7T)D8YfAKl`+Rj^iEoM3zR@?wZ4F!%SR~{&}ApH|>=jA}B$5S@Oa$!IE+fYL45!6j@!-WR`t83{(!!I_l*5NXWMybc`X<Owe_DR%XD?0Tqcaq<_B^PtAeE1qcudrc!BU6jc*0G!{`}*wRwb*u*87+670EY{HG?A!{k`BNTgS33%kKLZh5jq=5E{-(v@|m9f7f?$B1QY-O00;m803iSg*=J2K4FCYFEC2u;0001GcyMKMX>Db0baO9sVRK)0H!pH!aBp&SE^v9xT5XTxwh{jBU%{$h?5nMPNv>#iqX37yO>;nc7bHQ8c3}v#MBB<rq9Rhe>n`@ccZTnJv2X8CpxqB%i5w1x!{N*`Lq$>aOIEO27TY2tqN^&}7ECOOY-ueT+Cmp;ijqqrSjjS3@S4y%Ck-uHmXo5QI|dE3JP0P@D2isYZObc?rrS<-ElX3NX?QDvsOFL)12LPKwoKgH$Aa6ZipsralJU~aDJ++qZE07^yvU@cX@I&cHn#sK9Bet&UDX^26{K#ASdxmg?O?eTi)jn1rkQlSspM6WrN0SYFT<D843617EjzUYJ7hj1@Ry=Cih7scvBOfe(A}<OJ1SY4(FO;k6_ag|3Ei!=mu6)rBx}<R=Td;io8@dVo6TU2b}RnP<}J&3n~TMj8d+=!sF9dP-G+ZH18vvjDatr3FS;V<OA-mtcPHQ_s(H#g*>qAw&ofOgK#e3ZD>x~$6Ty*!6cOM2273cEwc4PF6txxW_I&L~St%HZRa>SgPRVM=Q;VZn|B-wwbJh|;@4>bV1z15XDKLkPfX%Ex?Vy&N>{!iOwHc`Z$~qS@`E}0@B%`(ACS*~TpvtC!k#gViZnp;=EXk&mNQRnZprlP%h&{~rdk1f48!EFsDM1l+c8F(s5wsC7XF_hh#`#0MV{#rjk<zh9(bCgR_MXS*3$NELZ;?JwrRvh!G!hMkxQ&#+sp)pS*Pr5QH*$KoqUI05{F2`>IEXO)a7nOq-H8iYOw4B)NLs>pu<g0g=pr!2xNCCY<qdMIutc#~Itpu3X#=Nd&$~8#ply|W_2=_*@~UTM<jhPDE7lLnFhB5`Er%1j<s>Mneq4Ae-Jqs4^C#PqQ#r4jxMH-nBZ48#>+^I{&3+`X%kU_bY;I_l<aKBpAh_x1pRN89nr~iz$xPq8{_L47f34wE`JDt+ZaN0b_ou`Nq!g&)$E0Y+K(rG2>-%C>xXOSPqj%uGfY~A493&G-MIYzRwnyE^j5}bNaUoNY@rET4S|Wt2jH5?uTDApW^e6p(P2T?a{wK0!6hkCLTP3VAr6o8m-s+&5L(G+31M^_4VfYFW8MIp-qVYbMX|1d3&qy%lmEZ#zdD!!knZT`GosJ#i8X^k;7uc~gHSd|1ieSJrbJ`=T;*CDJCT8ux!8W)nDs~+PkxTWW`v^k#92~FI499x)!A%eKUJw7~%Jo5n+VebGFSRFZS}r-D2GcO|yA`M+!oWzYreslI^Xr&44Xblx+gqWZ^{F?%o%)LChh@LlNpeN(<A7o)7BGT;eWF*d0vx~}LQ{M0x%asd!cV5SEtDDrDpe>Gnrl_Yg=CeOoAs|P=1>SP-4+1nU|(F^Uy@f#qR;mwKHM`jQ1Fo@Jk%LWAc$R>T?PdB%O0je283?}YcZ7XX<2p|@U_^f1ty3b1Q;fs4@yd&-2tWLPTH<y@l5f2YugG00~MmuoIDhAPi`(&Kj7lfu3=Br4{U7i)DV-KM?hI{02*EvfOK`}%K_Wf=;+A?AO*mHJ;(rnATKZi5m0D_Cqv0o%NErMBw(4ez5`?kx`oxr2d>e<uX_+IXZNh+4K7N_j>00ophOhA8sRmn3R#<=vHHCOHOm9pb#SO_$yjVP7$QAH!*L;CQ(({LW92W9B@&*KZ<C45!PbiaC(;5u`QHJ@vYWQ$?L3+oQUNR_8*o8HPfNBf)zBYweS5yVSjW*K6!JpP4HkkAm#3!l)}iU`To7AuC8$)x+hrp|#4`nKj`^N2s0~~MNfHwM@aE^gef;qLUpMa}<#83w@MvmM<1_+9w4rch3J^RYHG7nEJfopdeg$@R>#r0;wB87gD8^N%PRA4M=YvrStq&d*SpwzUAmLazN}L%QaOQdR_J=p`e=gA3sV9tHQ3l4n!+2)*+-Q1nV#E4Q>rDq~5oiW-MZ1rX_bBKL8I2$igz#{Bt-2Gbj;cDecJerKytNqa0KRC)opMK8?37$!LUeP9OTYO_{eG=}U*qrh7wT8_ey@7J|2pzs6^#MBR)76ArdYM3?W5{@L|?oOUK8u^aCx=7c$+?O<tR&rSix?}PnUs<V~^<Jm8{h5DLA`r(N@|^yL|t|KE)VS{M@kWq@>j*r>LnbGE&~H)yH~yq)QCDR;q8TvPm-8cFbp_Q<FHrFj$jWnTjrj;PscA;Q!cY0`VpdM`_c#{G64NqBkPPv`kjc;Y{`)Vcbw50=~ofzDvkzBJQln1zTNQ_IdWOLu#;ta-pZ}i+mf%?HaXiys3?;!n}FW7X)#K<f+3BN`#(k#y^E6LOGJYU_D}VGT{I*93YMlnA)uGj&J0MO<(Lu(N9RO=-aXICuHvrnH2wJ(l16QM*}!Cd4GU=L~t$*jXr<jfgWr#Si`Q^l<b&=4Qza**ONv&qBg=zS=T^7?T>X4svqVikGzY9em(2kD}}k%VSeMYnn(cRttgfYB^4_d;8*vTLAn`PIe^YS2EASY==U!Hg*|Z4Zy!h8QE3RGQ5KgGAWqK7y--l%+GY_%n<rY9wmM8>YY*^g#Q~sLIPdZUlMP&JNUwREX$C&UIm38DY&=<#_wPbQj2##*dDJ*Hq61W|>auhrc-cWbwl?qFqQl(RSR=fET}Oy#sv6ke@S>Kzjd9gpw{@^fXa~-8R2c`0!PY2TvKKM}zN~G)K9uz$!!WrKKE12U!!_O%<cx2(LfNxPZir283^IiO_XT@UB?gZ-9*P=(_E|4M%1T;PMw(}nm@?#+maN{%ed>?++}V$}G@pjPV<Kw{kOpkT+*XJV5>k1MI43VqOvJ7rtS*fcJ%M!*8SMluF_Yxm*d}EFj|5iDpTd&Asu^670r<DVJ)6^ulfG=ETg<bCf!+;aflUdqAb&{Io1MqIG2nz9dP+y<RTL5`UNMLw=894eG7X=Z9NgeGt&A4^HbHL0a!mA#-#geUu!~)7ua3^3sXd$t=Ii|<xM|OB2*^Ob*#vY_s|mrFYPf~o^=)MA=K&|t8JJ$|WOsn5#*Wfl-xz_-g@)dh;20UX2V|yoico1JEflFK*F1r2x996{jfVPpG3`^2-{RbQWlZo}_flokc&h&EUH$1<110}grd2d}er$kHg`bf%p)+ES;6fn*QvQSB&g!CrE_GU01x~)sw5$7s6Z8(nc~t9Cx>`@7tw?~;RCs4)YE$Z2mvOI2&YAWw{_1JyHC2^Jxu1$<ds-J_=n@~=0+g*;>SLILO=c28$kJG?uHC7ZSNh+U79|#(lesY-_SmRpBwoj`Zxl9_22JjI66a%7isa(_{CwDlT2G+W_zQ5G=qT|%cJ4%5xURUmgVwpeVHU}+ExHDkl&1Vny@o{~y!1L*;V}*#T!cQeVZ`vQn?tI&d{GhF4xTDQe{S7**-u7Km7!I2w_nv=nv!!+-L=Bm!tT|mTtrmCNx=cjO%<Vjy!q>!xBrX`G*Bg}@I>}qyq*`wS+=tQx*v(^!jm%jK-A&L@rAM(fNKnIl!L|ToE_-v&PyWO!;r#-qU0!+7^Nz1zmG3T@3uISoUli{Jqf}keu)5$Cxm+nKwMXAqei=F85PP2!BLw7i-oUKh3m1#)0^OFv@#biQg??jM&e^PpPsz&c-gIo_Ta~^JM@)@*GRwjc_L%O)1f2H=TuIhu@~(J7Bw=U-Uk!MzTFsG(2K2TawJ;TbdO$eqW>KkY#jB-bx)Hk-`#iRW5UI#MXO@c>W*9A%M!ukW_ZH>KF<zcW{=)3wFOL5Jbyx4FY^We*E-earYVow-5?S5KI}O4$hUo-hE494^a>doA|Oxwv>al0Gz|3+^c;x{xR9_b7-8`gLVLgvbNe1ba|^ND_g7aAb@eI(x>_HSg`cm4_}2h5y%XvY!{^g;KvNb~VZOBkvOZ#o-hO=hG58vfs1FA)>n&<4#x*<(6q3+tQ|vn430DS*2b-xq?V0tmzIbBLSI$2Tc3W1`Y33T9C^R2jDs_NW@?yJX>ifhU<7ktIv%0GoCLIdiyBpk=3$b#TN7Wa`YY1;XTGrtg6c}4%r)mTn*oKPrE_ttsDxHu$EAtg>8c`deVLv9jpi{_l!->`w!K>Pz|GaRnpz;}>1Vr;Own2!}hOih?myX<i{G<ky_@!Bhh$n|w19d~3#hU5{ET(_+VKHiWf~<25+D2H{cLiVm1+6wv4_bOO6M@@z_OE(E+YcjqIAHYIR)4OkFUfk#KK2b)kLcLvS3^!GzG3Q-jz2;QDih!_8LeQy3LgI=v8eR^qp8}c{IG?tOx(xsAj|oOYMARGOS9FOazVrwy3&|L#jfV9y;;&MV)bv&Xbz*t9f(rq#>?tnXKbc51_`2trcNbZlWQ3;Ny6`G86=|wRK;_UKZ!cIUHv8MeS)J;!t6g#O9KQH000080000X05s(=XT~M~0P&mv02=@R0A+Y^WpZh4Wo~qHFLYsZUv@Vyb98lNc`k5y#XW0t+c=Wn^((mCx;inTWhXnkv!{%%Dsdh=wV61TWVUWg<zmqyWpPcBT0ZP}T>kIZ4*&!}P@*z3clj`5kwBx-X!IMP@B5$9y4}WCdFrjRs%}oIvMmzt_Is~x+T`B5%bJaMm6xlVthn|*o_VWHy1J>o7#fvD>b?8;vsbl6lWo(P@B5R<x+=F`6s_B)t<os+vhA*{8fagXP26N<QBNk~SrRw#Dv#?rt>vSC)ny_7F4fa|(>7Tyf8kg>%&N@ubVWerI7zyW+q_A#Rl{v|akI&@EBXE-{1iwwaUDt2<EqKl@v2d9rV|hTy%dk%R8?7p^eAr1ZMKRKQGT$Bi?Ya8aUN~rdc)7^P5k`Ne~H#vo`#d4KisM;*4ecj>oZdNoE}Xk6QYRsIjw7$*Q++UPMc3@3}jEHZ@#Y59Zolx(=b5rm)<T1+NOzDsfDbYau<3<dYe{W?5$(KB7w2u>(nbNufEB4J9v<Ojkmj8tl9+V=GA3et<oq<QoypgpYE#iUqIfdD$7QUh<|zuqN4el-r{^~c>VCmKVh^atDsF;-FsQ#O?{0Be(3q>*Iin{-$hfO;;Y;5eKC;ikIRQ$YakR*U`hy_c3^|qbzWXg{U@{Cy&q_u6+X+lsip{KR;4*`;B6W;CBX@TOF&jzD{2-f+>3ad650x=pXc#ye7Y@@G(QEpU$;QiNc_ZC5&U->BQD`UCu$W}tIeK9Tj1af{sf*%ixoVoPFH!hlRu8@dDUk5=uToy%l+MeeSx=X+Rj4_>L!L&b&tA@a6j8G(_26QxZt#A#vFK?!Po^ZeUzjP;X2eZZm*HGTxeG5zgu7-<~$H1@zu#yT&J_<YjdC(Fs#^wBO1rsD<EOlPG@f7O=K~f2?dMeE9S*21zB2n4?gwr=iC>z5BGuxrVN_W``}S*>s^)Z;wqi8(3$JvCiI?!9x*|^_)@iL2qJX@JFQ+!{g=2wU*Uhw@jtKC?gD{v4pM)&L4tX2(`L2tI{AqTqih?GSfFY_I)mPbi+k_<=hrWJED||HgPs@$UC_zkpDtd%c^`fL_~!g`=q2S{k(Y520q14wEKNc=v50%QAzlG70gremfyz!rzH|3Lt)#TY#m6_%&!67I_`7lkq7j8C2*|n{d{wT>2>VQW0+I<n1CFV)x&gLeQ8@#t2kNt}(`p*1PP6R|2(>A~1pX1Lk$#1aQF%i@x$Q135+G@Gb>EPzJ)@`DS}da}J&}h(uim9Alw%q_XIYbO>!~d-p$onsfeW3Mg5xgDW`qXH<%K31^?!nceO?%rGPN&w8ve(ytMoQQ9p_)dBOI28!#4h94{p$V0&@nNf-sp8p<q6qP#H52Kg~3N|3!87OX~e$Ax6MouKDnsfQ#cCrbQr-lOx22wFlw;)bnT9%n#fUjKz7n@V4<+H;M4ySvdGgM|QB$i8mD}+tUTo$Tx~Iumq`h7P!akrn9cnlmrrr=vk80H&I&y*R#f);p>UE0_Kd-DkK0u*rf2Dojw2V$&){sl&|6pkov?skBqE7wbuSewSmyIsnCuo(lmkLu29lEOb5mafUlmV|23aITR(d5UcnBjQbw=d#jBLqz1gI^b53dB^R6b15qehu9Dbub^d6|60dMC049Le`9#_|3X?5Ei0Hf;kuF^G*C?ywQIP)iNs_&|-Xw12<{hu<h0YQ3_$-OI~g$I(XsMl#V_a4ki3(TGqW-mM`Bw2zW5Vp-gy1<k`^dvLi<b-wJ)*BW(6B9ooPqdATY@OC9+)TMUHFvuobi#;v?gRJ<1PDZ~)*F7QuJE_mQR&xJ-X<*FOHlN2u|jrSUWU@D1n5GrQd^WDH7U-Hwb+og#IV)6qrDsw_bZz9GhD2C%0R$q1fA2QUtxRc%O!DJ`Y6W<VkFeDt<M=yU?C6YdRl=kih9Me3TVED9KVgL49uy}YpWcEv(ig$IWB~APi6Ku7r+aZxW*lTQrbtH@Q<}m+U-FM|2RYP<UXy<HB6vh5Z_LQ<uuJ_7VnT*HS5|YyQy4-{o<r{xn5_h3@ugcBAK$*Y5&lp1<CGt-_kcc3>s8*WnN=zC=yD&BOg9cKP2V!2S_*l0aB6-tCI(@{`D@;Kw5(wdxU4E{U)3p*iKVPr635F^JmX59~mkWheXz*Z;Et?Y7^+TH4eJyp%I!ZLpx=mwqyTg)Gf=R`!lqLt{39}-HW(QV8~SoQeR?`Wbt)Tf}mRQ^H7^)&p=ax7Ph#s)2`Ygfenar2ck_BaUy;JG%{OEb-H++0}JY8*ER+Z`89~&Z45>>I<F_1f;hQ-rknpl{LL+Sx%8z!4a_MeV<%0Ogw!B)Yl5cz@H+W6*}CR#f^w&hF3?{4Wwn9X5Pc%)HbZN4r8UwyJZc9JHGdvWCy3S>RlpoogyB(jGP0YS{#1AS6xSl6t{tQ`20fY;s6Byit31Y`{kFI%Kw`e|h}LAQ)*wU4$b_Yeukx(kcx?fTP<RpmO-ZnU&`)A^z(v-!X6_2?Fp|=O8gJDsejg+-7QG;Obr^s2UV7w;7a45^G7-S;02zJ%f-IYLo9Fi!87lKzkhIVk?UPf5Vpg+pktPCQd=1k=X^0Ug5duV`u8y%k1p_m%h8T-_HfAa=%-P&ovme<t1q9LaZ$MT*Af!j=*4ZV7GtAJ0hYJ$~yD*LJ$_m4j5$|wh7NHmqIGXe;LPb!I^C}%8NSC?0`_4=4bcR4uZVP+U$@gjt1QX%#NxL%i@^VG~NjKPNM>BMGA%KIeh5aXFRV2&`&CqqN^hEQ`jIPQR>5P>__J5B-fMK_670<96F*yvBf}4MXYzysA#R8I~<)F4^c1!;2KpIA$iEz*MlP;$==WF0KW%~Y#nI&05u(N{173&FMKhad>v}Azg<OCzRC&JPezDAtmiju*Pq<Q!8=ic1W=DA5lJXTXRp+ld9mJHS=n5}zc0H^_a+VFSWkfz`m6F&bYzmI2}cn<*K5h%0<&!-l<af7mp251EeM5T_boqvZ?u}Db%ktyQz!h}TVc{iOEYOuXnao#2<IYnNiN>^#N19OMnth)2WI{T6~-Pm}Kh$<%NRJ7X;9cP4~AM;^6mlEGg=`c(V#(XALoW)!rmPEf%j3-c{K@&SLY(EG=!*UeD`=O(aBfM~M1HPvj2#N0D7${wuvh?aLY$Nu&<l^0>+4Tn4V+Y#E6KaN3|I9nj(-?H_73f&dT!Jt)r*CSEN3AwxsRs)5A)UK(+zQ(6t`yHC-eo7!Y5pb0nY%^rvN8*!H_+V2hzHngzGMhCQMFbx=2=Z~u@Q(Kr-W-*hR&Tcf&)#NVW`!<xag+TXS9CI%VJ;TTQ!X_{n?dVh)pPb&(I-O&^Dl?9QYKLD?wZ4gb^TLLh~W)MStvBTJ{DMaHIm0R+k)~u@S2%{YHIVG-19Ru>X0(CMgD#r0!H`sNey)z9;~?vB>|P3>!hyo{bag!-Re0qyuvqOf1{U;x=?E!Vb$W&{Wp5w7AVGg!tk8m%m4^Uw(PXu^Uibjt;`kEAr-MBEnhC4P^^vv@0@j460^&7CHdfL(bYAbWA%g{@dH=Z$H0^F5bR<|Ko=@3t#svzkGW6;q%*zPd~l+^x56wpBL{hqL)8^x%lzJ=PxhczhC(47&fie0^wi$^k!k8wppL7FtO}w#^INXPv?Jgv<336HvQ%_^7+Mww?Dp%UjO*X?9=ApCFS>EPSn0(QObl?=Z|R%4BW3rVI2-O|DnBMDL2>cczb{7TfugQk;~CJ)i_Q|St<(|nlNu_1Dh5%qTX?`@Cn&PiLR@%-PO_6J;qJ8yCwqRat)HYD(}eYz-P4(>Q^neWMqtJ<qT7BKbuZTKtB|i*u9gJqT>=^PX)*&v{?H8Zqo|<?aSq}%ivPR<KT_49!tWJphuy{Kf2VrfGyQea4Bb2uVCv{Q$@;<kkUP`8Kz`gVZrk@ItM9nzKOHy4ZkWY-c;;+F9^i)>~hH#st`S(p5o$uN;4AA(ca`f7EyY(_&;DAry+1rq~NJjO4q}=P{>_XmU!E?ZagsMsN4m7q&!=}Xy+H_7d&uJg$ru-XE;M*L|r9Jga)ALmMJ1)?!p(?XkBV;Pt)iN`fvA81yL8g@bWiM$|BIE@-k(~33~f;A>(0PqpSemAez=`z79QS_E;|gSzr`Pw@RR!EOc3j?&eFqx2vcGjVF2-aeO@rW+Id^4TKR=FQ02>4MA1Gf4iKeH>TaOiILHvC;9l>5{QoBo2k_<0k1I63K~N1_f!U`w(-|U7=Y0>D_YFYJn{Z=W`dDs;G#>Abd*s=2F}+chMjNnG`U6}$;co`EP)YZI*V#_)KZi00xA`2jfhO9pJ$W9`|pKG372!{EMV%{QNCzKV|a`KY#?g=LWP5{xCypG!ucyHViWNA(h7qBB^N)XJ;zSX5Lr2kbMTgW^Akm7sNW`97PKGnkt-q#3DgtCVtc#W0i!Of`C<3VInH=b_o{>)fPzVM7Ip;Iny6UvaCgaSqVs#K!V#X4HD#BiMj7@V95mo=tMN8%Hf5s1XoAR*cs#K^NTTs<)h6*Qt0TZ6%TXUszaco_lNiE@iBvGwA)-P3f+7IQ{i-)*n<pf`WmEu5;80@?=3=jREy{onS>VYi0Y?`7pQ?nVY_hbTo(1~+G-)>zZfO)hJ3FK3sikmU!gF5)dvG)^Edadm?*=42l5mTz%n#o8XCOL+bKB{eZ;7Uyi(m=tCPcvd>)E?kBEm~S3k>i+NNgTYA9~1~DkmEg;P0|mGuY&)CCwm<WHy+O)Rq*3zC|RlCCYFT(O{P~`;y{7RzzYO{|jLK;QjR**yIa_iAva|TiE)Em@e-oKnDGB5QZ<eu%?7?k=~`%3-8N!vcLdvBug}fDZtLi&}vU4fI-BG88AwKhKwUNtuag2N#AFn-`3ezs-Pv=Y^$J7cj$YWVwffo6ecPZ5cc2eI?;R$b^+MaWu1|j%ggJmsTVq{yBgT_1MIw4X)iVFQ;G#kF{Dd;`@L0_;9sXjs{8`2V0aeSWEAVjbZxCF7vF^@W{F16#7JntIwn?c@yhDJd@h^vFRKZs1C?u|t(SED-T5*dA+w+EN?k7dSvVU=on3Fi=EM%xq_j6ypZhW|BWz{$P|vOQlv*Hr$Tg1j*2WKaBKlsSERE%+7eTMJu-joRLH1zM@u4pk%s@<yQDFpsa{Jux$9jCM3`H{s+rN2s@;6tT6($nV!fMJK{wTU-7ufb2IG~{AxCS2-k+B(6VQFV{$<4Icid8UxSn{7B$|Kq^T=c*XJ<9bNN_!gE{K1LYVFttZb|1|=fxa$dW!hUfvQO7}dFNiTxNXX9jH!uu-K14iZ_68-#@Ghg541{~>g_@&JRoTHi<B*Xc?Z%%S6R4V73Pqj*)=YTN3gt23zIlOg_hoVyBKVX7kDtD0*=bS<dBCYTwL<;PtnKkqK|*Z@sg}Y>nHICSMpK)^+)erh9$MzZPuLb^0<LD<f9ELL%rI>#WgF~dX?c`UuP>>35DtB7aBB>xi>)W`c#@tIf4Q5V*2=QlNMO_mdE#0yNenfoL{hHN`MZ$1GXychM6JUm&Tf3&UWQ)>R(fx2O{l<<g5gT`^ee#deB1a$#sCx(~2PG`gwP387$||&n%Zngm&6OdUDL)TEqvl7Ctnf0I@bk&s$SbARCw+AsPo!VKz2PG=R%7-SyzI!$vUYcuI*uswy}Un~wv|M$ZS|$k5rlwH6X7^n9&G1Xt_PWkba6*PaP+littKFl*CZAv3%R`4t7RT#9utDJHB>ktTicuksAT$fWo1|J(A43jShbQ=iUeV4B!}bnIQrp^sEZ#Nj|rVUD3&qXzB}K;MfTGyrYB?#35!-Eh!!bTKjYsZTp7u7c7C4<X$zp8dtOpdwI)19hG{hNe`=300ncW@2BhJ|*j^Kr!UJ-MvyaTw16(q*^hnHq9;=OcPYag`yUR(UC=vS}ovRO?JK<UKJ5^Gj-!5V!v(-dSt@J9ho%d0cIBj!+@?ut&cyxmjwC|2R$mpVY1Rk^&Za*=9UGby=8djYYkuAm=h~TqgAaAE=U*8)KbT|MHd!yD((yiHlDU9BpVjmeo|Eox(g~qNKIN6Rg&d~^6J>c;aD$0of3o~WQl}A*V@W@^sOIM(3holAkYsO_LidW7(4tzj$S4hqFOj5(15c`MPbBSvvP-d2db=%Yhins$nq&f8JnH)L9%A*e|q;y&j=rkf7SQJY6F%7c=a6vX%BUW*$nFfrPgdgtK&Cp6Meh~gRme|2`oOW-v@Go7Y<#o?#)7@y1fF7R%u=9w(U$W<pLIvtfie8gZWH}1i=BM83G2_Vy>Cg__$7iY)Kup1+iB`NlVAvyc0Qhhw7N^T4ME34i#5bd|#t7w#M!YBHsPX&Tlc%sEH*}Jflf*D;dL)!<{Vcu})=Wri|h$KQLMHcvs`0fKAra;yA$B>@4)oj4A?WPVCS=>&Pvp#8mCC9tm2NGt=vEw5lPRz`%a1T!PWsjl&|Z->q_pVJq|E=oA+TNG7TQH{44{fpNc{A7?9**QU^25be5W3&zguF4hrC{-*=eQGCahb^G|9cwq2c7&q#@<zrB@vptg}CVRSYL12cI2~?y;Z+l3dwPRiBUDTfpddGsd&E_<AwBb0_rXP!93j)I;X+d^Gp!<z#KT@r|o$}x<NUu}{EY;)#Yec&&FPn*N-7x>+ruJX@;7ifLHl2N_o3Lv_?|E>ko!;QBt}_P+sm}N^WCl$|lgj@<C;K|i^R%LJ<#^S!;E992kR^`=IE$FsAs6F?2UcoLr#dQ8;0yw)lvU2N{l-zq|9f4<cO3;9_mLbxG*o3vzhpbYd^!K_QdJNq_{(v^lFPeT6Jh{{YKSnH;4JRa(Eu!{8>T@~Ure7r{|<=#{JS8SVR_WFzZukoX84~eLJ4M6(&{`1I0bm$4V)1K6By;uf6A2de_lR%50otM2RdYVz5qWl8F*heRe6&}-0TjRJl)^$E=!ut;#oN2Y-ppOJ8F#MUXqy>!f1hw$@WJGv-h5OL{nt%XCZAa0u^|tr$88f9ayz%MUsNoMhk?M^F_PGFmx(<noB80pY7sWw18o!1j73dH3B^2NA6K25Uk3d4!GyUP_tce?FW6k#M<r-mr?2}Lz#H8SVivWfw2j^@AUW(0jijT{*fgR$dIv~^}2GH4KpbM#R$uGwm@&{Zl&75jKh*Xaa#yylSLR$lzh}LJc_J(L<wo<$8>b!q@ud(a+}quYdU(!WFIL`>A$ih&sL1wlcLp}FJcl=ARCFqaqLU$tmxYRcm`6g#-p6x`x2#_mwruGEja<)Y3@B}z+I;)dS;zg6rf6ArUt5^ZdqUJ2N;k!Iv1Pc$=!n|<^i;NVV-)zC?sAxz?VPxp(cbRztLfxD#LbN-{VvH38>yg^Ch^pfJ~pG68O88P6S0<a}lacAOHTf&WuAzi~Lr4z?NppN9#1k1fq7LMLdbF;syw00^!0C3E~cvrz{qkn9GK~M!U<?ekz7NR{DeeR8cuhX&RkNA}xx$Gy{hHka>owHnxKkml51mIL$q);>ppN>)=F7Q4}`>(*WpfrtaS`j{A-=<}DTb*fSC|S4O?lO7c4bKG<oLHo2F9R@uyzYN6A7#{T6acQ7DNJR0adKD8uTUJfMm=>NY<x|d{$IEh*JQjtPNN+Pjvw9RXDq4Ay>ah|Zxm{{pb2U|EmwDhIl;<&J(AC8{(Wg6BWJUj-@ShO=K#2GV6c+j7CImL;OWoTylL#`tqAnwip7<Jrk^(BYs#w@^D*_5I-wF7{~it<&bDOnO~MG*X8YGq%#al&qZP%(MnEOC>@-BgNS)icn~nD0o5_6!5%H5pHz&~w4I_A5MngRfbh$)mj9V&O@h#q#bM?*e>>djMH5aM%u9HQPf>R@1trh=``xc?w4*0^wzzLRa3eQ8uUrU{UJ>o60yxC^m3NJC*MV3Y@{u-(74oBDkYj;$(?S8uNkaeh5R!1R)Z>NiRHVdXTMG;lqyg^D$nv$HzY#a`5dJQi~5je$-N|U~oir1|~~HcAvT}h~J9ZN)8y>$IFDOUZS#S+8S+ZV5suO*R>OGy_qP^oaSJg)Qx3b>UJiY6Y`jnB|Z41c7jyhmSLf}=r%Q$)@eW;=S#a`&10T=FXsgxVsuxK3VSa74li*Qnkgi4>-Fn!UHY=o5nU~jE+KU~N3Rm<yTgmcpi(g7U~_jc(I_5X(l2>ZMk8h5pcWr27`P?s4zmyUxFiF&939DP*6KO=x7QejDjT{gcu?J&boit9lc;p1v+!i&k`MrUdE4^&E<SGIAtRms_`05XfBy%nHE=hfj|EK{@>ofjVv3;_)gu&3K(&Uamoh%yOzsNI+)StO;f|j8!G9#a=7t+T<MvZ}WhR(AD~hr)beUn`v|y++Vuh9%<+ngt)Kgt#_6}hjuyWNIlcSEu*ib0TK|^8*44%M3JxvgUP&e|HKDj<3OzB?+?t8$f5s;Q3EE)P-jPi6Jyvu!;f|(&idk#ITc|LgUSI@i{ZmKAhrU_R;2tUnjy0C4tJi82o{V=t|I0xI&QMs~_8(0a>aXm$jcdtz{3edP#8vmMJ@*vpxD7b<Ll+o&##h@6Y8wa@m@L;m)%^g2IkpAk2>e05TodJ3xOR!H`n_+Qk-^C2~&`aGYV0cBOIIWXRS>b58e<5VEE>)*>wU0_&?ggdiz$A!ag2+o`iK4?eYBr^~6ciIs*oS|*nS;YIbW#Dt)9Plr96I&a<Lt`-{Py8JXlM^U{4g26t(1*QGIM|C-MEgd$N*<qi;BmKvFLXY$p_^QNY6l%r6C?-jEGlw^E}ST{jLYb-a8ZhVhAIQHjaXN%-o#g5vb^u0*>W5uoCPo3|0ylD>gaAp*<&r*wuP?K-cNQPg|w=eu9pWVnSt5&hC)v=x>WbkXRfmY_7exdQuuqC_A=N`C=3i-r@UV7#3!*gM-RjR3YM&CI=L<Q_mU&P2<%f9Ru#HDmi`Adc=ZtF!HM`tq*9zo>A>|31oswTFB#!g#FbO!@M+5BH{WccxP1G%nobr2dX0V(=KGsZ-Q}PmZL2xE!8pJ{F0&n@TeXKgF1%^;|%j09CI+L|LZKnZ?X>AzZ})<x&jj4FO*@<zx{EYRZx3jcqd(~djt}P$sntDWRh}5-fNyK<?>X%V4robA{aCrMHN<4-lPRxqZHR=A+5<#)L@fyoLw%}@~GA*g2%NKbf8H@qve6<h2L+8c-|vAj`To4J2*ANd^Gp(buoJ4KK<H|1%{I$<H=!{j)EkwW?0xZDX|y@?j*0^Six9oaE%zj9#4b)S4kY?bN>wx;a*LNsz;~}s8?C@Ldfb<)UK%i>n4p8uUvy7P5FYin9irRE38AE`M<Ha6LC3bfNAL8!Jz+e$0L#fgFuL@qF;x@!JO}4cd@4QF@~Lahs?pgu<v)U*VAw6JSI0X*crBEQ?AOq>n@)h0c3CDWw2Klc#QVuX7=HZHuRij&G>#wC}UTa97kgfju3Yh6`1@ZIEHAQkVmGYKaS|mfk*T&iq-w0YSIyqELBb@xpe%|fK5}NgKIuN0&Ki3s|HW=M;Zt92rwGnDXkwzwY~|9v6TD2!)P4Ie{XztX?!EDj^J+;Xq2&E9WzZT0FI2n`~2H5yuxGH8yzuz;yPfx7ZBE3Mk*IK7bU%2a_%Uq=|%Mlmi71>A=)S&WxQyB#v2=#s+@lu#*bsIH!hPHZ@6{#4ELPT|EB+UF745Z9sfOf@^Dmpa#H5r?a`em&J=pL6o_L|ISY|SuFj3T-R7aASJ>G|k4M3L{H<MiyyY~HuhQHXdCH^NXes(%@=bN>=O{QvCY#S-39#eP@7HxaU3kw%b~URhMs>Ub<K$+tKm34#!8^R0^xJw4uWILJ9d7#;=!ag>N4x`nL?1e#2zbv&1t+u+?ED!2@>hD?CZ`;=X^ca*WBwg4XnedLqr%V8`47$gF2ye`1ha@fnH5EkbMFBjJo@4BfcF%r{)WPW;+j+)vJg}GEd`BP9qy=7JP6^B-bZoyI@N*XR*nBDmnm?pZ{}TG{N!cz2%zeVpvK({-S<Lky!N=ktFS!j3(m$A17vkZAAMXABZT?|Y8P#DvHpqdJ=&Thw*UoO_N9yal2G1EDC^P%wfP!?{iFW61>0Y2*AF}PKKIp+6+IC9<36&)_T%Y;EEao|pW-mib{v`Qp$`q%rv(!0BCv+9OBh>*B`<7TcuTX+%c$vMs6(Y%raBc|%K*nf`GX6Jss_}LI5=(-cijbS&8`0eQA8px7}tISxoivhN56@*mltk<ERRlcMIN1+n(L`qw+PekS8<p155vcbWcvJ$ziO+BKEfyN?BwsvY%;9v#>zrGyLH0xOnF9EJI;lPXHd{ushdpqDYs+*CCUEsv@4T8=+%Ov#vP)(LocXLgCZ9I`<pl08E-+v#!ff2Yt=hYZNA!fsQOTe<(^X6Ud5_Gpg(8?g8jm@nNQq7?fo|?vPk+&=M02k!cUdRBX!dUePCTc1urmojX%`(P#;EArH%y96~qo)<>qq#*>#(Ab6ReR`Qya8c0mG()2Vic>?$wo)OnWrsDok?<M3z?A9Y!gIOp)Kvr)$aa5D2lleBwv+jj3|BQvH+aNt4`I&jckER8o_VKS*Ib9qUGykZX)Ro#E5QQ7x>{J2X3YCJD@c-e**U*&uwHf?>ZO~6<nuvZ6*2&}-VuMVqav+u8^PhKm`LJuV2mdb6CNWq*;^mG`u>mLp8YB|;5_NOdF*QX0>P*E5d3^wn_Z1hu7bZgscAgm$FU$SJOQU!QsD0=n&<@w+JP>!?kfwZd|!U-?6>Tk;McR=FP%MY(Fe$ufJH&s;WwKL@O8(=X|RA*s!e(~YUoB#O|U3_@|&!Lm<^x-+#kAG2LNkexxI-|Xq`WK&meCNxekD2Hb;oE_5rlOH-i>lZkZVOEo4(Q<_qF!n@F}#si-)npA1jp_86R%k=_#-M5UW2`f-vDcjYcy2(JJN^?x#}0DGjf9Zg6=QW_jOj&=YNe`9PkTIdXvJNp3S}wy=Qni%(K~_><^>jt&L@?DqF2v={w;l?`RvvQ4(sOoZU<pqj^(8(<<{reUc09??dZn$`NM?7DDU9o3|LKCUT*#?Pc&*)TViizC36+KKj9ks&|*Hb;KSohc3J6pH(+47oBNM?84!aX}<8}l|QfxP0|g)!K3a|wWwW%y}SLMWAN@pbs9Z=MwSRnM$yfeTiSIe-~d%R9(vm8J^m;H>(hyYF1<tK4${~pQMFM)?;1I}K(H;`7v5Of-kh<E|4cNji*hX-xsvY+>knL49l8$L^9>K(eZQ0814w+DyZeTL!{FkZeykUbxNS|RGCp(Qj$6NX7x2D=ri15HI(SQ+EK`gN21ZT11KeeANS?z2Pi{A91+?-Vr-1m|1)_nv*a7TJ`E1hndbGE~3U!|R)f^AP7uE|b@To#4QtoMQRpd?7+Qo+b8$#5*&eq{aWB!y%q05nh{rT$S$N9I(LeV|6?sQVK5BKlQ5tr{6tHz_z<@#ka?zU!koX)id)O=_G3^e#kn)#@el^>|y57k_f6^Nw5xhC{RJJKNkdOB9;F$6rzuo%0-jLo-eHWTT+nr(~(ce9DwAUrQa7Ebq|HDu=E_rynEnBH}>uVr5xduzLM15Af#ugL`StC#jabZfOE%T`zY)N*%O&_D+A4~uT(y^RL~^Bf?0SvA=@UNr{_AN`joj(<rDPc4ABoIH6!9zsmH1^yI=3`P#rFZL;8UAd%@B<$vRNF?$0I6}<ub{LijLbSQo_S?g4{%bQnroKtieHMukJMB~meFGPT5`X3f-}XL%yvL5r_|)HWK&R`9+<6jLD)wK=x0f#Z4#ZMKrmPD$bssV2&qV>tt=rs7Z-EqawLO0r9bI$>pw0abZ@I@W_hPtG&@d~m>4WO`^0s|k3RsDc&lOs;VV*NIg@&I0D9)2>pAKsAt8l7?n?>2$p$()r*3BO6@x;yMW+S$k$7n}4s}^Mh`s+Ho%2Or82lNl#qPy@vARRHjgMn?JB~Wq&#GnuC;6$ntzeiMFQT{UlRhj4QN(RB(0<Y(#b!15+#WYMSPiNj|`dU4{$E^}qS&cVC+TmvJE~V=u8xQX-PLr8`NVe~HlMfiXQ~%8wGQb=9((BtZONv~MGYH&FMch+u2N+7cjoqd0B5Bb7vWoYE_osdOU`!P}e|1WXEx*@HVWB*YYy5H)e#se}(<EKxbcq=sQK#=Lvzs?sHH@>3B&iM>N9RMvk<-x~=gqrZSNDVl*O}e&!G)&R1D=5ycz1@}C%?M@`?bQ$CAhR1fc3e@3!rA?jA+krUe0+?>-o_A!XsNBxnC}Jq-}UY6m{!%jr#bg8jce0gnO)vE}>FoVaRX0dzgR+_l~-x^Z#`ZxlB1}J4Iv_MHP4kiLYN6MwM)?k5@Vi#+=ro`1B{owj=R1OF*17A0~9MP%7A;w&djy*)<j05<@W0bD@I(YkhoG9i;7D4@kukbJ^LtyQ-ulJO0nU7LYzt1Mgr)anr~TPRQrtTRnCsPG+m`pXnKH`&K@#+Sh8<dmkM=F}}8He+eyJr&UGAP?oOGE_rc+wWqhK^Q%nX$>V9KTmatL^3LXeM&(r%TtbDy6kJ1B!?*8F-eHr3z9|RWS>`N#=j~<K5;n%_<_ho56ki+cx9(l1sJ`|TfeOvGSg8Tl)m>JQAYrTI^xrMOYwk((W!D8ZM{m}nE|^-$3Z|fOS1zV_eb?P$AOkmXt#g7I*zzsEXT(jpV%rSelFKDi<X9x_c87O(@tyq{*>sEN!I;}^{})h80|XQR000O8001EX4C;tMA`t)pqdx!uCjbBdWq5F9a%pX4Zgg`mbYXK}b~i6`baiBTUu0=vXKrtEbZKLAE^v9h8*7u>w(+}u1<QUR9g4nlrjJpo%p|d$x^?4B+;lpl(QqIVlGvjNh9J+Uc>LeHy8uXl51lSa>kmE(T<k6ui+y3~`~LeAD+-U5rD#}_i*hMbQSr>Hc%AYJM?BVMc>^Gsx6dDVwybIn4_;pGi<Xxu_o`a#xQrKzui8eO08Q$p1(WjmPCR(2sB11OQDz7$JyvtC<}jK86((`=eJ+6xD~Lv3%8I8A+ZFtaMRVk5ngQ&=n~YZ+@id;~Csrb%dtT@JJSE;eQG3iwSt|ioTfq{%ooLG}_Fl6>@<q)L2n7^)P4G4Usm*~qKv%JvHM}@$gd7>ETD9z4YWnv;8r~dv&Etjd`-{cC7AG%B_HEPFJW0I#REfH=<S7>mJ@y|dN(G;aqJSmhTQec!d)5|BmZwcOp=?jp8AVi9YToy6-WvEfCwAbPe8Xf?w;+uZj~Og%&(fwt%wR|IG(qB#{iBFLP^BnAbVYJx@)&vY$X@*6kI6nS_!w~pTbAYJ!QkBs(2p!px{ws&kT-HX%hDn}@B41#2QFd7kv*bj>1D-P&JLxJu*=<Qf`teF-juRopiHmsb5g2^MoBphL!EM}cWUcIeNIT3)nLs};+`jF!(om>Yu;7`OL-#MiFdCO-jNJ7m;9a;Ewh*S&5`BxTmF!zX2atXZ;m4CmZ5$oyK|Cf8UpwR6-2dDAo_Q*=ZsWq$C?!O$PDLd{}-S)UIND~$weFN^IC#HC2Uohg)3Ud!%gI!Z@e`qQG8+#`KdiAxXtRt*1+O(sK9Rl{8oeKr3L<C)q}&c0tflk1*-raa0fu@;nrIM&Cq*hM$R~LzESY3htycN@K!zEY^_;p==1*Vt){r^F&JhLNIyF`^w{UtdxNFsAk?yB|G5`1?=!%?G((}6?>#m4We3K=h<G>gjgk9_$-AIG>Jb=*{BVRF_>gDKQEvckZ*@*ZDjFe*UI$<@9&Y>#Mgl=4Kb$~Kl`^d@BeL=BSN>M%^d1b8QOv*wh`b_~ji!(PKw|}(+qI$w>R2Cmpq}^j_vd%J|A0M`10?Qhl;C(VPj=&{U~L=eZy#mgTB4e!<Ws2ZiT1845@fJ=puxkK$>v<~U<Z2;>J2>MnSz7g3%Wb#Qdt@}ZfKH;Gs~u<?QHl#)H#ReLLK$Udlq?cX7DJkZ{j~J^!mY1yyOk9@i?I|^F+y)jab4TuV6b~d^UEIxc=to@rSC&Q*ge`5ndiek;UkBh%>M;5`?f(%0P(lch4~z9hR|%#RuL5KKR*6NLZ}`^&^<7<O`;peCtExfq3A5f`AiHg0NfA6<BNVb6y1&Zh)hu6RyzOX)TyDu%hJ>qw%dJCpHx74jPoHn*d(+JCI3_iqHvg31!Q>L1eby3FOh*7k4fo51awAKpnIaOD@w|e8@l?H{l2vX!RStvCpI78VtC<-Fm-Uck6JFgsTzG(Td3If}Mmt2lXGE`yB{cpZ9KAq6MJN=+lmYwPtnA&cQ}udH~A>*v|0hxDCUhSP;dRYon}V9!a@kR$;Ui1p7j;CIG-NGJ0aOh!W5wAm+;trSigBQ$R_&QRf7y!zbw+H$fPgKU$$lDMdR$5k!vLF2RWMtlwbbIr8rKd0nv6E@NKa@YBi*miQMuLqr4M0`!A%dDb3{G=_-l;BRtu3s<+Y0iVN}c3e24BU&y-`lG%1I2s`%)AN)N)BZ|JZW`QS6JVX+Gjp`YA!gW^5zHr7^IgeIh%RPX77&j!u$Hh0*5NU9Ek4Oin{^026DxKMWYcD63@ISR01LcEE2D)DN1rqhA!7Uy_!J=e3VjC2LE0#5reGkTm4E}>4#v0;IQSMN-<=en?%+AVcn!Sb`k!?Rb{%Y@Y!Y!tKlQ2U*O^qTK@-q#yCVBR)OQ;njd@ZF(fC`W44GOD5xFg==bisbAS``xbNi<lgMG!Aq9#QP@ix%F7cm^|Oa?;^;+nH8Y4`)2$T5#9Pe76M3Bw45nx<U<z}i7^pkDT-72fx1O^1`L%4+CBHtJmoAvs!IvJo(%%mFT|Gt7Wgn@y#b7;0n443w-?E1)oCG3{3jMr&ojQpcvj=(q(MQ9i*O;QqCiB<dF!Dhw)XNG6mMkGzlm%jG}5djEGgULQY29{l?H-M{-E|9JEMJ^lLBu?zZ&e4^T}2KoZ0A-zj>ZH7LsKVEh?12Q_`iAca=MqXlTxK76sAQAxfe!e9NfbS##FwW9D_(F)g7PEto_NLO7*?VW9?Ttw9A07|Wrq=gZxV9KuDplxM4@_*ovXwVjscQs%%`(90r>XaFVgbOn%O1mk5bGw(H{!=HF^r7B$ld4s1BAR36Q-<E8&JTS>MyRVjQ9gix_p@2gp(VOJ<dc(xv?;8$F4y8=8snJq!eHfHq9wPI7VrLY}d&QmZdobT|r+(*@%xuq#T8=!<m}8RRwUSTg-){O_g`azGj3Rz*V$l%M1WOa16}eaPFQJ#g3(SFl|!E1wXL#oOEE8s6V+{+r<m?t5<1K87daV0^SGn?xa_32<I>{JwiBB48n0^L)E(&HUzW`qUayK{o3z|{+eXOQaE^R$}MOk9Y%&Fpt07<rq<ELgfbD;4ZYq{KumGoUuds&PNTAYzb)@d@nLv?@L$tb1?H1e1aQxz3-Qn><-!SrjBA8+It8;vI;$AYknSzBn{JM59tw}F)n*)6Zo`%R-i;);<8=}oF3m(~6Dr)}uy*|8r^O{atHpF)hnXk&oy|0d+h3e{O|ZE^NsO|gE#SINLq}7`x^z_rI-VAcGr<Yu;~Ewn@z|7M)Gt+xi6%S)L@_0<a_xWf?wfa(g#0=6$F00)bq-DnH?s?a-0g&*WTccT31N~aeT)K@(+On86Z-fZU^U?huyKIvWJP9hV&)@WpI|V0RowbmPhOgny0pT8<^79+(A<m)>&(0h(^3Yc$e6;iM+v@y)%O<TstTzU*zg0@(S{}Y7)zE35eNv8i{35A^_jA^J+9gBhQU`<=U`l7AuHz2p`kf=OnQbB)Y#k%<9R0a3I9ENHsPmca^UN<=~}%YIQN2qq7F=xC9m&!)>FY6lgR@fPwMFw57kWv&%IBzc1o1R`3tWt2QU;}LT1f!$inAnS&EyOxHd8OX^9G1fj%X@kCX~JomJ~xt94Q;G)Y@)Zhsd8DIG9N5y$0Qn_v>(bjT4+nfqwqI4ALilcQMb95B9`Wyd~EWfjAZ^e&(nn!-sZa2{gaPSgLXWkueoWE<1od0Duj8b~tfrGxb!2-eV~Y`td`FZEGJ_ylo0?$IZi-^@lO=yQ>+#ib=}CYFN}JN%E#&dUamWl4m!))4~zJep4TC9Y;;T2H0Ii>v-WDfukyl=9C44=yh*z?}dH#J9#`;*4pT))(U^x=Qyf5p_&30em<=oG%|0kSg8^tpO|LZJ)n*0!zO_Ewn)xR$Qj<K-`#dFk?NwKc5%Pu%L*AI=ZfdL~gG{vew}R4jkgMC`5wKvnZ!UQ**42j!Q~049~Mi!De*9D4Ifr2^!Z=FHJc5)FV>K)Kkp4Wua-ceW>$`ERe!+5$X!Oge3tbJc&#-HPFI{0=nXB>=Q*8j~gh}J&|?wIk~yn{sK!j-GcGk*f316G@Fxo?EK7*AqvDGAj>nkzQ)Qut}OFQ`;8p^Ot3KvOLdN^kbSe7+0p-rIkXIX#tYt1X$1=$7j_V0w1-@-Z>AaQQiUF+8u}GDL9nCZ6-6+5@vBFps?BKrc2wP(tXExxAU)Cd@opLE0^RT;N-T?E(s4mnv-7vJ!W-MQr;2adeqI)!ito<Xh){#h32}n@|67bbGa6?yM(I7BV2^fW{RR~@3;8VTgR0x&y5*70o3oL9ErFHi;E_}6W+sbS!w0tEXI=}L6!{$=mhH#$=F1ClTV<1FTeGh2l3=Z_2<GnvoLT%-HRm<faj<ns&BFCxoV0I?Vr{_}#Uy<AgsKo>iuz56JZ+ziO*Mj;sM%N*tE#1nRb(isXkKFHhZ&hh)HO)1-6qIjhMaCejTV;z;Q1?%XihY92vx*9ep^E%B~FRtJac1fS%ICQ@k=)aX4yu)`dE5x7_xe`u?D&x6$HB)>qbX<+tHP6FV#%lQd^zELEi@P9K2sgY)N-b)L>Bfk-mpgluf@Y*k!xdHSk~E_+jJ?#%}$vYiTrH8q>u~jn`au8DQOvY6`)sy<FP2+I`D)uP=7%#`5Lz`yXDtgk|7$y5SJD8}>)819#_e3_FTKtFOYUYe%a7DxLavjA0z;`RJ;Cg0m5KzW3R5wUM}84D1nc?%wBITtzggab0pG*J?S@;L4oZ{u64G$DMZD$@|3&Zqtbef2CKCv1tSDc5pFz;3Ya<RWf(D?%xci@CTl@&WuVFc{;?b#REWEGvh`i)C&@-niLJ{m;@n*m;5Mk)d?V{?~tkddhHE@I)jLCT${Gd(aLGIx$+iwF%{xiT_f&g!|-ldH|dB|1)(!8O7|u?*1%FytJA3KzMq_9e?_wFK(!q)c3?m^roJgSKKdIG8-MrC>Oxhs$y-=`PD7-lHYb-jSY2~l-6^^sVw&uzK62#F7&coNZnawU=Lm7x8|yA?&>5(99o#P%Z8)Q%Zsf*kKiiVItB1=B7jw=Sh`~v9->a578v4EWBSc8pmSRlnQXfBcS%6;w;YT*AD*%=2tG91Gto+Lu47W!_GVj%`#-_d!q3?Mb5PhRQ0Ai-4#p{vp&LCCM909pH#j&ebedDv?%<Uml9>Df+D?R4iL#iP%(*xgOYlA_i3<64d?V%&H+RJ`%I~G}5r~OLDHq?xWZrVD=dv?c@S=ajdcI;YTzc8+q=G)(rSaFmy9=PF?#iUKXXNtBPg^yi%E0@Lw$5fjSKB0I5|J@I7z6C$)zx(=c%fI5o*YyAEZ<jy%VSh7}XX9##!y1SR^t=hk<o(ogs`t$~3u;xtsu&M9w{G4Mx21&X{AV5v(g|=4t~BC|-{+}yAL{HS9*1?WkfRuD<2T}tV>$#!b+h_&&xR~nC69vK=+uW*m!yD=K@bbMZY3WE0JKXZKMxe#vCfIEZ5O(+oa;7POyB8OLgxD86Lrj;1iD+{z@{lCxTQd9WV15(T=k)Kd>WttTa1F6{Qhg49w_!A&p>7pPYWVm8*U;GVzml62UT9*#y1xN^xlCzv19w%wnDqR4Y>0sL=?^3Yk!=tHLAm^d(}XI>ZsxeREtXmlKvRK$9n<vop^ZPC#oQzo_9rC`_p(n&0UsrCPCKtJx^lE9u^DIpX*2Gz!68s+3<Syu35Wa0IBd!TStg$&BO8S^<C{5FI-zUPMch39`%|M4q<~~;9^{NG#5pu;t2jfk@?SDw<w~+2p<i0^X$4P*N$bW5Bdoqt(A_dX8P59);JR>;gpME;;8Y{iqKY0cosbpjIAyOU_+^x@0(dsobdwyl&s&Sj?cP`X(&#RSTZ$vqQU9q^38Yu!u1Oh7c0C>d?+ObOeb|q^hN(&m~26?_ql|S$?8mrrm(*L>h1eih_}SN4|dV+8n(ZC-s88@OZ~0%iIw@D%VrAan{N}k5=onqOPPo|Il%H%dvo_?b%$2h5}&gj{_(0~%_LXB^N|<XQnU@G&3QJ}J~l8Le%IbI`fxpRJ}4i2e2(4H=KFB`KJa8LD1Pa6Xju*UHAn}3v9#d!C*&4Cniu^w#kDc{<*dm@=7#3~1yD-^1QY-O00;m803iS%eYBmW4FCXFDF6T)0001GcyMKMX>Db0baO9sVRK)0H!pT+Wp{HfaCxN~ZFAeU`MZ7vrhN#NQj(K)*PT-J=A=&Qotxxbr(MsN<3S`OaiIv6AT7ID_S^To03awzcGl=F5d}VPc<{W#!{PAkc3b3WUW?0D&&5ryb|TfKsjDWfb6pBqX5z)g`CGA4dZX&<PAqjLu1<dzH{VVMgO{(y@5G{LDkJK(5-;CfT!?%lSBel*z1eD$*NWd($}~kS-n_o}P2{D?Glj&|s?trFO$IMQzNNZR6=LLnC=-jc(p&f{b+wTNFM$}QS(rMn8)P6$p*Gw4PHd$4Fd1B^l}zu%*%ML8-FTsp(Rf?wR2c)92$YvAkprU0a-dSCb-PDUL0!qTo(zV=;b5?=^hPAfvZ<R&B?;_stE(DVmb#WGt{DvM+D6uEhR<Xz(?S{p!Z`$!<!K$auNbnv+mb8{bXwkdYGt$8-T|5@w+#CB{KCPVqnZbU!9P5;2tfW&<@~B@)M!9U;uknkQ0y{a%?1KKpj86`&jhGi%*C*lW_<JQc)4oEH&2EH;<VslyxNjfmN(Kcn8{ZgnN+y{nOGKD)&P4lIr)Z($i|plmI<6_naWzVFkeqj2xOtFOjVvsrIv+C>nA5xj8qrE(QIX^JPg?A*x-MvOm(KRag&$z-x(f{G=ZP0+UkN%#iNx<f+f`qM^=&$GyvvlyH7kj1^?x#Ea21;I8#eFcrfA`NauBuL`D_M5$h5^`BLD@#G4M71Q@sY*w<Pth6PzAC~Y_V)Y8CDC3B<1FS2OVSykyO8onz(lzLb8jdC~&xg$Rme3F|bil|7!NpLzNa+SSB@trt%NP-Z2)0kQ;z+s68d3UMH682N)H|n!Q>X}3<SnrkiZvL=*eJm#>MV+2p#EPx1J*L3yVsC`}h=CEPK73R^H(PW`m06oMGtrWb-7`Kok{!9C4tbKYZ_MH^`-bd3PKiCkCX}vGJDkx2Tk5>U&+x@smMhpH(g%JUSprD|Tl&4pD=-LH+e)w%Ror2OD%3`ms1ZsjHtKZc?w?KI@+A8acYo+O-jpDN3iBG_QJ%txABYDB;r+bU35Mr=53dwN{nEy6ehE<*5f*tVE7o{~|IHvwPs&VImAs3ES<5X%uDStc*T6ppd#m4%#Q6IFjM+ev0r~R`O;}p8;>X)xS=o>(Yb7(GK}&f7V#Dt0HQ2#g7a1lFqn9;-nhB;Wq)5aerU55u=m$y2DhGAb(u71rO_Puj{S0@$ui54BClu&ofc;5i8WNm!`E+fwKZiW=kzs~=d^5;+#23U{Xk8>Wp47OIo4kyy4zEvQ@#OtzB8wurPcR6U=pv*84gwe~B{;Z7TR<L4meZ5z6LEuB#!BSQmZXvf#flIZ3J$%!o4=HWQSP*Tv{^Pqk<d2TP3UpP`FKMA$8CbNX^&SP^Vh+Y7(b8`;{lmYz-8u?N&-wa;33XyhWF^I<2*y6<m0Aunu4+ocrAKxt;vVDhU%AyK~NbSPXMS3j`#S4*$00d3HSZ;JqL3F2NAFAAqb2$kQO1Xw8q7x=Uq~K!mJ`Z)X0K!5tZIN9R!Lup!AO(H&p)&K_}1OT>fx&nOvT|yf}Mtm0)!5K_3L|fVm{F2l)2l{Ke_j8H0Xuo9gB(mtO`C*9~4(jN>d`W_g3#+{oJqauXd>s`4W8XYrU7IbsA1niW{VoP>2mpTlZvNkNn$MUUF;Mo)pBW%`u>hTY!*Br*NhoWAx}f7@T>EnR`T&aFnS`TIS?biA8g2Fqs_(e4QL4jwZncX2+RRthwbppBdRox8z+lk}qg{grVi3Zz_>2Nk;bk9e(11;F@a9UimJH8@)w%JAOj4c|10H<#zHrf;7AbNu7<`Hxd-%pfFU-vyb!P}L)7bqd&Db~i3``T>rVgY<+Q;Mcrfi=XBvGocs%1~bM)Np4@r^g}EhNl5py08899^~57RVR0-Xff$BCM5JV?5Q&_-K@6kbNP*Vb6OfP#WTIB;aRv^~?cfr&umBQ1QpqiN5$qwTI&N}l@e`JW&T-;2_2;7HFf<8Q?XYHOgVwFm<qAE7j}e%-C(ev?s;WTB{;;FKO6Fm;PV~0<Ex>mZIJ=hbb>L-KfV2h~TWn6y*R!YZN5SzV{4y4(PHw#u&&?uooQ%6a`!ufqi|pQ`adW8Hr+D3c-!}URm16S)3_y3(z{?U=c!;oEbsCrN!|+bI*b)m=)yZ*36L<K@Th|Uh`^+DU4OAyR{q_-EG@T=bUZc{7=kP_){1_jpmD{I1b$Vrco$yK+3WMB+;Q}&Hm>7d3MR8;}>&4DJ8ww(5fc*RrX3sc|VPlLqfWNof1K6Kz5bwu;qoMp*7>5|`L(YuKN6;%(gGZ_`h%ZzcU!MNzeAm`XEDg`l>_`A=_Gfq^SfBbf#j&&D*!`qoj@Ur;%288S`3k(g04`fq<fXDl3!$)*JB~>r;z?K`7pmZ`LoHX<8oCONrG((X3eO9DtIP<Z@2MM~8MVUJh<i?1VC1q<45ZgfE@0QP6<FvrwFwp7wSv|VIu?%jVz*Xp@aB5@O9Qf<?D0-rI^ZP!8KIG*=wz+()f!te`nAimdOh+1$e;X!Rb(ejw?W#r4?Kr!;r8R@k^1C-dsxd(+L@A3bq#hD`IQ+C4r5V5RZQ(ej6JPvih7RO$2L!vV;lZF`edc*2$4Nrr#;7}%+#_>Rm(A^+N1@e1P5efT~{I00jh38?S9;p%MQ!_e~;}dg58vR?+$v1g5tN8)G1UH&~?8gSAAsc?0F1bHgrK?WmA#j$e<eIEV`Y_+3x7{^$&LPO!R)r(O8YRs=#p}4s5Wu_-69;-zHCaSC9(q8-qGam|LQo8qAuLRDvtq&0`uQ<_R&6pr{1r#re<*dV?$nkHz_lnd;>dEQ36n<{rk5-c7^x(x7wbu*<i3QRp3RR^n9TneCwgQP^P<l2Qh0NHz82cLAAit4g9)Kt8L)4lBWSzMg5jBldaP0jvd^2!nxZSz3Ir-W(4C133h^U=RZAD}4)Ug7q&AD6br^qt)G)#*}Kc0Q6q;;2|G1;PYM-;3|km5@576!5X=saA5^=*?>$1h+G3FF9$x86s`my3j=%Q{=zNNxubYx9o-jTXe7$eL)VO4@&dq3o}a&X2mjhA`T#9&AwY}*<t*s=XfqEGAl%;EIS5n;U}F@|NNZXep;3GsI=@a-wt9HP+n<M?-EP?`DYKKQe|*#-`e-Ize|iQ1%lPrIH|za@%dyEQ(y?cQEUx#&e&5+Y0e{Zm{H^yhAKeJ|p7A*uQY_fEKq53QAD=9Lf<XZosw>4l0c1!c(TG?-c5kK(*-b0=lM0Bi2vhqG`~}y8G~T+vwR4leMOt6Q?J;x|b=Zw!?=y4v(8}8J==O&r)!D^yx;@`|eRg_z_Vd+V4}eiLfdVe{=g;!HAPWCk=Of-<=z+X(ylNp$JbcaKUCtNHZNl&ZS$$5KW-A>5tobH6*>!x%3Ic6hu*mpb2V)xKV#4v<eU`_)z_i`w>H9qGuXM>vd44N1!<qFDRq1EKW_P?ts&v+*3d$A8;pT3GQ6*0i0dE8*0Wm0Owor=OIzVTrJ*Yhh6jVvYCJu@gx~}zRTA?bi9}w<tFQCO`2Z_d_h@EbVj2Q?x4;(n@?X*x!a4e(SIdNYyP^V8&;yb5HAm64ix4iCZw^X}i7t`0;4l)@#NO_nk-TFEXXpXv9XS|EKh$5mY#sLX2or<TB-I>uiBtoycq*$K`0LAXvmJR*n!43{|BPvcT54We$fpJ2Baq$rRF}WsNC?`}z67t3mLQ>(uplt*@j6tM2G4F^LJQiC9*a+&SJ^!&if3Nev2<$t19$kOt=y&a@#>7B;C7eMZC;O`#<M-r7eh3xuO_(D-8ijC&A-I%-wg>+XqkDhgZ|r9W{)i$NdH4%S>benny&R`Sz8!PRFm`nV{*6X}=gYtK4v;#<VCTYtvL3<s3>dt3TS;#3Ztw6?!u}3HNMt5)q{i%Q2fG(vEY5sp3t&jA0f@ia4q#lnL0U=d#1~+xa|o|VY+Fmmx_9HrCpQW&_wY>ZaoIK$ILm~tFVS|PSJ9hex4Koiq6LPUyi%pAWUZ@X_qu1daQ<UtKz|JdKS0_Q%~on51(OklPHB^vfHDQ0RR+d@eGZLq+=%{a4c!pgl&1MQByk+;5HVm7Q~IT$<kVfz6F9&?L_e?HAV>0*cMhRBUo?adtax5FF7gk`UM3?)cY6R*r@G2Gi)^6_X3sHdzyI<Bs^gc>4g{2yZ?G14dS81+b2XPQ>z!`Xs4Vu{puHZ=EypcKzTze%NDEj7bYIz<nWl7gW&66>M6R9Vzhk2fVq!K4D$Rp-u0e@bZ;-?k9Gv9`bu(75Jz(Gh<=sIh3%tppY6i#BJ5GA-l^zp@YS>mbG(P3`=J%@R_apoa{4l-JI|Ff3=BUjqzk!C6;bO-vWW9FtW<-NXi~bcMG6Qj-C)SYr?YzQ7Cs6f5Tz4e!9+2TFdLKsEd4yenz>2&^J~JEagIy50nac?l8|QCW^E&OLw4u&D!Fxy{PKQqST>l49O9KQH000080000X0Dt90I-vjn0MP&d02}}S0CHt<b!l>CZDnqBb1!3WY++(8VQpe%axQdubZw163c@fHME5;Kz;zO&h@f<##ibOyKuMDy8~Du+NlMM_O<lVghT$>qz=n_=d4-Iub=uW(d)^-}mNZ3(EJU&n;mVvh5TJ(76=epjS~0VJ=m(baYKmx5P`Y$oE6(E7%sGd2k@D^GyiP*XBH<H|Qnkq+$rmz!Nk%|W36eClc_d{d_;I(78%!g<`F?o(JMvIk?#TmJzKCtc<_l0u0|XQR000O8001EXvX8%%O921?@c{q;8vp<Ra%FLKX>w(4Wo~qHFJo_PVPY*}VRL0Jba-?vP`gsYFc9qZ6`G-ru>45EFe(@*Va5d%l(Kw@HIXI7IwzQ~=VVjf-tOMsKJYi_t<eZQZx<Yb=arBJ2Y<6u@FRG9=AU6Q1GkA=pHMqT+yqfiq;d;Cg^9nmUyo*%r4$S&?Gs@mOi8I)RE;PZh6{N6yF!JK85-vE+4}QI8`uqVm+>;=+8A(<y;rCfk|lit1fRktuSLTW1p{v&S}+??6wJ?KT$GZc+>4sE*12$}F)IodBJpp#xIUSK5nhO6&EjH$2^fMS_cCM5JG>}Wi6=Ua@PnNTYe0FVbqom$mW<tQ$((T17+pYTyG>TIsC}<j!%|c%rWvhMYA+;3X_xPpkM}%LZZ^SnhJ-7(%4EFuV-C1KkaN`LqH|HSuCwogL{{o3Dx&%>H?3)I@;^28f^+=;<<JPa!0rQpTYDo8`vXu*0|XQR000O8001EX3YRvZz5oCK00961ApigXa%FLKX>w(4Wo~qHFJo_PVPY+CWpZO>aCB*JZZ33qbal*4Ys4TFh2gz_rBv@#qj5lJnN^_!h3>jDencVB(3r*i`)0c7s=M=^#d9qE+b#EA{BG??1(iUKzwf!{wpq(5P~@bSb+s0#c|<I=ZTlk?MUr~DTdPh0$z1wV?tR-YjGa3l6ZLInZHv_vc71>Qpy&PKFn~raM9_%njR%woxKzp{GZ-8>qKrcP=UuaFwbc15yE?O5o5~VDUvIy5(x6;0UdBl$j;9U5zyMJy!N&o|lqNL<PJRGTO9KQH000080000X0OeApZ(slb0BryO02crN0CZt<UvhPBbZKp6FJE72ZfSI1UoLQY4ava`gD?;Tz`v_lT>%z(M`<DrSOykz!km%MB<&~j9T9o+_B>ZYZDXe`#67g?Q2G;kUab$cg0ZXP%FB@pVf?V8w!AvdwzA?&E&K2&A|a&oZ`Woi;RTuEBTf>2P)h>@6aWAK2mk;8Apkq}{&yA+004(I000;O004Ahb6;|GZggpFWiMfJbZKT`V{~&aaCyC3ZI9fx5&pivg4LHw$)3&$+9FtYiyE#Av}v3mPSX#UgRGUPTl%z;B2sr=U+=&78NTRUduuzWfy66`9L@~q<(VPvdET2toabxy@`rz4ltq4I6|bd`iWdoIRV@xei90S?R@SWMRb8e{B6c}v{5?jB122>eSF3M1ubA3%7AG{G<vhLNH7g7K^5^e<FIiqDan4d9Z$tJIUiKF{fGlWM%`O*m&*iG%_gF3|>r^sPFmX6EDyC(VeN)^@q9CN3wvQz%N_yfs1D~p_lRf5CWv#-M=XtADR+k4BMOmYonn#iS$l{_XRjfo=$koceOJfx$c`PN^u&=PdJ{LRp_)eCE`+bPj-u*76S*OA~40ntC&>N!op~^(g+h=IdOj5^66gO_pi{iLu-^CTZTdh_p&sdbiMOg?a(Z7rHhHq$4z%Kqme{WU{e?->jJ`^%)AI+Qq5~t9WVnT{Ssu;>QU)IC`4-6gpqu;XafM6_Bktj>btvIDb0{1A@97->kRP~cHt;D^_7#0^tf5h@Fhk8IT&~}L@G7>|)o2u82h2-ofi1w<k%i8x|mPICRnp$r<c2BQI6ktt_RBXqIF5XEm7)$VgVVnLw2yAD|!kW=z%ScF1(hnY&rS(-)<@g&muY#$<u4Y#nEONa}Jj=^ib^A?r3WLZ*0RsM5RYpxe(_5>j*JW{G2D73$?BJ^`(cZ#D3ZEC*5u|%Tz-xEa7!JG>BsjU_jNnl5n(fLGfkza|F*{X(9>1E~$jtk9A)AVvg{LeT?5cH-2eN1@Y8XXDe1PXaf>Gm7t#Tvs=dn+LL&J^bgvNwvbEu@>GCB6hVBLO_;431l+Wu5S;_f}KVsKg3a_f6*Z_T`o*PF&66XH`W6Ct*Lk8|0ZiZET?LyDsRVJZy8s;7jp@hp`4`0GFZ$)D3i5F#)mp!!~;vWsuLAPD#TeJXCaRK8o)%#SFb`4sdvv=aNT4ghNtsv{{rfD`O8V#y!a7woHNU%S7CycyaUHR8?~*1b@BO;;}zgk{AG->Y{XOk840k><QJA957EyiOj9*M1%!c4@q^Bg2}<soyaez#2Vo6gvz<VN)Rlax*ySYteq6swKh(A4C%Ej*80}#ZXAyF#T1lfZm#=X)H9*$=SXQVK;EI@Nk=o+BXl@xZs5dAW?a%AI+4M%cK?+3~jA*PFctG5tLp20Ub%d{i^1fc)!iO@Ify_Uh){iFtN<Gx3D3r>SL!Py(Og#vs7<Je@X9tVPoiz-Gcj?A`$5#nERaN4Geksi58Q`BJtfANK%x3(2c?fpp-mDtZBtwvrd%=Y4D@tAirRa7F4TIcJoH&fMHFMi{iF7qHhjF!y16yf9*}v?H@U3xBHgR)O-*zXML1?$-G|*Z|bX4KJpnx>hfOB@d3p2AKDxiBStqxiU50v3z5ObFQHuxoHS)p<(d`z4p0x-z+yk-RF16ku7ogg9uodql&;mV5!3U+ccm!C(t=SZs{x|rJvlb<Kasr+iyE}B7d3Jf7&BSp$EF~}r5}P;*cV{k4x_v!K=6RXtl(NepmkFrLt)h&&r9F}z)J{Q0@v2VyrU&`>%Oi&yo{4QZx{KQ;4U&^N<<65UUd<tY0UwwQcAHE5^7z?z#$bZpZ)XA_pjN3tC$>wgr#JvJ$I!ke%m4zq84KSHUn-?s)t4dAiFOP@eQ|O3cJnk1aTnY4}jMCw@D(%;R&yF>Xck50zYeZ!%Hkr-g74a#_tUK2sSDSgAjVvyqXxWd&i-=l=7`6h=_sZ=p~6u5QO5b1dI1HsX-=x`_|;MN+h>AH!1Dy+t6uiWuq8^X+$U_ihRlQjBK`4eJk!cDGdneyNjUoJD7exCx*7mlU>@SHTs)+%OH1+@-U`MP-MsErJZ-R`T80UnfG9xAH$;h&{ILxD1RVE2MQcdJ(!?3X5*mrm4;dvaor7ZW;HViTGg;^tN*eXV=Aw$Qmomc`l?p_i0Va=TO*1*vR=#83^wnB&i*v=M0f|U?|9mS;;?p@B4NHQdw3iqd`eww2S44C4u%hx29KkP!c5jB1(Cu>dPBdxOD#eB_{%Tt^N{GgOgah}X7rq1ogpa7V66iAL>*`9j&?sGY*C^7(hvPAs$VA2w@^D_2;(zOc@Br)8~3%*hkO^>+=&!%gaqv{KCBzEj2Ucq<ch?*2CJG<hiS>PsL0wH)4$<r=ojF<9V4XHJsIC9`{rq6I9%)BZD`!KlLzA^ZP%9h6Zwaew|x-bzQ#5`JO-1g6v#S0LW1{xKr~tv*mtut5$~ql&>z}aq8RaCn|NW&Pz6TQ*47cbXr8VNOyy~8;8tK!?4Oq<-Ym^v_U()(b>dlPTAd~p%=wx;;V;7^J<FE&f>mYJ<gqUIv`<mlZ)?&7fbfPtcUcOMUk-Ai_XNlN(5xf=z(Ds%UP$%`ID<)^1Q&<c+b~bX^1^J*#obrldN>TUyg}Q5h)>7pjDdkODP%O*+3Aq#$a=dukJI3JHH^^LCcaP~YS&~LAML>ijTv=JfY;gpfGr3$DEGcuO;v5nn*}w0LYea^rH&Wcomwh#CWX%D6g$?zWqij!M1(61;<FXZntky_yD3w*I#CLpR}P(163v!HA6FWkERQB7l7lsr2xgF#Zi-z|qp^ow*znoml=BeDX|uEOsW0caiD<Z|1)new>~vnnsV&;32v?g)Tqk?fNmCWJaYi?rh>-N{;IYGosx$>u5T<-o`;Ey$KtoxlUKh0j_Pgyka8c;!4V@3fZK#^YR3bSOzBaRj=0bmF;gabw0uvpCJj5zN;i}sckKU|!qcx2PS32-s|8`7KzBz**i=)Qbq74R=2b~R=L>?w(b@WfCV}oe*B<<&Nb~M>C4W_Lf7|dk!<D7*%faMg}CYtLWnxj&2sG%8HKOX<;y{j<c0C+c72~4$K7~qic{HQBPoA{0Du>~N69bo$W1i#zj`$sODT>VR*LA3?=8$iUIU+A_|J!(=I@11m!MQVPU3KsC3OIv=wLdXVUZ;}MzJZo}C5BODwWPlWX-jyFwDzl`9)sYdhoX6B5L9~IeSQqRn+a009u8l@6+Ostf2e7MSctQbs8hz3Uo!ZBWHPoay2t{xR<$rF<r9+8o$rRB)P`zI^5)-<S_DGGo?n&OHgt#+IChCg9w!$Y`*R*s}A(OaH9mN-U-a6*Rj>om0E_j~yYMQ31ashg~qdxhQs!6eTO;ParIo^Ovy+tdFD|8iBWOP($S=6U^m(+8(B-4{zT}o_H!n+2>(_I4Ob(AzEV)UTl<+wM=X4E-o=eYJk`yyBv_%SF&&uqimpzLsO7Y6H)HEk_SXSp7oOLd(<m&|j+L&_~)&pj*MrFjb6n)BQlaO>z|;DGCeVGiU@C7i>n2xm^Hh184@y=2)AreF~%{$BYSq{2|QZGNQS=*0evf+X{P<XIzi(W)DU2o-cdUF+B*`zH7>+>Xan)d&Yl7;@EbCOZMwtpp>Zaq(=<1T1IT0&;ISFEYJgI*Gg*L#-{lx*m<DGEAyX50y<P^|~`=k_$2YX($wC$7&SSYr~lr^i5;L5%#kcW={bpk=*bEV=&DfD}A|)=<oLkn2fx81_Wmr0+v3*(uiDf6#di04w%flL1e7-#-GJDBpH33$16KbzMXuTfYVm~@_q;}iSj(~fcq!6vN>QpW*O+{-gygCOkaBu`gHKev=(Ms4YZVPG7P{>yYejI$0ab-3Ngiime~ayINvEV>9eEi(BOhTz`$}h2a;ZSlK<M!1sGJ|;1JcD-+5!O+z6`8>|s#VB&Kn7zV-77QsX~L(=}!DDL`LV!AS3#^A4H2M<;`oZIL}ss#rC0-VUx!SGQmfUU^H0>zpL;9zQ`^0{h^_tagT!t&i9!XX>kA;L`&}0<YOKI@<c`6IdifdbGDQp5m~wXwW4F8^Px|S`@LR(QDFbRO&ulxy`wzrYLQ}M@Y3eQtx`)1F=2SZBiP$N2mt!tqI%1sl7#7Q`u~N;&+^^>1@sZYVVr_vvo^PITT1kQ~;84ioesxvQJ5W2FmEr#!GV&+0%o|_0}9v6j0&79!wH?JfVfIVw(G`%ZV8Pk$K40Tyw$G3mSFtK&Po8{Vyu$u2Z-Z3)&wZ`_kk!TTVVUGfK{HeIw4vaS9<Rp-UPkM=iC{&Tpli&bnRb4ox+v&#f1R8UMQ)`QEk3whMDy99fuk)yAIdQ$XT8EA8*Vvo-1G&8t`6Mz3D~?WB&)r4B>*|G5_}P&9g->njcs>ZE~apwT2a&UB6@ZOtB@nz^_T*z_<6n*{SsL$T%DT{=kTn5VmPzX`8nMs#aKLy=MF?ie%iWyBSAkdGv@eKONcxorr{D189gZE8IyV?UiNyzi+qQr=$b`R&zV>SeUI1fT0(M^n2aSY0!aZ7EEZx1wVD!o&z$1pTx2g<5=gjBz5>Iue8K^JnMPPMP-eXtv!cGriY%u<vIgz>|jUh>x7|ti0yE=xp6(aN7#n@qQkE&JsAczK5jmgZFUf(jFPl7`HMU%HE72_Gfe{KILf7{Tz+3ySuTwIQr^>`eqJIwiGRl9L=?v8FPn;dNj-#$Y#zu<{C3PVzf2c_0%H<nQy7gj?cX$c;Y_T%-y%&>@wjK34M^r-A0Tv_$YIahRvny^ODLqJWCs&U1x?82G+2rPKd;Ft(`$ep@2n${6W;xl{qX%XEF26B(F|6AH}(JtpByDVFk51js%I4vME&8>H+4QTnir;%41zbQqY|qqY*e&t@I7#GxPu<^{sI|DR2>)ZZF$gLC@Hqw}vGB7%}qH{ic5u`cR=o?o}xT9WZzw0V%rWu8d^Ytwwcc{lL|}OnVxUfbTqVR?^)C$mMx`^0m{de(G+7dHqwCUPhw3j^^luyVcu+2WA?k`ecAwJ9U%ONr%*L%s&tsUmMl^GV`yAw#0PTmhbo+q*F84b~K;NP9z1f2o8U31YraEXASy3mf^>c>+R6sQmtLgGTo{w7_r_qG$)2kUswMFP)h>@6aWAK2mk;8Apok$K`T}n007oj000#L004Ahb6;|GZggpFWiMfKb7gdME^v9xT<enCxRw5|r$CrLl1HL=zbMgEc^yyIRpX4y_DrhM(<Ord*kW2uDkRnIo=*SW2iQm4C)slVkRo+!dF+(kuCgQ&H~`M|JHR0g!+4YBx!_|`icLk6l^|>>*!8-|@|y7CE?*UtSL80Mm&K+g+0D8r>ny(_S)Phg<V=t?W!LmdROVptTGW)&nvy$Oky}w_X(l+KS2WA3n$$}{iZsnwMpxvf;9>=%u8S(Gi!u|{v=l2rD<DMvF0!j-JxGi4hSn9Kr63jDWHnF|AJ;1&QLpaDq$&tq)uPO4Ey(*Hra!Fc$LX8<_eIH;gcWO`Dm&*zJ)n$bT;w%ft?t{E32u`2FfGdJ>{gJ!zWJGyo4n3$fDbe$n@ZrUD_mr~tZ??LEEnd$FpR+<EsGnnSfrbJQ;NkxGl$R|hNE>><keuHuPte{TxCi7v#!@BTV*hv=F&f_B5%GIm7<M<La&DK8~8aUZ(y)Dz_gF}v*}Ta=3BMg)Y+>21ruq$*WzZKW~;U(uCn>QDOWI%Dax`acdo5zS?#tf#lJS9YAOGgDZ+Bn_5E7vhr0DV2Qj?>o}+G!$tw^Re$&h|DJWA4%&*$^`oc?`$(Yc(xXIW8nKMR-6nVzzYJuz=4E{)d*sM^8ZnFxx4V>o$C!%z?+T2uQ6g}wLEl%kM)Nr~2RjH6+T75xnU;p%%mp?Avy?OcK9hsAd0f9dQW_(MHnf@s+p|UPA4i}A8wpr4m&L|MXhLc;{^e16rjJwU%ys*9JMK*Uri!sMZxz9XqCyCEf+ld3mv8;eG9y_kr?T;&|pz}q03I85?zuJa5y%9JNEAlkEGLc~&T5VI1AGSjMt7g;ba&vVBoB@UllNHlrs{+fciq)-{p1X|c6Vs@c)bRtHNov~;7g6L0*S15O+LoVM+)aa+$086>Xj#;=Y%fl&*z-~^@vV@%)U(4V7GCTeq@MkTys+)S?%#bJe+nzFL=MV|cD(pTxO)DKwhR`Y?^=E+Xh0+2spas{b%MmDoF~9@hlVs@QA%l&hG9s3$7XTra6fR<Fis+ViouTeIs9GU*XXAgG7?NI__HTPD!({pl6_T9|DL&Rc@BD+@{|THOI_Qu;*iHNr;$xvKMJ_-(Kw`>*&gV-MTO<Ne(Ezyod6gfK#vnVkK-^1T+jETFUxx88)m)p39M(9NMcVUZc6<yq8<xeKL}GtFd8HlWobY?4=llEjvrD+EuKbEn&K5MLPwlnz3Y3969$fNe=*kMddw{6;^SBMEuT8Cj<!c3uo`@JC>VGTa5jD%a4U^D3vDa%0C>4xERra--GoPpPtz!1PT&O=I2zYJ!E%^*@L7sptp%dZMO|EryjtAQ^;c)~CroL}<Is*H*G*F=pezFK;R+TB9s>)Vh}y2tg#*q7Jc>nAKTbT$ibBxy#7-?C;v_u5W6%984$EZ#rP7p|^EEkp##FW(jfL+AH07S>fLaC#5EV`m`QSg2gvU|p!?2zk(#UdbngUxRA;L7VBO7$xJIPUS(vJH@nEI_CkFzJ#@x-ZR+n&z?F8naH0y~cFG>t-AL_W7X%VBN=!>4f&aVls83d=m#X53<7=vnCr28UjJl)IPetG;5i5d1KXFV?{GYznE^dUBi9lNBwm#02s#wzR#8>juXFJrg$gx7c<V^%)1Z7tz3XSr`k54<fQ6#%RL0{dEETlgn#0;oS9{kL-Oru>s>y?3M;@%sgtjA-E$BAm6r<Fm#hd*gOp**N#~T5yOW(05p*@2-Mao1A~C`Suq$ipe!KwtBTLGu>SyqKV@=TZga<XBd{i*={TJEUcey`urzjj$e$u`c5akJf<g4P6E6-J=PtOvIF6kFFzG4C>Ds{;1w7vdm(Q71$C(9|3#utBfa${J%;8>|K*UAp4mclj(ge6FSsn^IauPdXUgEkAAbLBpPGKC#dhHk-USyx^(-PprQIGbHf|E&MZDN{8e*KG15?YC4jx3DA#N~h>AV>6KKXekXSnfDc1oP)UwR`~0R_J_PM|Nygt>DjH#`*p$h+=Ai-q1LXz+0t^auBBz0>T9o^Su->mz&rC3TSMxAWSUF1<?l~=pVD<=#>BR{rGcCN-1qcQ&9tiEXwL@GW4v;Y}x9%VH7|Ra6K+ki^X6+F~9&PVhJG8AOW-*J9Z4&FqoU;r#5gCplkpa9RMe%aF>lP(6ao<`!<5Zm*es8iwPE`K_DCvQ|1clCg2f)#SwFTpeZ0sB%ZJtLkPxTDo!G}1<(ulZ99~UQ%=kC{BI&H`*JM)U1?bofF6U+x<NogZgUTkG+-->Q=Ym>8pVMHa5@TO-*Eu&#4+ay^<xnRjvET$p4?pXzf4;8wHf>LOu8t>vuR*)Uqq?pfodjB;)+<LA@#rkBu<#5z7y~`4J_ZHNo)bY2|@}gEFjYePUQ@??Z&}3kd*xgID5WJ@FH#_JhD;|z?aYi$BV4Aqd2i#OT=!Ff@;Pgz^0U2)DB|{{B>Z1qwoN)pJ1>bJD<bd+X9f8S%0j*CPz<BqT^UY!61T#J`<4Pc#uJ}*s?%(Q&+fd?1d2*5ZR&>_kqD3_`aPwvCjZ(TW&;8nZ4uMz7?H|1n-Mgv5N$^&~CG;OS-zRGgeJ#Rf)Qq+(xErTE7d+b(Rl78>}!$k~j%`F98m607|xq8HemHjD(dsEW-Ugp9j7pC>V1>6YzL|!dZH9!_N+4*Y=*)fvc=uZW0jx&9t-5saj`ylss(?dZ%jTQ{>ofR88)Ra>dQ{K|y!b@`?73$0T!8BtT_R(VOG)Nkc0D&M{8_yadd{EQ`lrjKT|CC-UtmO@Y7M`Al&IkedKm$Ax^wb;0sE4^x}FJOz&l8KCdjE|0*u#*j17$hIO-S<XG-hCE=-DRFsVG?C*3Pm8Ohccv2ArV=IiD*%hNl-`RaEpo8te=KQ!HGM7l_U`%X>ABpQ-pOKtYPx0l8FD?<_^41Z0bSGr+{00sdKTo>erWr#?XsBDIPybKXo&unLnDXTQ6g|L%%PzR>|-8S67DS~G=_+les-GmOZI;9UlZ3k!h;>LsLXo30l&#i`rD=iWPT;gT+}k#R^Yyxe(tKKPt2+lpuQac_RnwJI0Sd(JT2|V!Qk~z|Mlv}mkZFu#oL#^yn6TQr@xW;C)n)Z%};M$_PT;yLF`FlI}|>JEQmT_d=QnAz=>Vvy8)neCyo+hFc?5?L>8D+Ex?metW6oBfM=wt%MqD;FKa<&a=J|e7@}7iB0G#^lPo2}c9l(gGpiOffdILw#c<T#E=pN4(Y9LCvaY&!X?{P%izLg*gJFz`VaoqZ<B>oRCe6J*1vS8Eww(seDnWjstBsJg9>di5ImZ=~rY3|mY7NGQAAcwxjdnf8ufG~N3f{n38MF51ml>lJ^`^`vpNG$O*P;c4ok2iMWb1l}#X2)xqa!}9WwFwX@Ql?LAha=7!dyzx^)HcslfM<YXk;mAfhduhiB(q%oR%bR%C&w6tZpi+6yQ_yt;_rE!m~Ur8X{N%)RC_&&zj3}BU9FP4fUXADJYlTZ76b9aB#_UV^gP-$QW(A_f)B&XI_bVXf%s9Mr1y3S<q{Y1v=Xf(k^9VoG(mtfmcp+xWSU8MPBtgHCNk?9WCof63NKx;v*Bl<Ka*5bdk|kK{`FxfF!z#fI`x8Z!pE8k%Qt0xK?SO%8Mv<fS({>sTA_+GxeeTU2_{rgGlg0-v&0!2v*=7xOv0<qaD{}aRs2anzbcY7pQ5MW7+2NP#0Jp7T>Ai=0qbeWNERfXC#G%;`P{4H{_D<7mTtcOfe~n72N!Rt|}n6<CqM?cq^(6=qznnHr51)MgU_eVO8ZN3-e?ss+O`=FwJthk``(XB*phXz9U&hasfoZjAXqcH_3m$fBy!rQ-*>SoD?}wLSM83Jil5NSbK;?UupIcOZ6&KQhSxc!~l#SvFs`@;0AdM7(|Ta=xFMh9v`WfV!eWdgA=S}1XCde^j|~##3DjT@C7nORiIU}bda`%Xsu<-sogD+CQ5lKl~*#MF&I)68%Sl8tZ|F`PvpagF5Bnt%Z>Q(fyj&pR!ve>1&2j(Ch%WPJGqA?!&NMLgHa_pVA(aHX$=d82oI3<8l^7Ea<j&wA|;Db$XZazM#+dATmrPi+z4Bljc~P^m)LaPs}?ljhx{^xFRjZ~O-;mqjn~$f(AHo!{DkK=UX%i>H(_ZNx!Hg)F-j$Zh?@ietrW}FtN;A=$I93pxB}k6uz#3q_+!ZRBAH>w>bp(nx~XcZ@1Z)$`OEm%Y8b{q+X+nz{@M6rHnWY#C!?Ox6g+l{ZLzvC12LdpzthV=ZN^c-fUxtJK-jvfmpF;xu)$Ab?O4Vn@J3?-4wL(E5&j<&*BN!@0bG%Eq&Wcxq+4j%Q3?}PN-E#5OpOa`64Oa~`S9?dMtyvIH1tGveeHh4eU(GQjH~$i*jmI^JlX@1KNA~uVu*IJH!ped&%`>(%eR`6)DO@NSzC<IsBd3LDlhU}T*;uWXMzqJF%$-T{Jy=4N{!jd%{9-;q54rtE5t~jK?o|Y<<F6hFM6Ok=!e4F@c+1javLd0`OK7j8kCA3kILO)tdYE<fY2DML4H#hL;Bi!vjz{3PI!zF;j$;+gKTW6Xjsq$U9<`rbq+;N<=%<5cpblBb-Jk8RLkjLj5|TEJNIFI!x2sizVv1UsfAF1we1Orr!z@RPliUBK)|OJ(98L1XNqVSY`MwNlaUO3*>FW~5>98jwL~>Tk31u`<uo5)8;N1;sIFd@xmhcIBhk@b3s{9*U1u5lWEvhEp~bT+j7D=g+){j0zdQWXeD4FTp<*+RESXo7Eup#AsL>vF8a9({y2xpD#>CDbmTgbqjk>qhK^>D@q^s_4bCEt1PGHYYHFtk)pMrLu+*Op$o5~$qT1_w-Pi_yMWwd~G*4pf;+*B?5cz)Rhu?JeU;B~f#T#s(|jRxBo1}uS$4VN?QI>9z$kP=3}-hm3r!uIo&ZHCfA_6@P4%B>X>Nwn9chZ=8{y7gFFC%TO|8SfK3tXiWd2mK>?-hqcI95=wQ0|S*4q$_02-BRRTL?%7AhJs3ORh-((y;9?$RmewEH=8`5>`X1d;iyR)Ws+Q(8Z2sN>pohV4X}J7$-`!CXp%h^t!c@hy`Jhy+qsh%DgY#Jt0i~Ur<XNK4~P_c>0x*9lR4tStna*Jt243;HeIbjO{_FtJd6#R<?V#D@1^v2Mrqn^@>P~!Yv0uMOWoz}n#qa+ZTXDMjJ%W!L)6?$s#>r{Ykb~D?Sx=bG1M^U6!VKTtwi0C@Xm@3h*DX&)gyz(!HOHe2DR7%d@zUxVyY}87lJoWX?3Cf0SwVZ4?V$yU;+P@pdO;E+NWstoZ!y@{wL3`06?LCe*gU4M4oK(4a*>x)29^EL*80;e=bdX@>1&_(vZ(|W%d1mE{}(zw_3vCn60OOVaPPlxP*12hD@a9531ke#MtYpABNiIK>C9DR(%d7`qQ-M`g0#yNa(Xe0e(sdq<@c_ynL=dB`kuxbzS7u5gdXlv_Xk9%}jOWU}F@yZa*iE6&xmse>#6(NZsyI*XvUoHZN4GiQ*?h$k9=XGTo4rI@!n3){^l`He~oK|86vq|4;G%fB*Vy1cUXac$Cob2_ayWbLtoXSkq8mG_T5HvmV;u_P`oWDOEpR={?_edU;ag->Llx6$9~EN<g_e8aEg$KCRj3@JV`yS1MLJLbr}DqxDbL5mLvL@YwosKekPmTJI$HbM%xR)aT<7#gVw-R_0oV=OnGFahfWfpoy_{1j>IJcPCPe^U{sh06r<<=y@NNu-ui|@$}>1+P|CQZopGrH2HM3sg}dz%>cZl)qT!}O?$*w#c*`EM`L4pH{vNj@S|*YM3x|v9df&YzFqMj=F$@T%}1Z6dnb8oyy}F_8azmq3JCiBb%4eSHACem+UZHRWSr@8PT*&$wWHmji~}Bk(oN*A%S{g9yfbh4D05Y3s}*jFlyXm&o8<3hkzc9JL(Rwo^qQ@dQrr}`_?&oAFGYEW?36Bt+!dP@C)Yx(<wnlEeD=4lNKLbqc_;wyVv1wv?HI;5fZ}els)BL*+b~c2p1(8>r^yMXa%7;U<cY@Kf&M#c+e7(sI%CpLdCpC6dI<ZRB_R%McKY|$+T4K!{?a^w-&qA2K6Fy^hTXIAqte=u@pSTa0@IIoAK>mh>29K5Z(*}^w?L`CfGxr%m`t$UWoH%czSA6-4tK}B=hhW{nOW!!KV||>l-szT9UQWjpY!V+lwhL$je?(G&G(Qqq94BpuX!u?@f$cT(=vS~u}|+U$QJJx_+EsPqN$J?9I4Jp(KfWHwQp3Qa+_?Q*r%vCFs4a2z9TmIb9<<>M^z@R)e5W?DCR@w*)!d7Od{>#Px{J^(WGGD2uS2{!h;_1v6)CUBM%^)NBG#v^ilc55s?r7Z?EffHF@3*xn&9CE`H0_YFJvbC3w~SvosFO&T^T!96e~e!Q+7vCeyrlKpux3JXWJ07t#WnlQ7%S!TZ@WciW^Ts2f?<!m=!V_63$+U*H@r+Y_*tYB?a0tIdj*`fy0161V}-u0p?2-QS>;$9lg4l=sEk7uqeE%~4Bdz0E##=hA4m&AHpvWl`Wtx;=H>oJI-ln>WBCLonAIP#e1H`-x3mElxZN-*N{iU0e=Yc&1+;Vg4gq|A5<DTH?|}sKc=S?jmM0>bi+Pk?yd!(S&G7HXM=fdr})b6%>r;YHT!0QWpjE&hMLLc3h(j5G_l#*XyE|J3Cu2S_zTwHEUu#-*TcsmvWB>!6nXrclWKW^w_Blk&&a>2{TL3aj+g^YV4r48|e+z_9F-e&*4@JFxZlO|2?sTUfb&|m&=1NVZK|u`r-BaA$H$$JhB}MACJSV@wUU;mw)a2`pc`gkF&w=#${<M81$d`IvH@BXfjRRwlpkoHa$)}?qjgM`S&{DB2Zwna}mS_-$jEjpV2z2Rb_u7{c&*&lXBh{T)$0?4>|2vCYiQMa0nHuE?e`(4dbT$qArGL>Z5bl)-YUDjIP0F_1x|0<sr&zoJKEs-ea|Dv_B<nUr+I9YjWSsx8(QBNc5ys)9*XJesFHKuw@4T7~D6bt5tCa>)Xg|908X;T0D^5wB~NAf`Q>s?Jg>L^{Z>TjO(9(Ha8{>cb;}<gX4At!1cO4YUbNO`A`)lj56FGZnR@gqdmXXtO#Y>u6cIQ;y^PoIvvnPt;A<Wuia!c%QxbH8GYgrZhKM($`7uzMkI@{rYLE74<^~XN(tyy_<eW)g^M)%I5$ik04D~fc;&n|oxyR>w6{0hM$5wv64Q1r_cyw8Zud0+X|6Z*;ayc7N@Zu1jqH%+&|V%K?RPqoXsb6*0pwoaW4|K#=deSinF86}8acci+4$sUMY~bav^}~}(aF?WA-2ZXKTXCXrPTq><9<<P%EU?UytkB&a08la=)P^)YuF;A#wm0sq|>gpjYi1$92jWMLM1B07QgM|U-Xsdz}{z{SU9#W0r;L-3f3=lY$0)*L9{MrE#E<>UI|zIshe|FWDP1n@YP`Q<#aI5TZjXYUNt3QOR?P?{qUhV4fyaurd6^6N`odvP8LCQ-VP>D3?Wa`03Uo~E+!IFq!}aX5EVIW0&z(JC}W8f_!^i<CSQQpRMF}o?v@!-$ui`Pz)igYQwn@0MdwSvYkcV)<bzmY1&#q2QGjcy3+VKs011#^UiT}lG)|YeUj}l}#5F7j>!~2r_c6N^KeU$ExbuE9iQ3i8CNN99F5AiL?V%xpXhf~*s@&$rOX_D+2rMJHMt|Mcs5RwEO-*j~ROPSC!>7E(Q|k4AN<$RlkYw8GTolQf&RiMGZYeIFQ9ZULKE2!0R@&QpOWCdj^#mGr7Jh*(E(Zq?yOf)Hau@4Yr;6rO(zw_@LU7rs5^0oTu4I1z`{CSFePXT&sb>Z*cZy^h%(a&|;GK)@^8=SV;qP#v>n`aFS><%OA1Jm<n6`5mA1b<W{9vvq>1vAu#1LCGjSyxXXwrDGdGlqzNw1Ky`(u107)yyQbaB}5a6X3h=*=VejVP2|wN0jd`CM3+UA7PQ$5mRZ^a+*m*)t?C#(^}T$2np3SUW+JRtqe7`#2mu4!Wf^TQZ>9L#<5Pe!9q@@v&I8)~}&aui|PRA(PdO>IXDXUmmsp%(GPOV#9nJg5%C1?QG{Ij>D$W2wKUhLKFG<xOXzI=dFjHG=u4<9^3i%0%+4n$4}W+uK8H-Y<tWO*XcLM{5t_|TYV?1o|UI36~)tlma#2+R1A3W=I8(WpZ}Gw9F@^P?Y;aicQ21BHF53NXPv!@Rw|vcj(%;~N#2f5m3{1}uIBsp_`iK@r5R(#12z7ytJIgHer}-(U!_%LTbW)oN_c5zfUzpgi^swL15ir?1QY-O00;m803iTEBmPWR1polB4gdfc0001VVRK({b#8QNZDlWEbaZ8IbZKvHE^v9pS4(f)HW0q+S4`E5)|D08Z7()dz<mS=>I4Do<Prn~SK_YNl}MSS>}-Sl_YQ}8de=VWQXOn*hQoP%^Vm3!-ze+2D#U563tq$RpYN}Mm%Metz(yNzm4H?%B}%w6Tv?<w!dUnv`}tS6sd>Y*D7y29jQ&RyPHL5c(T<xvh-NLyQmQR*RYEPcyx7CvAHj8PEx?L~D<=yJ+z2qjafw>8=$(e&Z{ECrb$k8x)wb4aw2C_?l)oBegQ5+7@bGD&(WTr-6ofx`QK6aloj~2SP+M2QdDK~f8tNLv7ipb0W+(Rc3Y1o-_P+DFi>gyQ+neA$mIhC^wJfB=V3g9y2LTPYyDW<1IEpq#H^A7YbDa^4d8@R+qe|&un2n;I=Ju_?uc#CoV76>I!w5cdQzjAMV@175>8+AwDHOwOIAH^6bennI`8%ydV6@swqRZ0G$=o`I-mwQ>cVfoSoohSC#`|pkn%A~xVt`oTLpLX$ob1rZEgG`AB*Ng1Hbr$n$dt-9oucgfbXj;!XDCi?2Ad4CC2E`11RVOnj4=8fqu93fo&^Xo^pqit5K^KQ9KCypg7(d!Q-E;w@yrUXHgcP_m{5Zm^%`6YKjd&e9vK+HIsf3{&Zxj;j-y(>6=s>O;W?74;aTQv$(yI`31^YHCGWt`C3{TYk~}4FpseU^&0SHYaL=$nQecoToyly)TR|s79px~&PO-UXzS5J0)M`AEJxKAGER~R$>KaxFj*f?K|ISL;%tQ$&7$cFKr*IJ%;V(w!(2(8t8!|@v`%NP8L!%!4SEC3<cq`g<=1XJ=E8i`BPXMey@r3v3>~l!{r_+nibXTFh3DVz1hV%uA#h3(;BAHysN3A~vpH>rm(mr*^YC4%tq@%WUQ6#IO)(;)QPD{>A1spVmepGOmBnKXnu2l*c&irBu=#a|iMbA-`-j>4HU@X?8M*%qzj(B^=q71S2hR%RI_Oc*$1OZ#4yVfp|Hc5@~e^Ks7xT`Qww!)@bbj}A&D88<7L~2#<VZF!Jf`blcV0{|KWeS~Yap2(*sW!q0Tnw;d6g#1CFcT_xXR*WQK(k^<&lV@Ab0(o_1~IkI;ifnZXIE3EhJ+GQ5vrFjb<jyA&9eY8_zfrfAb>Mu)06SG^OtVxZ<!56?@DHI^$}QR+WGP=yJFUErvfLu`r+c_w$q(WR_Ppk&u}O)&+t%5Kii)h8>P`L8p_)IEIXUX(y|biJp4;%O<?pZygY3e3sW*gu&?@KJ!$A+@?Q}yo=->}`dm<fRS|O27ev4ASTPGD>tU48&`$?cU4nHcg_8++m$24)L|Bg++`4WbC7K8vYRx@BZ^F*dudjgI3a!Cij;pQT@*aiUR*see-2QO6rQMICRTzL0$c)Tyb3Gz^gQ&4!*ZUgN{73Z0ert=r%{B!tv8F%zy#8g3Hp|^0{(wx9rDX$UuW*^fb=q%6uWN3r>v5YcbkoXuTxGp#gY_s4uCwvM;4XG+-@WtRb}TwZZwa-Wid(9|KECnDj#r@x1u6O)mNPW3=6kdg-5qYvh+;3JgZe?e#LImCeO}ePp1tXV1*6E&3KpAGyjm703XcLAV+BUYGKO!8H-5LqwJoMxk70|K7fdO+!(IDGMuuk-e^9n-{gR3JnH(dP@bx7tzFiGJf<k5k_>SainX`i+9@#AYbR5||EqX?9?9TaEE=P_|7cb*LKSi>)0t=<|p-L9NN7Ctr)zbtGwSnz5RKwx`q3b0VpCCB^k&Yd-2Q<2Wo41^S^dC@50|XQR000O8001EXC;5&W3J?GQ!7cy*7ytkObYXK}a&>NWX>DaMVsCGBb98cHa4v9p&06bo+r|<9j=$oHegF`IuqZjMLl>P<Y^BlEwN+d8Oh@Hlj=+&b1Of;Spu{lz-@AJc;EoT)op#zPjs)CeZ*T9n&x7l_A6ea0S(oOsf)U=$;4NjG6IQPCrYcvgY+sX-tyx1FRu>dH$(yR6Gmn$9YT2x+?#VrCN>)UUbGu|iYsX=1-oiNA=4(c7Htn)1NmUjbGOrrKS=-fwmKn)%4x~2`xhx7;hBMlvODC-|M%W`uyA}zRgpw64>4IgN)I~HtGdU#>dAo#RbzYV*>b3$hWJc3_R%Q_}mvchbRi1Ih@d(yxY12N>S9R62B(2JMk*C12Af)%1E@`>Q%LS0)jN~g|A#z;Tb)0!qtw@s0yS8gsk`P6QmLLS!bX9W4(QVD5rVVHM{U=_P`nlqgQcc@sk<T=l8+g@2__AyBqJQfTs&CnF#ZCkp%I6e!p_gu}Rh}leVn|Y2R%M>jB3V+t41q8`|Mq1v&kN=_H-EqV@x%M%%hktE*B{=Kn7An7X#CQ3K7TQr&!Sh+`MK+Sy!zYc>yKA&p>69B_;K5YmK=_E#TxETi2EYC_%=FsL)BZsS>(T{KA>#$mEEe|VDmUSe;K{d%{O^bR1c#4EP4^0^}~zy9$PP>@1k)}pFOs-qM9k{ucEW4u6T=+c8=p@Y)+D<EB&-u0hY3meDlry0~`$xCW1iS<Ynu-&xlL#e>eK6%1eJmYoE6b2v1H6f{@IMjxXcerelGa*#MfG(m0t2*C}$|9-W>pS$Z#~8$4(ZyS3j`RjYWCDnL;|V}*n0VVS2(-|g0}Ts@x~!{jvl0k!3TLYz17C6-RAuC2S)zZ<57#2t;)40jq5l%?EY1l;GfAM|u#IW;f>&COdsxSO0$NgR_q<S8TrTBZXxhM0y`CHp1lA?SWgaNp!P0V*RfA}uUTYgde!<t1%4FhmcAQPAef(XJK)Xr3Axu}6?T_k)Sa_c1ZykAQ{Mf<Lx+>b!=%xrzZ-2n_7Ug0=9hS^*irP8@Fs5$xQlMR<@SEBM#3anOQ4p!1Qz-_bGdXyu;h5xY6w)$i~G%@J5^Tyq38G-O5sSEr|FK^^Va!aLe|M4*QQkBJzL`y}oGS-!+c@D%V3bpI|k&OyMue;ZhnZ3gHYnuF8$*p+R*VpmO5HNN|~WREpVTkuicn)EpDb;`9(%D|~I5U?Y(*GZm%)63h9zXDeoV)IWxH9;K;g$qk+)}*XPPo%T}M1q5bnt)5)x@p}ZkQeZ0x6kz#T6CIcF>BRv&@M(`5&;j6lN76{wz2S7zR8B{MP@m4-)Ze`*w0<wF!b5{d5Ug!guXRuA6um>z~|lu2Id60h}Jc@b^i&H!(2hw66$t1Ne;kRQS=xs<s&&2f{M|4Y@D!Li;x3G**?-3(PFrPWX{1&S#w3tIv4noj_Esl08LX?um&7;0|UfqT1cB5AWZa_dxkMvj31(suGqwM>pnIIBILp+Ja!?$ow>VSR6u7>@N$@TpKDs=8Esi&#9h7=PAEAi-yDE1{(*v}3rSHe667I%UzJQDlBeKX5Ih)Bsfb1Xflw$M9-|d&DXdIGA|siKha&7ju6WC8e+60&u^G~pe%!mO8wL^GV%biK_e}NMzFW-nU^HuBF#MjtNZYbL^T@BNQE6!4$OVv*D1*3#`&&;6LF|uv6C90?p=Sunn{4mdW>Nul*%*OI^r1-}4?N)M_@2br&E@TnUZ`Gv{P5Q+czO3Wd3XK&$IFj@PgM7t&u=f2FV~;0zkhd?yuJEz{pRYE7wmRHXl|2o&SaRV_mJe}yh6hk$Noh)J_|y#*U=U#7JDoznsFcd!HTc%##1f0jHLx_z>w7tW`l6G4dWiHnYJA*VzK4JW7I1hY+tXu(@ZNmvF-?oFPlQ$5;!;?kuT*Z6b@u#C?33t025n{5EbQn{SG@JoWwpAK*D3XK8N4+Cauz{t*QdH1jtf&kvR39;hJDr0Qg<O;$fHaDUt2I8YMx4FV8#yGqCIc0NTP|IWqK;M4wLrt9wD9(RXyLa06<fb>50WA%Kt=a?nB*O_8yh@^#)bFSt7cP<X<OdEwzmi-e^5Bq3?<yztZn2mQ2$7UIzZ$Kv&}Mu0=!MsqOCIZ)n+oUxG}RpSldMy#EsT}GpvCu$-b0s!3m^UY@~VN`JkVrtg4L@WhaVpsOlB5#kdcG9>2usS|Bgk<(v?Ei2xehHS2!b&8o+L27UF*R$^g-iZ_jwKa<XIA>-@J0A4yf_atw$4))dq~s^vvyN6HE<4U*^XcCZ$0Ppyv$qX%h$4u=3SZMDzu0s6A3VpC?=Ag3IFF1{{yL|wBw+10!u*z62jo$*pG3N<o#biA@{TDk#WTs(Blv^+})i;A^bbXf74LCjq%@fYVQllr&V)6A^I+iULB4-g21ZgQMuy2;3Jg8!sy!|0<~wr|DVnr6B>L|rb~bUB%`o56M+|qzzySwK-;Wz1(=Rv)8;AKja4;6?c;QX#-hk;G#~kDCCtPLJMoo<;=gGrbd|u^TrXdv%~V*B&#UMndT|E8KQTnPhZ+fG-V^>(dP+4<S!{1|U~@uw^pit90~0M>4;+|u3%ei?<(VO&pSy1`_{SQ~7Gsp1X#_BE60AU4aU`whutPh6wo)`ZHAC4;Vu78axTDV9Z%OxBWF%y_84}aAmOaXLv?wS8c-<orv`9CJTUN%p)8VmCdxj?7W{@w%#@=tPZ?3F<)--1Sr`xw5KHtg_Qi3w!!qi@fGLkegeQ^7vUB`uJdoeS?@NkIt`(Bk`==b1Y1Vmg7oM^PV&vG~j@xoP=MN|Wl>fSh9YwAM-CNn`H_>yIqQl?b~po!hCosV9r3_?O`7?Z`WkhW}r$S&h3c;Ed;=>|&X@OX$Vn4@GOlM*a=n!8R#2@!TZhxtErMM3&-7TE^eYUX#YB}KWW`&16<*a`NP;S+~N=HLUtVfi4smTV7r-38HBbGiFRxs?Lg4eMGNmk~!QNvW!mj2DW$WE?W>5yqA>)c_ZOWmPlLaC*cOndRC!?~%*{p%juVr;D=UXr&T>hG=zE0s^MuI%8l2(F+zEk~R$T8$!hiU_A#AfXM;o2$>Z<U>qW?r!0Puyp6QsIb=`6SLDTzUeqwX12YQ}0MS;(T7)QvDfH0W6aHF&8@eCMyg!a*u7AutfbyYSBz*QdC16#(0f>E73Bss8i21-XbXKB@r}FkWB-R(>vyn8*qpe+@_BHw7ZZbZfDriXq3p56LvKIJUCFu%wDU*q%H4Jln#e9U2l!FZuGpL3*iaq26K*R3|iewur#Q^MuYt={h=9!kwD*xHs^|~hrI#wx5ZKyA63i!z#A1QUV&Bm@Q9!=a2%4l{bzpqS2QDDgB^}~r8r(psl@nQRqNR0TVVG2!0`$mN3H}Xw&4>hTfeE1|D0(+y<J|FDNvT;fnh<a({UIULR8Neu^UBbcisjY&`k*ech(!(R$@792)zM5}Vs81f7sQ=z|>yWr>I4D_TR<I8DYfl4qrjk1E=JWg!Ybj|rJ|DY%xvy7y=^C);=Ny)s?N|nwaDJm;e4<cnu<RTG*PibYAzd7cbo1F(ZehQ|J2z`+aV#^+Q-fjov?{Tr3P9Lr!8FLfQHeaDnwJ@S9Cb}`z}>QKYd(4YJgXi`^l=evRYa;)vv~fnES?`>b!w>h|Nb1@QS%=W)zmgx$LhDgCm(87US5+o@2+n=P71c5=|<iMgW!})<)pq}B<TA(Ai*Q@eU}&DZ{;noK@4?O-S`R;qE<ubtXeaX-!qz#Y7W*C()xxK6umh9=8*n_W5sua+DxosdU;=5wt;<E0mTfKqpeEgg2*|2Z*zEOvwx6J+`>F@19R$7PaUk9Q&uCfwUOL&esumJ2)+^sd!=@p#&j<rlvx^Bsi$4izw3(B-9y9JFYJjM&_U~j)m3+Va@wBxHL^SNn#fqqxN08=E5|(uWxJ=mD#&pseT22An^I!Xuj-z5uZj<|L?^;9Q1Wku*2AownV;Fd*^0dBYQ~zrxPKVY4gvIsT9%h&5vS30HGwm9tv`35ZWuj30z)aVv<cN$CE$TkfmdLw#q~rVK=;mb2%7(YZRKIqenL@%NS`p7Pi^ivE9t@h)E6OuhRthI7IiEhMegN$tW5BTk_R?RiiH=v*0mlv$gc8SM1P3B_4*+|OO)^LJY992szGD#$bR%?CBr-=fysWKr3o>}TcH6&Zzw1x=`B9`4i52=$jV*=N+5&=>ENe89P~uMMjLR`VD|`8g22kYb8t)fR2KFx=UKyr&Yk=I-R)nJx0kn<?i8&Km<%iW>l^eHEV37g3lW6>;3e($2sRBz;Gcy91(o9|rN{eKx9-Y&{1`z#BrTl{_o+hW1O0y9Go1KPHCZXXV{j=PE`DXDtu5>d2rVE720Oh_^j5UY=jcL*Pc{ZuJ>*qz`0KptkU1m&*ONiwG?OKlAP}*t?&xzI5wC7{eK?`%iQGrMh)Xp2Jy3wR^g?n5kSAYV2yuUCs$W=1-oLxqztECjfnlNmXtDDH3wwLM4hzU4=*ZWi46zVlN4dwlysZV{Z+AhIw`^tQ1S&CAa@N$$WK*OR0wr8t`36{DJG=w_U94Ex`zYr(#Ii3MebQvM!B6v)WN=B42U+EuLk@*zpF<EVg(a<k%(zn2b3;;tw8XJv{uN^==i5T+@da|an+CyAOo(X;*#Y@u+>_i>EW~@Fzee-jOJQs({MZ7haM&23kU~?4*;67DawEOhE>kVsAGC&yaAlGQ>)Ruv|Bx;GfwTo0yMwcpwm{_;_Sq<!fGyQ6>RY{y2no0#rs}%#pcBvuhKD=^FX9A#6BBb`+lJHEE2<P)e1IUB*(q85dy@V=`N+UkVbB0!Prn)f&B7qcOnjw^j-V0R367$g&oS(YtnW{@j#Z~eh0)ps94e!F<lBki%M;@p3J$)7$?s31W>Hi#-~A@~1;P!5%-voU>f{$E;28J5HBp~7?%Ofz%MYy!6U|7kJHOgWV8nge&P=34LW55;aMe~g)ZANvqH0s#9pg4_tHk%M4Qy3^>l$4)iw@1=jp*?+Ca*X!eovCDN|PioW=1s25~`;75RhTy@8Y+TRSsK?ee+`_KsKyTyP}On^OJ<|klc`{cq}t3q(>$=W=tq!I!!?@iTDv=guYlaq6c*e!Npe+gKay(8lJu&{^Tg(yJ3AQu5n{m7}lhPZK!$p{V0}Y<#h1}32?tRllT_FY9c%}oXh^joEfiKb9B@u8pUIvpl*N0C@CH(-iwJcn-h<UC_Y-DGe(ZUaFSqkHAzJADoGG&N#aga_<^u<{s~Y^0|XQR000O8001EXhN-_?y9WRO(H8&!6#xJLbYXK}a&>NWX>DaMV{dL|X=g5Qd7W5YliRiqefO_mbsuVXY_094NoJFsX`3dUKHO%fY5S7tXlRMD*_B9@q`cYW=D+uV)R%2-ynFq!D*yyR-~a>|MbYaIRUuR9=FAEsc_BaYO(F2%oK#!Oa&4H~36>lEQ7E>5#>9sTlBH0N7pyYc=~NeS6h)H>gi4kqdF^T=l7z{!(#A2aly=-nt?Xpd&@%3LT5xNHZApJyt%8b*yImnS?fPqcP9~G@2W#T)|7o!7>UUF%*~F9B&q4{~H+-d4F1MExhF{4)Bud;SPTvS+FPT)1U9r{krRO?PR<sC~U$VT=+%w~bsjH+KC*K4`|6{*uCBUOUo|);y^0F$RZliSp7MbW-l@Y15Fz=FCXTXW_YV|Z~c)F}y$M{Lu)X0rM5Tu1e_*TS0Fx6_GlsXfgY*Fw%Pxh<8H)@+y8o|Bm1pM+vDcnwH3oAsnAmL!G$Pu|pTbD>BU6M?#DDqkG9Q|a@h!P)qk-cOq28bZQY>5vq;=RRQ!cYBjZxNSPus?ZGi`T|zGmVavm9=#YXN4Kk$-Q_H9N*EUV$<b(^;aCtPVDO34Y*~E9eqg2n&la54!HD--jPD20W;#CS{h+H%Z6}?*&k5U?SJ)#7x<>c*&6uxu8CLsJ2l<Ih#|b>MGvF?2%h!wtnE<{Y#rs5_^>D?wwx>;^I3;O_En3)NBamB!3cmht5ZriI>wvp)zg^$z%rTV!k|#}fM*IGv5YgtOOXxA#~By(VD_et0{PLV|DkMMQH6mezy10%3md$^d2C&ZKIne8i?G{;snx<(I8H88$}7Q3dt&}$iq$(;XLrLo1zcHa9sHIUtwSmBj6$cp=xLPE7pMXxdEvNS?4K=ewYdq#SO=r~`cmlIKqp&J-8)VS{|8PuAPS?a0ps`J&siMDe?e**B0`?-_^5h4e>rEXIa>#@ZC8FTPf9U_hn~r$D6%Pv53dWC<UDoS++7_nYIc{~k|&<tm*Qo-eB!xxy3h%{kq>$6P^WQaaYgVOS?vWj+@-t3%8z3FcyzWodopLwgDKpR(@Eimrzl<bcr6l$fr;iUvRTEO1QZ`cTDxKK=e27>0I#?t)EMY4^j11Ma(&1`yQ6Dt{ba!MfbCe%yW7tQXJpS<2e&F-*&SAiy)btz@7son`sWggd$(x0e!5z&SHW+<A`Zc;F_HC=2+9QVi@V-{)p8m9kNu4+0D45QR_@TiRKqYF2q!M^k4nWs0N0HDhv{1&hVcpoWshujx0>;d)kW<D<x@)C%9bt`D7GOFB;P7rcQQTP^VxSH!iWjM30+Vv5}rF@5?jLMr{O?F@W7L%Y(RgHgfqIZ<J{hwdjxW+c+s)^71N*zmVuR&T*+M6Cb3W3kxtNqV^BrLMAM?Cbg|gB5ci3pGzzm9z-u+)6<%)OGRWu}?zi_T32eyi4;!MUhY2M(;XumB;R(~IB0jTA1j<9JYto)mc0aDU@kP^L!)P@=9TfFYLg$AxUT4xB6Fp2$h$(sjbxdtwGeY)QIi?R#a?JAhI3~6+AJ9Laxg~SliZ=@B)LtN;ddGYoYrDdA()ldhC#ZNn{`3#QBRaQj)aq$F7*aD2AfiTwl*C`y{;T#T>szM~OJa0xcyx#G!+(E#{oBR$+w0e?hOAe4^>wT7>lf_R+gER4E0r&FHRv-HRZ-x!;2Tk(5^<2>n;nQ7q0(K+&CS_vr#;ox>8bj=H4c-4KjG+<@aWUNk;b}(K{d?6!I~YQ60OIL$YORKDCv$X+%Ro^qMx&(;;>6JjU9%_VB{B$N{|VW@;xD<TnL5Cbkm+Xo6-3Ky`!DO61C$f0foSA%BbGdO8!#|-@Y7ysIy_v0gxrACevg3jE0?adk^(^EK@FXWJWhR_~?X=gL%0VTsIClZ8BzTfebrrD$ZusyROIW6tcc+@yam#L)BP*b<Ikj;J#a--Tixk&v4R~owuY{)@C{BYF{3FntM$nF<R097a-7gfquk1nLfKl@Ie#kFKW*VnXye#`$CfmC&D1(YxA~ll5NyC2VErr_3<O-?>ic6febhRqf|Q(`Gm8_jqBC&1>I#*9c0`?bM3p{iO0>4ch&Oo)?Y6dKT&10P`-eI0B+0tD-Jyyg}O0lwwhvj&el~Q@59!R=<9Pq9Xd5!ynTmhsWg<Z#W#t~%=1OWo%Wv(Le|K#4)!c3^XOHLH&e4>??+xX4qYB*_CfOx$gdA10N{a86O8+W1``2Qwo#eG;ov}Q_R!Sw#>^RQb>|f~I=e^4vE&+Q?f$<voH0eG?@YC#h}a#l>MwcQJ_0CQ_N-ZdI_9^{CAa9{Q+M1};>fj82Lms+y<k}Xy=Mg=b_ywhg37e8bGD}w4ix)u8l_1hFgsq~Awa<MVsuZ_sHdZ5fBJmja`fH`=LSZnz1aL@cs~lcSN?ven5N$TvlA9**dMa=c)#g4-}Dl_!Su9tBkufjVAN!AxRFHoXK)GoR`lc|jgC$N7c>gJVV5T`3id*V%=DA~v?tAu1CE%%^d>I%2N{D(7UD|}-lw<$EAD|n&*L#Iu~T{wpZYbi5j;yAKsuexW|MybP)h>@6aWAK2mk;8Apmx>18Yba000J1000vJ004Ahb6;|GZggpFWiMo5bYU)VdCfd)bK5qy-}Ng{cJ7qS%98CQZC9!8juX3E-)Rz0lXhlTtDz|pvRG53mJi#iR{#B;2S5TOWxL(IGdCY%2{<?a4i3&6avbN!EWDMGc=z$$M^Thjv=w2t%JQ-hK^lo{Q^r{uBtkxvGRF@KnTD%%kl%`Rkj6_{l*MQ;xXR@w$O8lxaUtGa{d|<A$#x_zQvn@i99Bt?4@H`Zjm$$yXvI2<WP<Q<T9iRr#z6^7DYL<b&`A;0lZY%AQFfmuSrCb3oXBD%-syz`WWw86$YloK7;#pWo2o1Z@*xHUxSe<%+(;2e66PwmICCz`AWmUq5aw9{xRONtvyyo%i*X@k1e<dlXE0di*;@GivMQ@w`aWRVz}$c|%}UyKF&L<}UyCed_%KVuD$ilI5f3c%xX-Ix2GLcPB^M7etja9s5u2b~CGkRoUBOq4HkS-iZZ~myqan`H?ND68*#wJ34#oRmgS~3KBc%K?2+Mk2cOc;3nQFes^DG}yQ&48>IP`JtorZ*4!XVAkI1Cbh6%;FOD^|hj>#u!0km117vlfMHro8%ib>aVX@yq3>%a0$1^&$FqK;&gCqoF8*rSyTpa#)i8Tf7+z2FQ;7+p|v>{?9-CD5k<$mF1?GjK{^xQLqkv1D?DuMq#!d(~;uGFpGjRfSE^Ge&aBPvkxCWem?sQ6Ms^OY{Y*O&e%9Y^Ro}jiPO6vIXyo4W(2G1LHYJ?pD#WE($m+c{-2M(@n63>8GI=|E|+l_!&Uk8QWS6)i9Et9AhN};GAzF;#Kq^}W+cwSvO=IA&VYi*ia}CjaLQqpN5GH3(7=n^QFqq&=dY0>%U6p<5PtpY^!VGaBl&9i`ZRdCjH2K)lCPJ+Y3K~jFTOwf`Nz-xyN^G7{OPm*>GJQi+MRReOvKCMp>W>9htpU1=^ZuUr*rsp^7?o`Xy)|aU%Y>N@e>X%t4$)Ai<XW_y4`J@Mtgr(Y?8R#6VZ@&90rBkOC502h3gP$9heKq>v%N;VqkzUAOTSz49kX4769oDh@0T(Xd^W8F34k$9TQlOW|BO(AKk>|s#;V9NSiDz@d_fhoMrKO9^7Q<7!i$k7IW^^Vx2KdjxjP7{pWA8jr4bFs(ou`s+pIbgPh6MxNsPQLJvO%5pa8fV7T8sFnA=F0!08{Fc<Ir1mtfn{zaGG6G#6o5)fOnC=SaRbPVyT&*u}GHP7w~Ai!)+-|k_nG>Xu8BUo$df|$-C;J&F-mCK`V9Zv*>Si#~F$vE`t(`vntfIa~+h+B}*0OF*SE<+E+iPwz4fAJFJcu50Bu(EjLwg5<40M+BPl2%8VZ@ZAN8Q2TbVT6LAa1pU*fyoC{8RC~9sU!(4+zd4KAPeF`+5*m9I+u8n<4F9^r;i_g1komTNU8l~$D7{+bx3pp%D$b(g=Rt*77H>A&cu7}-0!sp!+owR_^<%QmMU>D=`Ll^B2F;^p)S)1=jKjOT2-D>pJFAS#bAezhJ5X)iFk_GDNak?M2E5dI4hy>HIGqo4=<QkhgWuA-x{ThQyMvQ7{r-37+Fn&ipXIgbpur7%m>ElxE8TmXVoAW!aSu-SX@Cr=mNBI7q(dpESVQu!?bvzj}II&uo{7>wtsnE)I3q8F<2hBQh5W?!e9hcuW7KB4IQw>$EK&qkxMW=l4Ko};mXaOe?+^J)BQh2ziw_oE@`~o-KM3hJ7<%lS94E%F*RuLdv?Bbeop1X26$bWnW!7o_VpZa3-re?tSi8To;5SiYq_ugAx}(<wGCUnh{1mbjDs44=Nl_6Hm+B-cUnV~tpS4^R1mnQHVEMW0Q%eNyN{!Cg-hb&<gfD<&`QQPtFnPo-`ikCR)IzBS9wBP*jXmd{_(x+O?jfYT1ga8WMEOl%_t_dfr*l1J`^uNtblDi5hO^85TrySVMuEs9J9uk;s>x61sXLu+O5G|1}H@&QKW%n0Yh8Bb@=JSU(sMp<QmS1O$ua06xcj=WUjOT!xGkJh>D`w)2-Z2lVH7w0^`Idt-E0xvu;P0?>a3+YouSX2}0x+2RPv9u6^vyu>ghe^x@AT2qYYW;NJr<&tTu?9P3+Y1N044Af7tyMHMHJPo_@o)-h2G8*ifB3sxJAmQfs;XIW_}A8LJ}JO#}#XfGe1etmL$dcr-Rbd##!2`okfNY`t@_?tYt12zVsJwe3<Fdwp1Q7@>XxP)C)Xb$pJ-@I1N3;BwE5&tIJ`=F`*MQAK+=mu_iKVWX3%OcEU5Yy!f9acP+5gdJ^RBKhrd=FmZ_qmk6fjAtKy^9*R_%*|2y&b<BpO167hGtmC7>tu4`KMqO&>lolEkL@6Y$?vcyGDH1*TzX)U$fRgBMWdqCrts(h~@ek5W?rS%E8N2G-^J;aeU`-V4eqC;cCm9eQa{Ai#0eRp!ZN|)8;o-zR3#d@s&IiA*lKr5bnh4Cc_Mkvju)Z+T*2$BV2Dv_P&2*y%3-)txg)`2MP-0Ch+lB8B8Stk;;1&pTG{%k}1a~lC3B+5rQ>c)_^GTczK_5Cb9sw$&+Bi(<&zm?3gOOO~EA{!Z-jBVG_h^>;d|x3IzrLG=Bptka>iky%Fd&AUHZ1t~UbrK1=ST%i4=~J0;$5dp$AG*Ctou!iWcD0vk4alnu260+%kgpgKlBj(j+B-^g>@BYA^#+sLzWyOHi4+RYwLkTU56c`eW=z7r?Phe^ox;B}AaYoWGcD0x=!jU5hl4J}k{mzc5uT^}#E0(c}Y<Gb3Tfr~(lFX)15`$n1AC*r%Qd4KUM;he#?=8rAlj4lihs?BS_nMz=?t$CB|84c@?Zu*cl7<xp|OHEzxrf2$QDs^{wEc4kvt2hTW;59-F*>0*87#?VIMZ9`_c!YFvjp%M4pJ9-GCyv>owro^A+oP+$s3)&!M^Zx{S#71NEi{*#^d{T%HGVebjF^yX2!k2+f+0yhCmOSdLIHavu+wFHW1)XS`+`1Qfk@RR*^!E49^IW-Kz(Z2Ay&g?{A;uu90|R~Hv@)_8+w_GjIl;AiD`l;+5|KP=o9m`(#I9)#x`6_8EVK@voWevg3woj5b30Z-y0SFr3eiXSwtBz8`uG$j$<(3IHanP0R42l*CZ~5NeuiTACou)qz#h`P`ybEXJGm*dXse<$v_8JKs1-p8@(bD&=gb@E15(`SyiePFyGX{-D%s#V~%1Ti{M&@WtJltiS@B?YQx8|K}PFa`0Qdp8pPK0b6&|It0_ObrLWp6zk#-V>!a&Tl3&j;Mvf$AG$Kdw(jGA)l!Ym6ir|i<g%KOEhfoBmln-UsNB2zn5~9%!NE=7f$1z+wpcV$`8XzWBr4E=Nx#?+luVb288HPHXZ_S`Bh&Q6@(e;jP+giaZCM()-%Z#)a?qQ~>?aW8OJL{qypXxavM8Iyf;ZYmfg2){gEY(O$m|~8CB}7j`qi*~BxgjD&Azrn(2de~b8wJP5>$t!eGC0WGYdXiu8LBOIUo!_;CqpI%gVVi*E$cXuDVLQD2EUF!kuU41*lKcU5X$R0%WY#oEy7UePDQQXB5NTi+7PdDF%gQ|!I-u%u$wt$yOO-5Y2Oli&_U~SklBleqXrMzt(WM5W>q%GnKQ>T$8I$tbyn+!-x-pWTuaB97TN)esue7{OpA)L2yi<=7REO*_(VF2#&)>Gh~;lGRTqU7nA!)CgTwswW@Zy(FLRca*>cI}bUd&xRL{E2xLh$~F!azAIc5siL?iu^nodDBmrD0fZ2-OUnN#~kP8a{gfFsR7Fh;C`%*DQ=zB)cSKVoCh@n#c4mCp?au@$4N#fw(R6#R>du2|t93?@kErq#B+n1b(k=CDG8y$sC2E{5<e5wUymf)SY!uVLH7(AoqVlu)pcEc?TixcnNfIgTD#xOB9zA$65B0@zFnlF%E{J{pmvsA%v<D_BPocEaKmZ%8Yar7bPIXoR^P>(X{qNBSNJ`xQET=36l$rkK{RyGAQcYL!T2Hr3!nDXp6RMSTFpzMV6*6}2UcpDDIg7_J{`qNDo=J81<V4#p(Z%*~MiyCDL@f+K}XK@WCu%`UY7quVx{I(CjZx&fQCctUv%hftcD+3PSm@z9g%zd3hpDl&B`2fb}*tP~x==teW51%K)YRV#n;j1X8P!A66vVuAs<jtt3114!>yGQ6ddgj_Bqn4h7HjL3QMMl6Fkp}GPL)M#s4$1;E!OE8^BhvOG^$U=!EBQ2?E;GsY>9A@1<6fUirQkEVIIgI}{0|z>PVMtbqO3uh%;yGe+EasHfq%ZFQ9V$oUbYz!e8x3t5zO>gm&4C77uz)RBr844b)2^lj!sIduFmmCS858O`S&mCO7g~}m!eE0=6ekH3R~P|!%Q&xh3Hv8X*|T*LshuS1K@*i);K)kYR>8p})JMn9vr8RidyYA}!Nm94Gb^U<V5EHN)#Z=641~xt=zviqlKME;$d+`VBCM0%pg0F#`R=;je+Fecch-}wQT%xj3`4IvNAnz5-Ub~Ccn&UcxK0Eyzw3mL*NexS97NyGp;eQIi{iCR*&p9E>u4vRi5q8{KS`TYq(m!lo3)0IsprN-QMr8g<0mdZGU8D~d&ypJmY=1E&WZ1u<ege3EQ3Ilnu77(zP9gq@Z;9EO#<g!U1FBIR7b#0E6QFVrMp98g%NvurkMw~om*y;ljHfmv7jsyU`J}LP>V0H*rl;UTG0!}JG4zCZq(Y$XY<xK{4|%%No_7f5@YMsRa+1;A(p_980YOsBVi=E)<ai$7F8hzrPmuE6PnbR%Ns{=aFb?$GBl(9BC06U_wMRvv1#f8Yc*6eA+ZxDFMAn>`*6>uHCKmDdEK5)VTLl^pn5a*I6_h2lPmS$IwQuXOp`$^zZ!5IfuZZEF!qFHU{iN-Ru%2Y48uP+CZn;auuQtCz%fDbT+;_vVM8f05tvCSi)D=+9n8}c|23%~MVbFlwFl_ZDk*y+!<1DRbj`Oq`G9CUHXEupRF53|7CT)kGPFXh&VFDfv~dMZQ3=`_%5FMbc|tZ$Q)WW;q++tUh1HyOZuxLckF`s)>_g9LAu;#bbj+IUueF*%<H4C8hjH#hwN|gkEg>!)A{mM0CN;kulHUeil%y<cRD~|5jB!Wq|Bn=~GW`uSuoyzq&6aup7g=K?z|Y3q!K8WL=(;+VnX=_8SP=JFek(zjsU?5Q-9VcgIQv3VW|L!$#-m=wl`6K*bls|z!A;EAWWi}_EbXqjQH7;cn)>Y)<j*p|`~UJ-2SYe+g(JWI^DzY#brVdxlx4U=G-(vy#ZiR`-8UHEB77&`{`IXw6XfBF(usxgZ!{^u#Sx!qHRAfZjvifK*A5Y^7OtUkMX-pS<cv52#8px1Vp}W4&WDG2c0N7fE<r|OZAl@DDqMk3RB3sgG571p<n^@&Fq;j<lYpvJS(!_UZ37J`pDs#_f`eY7vhaT+&rE6ZXJ#lY>B2kIS?5NyYcfx}rP{Wh>8r3Fwu%?opVx>uMDS>lwT%ET-n75$+6G)lucv*5(l(U798>*aIOwJoR3SPF>n1m3tog_^BP>^xTwFS}wZvLZw2b9?qLFYTDFpQqn7UhJz3;{pTQM|sLj6XQ>TJiuxH|0VBXfX~eSj7%n<d_48QO8*2jKbJ2|Z239{cG)YRee&EE~$0-$-3C(UghA1f<HEB<VVMR@#m4W`l@Lb5N9fj~Xa~&^Eltx9R%6wPi!$fH-CL=vX#3<0&^Dx74PU{XRmU@lbHUkh!Utc6rI3%CrMIYi_7Nd~PwEBK1Vk`9KAmc5!yy>Hzl4pP<&hr`m1ULqZHbO}h7yXKhNFd!}p4#Cm2U)5e7yy%+p<+-S=ADpW^mo2XLU-UyRXsL^FH=RHSfOyg1+96RMA9hl17_hnR-Bi2-P{L$kYX#W6R(K_GTh*H&cq`T~tczfU@?w8+oLJGGt`u93xgwqrxeoqw#WqK`o{gJfz_bPqIzO;FaN^Je^q__^y15FOsbJBfQDPIob6f>u5!|2>@4~$Nl&W&qGf~Zkkb@t7S`X~#3YSaXFyA9)h`glQ$VYW|u3v*24eQJKurLT6{kK)$qt5t|)I-i=vGb2Y~r+YPqN2;!{BA)7QbdHcsX$s2hbV5l*DAc2Xo~)!Xnf;(ghtiuce($Pvj;<woc*2<(Z=ZAgy}qI?AH}>!RJ7K1QQD<zb-iC35Zhj!J|PO7bNvidkJ6QE3t8SvxKbyi%3q!W37dcg_^jk<V4Sr#VTN+OFG|v51`~0Nvitc^OE;5j#_V5;??HgpnAq_Fhes8`mv6+4Or<J@p;CD5<A8OH#(p#!SmEvN`%!ltJZL&jyC)D7np~4^vKZFzFciN1`b&Y2zyWfJazn)%b#bWIeHvDaQ?{?^AY8&P{!6?*rtE{zQ{p#_VlrCqKs;OHf@|}OIsmRE(%O!2?Z-O<l^Mw9QwsxS(H}_pwYJ6w+IolLH$g_N%y{&XIY-+u(3+){1XGl<5FfCWJ17i|XODkO-OUCEOf#!}{zQh=-04gqI8bF}#_rHcuy<hwH@QjG&ZsJ?erD)TY$#RAm8%roTWrC<)*4jVo+eX(JJ1}+ix>7nhT_GG-hJ)|Pny!!dDv#^Zg9{xu9wrbbstFy)T6wfVP=}(5a7Pg|810ojy|gS=lb^4WNEsTF6H!qrL5Y|e)*-8Q8zX*q~v8v^;!Z&-_{O2ftwTDQst>Kn2^`EB3PCnC4&uG2tgw5SFlY3A-y(Y=#8Xb7uW<jna$afil~wx2X;|$D*e*fFvdoPQIqJj1y-2#)gd0V5IXr<*QyR8x6)-wVT)1Y*1ZOL9iwU!p`30etjiv>YM&X2lexX%C*N96IyD-w@ug!w^L7p;9t;tw&dW5U9WFAWOW6{a_^&UI#YQMe2rG@%B(S_z)|_~8rW2Q#3N!=R1Dn;`A$>Kx1=V#42zb|2u>1}e$v~@i;MWI8E1=Kn4c!%d#%MN{9Ik_W%Tr{fTx8iT2K&JRQ1l@CAk0!Qm}9IMQLb<<7vLI23g2Sejo=GaSE$i`RVc&~E3G&B30|-0Ydc<zU}?p!ni_qq-={i#9BUWpx8<H8l4^<5DIC)kSv~imT}0NVcU{KS0yGbL(j&i{b2pw$cZx@z6CYCi)*B((_rLe-eO_RBr3XslI%HKDNW0rm6?@ZR(r?ezSd|VUV^jY|jPg1LIu2t$NPjWZ+(PZ<3w3;Fm&ltS+&a4E61a33Kh(X>qnV5kS-uVuEU2hok)2@#hJNfpJ<DOOkC!B?%Gx65UsN&j80QtH4)8@;^H3X&cDM#yRmD-CDk<X)TuZ=R&{4b*Q9Y6-By4)Le(yFCA2LvKU~Ix|%JG`fxzX#mBv{Bqy^+(7f*RGz5}3aM(G+|ZmVzSt9)nc2M@X!C=&3z`>$d`U4)s^PNlZu@T+MWy+JO1i8j3cl6>=e>T649^i_kVUidfa#I8?Q^5oEoPN=-UE<5B*F{#Bz|SL^tkcD#9W%>$_C{O*A9oUHw@zVo!K55W9g0;Ea{mG?)YO`V<UjjFOJvd8O}Y_!BN9onH_HQ!Dyz$58r-d7LnHmTsN<Lc-tHM<Va8I7E#Qkh`zH*<=AYpZcxk*8t=oEsEcLgTYhRCt)H!zW2<z+-Pt4E7oYQRE(G#x|)AT``^ER|3x)^!@V3Q+~vg@58Nk<gN7_g04a7ID*2*2b~XYpR;k`30%*wC8$q5FpvCP0+%YgJBMT=t_$clQ{U!$WG?lEcnHNEm7?zUZHMPT9hO=Dd{0IM{4t*gK2oLBsy=_jU)%N2p}lWw1?KV}d%tHr!OJt$7Ct_X%c#DCl5=+5GriNReFHYG?UD&yq|o=14xoW9I9mjI7XK&58+9BNeb$arT~I*kw-K?Zlj_)RT^7c#R&32>na@u}Ia@l7Yq-mhDum5nLFpF1I__U4l<{Pz^p)@Z#pPd0@$^rnsARkm-5Xi8#<rMH5|5|vWZ0kjwG@sG6VZ&21{gL0hP!YPUs2On^DjET7le;fP%z;yxX|>Yn>?#F?uqAlJ-?uV519u43s6e~1QY-O00;m803iSq7M46%3;+O~B>(^y0001VVRK({b#8QNZDlWHWn*t-X>MmOaCx;_Ta(+i6@J&RK>3i82}SQZ(@aLOGEEcP@iehLwkMrdqv1d#v|^Sb7y{gj6X&=0TmTZ3wChbf*@s<<0M7mL9cZH{db^XID#fOgO|cW|MisrSbd$nkTUSM8g^_!BYA%FqN>MeXY8CuytY}p??^L0?QoMQjJe$p4YCNU7Y7fh@7TewcDY4bvzOSXIjM(&5ZGni@!tUVnUYC8XGVxQTu8jamL@SF+g?wgL@NT}BKkH5a3)Q7?L^Yz;7nS9`Kp(aOE!`+lR>pSKhNMT)M5jQ6YA-KTHhZ^I!l+smmJVD39pJv@)9Olfw*uJ-y#*%H@-ZNbEJdSkzzSBi*({2p*=*bCy~y)zZ~IQ=Ic#oqXTdy;wi4N!+04BwrIkf3jZwzG+DWsktBwEov(b%zwqQ7>Zf{$(xO@MttZTWcRVsdzEiMf7t?r7QlPGIDy{+otoVf*`cDkwFtJ!Q;s;yvm<wmJ8#$OBcMat^tRhf#tvO8VE;{r4WlmF6#ZRlAnMHD4s{xu$1%mn<j^0wBpgcZ~xEBk$GVp_qURYYXxOmdFw=2z!Yisy-Wk|N#MGfx8Uj`2DjsHw)FwWTSlYWYmoMvZGmwX&073$u))6gWhSDB<OVes>C7)ga>xO_gtMtunFmL}qsK@!!6Poi>?j3SFu=NwTuKP{zi|YVqY7h;FJ2Fu*5bR53mmPo9V`J_oaibH{Hs`+MN^Y<P}=^Ly1S-*vr8X7oyYr;ydFmV$<Dx|cV3qpoubQbYa$`<{P!MyposA+#k1-a>3aTw2WBqHgo{^y^C|^qY8zsj{R$v{>q#@*Bu)w60+Z<)bB~4NC$306}aIyH*q~S(dF~XZ$!#TpX0cPA@U@hLE(n(De{O-+%Mw$5$_3{O9R+5p6Ii9iKG{eBKml;B~-p<O@b7*&(3wyw%3$Ra4nKhp4Kz$zpI%wZ)|0W>cGrC*m9`ihyhtXYhfxknO}1u-bT>B1cw6iJxTMtEXM3yEuB$T*<ofRiv)=P$GdE5y+zCi1}b)s-Mhk^dho6`xwi&z5+LpHBkBl=v>+AN`>)oinljE=~^}7*XW0*&%gQh-<TqSCq8^cfXOQUMBj);?xA`zIn;k7htgT#44yP7^dJOO+zxzPLBGkO3C+8+#wE%#-CfJBd<*qev9R0*7F+?dci>muxnMMQA@t^FDE7QlH5YbAarrSeoR4pm>1$Wwf&{3c!K~;PP+G6Y@P$TWFn@Um1p3;=zJhXGLtGcSK?0PRE^?92sg+jA%Y3i)y1V5n#d6~kx9GgkJ=EXf?a;9)#ax-+R*eM4Q0t-0v87UQQgwmvq3ZT6_6K<7D$leKxeS6{_w+^<w!WqQ8id+c!ghUAP_Ki6Ws99M)G47iH4qiwKNH|B*ljY-R2L>-5u>^**xX1A-+2jrPC1zH2#jS__lQq&%g=(aB0|vcAsAkN{+Lx1tRg7V+qSo%jDeN}8^BTX%SZSd+BlMW-RX_oK)<xNqL6g~2!*r<%bKdhu75oM7*Lki5Ouh2qrfgGN#>13f4O^urwpPVw7`XGP$W2>u_tnu8@@}A>Ge*x|NMdW{ELi`hAXd+S6m%rBS2wf`dLwfzv2X|fOGIz2$oQTT@4Tlny5cR5f~@_tZPN)5~w;m#y)<CuzQI)jkSf^3eg)n%g4B5H6#uJiLnl`P8n`cWlnh%|BjzhG3S)8nr(8(Xj*Ii+pygJh5F(lAJmc`9-TUMuRcK?{d0Ai^tC}sYVMO_I;eCQPVwYW5~Fmf+n(G-yv25zo!!mi^_?u*Q!E@5<5#X+<;6~Gqhg=UE@c4Er96EoSCT+BG^;a~*<#!G$C6Aow{cnRm-BOYuIo5J0^HJ`VUR;hdaa>)o@i~+v#=6D<Jcmqcf-JS1=tj7j1<fkibhjb&_NV8iJ+w=p|FR}n;0OI-36c%FSf@_t}s<(W7gK_t;L<*M&XImW|`3WUo8R-bm|Uc;Dqj1)B<T(*(fM^V5hFKJS{>cWvEn0ii#r(Ia4?Vp%q^S)5Am#-U3r3kk<Q3)^Vq=Q*j=|%5r#0R%h$UX^>0M{$UG(q4!lgwNV&&V?L}$Tt6hK>JX^rdzup{gk4AH9<AIIw82@nDO2y(eQO<-4nPa9;`9j%ZoXU5>=o{ElMpwg#W)di1K?sjzIH&LS5G&2cM<`m11^Ya`0c0Z321@norm)%ozWb4jFzAaK4%ju!VJ#wG>(s1BozFDH-H7elL#K&&`%@yz|~=gf3OkpW<g{3cd#lR?dj*&4I+_b{&ffzu_O~2Szdu_RLnELYLTLWV5VrhN_V-zy$44#@xqFIZ!9I(zQ=b=3O=L!(B=_j0#y5}0ad~8nRwLzEC7%z>pA1jRP+tZM<uAo5;KU9MWI>?KzM>4Adv_4N6gN4Ld@>$|KTu=!(m}0K(cStk)8dd0Nh3+W>C!b*d8DI*@!|CKSZzIym%gYc{oz<5#Z@f%l9lRa*IhF6Sn-Cnj{RUHK1los1eB%SuYsQ7KCRYEMwxAxO@T!!}~yLnB80D^Boidqto4jNl+I)04iW>xI!6+1LHDQ2U;7n%jlztljJ@<-|{eqq3JG=47i<xm`uNuSDXe=d&U&-N|x%XD%4aPen-n=DS|CyG31)?+%Lv_V#$9Jz!pu9$RF=_hHI9DRK!QrkgPKsD_ga7qGSZd5z-WWDWk;!PkfmnfVX)#q+sS4Ck87P+&WF)TiIb)Ea{#r1A}aNm;u<39EbmMtTzFKqMTm(%9{CNFe&$eK|2Mq+8CpPf+~TYk@(75;K~qUqtm&$o+Ss}YT$3e3ls#si{tgxp@d<%B7Q3rXwAOCNAwNJxIWT$V0M>j4@S+?CKhw=f-u0?A!LtjA6hb|>=<UmgG-cToJ|H)9NYqpt2E**nE}ie7r{yxy5Vgd0uHka1beSKZM-}oLA<g`r-YJV<~JXwKq8%@;D&m1Wi(YiT8mG)bogtsfsLhJJ%HfJ4cY_w(v8A!6?P)`9S_*@K#Cp#!3m5pFWjk{x9J7aFF}>?Hx?K`_hc~fv8qKz0EI48d`I|9hMA8fJ(d33oC@nw{Cad7g;}%UWN{<*3csef@oo|e5%Y`BAp8B>^%0#d;X>G#4hgB6eotBA%E#o)hw*ZHQUng<W$^$)@11l*%BS?!V^QLex)PFnI37_aw>pdjW5)+-mU~)pw_YFL@+3hZ(LLz$-ofi={ycG>Bkvl(kcxfQrl$*nJ2Vm*>JMNsikUSL#7)B<pT?gpMf|&GkWi?{VNupCUJFkgFakX)pKA;)$j22=q>J@rT%4T5fYgA8!1HxHU7{O~prYh;bvbH&s$saqqx7QgJ4x@wA!EPdx#RUP<<n<}7t@2{za46Rr-6|OHi1JPi(!?l<kydg@(g)sGDzzBHV<kxN%e^s{1s;UbThdiS<Fr*A@#%Q_C_~o_E<t$#qS3AyA%L(nbp+DlVcmXI?fdVLoC9oOpsr<<3Kvd{~Ie13O$O8Bed`(4y_61I$hunssTuG`3EcBA=H@;FvrLy8A9FFkmR27bc|Du<|9uFOBACNU#$4>`d&n$YT25*z#ar)*iV9sc0kG~)eyQ_(e`mNxR+T1FTc?hh*R6#<w7{52Db}vezoA!>+yboU-*JNij<B_li)i^7hB_t0~{87d&%ZVPhn~d90#T$z;uCndt2U#&bbtwrD3V`5O?G{e4GEEI}9*Gp~{?;xXHO!O)oOU5sFO(Xlg5%I{2bsyh#wBrlSB$ALFy}%Y|=sjbF(7FZ3Z7D*U&Cs=)soAW_5)sOgTbnHn6jpr>t&t!$6Mi(cDyN{Qfx-4D{Pm->M*7$npCrpJd&A5%;gSm*RG@}~?W(;l<_QwI^#K<MtYC1UH@QH^GphYE#}kPz1TqbP6SmB|+W%jDRUzn*M$0RlAG4m=8+Aiyd5gIU|kVuxROVC^_bmh^le7@Pvase?|`-~V4xr{qKRV7$pq<s{&Iur}aIcK#D_<n_hDt?dLSPUp+%oC$sjlH$&A;j(wmS#v}x6vZ|C<4t!Dqz8lP(Y;K1he00FP<>B!6O!{n*h&5O0zNWk{{>J>0|XQR000O8001EXi-(N^AqfBgrW^nO82|tPbYXK}a&>NWX>DaMaAk61WpH$9Z*DGddCggEj~llU{=UD0*nH?J)osq_HXl?7h@I9!0K2)`X^TQQ5Y&p2nA=@$$lbFP!~ea{ko*2(*JwXffX^p#W;ipPd1i(ehT+d=$+S@?m5Y?UeEs7+Q~8t3OKq-9<OlYzpI-m;%^ST}seJe14J%Y3Q<Y2hUX^ENWM#!Zl`NB`IEYfjLGZUq+EVG9mGV-qS&~YTSH+qgWGNGf{f5LG<WW><Y26X+@kf!SdyzacD>IRoDzSmVZz^Z<Qlu(TWz2qTwM)fbrk3eEy@DZyaOiWw$@WjIJj*M~<&!iF)cE=OwfM)2-`OA$g;TCDIx8UVUYbPur9>oW35|?|3l6f7_>q@PA4wWJR@U)p=y8pMFbsp>2qBE~V^vm0a?Vs%Xj8(bT$h4m+M0%3lcd60$ghdor68zR@+vFfbIbCgW{ZU>)lnp6%Vp2v?vLN{BW@!Ig6}#Z?q|=Mx6;DDn}Fey3H!K#)Hpu~W5m@hs@#?$Po$mf?Db`4)h61^EH@XwVF#6zfm_4nGDl$}+{<z0wV;oY@Knw|v<vhJNxyJIcX*8H_gL8STF7Xxb-G#)X<M}$hIZjoT+XD1!+mL}Va_qtqI`Zg=J5tMO#a&^xjuU9zjIdsznt}<bq8H1cL5e9sa>-tkyf(j81Jri>rS@yBxBEkIU1<x8F$`?59`^z&Swh*-aQMOwI4>ijiTM^3IREZ!U|cO$xZ7A&Q-2T&Rt-vSs}_Ztp1wN;YlTO8z$93Y;Hr~hR%-)35dnp_^#)wIuif^$$7k(q^fAwTnd#&<o2jdCQX|}FF|#0G#p~JS-Y>Vv}thbMQ-0qqw>>G5VFaL^h%)A$RTL_r#9(fNSQgX<W^*D){2a0`XEzI0A)*aU9)#0yEl@OMw=nO#Le?k*ti8^-e6~I0dTqn!avuWLFbJl0|I#$#pKagSuUQ0N(ro^o-<u&31ua#uj((xXfq5kmkm9fG*wQh%YL~3I}@b?M|pu&a2a4#CdzsY{vP}IUI^o`&;#{Bzt5352RM<?ABb4mI2@d}YQMS6^O3RkU%IG3hBQ0fV$1mzdyCQ}opN1y(UoDp*4OMy57nrfx<rw4Q<=f3PTzQ_MpqTL^X`v1&*giB_U!IzL0WBIYt0f_(Znkq-;dweK@0qh@I+lc4DZe*7!n*D$Y7T01hGZc!*aTg?a!u%F^f97*u}v^*-LrzX^m#Ia&gKJgsfX1vkqLJefv3uhf32YE_LJ`Ia;TridIXeUUjNrUz^Yoax|*7q`mNAwVD(C6{Gg4P&v$Gj`NLkRozOIXJwHaDB$dS-&uJeW+E{fK<VMKDgdOYEj_94s-j4*lu#7F&It?>lgC0Uv?St5D`b#hXLUf_v!gP$yeT}KP^vXq7EjIeml+dM^5$OYOHA~`tLizykyEn36D?Rb`S*(|x35HMdy%xeeHgo(@Nhc=g}d+x6#;^y&cbc)=nhVsTNo%g&}Hd<{n>rU9a$=}Nrccnw9r94>#O_ccWVHv!wG6e*;%AVLLao5gn3=}1Bk3EoXW&Wx(e5!xdy!fU~T`ZbL_e5J6v2UcU?Ph+QT5UA;P*k9l3L2uS*$)Z-00ht~z7RQk~9d#^0+0y7n!x$7m%Bx!c@5bVv0dZseaK9-L&!+i(8pVY;CJP0eLU1~P6yi4GnCjzEZ}40Gw)Ymh#SJHwhEEnEBhdVz*p)Ph8UG$gnV{~PVYuHWYeB~G~p9wy=FXg{rRH;wRm3gLbVffg=(#vJk_k}Ge9q6Mn1_%*F+w=yEd5QS!l&UF<T^tbVCzklsHU>KB*@vdf4Yti~=Gusu$9@C<c2@bv2?U)A$5LV~gP{Ik<E~3I=Fx9Q^c($3A6{N#&62KZogLq$2icMyMA;ayO{pUPRUB~kW&@E071M1t)5KjB)VcaiX$?yW2A#>YH=HIhjsQ1614GABnS?y{9x-ed|M)m*WJ0O=6ez!bzxru6oqo~`C<Bvhvy|&Vq2dWAKsm>yTBPYEc%r4i<HBO2uTE%W6j)t1}ab(OyXMTdo*~F=4j`>nMWNRio6rt5@XzxNKZ2_AO+^2N-0HM3uYdd;5xICcMH0}}Mf}h>#4+|2>pB#5UEwq>SVG<yZ^15LFUQwbnPHq8xpfD$lQXpHqxL8djkWSbpji$Nd;W0be-@^66rmmK@GI601Wof&CM4o4MiH1t)(w?EvU6Ng$sAHem9y*`{H^S`AY#8>+=`h#O@wn!+m%%Z@hze+Q^Ne6%K14x}o3Mbv+)Et7#h1iazsNuo7Qbel*gi~jt8BYvxBnA>&NcUM1pm*->@&K#7mv;pB7Zkuj=xPC>7dd<UW6v9=lJH0g)Xbh^It2Gs#1-Y-Q%erbyR+ohQQLb#pBtWau#+`&iZG$sp7&uBJYu%YsMRZCPf>3p3NHw+v4BOcc2>#VHqw?kua{SvZ%%#{d`spERRJ&Z>Evchz5ZEgWQy_bxp(P(jtN(;O)^naXy)FQ1TeH2Tr@fT}bmqZ8<WCFpJWsJ!^);*?cs@O-_#5E8@<LMShLMMcM7`D$b<Hqt$9jhAZ?=0Ch-Ybx+1J+p$Vh$2H<;PDL*ExtDTo!nkfi!v)T|iTw8XRX46ADez|eR$C7qr{puE$mDgTKT-FluR-&CN>H*>db96U>fU>m?+E>ash`BqeB#<q@@B6qdVs#d6Di;c;p^Y|3u*kc6$AkpRkR-B)Qyx1U6^}sYQHW#GS+7L|5CM&8;=E(C%Tq)@>1vhGkNb0EiB3e%^hIB>2$n#de*C9p&OCu@p+_iusF(a|EB>O0=mBuL{r6L?shm|^@R;yzUD>E@ePKLxw+$WGlzQv@=rhHlltAK?+Hri;r{cx{=Vy-T7qU^=k*_$_zcGvO=4jI$U#uiT@0JDTx4=H6D4u|8W0h|YJ%ppI^TkYF8*ganXvzeJ-6R`o1SG?SvI$>0lUf5O>r&HK)$8Q_t9^BoTe3;N#Ff*da6MUs-v{dqbB%6Pct8%Ej#ck>}1mnUrYNK{2Ne90|XQR000O8001EXwLeG~8V&#e-8BFJ8UO$QbYXK}a&>NWX>DaMaAk6HX>)XCZewLGaCyyIZExeo5&o`Uv0*<%x=8e0+6Dy=3UE!EHfWj}O|C#=7z9NwEyfh7a!LDqhW~qKcJ`HA%JQ`Z>JN7!IXgQ$FVD>EZWKk|w|Q2x`?kBSnj2PDH5cr@>i6u9cU4*OV$F8#&=kC2_g&S)*S=*(-ib=|yy=<f+b+A|aTG<1McK6nmZs&<4;@caRvnIQ*R!l?+CJ;6wh@bkdiRTH8~wW#H1e4B`?}icVVCfzNyL8Wt6D$xS%;FEM}9cE&mSv#B(8DR^;MbWz0i!83kHAh)ayTXUE8hYs~?UZzsd4Fr>DPjG1UF{ytv`&d)BvyDo;^%dXZ;M+f<;zbf1Ymy%YQF%in&Lq6F*3YOz=pykt92Va?Mb>odXorD%sP=V{lr{o2u{e6dO-8&K$F$B$XZS=(?{qn$QIC2m>P<@@T6Gtn~AH2c%#d)9F@1=uIMYpa6wRrA39&>>yNvqDG%N%2lh&9+kMu6V08y&4NCc@_M~!8Mxv0Y*N`SNPXt2cEcka2c`-`F&-m%eIqmSk<s)q%^cSWKC7Vl44YLv}RFt05=mGa?4zL6MZ92Px9Y7+wmG?@$Is-auRXJYhVDXQR_9cl9@!b0x8QVZ}WEZD0x1`;B%xv_;JHgRb%B{7`|$foL5IAq!BTORVs$EtUfFwSN5vZB>V#~6w4KaOraT=mx)nX)s!vpMHX@0f_0ag7HysO{6oK7Sq%gKrFqcfW0-FlRn2kealPBEV@;13MX>`!^^p^~5?MfGFIVtiil59#M_N(_IBFyL$oB*^TXr?|49_8~OxFwTEJX#Y1z&$4kI|kY&`U(#b^v6l13A`tB1b3^;0utzlM^Axdw~<;OF5S>tdV0CcQ<vrTSl*<m9OVgj*TlpVX&GgR(>X*088HU4r~uIJx0dw<&T<v9dFeCoPHdeo9Jo66IM=qQHe3APoU?4%V|h=eGmUgwG#C;l*T`C#LBFD_=8?0dZmyA^1W51!lP8GNkx<)!${KMWn;+BM@SyGVCJRE2$pvr$r;t#WMUSw2C7Q{s`FwgZ8~y1N5D8Misj^^#l&^+8w^H0ZbjcI`^P%W)4uhfauwzP`WpZ#?)diGtQLW3tR;#H+!P_h1y5HkmNajV52?23NOcsNYsF<tTou+3bXc&CaE=Jz8rG*5<}4&49zZ}`+%E_PNf-wlG6gY-6U%D{vYni&QEV-x)KG1KgHowVbUK%&gxTW+wWf@zeityNY9t;g=1DLS;HN9n$hE;{O@v{b?L=D-J%lm?4~*hCLjN-qkgj`S9k9zP!TmyG8NF{rR&u7qH`!c9tP09dMA_v~NVWHkg#O6MJ2K)IQ|}07aA(|_`TnE)hP0s^;994Aqe(59^J+p9A8r9AmsrL$z1aTL4KSHZXlZ*ZA6JXBz@VD}0|bHtKkRsihR~4?4oNS)w5b7J3r-}lUygVw-d*j*G&xKXtE_$ieii3R1YjTkR8sLhDnz9{Oj1kl3Q^&dDhwey+WHQ-A#WF`T%#Bwf6f4OzQlvoqN9*5WAcD=-N6jjX=GGIp~;r&g)9}m08J4vK=vO=Abm@F29!n}tVG&{1eSiKGJu_soG`lIk>9mL4=6U(5A`K6h8wbMcfTw(D;lz9zhSS~o7Z3J|9sh60b(EPEax_5YW!Y}0I#W!e&gT<o@pAs0Z~}SN`4PD&If4Qe*W$fI-dq2KPy;UvOVBs#|weKJAP<;XfCqmW`GK5gN`ASs!XM(XHb2V0Bf5j3SA&o+DxJ2T1>l(09FJ#iwS&8tJ_iG6=^Yrs$`~RFT#RFH<eZ^$|DxiVrXa&9z>|+Z<#-%SJ<CGhzXsK!^X?z^Rt2#RcYul_|bEk=r}aDeGE#E)`1UcpY1{)Gmry}T%?c*a(?W)wgJ)`=x$Tq+>Ay%CDRyEeRceEb^I3NN4e>2krr4B^bL;mNw^y0HB_A7x#Qy8Ls^~aLVs=tp#vYEYXF)6&N{Lhg_D#Qpcohg&|BrVP!z0{^cR=^{Pp<TQ;_fV<Sj@sqZ4De|4+SOi3bUYnh7JYR>MU!&p>x2qlRQ|NJwXb|A$I>X7O^7pDN-Q3zs^;z2vu68B%Pu(5p!W?8C_z=uQ{LVXW~=-&Y26erOx6a4$gV!}TmukJ=S1MW(wxrjB+i5O1Ty{ouC|7#k{7{11r-md>26jB-;$iWuCCa7ZU*>&wa&w@h)l*jmMXYYG-Lm@QXErNnYlk{Ct0>vbCK&!(#lm2V}L2<}E2D>L9#B<#_V^GV4SE>R}F5KyXLzSMFfRrN8TR&V!FK3C8j_ErC&{BoAkZ;9YVV3QKx`aU|^6Eu^lnxo(!{5?IK2tts=-bez9LTv_w3oC;P*P-r(+Ok>sUQQgQn|Y$ij$$u2Uu79nH93@JAOY~z-<EQ)Um>VZSCL+cTvv1*@`Kq@{yRIF6{H&)V+(eu#3AeRy<68STe}<Nu{JaG@NEY*n`1rAYwZZF$aGs7;27dyOX8D?qOFp|TGs2N9TLy>PQhtOrlqCV^pR0%XEfjQLl((6jG!~xD9_2E;Jy``yEk_03vhDmjmXEE64ZihMsJmY7o%Qn#;|u;U7^=m1Cv0^Y^^JccI~~~inTkeQe-NZr2w>q=sHj*JDMVxggB7odK)EB)hfczLZibe1ZfplUAv#*+~iGaH3V@U?71$Dy|wQHX0tE6qprolnuvm~hK3P!DYu630eQNj=k3t+FPB{Ke+|3=Z3jaUCd8%|N*0<=`>2)1I+6*xXPn2K9&3B7nez5<ta;C4duSmYN1jx`kE_j_YxESJ--_Wt8d1~M86-^m<pq<oeI?{hf6;P5J1eLF*6;b$jR``WolO%>37f>%dP+v&mCQt)GeD~`kqg(GnS@NoVTaYQ$#!&l7;e=siIzdKcwh}Q?27SQmJf$P?M%`c9+Z|PxGk#$tE+}*a<_|i7;KyVKiZq~5Oh?a&SgghsTL^D1Xtx*4v9{f3%3^-H^d5ilNsP0HNuWx5NT{ORG3hE#)#TE^bPpN>(Lhlq6q-4(`GO10f1h2Fr$a2eo++fc-=NPczO?+2jZv0G_PYvA`>&S&>~X#b|jjC5cV~VIxU2+f=hdEK$S`oT(Z5|n(86k>6$_JR@aRnc!~ezEiAklpk9t`BAv^~j+_g(3H8<|ppyVtT~1cEVQ*GfuRSy~TxL=2oUHt;Huf3BdIsfNS#p^U75=?tgP?l=q#boWe6E5JTejyde&trTFfLh4j&OBoo-D6nsG3`Daj;TPv3aB^KUdG6Unj4BS3GSV$(!gmalK03e0BZgz>0L`FcO)454W4P^lN4W8r3v$cEn>TYnu_YsN*r@KMH7bGUgU6cFNV5TviIn#6ia}A&Z1W=5LgF1}nxsyJb$a1|1YgfZsSxPjqJOyJk+;F}40;k;4pvSvS_#XH0_oy*!#@kjwfCiU(F54nv>q;MWn81|G_T-@piYtB5YHKw8%qSLnh~Ogqc(7*-$<N-DV=px}pWg)bxvcFP|mn;-xDZliZWa03R-13HS?Uuka1?a&syzWB>OSiAd$=L&}!4&$KQ>MAImKk_1Gf5Xfy4?E)K_)%RNKvETZMXZ;)P}#L)DKsWkpQcN}>(W1loSad<TC=hqpj^P2AGUA4a{EFU9?Q{SIjvgv7|^G5?qqFGT-gcGzvNIx4)qFx@#GhH;_Fuf{y?uR7?lPi_Tl<KHxnb;Nugkig;9L~rreNI$l`!SAa_@B%*~HaPmAfXyf@oSx92lGqsD*Vh~apYHy7Lsb>z%6Q#7$zQvklN=}G$_9_K&dJ9iIPQaHYa5Nw~Ux8h>RQ6YXD?}P4f;M%;>>EybR$W;rGRaBefo?>vPO;SxSIgC<<#~V%ZpyyYBrp<IMMG2gm&>)azRucSRb%f@EIpi?eS$4&!VU-lPHoRrxD&6O9kBuhLAXz+rYCLI;N1Q^(zB7%oyPQ<F%b#@6>r^P}%m|Y$Ddm}dI((?F_cUjZ^FQ@i-<sLK&@p?&f6^E)Im&nJ5xyvLYDD+~$ysyIr7;OVn46;DpcySacd|c$qti$HlS5GlH0BOa|7%wSOaS$eEDkXFiIe{D>@#-^%uUU?dj+1G)ATRDKyd0JfnrHR!*id$KQKBdv*;zS!pS&xcff2uC)?qh3F4dEg=TYmghaCTb#Mq@rZ3pFyR=9{&ZltT8Ng8hAIeuI*?}5uvcpSHUG^aGSgz0TGK+gCHlvY2vYZZGy)3)oX&BMDxn-Q@lBoez1#m^9v8MMfC8>QsmvqIx-cG840B8AiIngJT`dWcRg!HRq>=f>#q1-#VdJ9F1I;9AB4_)(>_YKYMTbIASxM7R6nVa8qCDhOOscA`VQk!1$KI=*>Ak}sRc5}C6y)t2XN7hPqZVJ6WBZUTaC#$j94EpS4SsfYc`Qv`)%B(8moeQ%gb*<+4JbQ80uhOl7;os36TK6dW^xd493Qn5FoSvI$EXQB5A3?db*`WazZY}>-;upq>-hLBY2nllgthoVTtNM6xMtkF14HVQ-%lC<0lxFamTWX(I9tO=3y_;E{+}m@T8}{Dc(n5uaekdN|=)?_a7a4SBK@X=3Ym@t2e(ULAtAn7Alj&3|khrFwAxhm{AbJdtx;Zd0a{Gi9%iQglkuBHGYQoK7oYy?-0LcFhP)h>@6aWAK2mk;8Apl7}Z_zyi005r|000;O004Ahb6;|GZggpFWiN1YZ)Rz1X>MmOaCxOwOOM+&5WeeI5IjkMtwVD&;6sqzqK9C!K%4fkSO^p~wpojhl9aP*u>ZY7k(MdjK?~FuYo46>X84g(`UmTK;)MjlR7XC2efUTuFgI9`pu8jBgg+2?Gri!#6E`Y;uv&?~nHO4-Ru8IUR)DK1rA4u`T9T&O4IUFT4H434>oK*`eu{8Kk#zB|7i}%UvyORowwRvzowo8UyP-8!b8r}3bg;{+fU<*=;IMtt3aU*(@Ry>zw){{;%`QqcA39bG*RT^4J!^X?tEugjg$PGNBBCvE-X?8TJI}0l0f~}^6tqf5g<qEWwyr8w4C*A%5Fl+NQ(_0smp`o6Fn(lvaAZrqzfT&3<E$k@k#atrPGNL)r8o(PU@z2`DHl)$>z3|E2n8>-jC1J0G7q^{4bqIc;#u^){+4To9uMkAFZl8lA>|@d?~e;Ry!p1*ZLTvT*Iaw8`$Z~kK=ISsjQ%ipn2*Db7a<4pC4Iais<*pWpAZ3rtV~|joCDrCn0xD5KS$Rm?SBr};L%!b|NZQ+V8u<P7UIaz;j->EKL9^AdhllO<s$P7-3w1wgks>@FzX-)zdU|?psy<u;N1RZ2Uy`AcVO){OZxEm`_tFYpXYR8zpXH3Y&o@Ce))CkXO)-p3AqBg=O|6)>TU=3TkNbrkkP#C<Uh3=TgFS&3AT1pgl5<4zXq`5dT)kp%-y6RMr&Ah=C|CPR!Sd$Ar6SmP1*}qH>MwAw{)!*(gMwk5(yNyM}f;M{}b^U(3jUq7b(YG6oEB_>fkMK2HyWYlo?w^_lD!{A{XU(BN=XUYTJWvQUKOwz<T8Ta!OuZ1-hCUyrbQ6g5Ph@rHV<Qn-o<C>2?HEg4A|w+R=l%ID33f`X<`$1aW3qD-iJ!LtavhWPb*9&yrBuUaiCX)nqK-Gp4fAX=0tt{6^JLbKbnN$_Xs_EZoz_(lOCQa7>@qgx*Z=&L)i{oi>doXVvL`G;uEiSkp)5fLG-LVtxja2`++^$SIroCg(NmKKz6amzP677tvy=IA#|YBpPcL9)u>iXVK`ORMR4FA0)HKyBU<KXNMU2Ui<@4O9KQH000080000X0LQX`Dq#%(0IDni02TlM0CZt<UvhPBbZKp6FLHHmbZKp6E^v9p8f$Odxbge`3ZD9*QlcofukD8|1f1(Q*WmKP#Yqcn7X&SFcU9|kqO`UX<iB@@q$E<JY_|nk!`NMs!{Kn=GnD6fA4*R0ilsayZ{M;n=Oimd%gTg@gx_VN&B{F~s+8v><-4rR+N>%CVPy)_b)F?z3oCZK;jn~Uv*d=Csfaw!Tdj6YRS-&dUE4LBQj!&Q)wHm-tXd}VtX6s~smhk$wRv{kkEkygh`Kh*6|S1K2N>xQeT2sWLt8b;LGKx{w&f-6(1Qxz(qWKlGBt3@lPbkyYS`CRbIY3a6VS+4t5uRSA;{aD9ob*hy7h{{9~|*KWp&FN!kVfBhO`H6@T|+*3>e39Qa8M2jXber9N;RL<kfXm@@OT|o#zNx6?H3sp#<mK<UkJW2uOeH08P%zeS3(=J0@;G0u3iD&tDu_o~3^#Wkn8In(|U2t67RCdRbRO3XC*70|qxhiQ+uT1g@@bc^Z*lz7T<jCD%N!%Dn(y%CjT@!JDW*t)dgO0Az}aK&dZyz6+IlgydOBfK@H+H>G*C&sq^*H60HW&+re|L`tJ#wbY(+;Q&b2ut5>3rZ{MVs;;fO*4R_A2C%3ZwTBv{^5_F(wp*rR#Ao4DKQ`--yf}rA^>zRoh#$`!<pn)(milT{pq5#yYS?-t6`$x0zo#JRj*9^bB-u5krL4Z%FO(S6=jLP!|B*Pbnr~j5Z3D%!pQN=!gW*!7%?}24f}ZTMJlE`0YDu#cP-g|(b78P{G__dSU<5iXXrA40-&~ls_cf0d#C28Wfyr`(H7e7rAaP87v^o7d@}b5Zzy`9S=vsE2b0q4YJZX9QV#gUe7~n`JIJwQ*gS4LlG(y@?G(q+zMS?-O$(~mQZ=3s<QZz3miC`8D$md=ggKZ1H7X(CMBa}`gNB0nc7en8P7OY&!n|J3wU(%}&zg)hjZ{EMU{QJsU3Mb#ALAH%=Xb520W&4nLZC8S13R>hfWy;lfA$iSmVcOH65G;e^e?cv|Y?`X^XAshkFIaLgn5HF7(uxakBH$^zTKQAJS4vLnt$`EZ{6LiphU^0$KieR%Mw;ptb$_$9*;Q#wpc9%1>F~a0wf~@^jYgHI-~qUOg$Z?L#_Av{tMV&vDxV43c6H8`Cjxtz;RHbpXrr<#c<u+}hnSqMolJZht0lW%TgushC(rP}{Ej7Ueow$w)JZjWYv8luf`h$t7E7lIQ5gLNICy_J1RQb>$%|`<@|HIO^^>*_=Z)kvRUDmb7YH+Abq(RqeOhJNu>CSKe7S>`*=8*X-v*-^gv0_+2YXIV!8ry&xQG^<Mtl-Jkz2vk03oXS-{5rGN1kk@hQa_UC7$~Gk|sRS;9gHGP5LBIfuaL9KZf@gD!h}f#NZ8b(#gQ1W+r(i2JZ_FD7o>G@X1kA9T))eJ*ZtMKP?cqEBD=cBs9a;XWO}RhY^0v5*}+%^|W#yf`dO>&uJZ~>EoHg6%vYuKR1I>sa3wkAxVJ%3ZaOBD;E^RO2?$kEiBUk8T*g?>iyCqG8vy#6kjwtn+!=X?j9or^C>|U4>VgtxAbTM31U2wZ$VAciCD7Mf&f276ATj4UD|Uh>7Z$rw3FFjnwZG*_Gy_;$exybu=I%ahfY5FSRvE2NAnJwpW2fQrme9=+SHyCYyNaQaWXTlwr7=wYZU6q#g9IT98DBzfpX7nQcc1hBQtdFJ#HO!rTDGm{44iQ{%cLt{Qpl>t<Rnj&A^3+)lnV%f2@?NpT_bt(mx@|0XuJ2sE|3|v*ex%RzMQb_&W2XO;xqB>ftg^O3gwC%B9Z`958dNnrxq8?yA>A%!j~_L52~LR5iGIhzz=b#xye|B|l<S*%!gGnjTV3NX$wB%M0EJ5MaqwWhu+#A@Tfq7=M@DWu-E_fs%MSA1T^aIwtA^88utgH9%b~OKyt+0eMM04$`A76COS&R8f6D1};(a1Tz$?q>pfv;*mfgIj4Q@Vk@GDcoFI=mTHQFbl?}H!-_PA$U(2WoX@va<7#6h!N-rlqBW8YktX7IK%s*7)z=8;=AbHR0np?qR2U;G?=zmKN}4E(+?ZNK*$d9@M*P}=(G@y$T4lLaP=_W*xZ%X4WoM8fk3%xJQIK!Kg{X`POXr8tXXDSs%3pyzSpP?;KYn0C4Ae=tsejub6KbWXtq~jCJcCn}ZE)Qr0!Xv><LhRJ^SrLX30CE+0ci<N#@G1@1O%u<!Jhpfn0j-yN>-*1EkXc$jn(q<J&x|o>qD?qBUpZ@(m{Pt>HQ>^X<~a2A(=Tk9d^8=OWFFwds@F|=Cpbrb>6kr)gal`Q|f8kH@v!heg66F6@BxLUcP^&YQJy^;?vdnuU8MlU;+8@{OaPbQ?S5AoNnhpDDP0|^dnBuTsHWeG)rB&P!;}sLUsmUSIq_M1k2yPGf6P}C0MBIf(t|uWrD)uEJv^HPfBK0rD$KVP^WlT5T}|B%FhD>;F4oO!N|quSLZm4D<gvOOwPwTzkUhP3+fukz%D+1o|GxU%B~Ah8F!#y?T=@WGqjM0gO&0m6H+%1+5X^7mkhmRIAU2w*Wm9D1#51=3{b-!fCkyfEh5?Br0wp<v&R>~7R%tOCDXeCCB8^T5bUU8rv%DYX*C;9s(=Sc8V0mn#x_AQ<}1UP;Ealb9wi%~^1hYcPPC1$oRdPRroC+tL5*DHM-Xu^cjB<nGN>7Olf28O74xed7;`MUOAEBSgRJ!9PVN1-NU-J<Z{oyZCsK0MOwFZUo&L!NLrEreM+;t1pl0t%9F;o*eXQeep10P|-{+a9U_0=%B}5dV8MK;qq-M-=c@LxCH#GSQqz`IUU=%h6RR0BB>%@y%vW{5VH){ONqW{8j2a=f#b2AA$QNV|(0ms|46s%*R<>(}4=2<oi9@y!--2^ZA73FWiB7g=3u8L#4_YgjIbp{Fzb5XW5Bpz7*nAYb9Q!lG#(Zi}sGMfo{7N`0;YO)pNF}@C)x<n)n-JKsx^$F>|0nEPQ|FoEaP;E1OlHV=lcm{2Y>E3e6x0vg>24~?5(+oQ=QANIX#M3Nsc4H^2SI14>q?R5s%mH&a^wp-G%((Nw-j5~-%ID@jL_c(tf{m4*qQ_Hjn%ex|QWXcltO@=|4uj`|jEtFArJvMgWz~FY?{lts&~hk6pfnfGObDEYS$qykN0oxfXDfdh_$qBID<_+{G8m)Jy7NAlH)VCJmJRgx%h%-Ffd9LZRv5&99yxe9y96}vV<LxZoOCT0@SE}^&n>X$PQ@@&gd?D`nX?%Vw*}Jbd*4baj=D(?flpxA^W|7mL2Qx@Fnf3JAEgIhQrn-51{+Y=3pJ7uC`&Y*(xqxsnq(>12QrH}Q#)*HL44t)3=AF0-M4=F*zDD<=TLajR&`IF;B~Kn_#0#iN{$VFTkWH(zkQI6=>hW0LZ{DSlj!42NH88w;+S&a(a=@1G!3cyVTvo_mC#~>%68Zi;5Kt#=U4^M1KjozdvezU@?8&>6ZwJxC|=r^0{s=CAiYnNm#*aU!zW2-4G}mZAKC-n8TMM+Uwsi)jxUb%(1dkq)-IM1ctKpYzHexe^S)1X1*FD~+nU*l<pC2WZ<TTX{$!8c&qdxW3M*Y5NDJkmjHr^`fg0(kA~DHII$`CJ^?ET9Th)E1_?wqjRAo1ind+<@#}bY&a?qry>VD$8)M$xO>Fl@fhG~|nHR9@+$eR-N;A`3(76k-9(}V@K@_F)fx_3KDZ);w$4s1e%PrCr<?*mP)QF(1<u*&S{>Ml~5z_;=$lg!9KVI*WmL#EHl#%@dC$`RzNLg2En4Kux-sA_{c?GY!|sh0c0nwzYqt`t3j5-_XnWK_Bde0vB{M_9F@(fN~ojlJ<}vnF`2sGznpw1x^lO)^8{b*aT@T%e6gt-F`T$hFKYj~Ty><*k~lqbN8eYsS59!R~0uZ+k|a{lVyDf|tS0DJV^R8l9Le&YITqp&MgYgL$6ky@yod$jRjidflO4etD{!_n6`2yZ{qaDTnlX1K2HV@czDPuy@?|>>tUMe&P4Gut0p$uJEGCz`QfIa$a<~v*IW9{p_MwzL;U?KZrc#cX;TN?U4THDb;(7iER6M%EK5hby(eP{F6+X2Nk7S4XAKcM7yfFp-s8hrYLlABplcLliQbBD|ftqG1Lm*Z^P75<|#H#$15d@w1Sl#7$WMOj~{{SH4#nc`k{9qR0nE6Mwxt*%7qZ^5=F731@2qw%~H_c&XcCmAXksS9B~60yhnQQF3r8S@~-jRrme<GrN!lFy0~oC_`Luu?O1*;1Kdb}8#lnjd)SZeB@f*>ngT?DK|EFxi#JlH+wckO26b9cS+{yA@kCvX>6i-Gf+m*ULuP^cRv%b=23`-Ju6fJiljx80{pTT6Unkmc?mc^^Hl76k{%>t6*>Fcq=0_M^#_Z;6!Fdh8g4MqOP)h>@6aWAK2mk;8AprjRaCa2~002A$000pH004Alb98erV{dL|bY*jNE^v8mQ_F6oFc7@+6&5G6Vo)-fWTlmmHj0!Jxkb5JJ-C_i;s>(LJof9?#!Mj5`hakCRk^#}&hz}4I0aG?iy%qUz{fLDhNC2%u|%Uk8{2O>i%5j_c(CAw(WE_M+q){y^DGl;FxHdd>e2bz&Kuhvv(6foe#wk$oh4LCI6*ZXRY}l?5e~@aZ49ySRdACv5qdGY6NfTwnizgH_60@<vU$t0td-y#c^CTnAza9kYF)h<K1wWpWh5+)T`cPimQ#naE6E@(DBT>6YcNJ4C-oeu6oRH)sG4r>OS0J(#V55hqxHu~oiqzI5PQTS{=VB07sUX+je@BoRSTZdq%CK&q9l{SBC+U4t7nDt(Dp$SZOYm02f3>r7hRV9P8U&i;u55o#iFOa(z(p!r9h7$nM36C=R-QMof;exT(z5irgAbZ?3ojPvEJR6pYTy|jymBF+Xau~Vij@$9#iybw?y@**sPRrMeFMA*C2S0I?ft9B5+rK)Ao2}Cy*n~;gwp!g$>4uNsMZboC|#jl*gxqv^S1<(}&{)uUFOJ8ef?Xz!&^3;p?VVO0m%)6N-gfvYw%iQ}hbb;hN@09mZTSB?kr^hR~xgG>dMO{&of5hnyytqQX!+Emn&8C7q_h51F1P-_=|A4>_bUSn~(lFcE<qUP~i1f~C8XgqtLL0j{1>yNRb+Odi=kP)h>@6aWAK2mk;8Apm;x>}LK2002@H000*N004Alb98erbY*jNUtw}|X=Y(#baO6ndA(R$Z`(E$e)q2sxQ9xGqcuU-0tNC=^a2#vx}oUyVgv*w9oeiYQWYsDZs7mEbEu0Y*-p~5o1ZM|T)%Ur^2!uJDcnlQB#=dEDi{2PkF)l_gQCo(g6_H2(mAkhHp_|A3CpWmJ6S+1DkpQ1I@`$!X950Rx5w|Q%2acI^s)T*Mx-lfo<4!CmHQpCCA9BTp^cWQP<$oqYI^p?8a<yyv)L?F!rETJXEe)(oq;huz7ZCpSDrtEJh0&YE<+0{Zx885k1@GvDp75N%id*_&>p^(6{OBo+b|llRmM2vAi%=0oLGwj(8f-+15v#-*ZOYlajHP*x(r90*73dfG6RI$ZFvP>&?w7Ak;r9jYW$}&j%2w+2!L;ds=*%Bhp0j2-hHiw3i~&Qz=yXiVXuP4`8>F|ng>kagLjqPuxJ;}?`9=xBt$+E5|WS+(;mue=~hF-J_%{j@_)zypEU#f4^shEr3Ug`rV^1sRk<?4g-i=gR^F%O^R|4Dl5_N?hnnrC0a8AFF(|I0(Ac^FACJ@&r2+>Tmqk%Kkw}HXn?|JzD(2EHD@+E8z^h;*tu*>f5Q-ZK*OrbqFFmNjr7L<wU!+iM*9%)iBS#iu2}tq$0?7+-E0YoZ`{%@N0qE)Fo~`i!IfwNa_Gv!vjf^HNF&2lZvJmyoQ7sUdq-bALfByLg+sy;(gsbgkOA#H*$@fB82wS;5b|gBwM{O!fhggx~*Fk2MF&o4l1A^rX*>`z2Fza(wL)Zy9Y1ct*^&Co2l6dk36ONE^?20PlG_wM|7tr`>!zg+LQ=|QooFM-5LtGw!Wf1%le8?!I4uJ~FI#F01L>B*;4=3t!G&Ux+`4F?wwowW^Ld~gExqbSy%a&nX8kL=GUL3Xtqt%v|G##;-I3clz<IqIQH-J8HGR)N2L5!7pNe!c*36^m@ENger!SAipK?e~-1jc2<j0QGAZ4H=~Q_NbnFQ%@vP_Zd7rc5L(3WN<-*y9w8VQbFf>k6k(m^`_~Uh}mEonkLKpR-bIu|H$)8E9rwJ8TMd>7I;o6PsxE2uF_;E=d~Ht#>}Ur;e1wlI_*SOJ$O9Zz*JF6w9-xaUyZh92S;W*@5k@%@k}6H|~VhdmA3;`n0wt%Y*S24^pck=Q%oHxzrZ76U?`q+qEo9%)<XM=S(G+QqurC;ObJ&XA*aa*LYi*kX7V_ry@K`B~l3Y;%Nq&Oy&~jUi9!K<Wc=$0>(HRUGr*#U};g48MbJS8*#eEylFPM5(%N&0Qck4-nlYZgdhdKc{N%aZ^vr{d+@9eehrv1sYmU;ssE5ZGU4coq2Cmm7Z{j3mV^@{JZ{fBG{+e-@<_w=*`^q{G#?vg!((}IdZ_nYhjPAaiMXQ~HS5qZ9>OsCY1w->iU&wuS4J<{DAA2uK}GR93F#QtwkP>`o@D8X7SFC0{p7gvDL+WJ6erG-vgbd8W8{5n)hLmAOQ+SY%Wrsj!^-9waNhG8?ck+M*Q&W6tN}{PaXS~-0>re)@iH;Cvb|aBx+u3ae|6cvhMN@2F8JUnyoaw2Yk?X(U5xR1<V8pnfnL!F5K4_&qqiEN?k?+N5DH<}c0fJ4ba1;87r(veTT$PE@F15rs0I(0U<$e|wO5*b@I+OKttDwA`x)Nsy$D^#o+Ml>p5f5NHJWay&#v%K=I}OmnHZxcw^Qal#2v&o0=_|IY#1W{1r5v-oVzg?F|K3HuGyGK1)xL%ii9pOBmGX#azru_=^77x{qCZlBxbyV@7Hds3drU`mqiF&qi6}uouj8j&nYW$!ncLNy$Kweg~Q8Pv_M~V_VovMG{sK!qVt5J(ztSz>x2D>T`%05i95B2WRw=TQR5i-p)NN;{C1c{XaVNHVFK)qMP&CQ-zyGmup)<1D&I2yy#~+DcYPw8gg)^^hY3U<VJNWeYLsj-s$s%1LDe@b;zuLar_O99^MG^uoW*%?c^NRy3n4Y<tl{fFxX~##?|%VMO9KQH000080000X0OP=9pvwpV0G1yB02BZK0CZ(@baO9sWpi|2VRLh3baO6ndDU6nbKAHPf7f5Z!SsPlB}&TVoT=wD(`(}Un7LjviQ9*&Dh47!2{B3V0LZfA@&DdkfTZ{-I~m_Ym4{g5?qWZGy9<e|kji*_ruV7XP7)<EZ*NQyYOZd%@`QW;H>UZ|*Vi9D^4||!o1a-8r(8|F>%HPE7WoceezGi+lI63Oxsz8ggTgCWAK+_6WO!C2BIS<IRMIL^C%z&+<iZ#@-4R#c!c0at6$2a<)xy6BC%Y8RG*yS(h>V9!8IiEa=!)-$^1+o-63k*oW+I{|6I#q#^Ao2(@!<3Kf4sl?m45ng_3qQuKCna-E!XZrERT6ASxmu2A`v`xj|;^MrZ{y%-80P+PWhctpb!NkrjuYYnM5hjct0TBHJGja+5!0W9Rw)2wCsPilo>aB8M~u-;sMt`i+A|Kk5XMy;s?Hkk<1Oxjkof)B<6;zOyoiv5zV4Kk8TPnax+WBojEA}pPYEFyyarilAv5(Xb`d9oZi%O8>iH78$3-BlZIflxXd+wI0YM)RHnv~{&aQiYn~>Zw~bSIOcl#_91cgqa1VLLl@52@_~awLA;B204p~v~JofDwAqIKcl^gWAWs*P?;6T6k3P0duzau2nMJhm_6$#eMP4#UC-xix-SV+=UWRQ%fntO}htG-N;c(sk*YUPo+*yU33nUzg02c_4B_D`_FH297xt_zuK?!R7K;4vkW)$H|RF{p~889wwR*@bf0$@n4fTq4d~n5UzJ3EAF5fq4(^+{f8N%5%R=<G`DF%g1@^(c_4)VDhB4wNk(Gd}sD#>S?3=&UsFcU_{(Wn)XLXj);fhQ3=B<x$DxYkWjr!O687bTx+)DHr1!zE2egOpsEU})JY2w%Kot$Y7xFqmapH23;4GLOBcbks`8*E!}Wt&hHv0>ZdNDp?mwD9M&4pOiW$#jzPeTiJ|rpBAi?<US6jdpI3gj~_Acl1TC>ZC?m>zo<Uf+hIL;c*)O&#XF{rJ??GFY!LPwYOVNoifJ_Zf>43O-X5v$?73wbu=*kPq&>K<&Yx-E2!^5J`-cx0ryZyaOUl$2P(*sJ}voG@n@?@;SK1LAUsz;Ca=yKrn(_5#4(#bO7*)Kt6j)_KjtDCI0a6m3oo?N#JC;9T8WBL@8{rbhuGIT*@vY4+g9oSUOmH?%~??qLqw8fMQulF68-;a23NChIluHo{nTn@fKNl(hUEG_ZZ5&0`(4s$AH4DZcO(0zk`;Z5qXm5?rsAr^k(Skq6s*YZ+Dh3jTj>vV1rzDRcGvfu%kw5MoJCi`70rNlW%PXWuPT=dv?AEk$uU4AFbqX@O}+u{iuw`h=BZL!>5wQ`Nw5=Lf->IA@W~wD0zo@4AT(&GV4K&<KcMVufwX0J5&>(=B8^h1t^KV<Z)pdFi4~`U7}+&^qjp2SbSG*OBga`ws4-y``>)EzY2RC1zFqG1w`3)mBljmzSH7;%jn;I56ih_us@q4RpHkU;v|m7});Ta7L9Z>&xY0IWSl?wt?ZP;{1ta=!Zvxhw(Axpi*+#s=>-p<0DIzQJ3Cu0k&wf_&Xr{P3HWZKqy(DdZK8n&*StNL*)~wGlV|pE$^eyv(V^kQ0WXhJu&5d6IzW*Xd%{mw%ovC0EKGBtss42Ip|=;QAvOWyT}1(Kr^&+%wd}fz2}-HB4;VZso*(%(F$vq<_QLzo*$XeU|MUV?rv#MC&1IKXFTv_(8&Tf)r&U+1kVikNU#SR#E%R%7Z+fDCvZP+%;_UO`WH(N{CN(lrc-{$vH~W!NM`dQWl$2(O8~0z;BMr>YnHj1i!jJkXy_im8NmaVz^-4O5iuiV{$jJyVEkOXdPRBib1-Q`Y6GaZq%wyDf8e<<JYYbP8Nc>kxJ%`>*&alG%Tf`~UXdUG!OPx&3+o2kLewk^%~&oHd>*l!ZaD-JEscBAbHO7BTD!H7^8N;}yx(&Oq=cj*hL;5PK@ka-&QXJbngiNfp&?h0m|=;%SqxBNvS}b$e{I`CMr@-f9-N$rook@euj!QIknv5O@++HQ?`dbiI}tp#A;Ml?hOvl@-@H+@5ryd#)*$7U+csQOc>zIh-zWO5!cBzI7Aut8!Dy&g*!qLc=n1C9PsnjZ<AQpXQ0ccOA_lh!a}OhLc~<jT(6}44nGK3GE>cIdsHC4!pcJ>Ri=c756-_pccY}g{ljawQzELMFpknO~5FfHaH;i;X13BGt22L2m0eHWN!lJRZBfL=z@JmA*R{S#%4B#+R(O%s0E)?TUaR|aX-MGB?4*xBRAF;8Q+%P8`NdO-8v^=PYx0A>{bC`=}5|+dMY`cP<9~mTkxx|2&hsVzC=vij?YnF2!j|s4zK+`(+-tDQNHlpPL*z=16CUwTMEjV$t9@TsEoP82Y-0I`tc(l>`16s__86Av`1uqk<Fr0H8v4W#_x2)MDM~08a^f7J!{t<NGr5Bg8p?J2_N`0G->6FWTmWuqwR*PMA0fk|+&9)Sg$<<1~`;C+eN_weM5qTm(euMWHc;b^%;ZAKJHP{HF7EapC7`zo-$mQ$B+qZAM@4xq!7iVLO{J4vA6V36S_4th7o}0eEHJmS)ttnpQLZ)urjJ|B%_y6G*VaK?m)y6b1G{dBY4Oip+B8ARIk6i5b2CtbjW^5`}>(t<F3QjL@2nxarl}0BI!2!V2RlnP1EbIC-XhFAnAQ=q%Slk)YkV^Ph%It{t*iJXp07HaFmxA0Zwz)jFUVh{|erKVtaV_PM|E?Fazpz=ddANA<_$P99L;VTHWKgXVf-}<wZ}-pf;51V*;1suUSM(Z_!0kL5mLAgi$)jo*YsXkmjGHsYZ^m5ips7E=x+xt~3r0OHl_n_E9Steqk>cJ+$TND{hiZb(x;UAL#H03dh*JAA8l@OcO3Po+PQPU{d$0dLP)h>@6aWAK2mk;8Apm7;S04KZ005#E000vJ004Alb98erbY*jNUtx7*X>=}dd9_$ukK48qe)q3n*%vvG6??PYHo?(B({^{8q9+Z~^r6Qv1X`kPW+hQ2soiyh{P#1%i!Qb|Nf$T<_9ANd&2TQ?3{5G0GbgvK6a_O}6SL!l<)&pNk&O|R_?s7`VP;3zmWf&$qS)Q65%;_#Yo>WAYOWJX>147|vLb-O@{(x{SfXmAf(^VWS<WZ^@x7L{lSSB4tb^Pecyuq!zUjCB4u4DjnpIpk2orFcJ=kl)laVUlc^mkB!<DFbZFFLo&ThX=SX&4a5bZ<8>LP1IDNRgrSv5@YOxLW@J885x1NN%61~2d=m-R+$1Hv`0xnf4h`bWDSliy8S?A^ohX8$+r{<UPc?5$gyOeWS{<ejMZ@8k`y!Q<-1gurJlb!KoW69q}h!Y=ah;+b7+STS6jk0oJ+{_Cm!H^d0n3%(&)mb0?VvIru$iHYI&X6lsUM{(1t`q1-{sGHV+2AwW(TQW3dHSNLI%;dYQ;JGZgR{)!0^5n_Q9aCF<tZ7LJP2NjUM-^)#ZB$H@yi1HMg*MT2Iwc#aaEXW-u~y)=?_B~U8*vX9e1e*7JmF}XM-?b9;bc{lVV-%NArk2*2vKK`-!%$GG>yq?S@YOtXq06eQSvOCCW>oW-tuUgVD><^Ef*_`G#%<n8c2u6fXb3p>w=NLTCVo7RrP}Os@-z;XcIayyw*}h%SBASi^=S4K99+HOfFW_Ued)j9%$aW^aPis9_Ko&fUy;;p}aHAc@Ztnysh#M0@OgM!bvcNma7<k>^pS5s0)6N&R4A7^5|?3LdylZ(*y!V6g{(Mo`dH)Thi{oX&-)2o3N(gu;&96729&iCHlhYiwU((foJ*?*SQjF4g@8T8bC6kAA6)=dO+)5;OE2W$@<J$4ampu?EBx|zPg6DPAq;v=7(QqKX*dp37IDs4~YJHdHwRoKeH1%o;|SR>U#FaSn~O?<j3PBT5-dWh4F0E7YBN<5StChG(`q~nzQ<rX(HF}d2Yz9;CGrHpndF~oSQubrbDjl{S1_=p;!SwQ&O4~ia6?uuLs_C%$XjNXk=uQ-K6t@w5w+uZJnMSu3PTW>A-`Ic{Eh5PNTDUfW!=%2D-8}%V8UkSVTGCDmoP7_dIW{b+kdbVpLb2t$bm}-Xrlr^$fVFV(JFT0VvccuqvXOA*WI(u6L>hh$~iV4&?dp4sS1R2?i6Yi-u9yhNOr4vaVTY4UlHO&$^bL)!^e>Zu{MisY=#P4C10)_<Gjt`7$QX(wKPtlQDxnba+UQ9&VgAeh%nU#v^<(M*hF}bP63o_OMAW7L!6|?l@^UmEl!`uc0{7MKUL1;}1-r+&EO*ywCn|3fLB&9u5)FXZ+H8mOLfYZl(?qXC*f!0q~M)14e1OA}Mt3245C*C<kpj%6B}!SyCTZ*Mcideb|O<ZQonA_Ml->%j_GCnR7z5$5Y($M9#C`g><!~ofAO|a4_cYBTxp8cZXbfS0BK1mDnTUb|{OTEWCe~^m2~XaJj(WSLbv!L@Ie51vDEYIJQ;j($adurR7j$Iq$S|-s;<6feE@196KV34%H93qPM{zEc<-`zE#k9_aR_kvO~8G+UO&l@304Z7xZb`XD}q?(9*9`Yut-bk$g&<MLDt$>*6HFcmt`WLFXJJkD}~nBVXPLjKBt%OA8zEQf`IOU6M_Dn)AuE;WmM?Tq@Nzb`LZ((46f+VYcQw7$CCExv$MnupH3g4c|9t`lA_6DzOzcj8xt%n^pdRVaW=Xe9M5EyNPKjpc<B876I3f#spW(D1!bB>R0`o?N1M69<)TRRSxoSVA7G9K|0Ax7;mF~nCd3J*iBlfpVw`Lb<2-+`0Qi#K5{c4{|g38Cl}k>SPn+I89qEA<j5Tdc=-96T%D7e-$D+z+TKkeBiyE$tjj&I7m|d$f?SbzbvWw`*{(Q&))5K0lWkeRbH(sJQ5&wnF8~M;_d^W16F}8?8<rrsg>xrIh{2Cb>#J2a0t3^!3)9KTZnl`?6z_fqh(zyL!<VxKjNJuZf79-beLO;%9$ulkM4PyKOqbS)hJnz)G}h%c4YD*okwXr-5DMZX)gDiE)VZvP^^hN16cZ4k$ls?fm6C9UvxJb+*Rmh0h?`D`Z)`)t{HOi2v+$rX<M#_mDOsVTs6Rz8J;B!$?{0WyhHEd}F8^Qc6)35&GuX*4Gt}Bi9`t|3-`i@T4+tOhDccWEcbLG{(5oyvKZr4uMk*isunXkO1v0pk-$U7GppXGqml6l)FpmO=)sSK3W>8q+y|nxtoucQ*67m-$??x#7s4?y!?SRi<vo+#T^X<V%fBGuIS|jNLx12>x&PEfe=AgIu)X{>?pFNyU1BVEy$xEA!KBO*3ts_soJDkJ<$aoSPq)YhnEeq2Qa(>n|Enkh64I`fzA&AQl;l4ip)?~SWIX))OlV|hui|@YuX4SE)liL7In8n5P<7(`=&*Ggu{{pzL$qy30H&t+_xX;TLv!n&TD{}Sb*%|T2gbP+{h;3)sZM+T8=)xY4<#wWolC8}+=aJdfo5cmJ19R`NRe{$EvfleXNiG+Ozl9xq#Ti-W#a<yMfiyCgWlDd3`Sw4*y?*uI%b)xW<pj~cw1I`>98*0MK&!=Cl)~)80f!|253F#b5Loa(P)h>@6aWAK2mk;8ApohnF1_Xj006@e000*N004Alb98erbY*jNUt(`>baQlaVQ?;RdA(OnkJ~m7z57?NJjtn-jyE5N1$+pa1PEHBi^R>vH3(XwY}OK~l9YFyApgBHq$EpT?e^nPA9iJGhMakC=5d@`Q<6&feWA8Qm6frc+``vM^XyJ;lB113RnoN<pW%NYKZ;ViN~ChqT3078oo^h3<F?jbm6C0Z@y=UOwHs8{<f@QdWfCm>X-1BsP?_+OOMO(<=n`H`CX-C&1o!MOcV0YG&MUpEmD@|l)2ilaoe3^<#+|8cDtU8I;>(I-n$w#JfsciN$TDP_a2`V8Q)Z<z#Ze|xRtXD7xb<Qa%wcD+z6?5{uuf-Tvl)3W3MZ$*eD{d?u4lfobz?1@lK=^x@xu~*FILwSVBH%{H)Oda^aWcyXN%XgZ?!`!Z*016jIP;BwiuWcQYXR8%0FvmB?9T_W{T$5!SGJ`eZwHLLOJP@FQTaBJ8O+?aKq2Ty!=BLdd!K`*P;mQbgrQaT+-~Sa;!6Ul-4Puxzf2=-&}7Hj^xFRES~pIj{&-asuH@|OOna70JbvnlPLt$u=dhcR>EJR9Jvk91<YuK+saUZ%h8HUFb;Vy)7pz|A!j5|Jm4w%11Tash@Aus0eUO6%Z)7|gNA{Z?8Pf~O~*VNqL`}+<rDfNovxn`XzPKO3&1Pm>0T8Xm&gQHjvtj%5MC(s3Q==jW?hZXNNMyzp5Qr!FR7>?s}|~}t?MmtkxJ(#VcFEcA%c>0K+`p#m=|id_nX!&M2?3W{kx`T)0G7k*a9FVDUHKSQvefAE(%fz=chLV^ABOSP**o3SNFcQlDK_f5p>83dqVC?<s5W(>pG30A~c9flA5vv0A}Rbv%{URJJ+j{wtJ!pv~hs<s-~9~(Q{3m&j5qS-5DHq`{DN86D~l=1DDUMx4-_j>YShwYpX96Gl|Qo$>ee|Bk}ZQWTDBk$02jl3-2v-+L)-a-s$iXS`qPyMO&C1u-aAznG>2>b(Hf=iJb<jRO;rv`k!~4Gw`I3=|Y#?rx;7163PyPUDCg0wBZu~C84Cxyp8rGodb!`YT10N^t=&6Xw4|65NF732@iut1#A*r14#(a;V5=QNgLRK4Jg0JxR>`n34$1j=o>W+0$as)3LdZso5FPg5B;W%Qr7^<L6v0TTFnbR3;khi25V=ry1b$OS516aB_nn4AB1y^ola^la34@N*bGsB$jGDmME|9I)U$u$RpDaxh~?P5pzE?aH5bByUB6~82jUJ}<DijoT~sLRR@SgTH#r=rqq=86TI%stGtfZ$<~}yysNPS8LT}ta_V=|{MSn#X{t&)DnAwE&Du=tf8~TJdrAql9&gdET_VugRVR9OG9~t>VvY6|*M|+X!RJJ~lHBc0Nf>Q*co!ksmNi0FOlb{nHe_8#^-@aMBL3NmGqvd_r(9`L3OjE&m=!7y6k7yjA3pGrN%_262HwUTJSBQ09)%xJsS5?s|1NOq4`B+!phU|rhO1m<58d?+2Z7)AZ02xwiaRlMUl5hI5gT#-Ccx=n0XEcZpqACxWvPm<{F3%(9Uf{qCdw%;KxiFS!xSO!D6B_bD2*U%19;CgrtSlgnBa)E!(fEcNd=GGRwz1w=YN}H}r-QNGWJU(jx69LRhZyO)eF3f1FTLamZ=69?g}{9~PaZDa%TmxAf;Z_El-1R7L1*Xp*beX(|1r*6@#Q^>wh<ket;e9+^o(PuU>VbhZzANqZn|KwJAvU#9IRs|tyIHQbx{q+at^7qD&4hAZ{Jz;>yJ>zZdC>&eDVy(nb9dK-1g*6&^N##-16tUKV<ktf*vU-=Pqhe)-VBL*EGL0h?f4&*aHzMjwr-vc!P6Fx6>074>AGpX3#T*5gE=lZQ#=<ofzCu0*L{Pz^hwgxUjz24DOqrHYE`#l<Wrg*kuv-5dlEb8xgp(3U0+M_I<LY>0W3UTAEfKnW9FAr}x%;l^WFHwu6)oetca<I{6DwO9KQH000080000X08PUhVPFUV0Jj|g01^NI0CZ(@baO9sWpi|2WMOn+E^v9hS>2D@HWGjLUm@_}AQg&s*Ucrlphl7GwnczL8x*}KpFz+PWiu;_DoJJI8vfsJhSV2j*}Jz*>xW&5<P3-N@tYY+>?)~D@JY*ZmMggnDrUBYSO>yQe1z9o^KAI8%0<E3_qr6uaBXIe(lAX`T^h0D2~$SoEH%32^R5~AK`E)?fEl?HDaCEQm*SLF#tdt@gtGvC>sk~UO=T+Sj+G(@5p-;43du6czZkAcRyc-kqLLM-V#hY{H)|GGidRf=#|U?E!-@ls0qJM1M85BKa$JB#G@H%Rf@vMx0qt)nMA&L1!Pi@+dGyxu$ao%TZvIn+niqL9h(CNyP_I%kwGS4Zj)6lncqbH3ja2(E^2wBx1}M-%q0c8N*^ZA^v$B+i86iszM2BNHn@bhwst_g)KsA9VgD@mUF;UL1Z(fr)P^{eW@Fos!ZlWj-Lh_jv{?s*2&S7?NGW<DJ!FEe;7=MS85&RL6k+&cQ2h-<*t9W`9Dv074VR23l{rb1pH}mY6e5y9&4C#Z&%ofProSENpvDq3}@alBI@ni&b*YA$H0~LC!R<rT?dr@Xq%Oqu$wFfAYh*BF?rd*Se4z2rO2!PRuQz)v)IblEsW`*J9>QqLNyDb-spx^QB<mqvrrN?qLX}r#ejlEF*CbpW_W+Qkb?}C(>gdCl+rgSu|EBx_7^Gl)vXTFh2@J7-`AaF<0xCbsF$JN(4?%A1v@5YZT8tUot(&lh`+MMa~>h)TFK@Y%CjDv6EV7_{U0<e!wJGw8iPL+H>5w6d#Jf59-hSf0zfgyU3y9$VwwZiCtXNY)o=p#K?*n)5euqG#523aLUj<L0Obh2)Yj>|EB>lMhs2UIQou-JAW??5+maJl#J9~4h+;r~nce~0hKS(gC}XX;Kql$Vo0k`|nmbp>$WQQC}W$>QLfZ|)zM+UNlT#Z9fs$-EfOE_<*;dJyzP4INm}J{e576lR0V<ZZeb5g+|m-`b|m_9F;S<rYc+YSq5zE#NO=btVj`0BVQD+@EUo){jO*(?Dh~WO~n0fw+;Tnb4o4C=GoOW=kDsU9TaJQ&wV`kruVa6OQevxULJ+da$#dCrVGcSi0nS4bj>U(@M$Dyu^I4I5e)0SjNAAL#D#KKyk@&aF7n{GwYHFerOXDiZYkW4s`(GA{&vo0P;OdVrV*n!DQ~_WR(bTM;%7TQ4OIrSLR0ssO%pII4{BdSHVBr<1IYe)Yv>X!*_$>_ulZtQ3gByrfMGArPc~S`Q4%;ouv5T_nH-9!AnP`_zqlHlp9zM+I|zf0F_S?{w&Rh(p5B|z61(7+3)~?<L*Rpw|(<8+@WU-zzOE)E6T9dbQhmwwFLZFMa%2e`Evl9CHcMPYLEMoI}VU6);us<p<OAEkn@VO)OHmZ7OD)eFnV;@R1;TScwdodk_#w=)a88%(1kT$=PYF|Hnp<xj>oWkKm!zfR~4+fXd2IbT9&SqGW#)9oQtE=Sej!j-TawYvQ9WH$`~&+2efb;sZ@Q3kU_%$eFC&F{1cS!JbMa2AhX5mUSnvYq1t|BLUSE@eblH_9CLfqbF)ptr1ODWN6CYNfTspJr;r?u|1HT^mE-kTaWc;fQF7`!FzY~QrmThxPxX!!1s$Xki?-PQu{D(aif@EA&<&mF)Qw}V9yfO3`ApNS&AxK7>BXnmtTa3Zhw_(a5L&i(bs@R15bG}i{cOO1dhxR?c|47m*4eCEnXuKz0M#wh_{pH7w!sUe2vZ4)GO8RJ2WokGMf0-nUWT<olvv=iUKL#ic!voGD;dTfAiS*`@f{PC<+^O!pVcw$PJ4-64}7#^iHO`Y^o}vfwe`CTstG3@>IxkX4Dq6kHdm(pA4-|Xv#i{Q))H8bMhQR+TWeX=hKDY%+tL}%*NQzlVgj{T4bdA#XF;vIJ;8hfjTOD;yrMe*4pBk)!Kijo39hZ7if}GFj>f4H9`9hR5{kZ9A71_VN3|upH}d35gjU9jjZ>2mArsvGa{EhDq`sur?eAv(?WKDJG$0ax(2#|>;sL`p_xp3p3%2H1CX+q*@5IeQ{JgWx|E{xTa`X7vO8NNFR#VW`<->{lyFu>;(X_B$1XYM_%@zbQ2_$h=yikxN<c>U247M2%P(a&KaDIWnzcUP%C<xLlL4B(W5!-G8h>hu=Zh4W(+6>Y5Pr7}7%h4H{IfM4bbLHZcV@*)6pF}-ryJyq~Pp~rEt+^s&Y|54VgM;gL*=ATJr1piTTfCE`YrcgZ-XT=eiFO5A=-JL4Y9OMlx?9qje<3p{3fh);qAVbTy5KZM$+$F_Xr4zDqv`+%Ch7yy76>rymk51rds-NssfD0fZnQA)*-pL|*u8dtvILloZZHQav?m^B#AtTmat?Z=rFSca7c=vzpc-%=U;8}6U!SEm-*X*|*iQ)}ZH&)EVYBE?Xl>gjMWL}{6+UA)oZKrsL-gs77vb{I$9}WDMS$Zf?sGP1j;Au?t{RR00x?doX((5cTmfooXv7{9&F|eF*L=F=={<l)Y5?YGCU(5kU8AvOddp8#4@IwkBKw)@<A;B@U#33bHnTxf$}1eiIjfL~4oTmO3cnMD<fAAGtV?wOUK`j*DhhC@Ugws5Qu2q8{QA>-@U-wv9K3>LM9}e~M6=J__eysyfSQ}gvrn<d&=x(jc=!or__@Th&r_=X{vH8w5~y#Un7kcowvCLH3SRV<&HwqzCiPb@|8{4cRM5YkprFQOXCe=%{iH>y{Zd0I-X~Goe5vU^EaE6M4*vsCO9KQH000080000X0GzOH{`LX@06GT%02u%P0CZ(@baO9sWpi|2WNBe%Zf|pRX=8IPaCwbYO^@0z5WVMDSUEXT6qRMQP^3Lo>ZymSSA;AR4_S8;JGC7a7V+OZwi7Vj4>dwbJehfK{AR|;uGhv9>A*OxTbpGvTGY_>N5U;peYP{YBmJ?~wGK{X^y#>L_AdSd^NTFH!l4HvJ5bJ6j$5{0RIummz@5};3jX%QrTGB|S!?xSn013VG9h`Rv~{w+PzSmaGjR122*#DtF2FOXtmCQ%8$msg8I(iZ3gK($(<FXgEWf}Gt;@1Zz>YYuF2xKqQZ-CkrWG)!!-Ffq;83@N0LyBlUxW>|K|L$|qFBqf&{|roGJ;ocK*&0<KfWQR!Np&yuU^4uOX;#C%c3M(l)Nb4`98<*AGvU2_>r#Hvy$NN9)FvX{Ihx}$wP$L;1Yv&t>$iicZ-A@wRE1?PI+&1uPw?RH<+);I<L9QOENw5_BYfn-<)Gu5$;z(Eh?~10Hr{T`xp+5Hb)jLvjv*ATYc~hF?Zbad_qlPA3l%0`z&_x-@Is*Hq_ZWy+!)^7HOFvMZGbXyD5uJxW^{;ntayI*ABThQkiO6lGALlno2NR;;Yb2elBlDjTUJ|ti`z}Yl`#Vt@A4|(h4}_xa0UGO-SUMg6y;jpNLdb{u}a`R+WCiHcQ%uqKXeiy6=&_1w~Kq(!9rVE<BBo84X1`v*9G_b$e`35L|V*C2NDCek>D%X@NCzkF)#)%<>Mxi89TqKZ<Gx0&8-ew?!N_Gz?O?WpU%Vo~^Kf@rE*xSseJ7hwt_4<c;$eUp7&QYg*`gIZP|Sd<9TH%@&n#hO0Q%6K7G3;ds8?j+ED@mDbV5=%FuSy;w=Y(yMtn@T*|jUO_9v_s~*p1enMCdp?rHOj8Mw@RVEm3ck0!Hhw;0E@+7_=Md$|Ej0dQAO8Xrp`$uwvitG|e=LQ+1b8JpW2_SLCUD4>F0tW5r7qE9pN$}uaK|2e6Zum!J?VT0;UQl|e-^Z6@MqvH9b?N&xr^*CP)h>@6aWAK2mk;8Apmez%TV(N003zj000;O004Alb98erbY*jNUuAY-Y;|FDX>V>WaCxm+ZExc?68`RA!ScSyIIU{u+Rg3)2AZY;dbn#+>}?Ai3`1Zf+GZn*DoG{lUF5&t3|}P5mh*yM5I7b!GvvHIGs93yFK;ey$Xa|ht>T)nx+HSF7KLCH`Ss@4EAnS6)SXU=s7u~(_^1s~kPWZ7Vn)cC6td(QrF1e8yGANQ{?M}S9-6)3+Ds;EC3l2>ZnzRVkfAffbbf!H@q1RaPLStFCIo&iw@j#i^UtD?b?R1RA?@mwW0A_Ov&}~F4KqA1Sc8J{9XCo8+P|*Erp_DL@SN59fh)fr)O4>J#Y<5jw|m>bKD^9}s?~<8{8mb%jbe>^s|%^PWlSg2$z<|_(=zjROyGUP7j!4fw&HX;DfyaIPM2i*!HQ5EDC>cm$%rV+%7em**)@5?%r*_;yfjg3!|DP&xFBEc3jR>BV;ipN2eSN1MG1euJ9|ut|9(DuT%|p20T$dyB^-UXjz6AZo)TKK9fz+J{0LsAkLf7kyNI%sn-gSj&myt{4G|N@XJd6OjSZ5~!=aYB*Dvif*}J9r@#as{zk7<|e0Id}_U!o7kM{F#<O2M>=8A)B-eA~+2D~J<qSzM|Cqjc8H^P8?!^v7Ia{2ScKR=S&){tL*`_CtGee=l+67{-5BH1dUdBYSw1h2p8?L)%ZAT949LpPj2%F3OnF@7>D&8Nv<h?Up+!|sP(z924eEYLt?kXx=r$q5TK$><BLABnSJL(mMR273^*<wQxj%jhYmbU8%61*JpY<d9-#v-x?3;6<m|gAxEJhJQ8*u4HAqYjo0)Pp8MSf>~cCDo}e#0S=BZE<@nVdxPhtP)Vm*j|ag@da*>KpU+kuMlAea%Y_(&n1UdaTkGYJ^c#o|K*FtJ8ey``wWv33#gzEsb`2FG8RCv#hDh@30k=RjFYXej9MhTuu=`g_+TcP-S7fmu??x#KR?AuxaHa^B9Q`b22)hR4H!^>as?-U++={A%#3K*P6kALS9fD40bh?~dX}u)_q74x-encQkPuZ~55JO$A=koRsUKnjn38e+jdJpeC0_3~4GNP%V{Fr^xfx9=Utu0{CZauRriA|*oJbwy`NC<T0A38ii{TQm6(G*aEmy6lJ0annGy2FLjc)NZ#LTFUW56J#uH}UV+tkQ=B^AR}~)x!H&fPEp{i&=K|{RnS|wE@op^(ZmYVA(zhn!%<%mqy6WVv*Eal_IPJ`PSDPGvFL&f;}R}031iWQ6TL^p`;Hdkqe!Jh@3g4bBNBay+Q0h3Y$vSo9Ch?2HyyAhq0o!*n}NRve<hu8eO3Q1Fn<JShB<urcV575SAU!(*bcvrbi=TOxQP&=(u~d^mliVeq{;}eiplf?;5l3f;(q`Y3L9z;Tp52S9Hs&sx7d<qPqMZ>KS<Yb$R`13%vmRf>$_ejaau8FhGwDQ1u!cTal~jsT>TJea>KTl56NxPLO@!5T_nQ;+Px(fQai<B3tqY0X5WjzKRI|)8uzB6Teal+I+G@3|~;|66)>wWb2<SR|Al4>$_S$)VZsYh{`*mvBpK19o5)16IJiPv@5fI>ygv`i|_4Oqr)9K`u+R&As2&JU#9jgfPL5Nk&Ek}ViO#949PO?O4=GqQ|WpUXbGHM!jQ3J>Mp}((wglHcjb_}IVr_ytpJ?#Rj+UJ5_&CDY_$}3L2FgQW?s^18j*bpB#4Hm$^OfA1_5UqLe?~#DQ)TY0FeX2>zmQ#wgwzyYV_UcVlBTpSYP2F*M-Z$YzXH8_`T!2(S5U2gjT76?Ts5@b2lwJGkPnMV8)5}noo^W^Jn7trkETZYZS_sT-4xgQ9|_U2G9w5i&9#6Dus3y%x&M;ZDK=E_rqs_T@qkP@6@lfVz<1)wi@k{hA_Yda8T;}ht9kgTHGGA>Jc}1cfgFXQcLIjmvQ{&{CSG}=y(`i{!oEsHNf0{UQfL0DY=K{mepoKSqXTg@u{K%<V->v)L`a>hdO##$8Xj(RuOC}VNmwJLwTJtp%W>JR<UC5kkBteBov?vrfakci%@0;OY8Zt0FI&l9=p;Gn!9Gi2T%7Rh<F(uRlMGq?Er_5?pOeTQ=B9RPJ3Lv2KE4<?YCz(8cd6@<DV^v#a!&}Ec!rkhe(gfum&F-wUaw2B>rBgyAOg4Gz_pg$A&0e7rDOQUXS^Lbkq&p0(w-U<|#B(bty_$*{#vs&UhPuGO>wV&TKTDt$ITQ{tU)SUY7?}7(9*DBHtFq%^ZGA(`Pw-8|b0@6lZL`@U2tyubugsou4?fZLME(YCuew)W9#NZUd7MH>SV_0BrPP{{77FH9sFDr<1#+C9e$2J5K0HpjhWYrc2E?^D)js<duAI6w%sNY_?Cag$5|ZsULgnK+w~8Z9P&tnzzT|aQXjbn3o|!Db}`GGchE*<_OCVZ|1APIbO5<Pc6>u2%4|i9!8C2_`aO2R;Mg7fEiea=_Kzj$D>;WoG*-k*~Eg3b5h*AYT2p+G&qEuE}HxF@ZWf!ipyuN??MX)u(p**La%1{jNY`Dt*hf&6e6n&C@{k^NU+g`LZ0WadmPUgffxibuqp#;8u#*|!b!WI!F@z8qpQ#HTfnH12;)6=8W+F(c=eH7-u!;`(cTS&{q0zTHQL`ppcU;u0$|MlA5cpJ1QY-O00;m803iT3?;M7&1^@s;6951f0001VWpi|MFLY&dbYE_7bY)_1Z)+}adBs?5Z`(Ey{_bBva33lUvep#qFa!`VH*uGsi4)jPhru%lTB2<=5-E_BlPt)8-|<DVoJ48bepo)Z67Pk_yXPLBniZvx8d0TtnS12Rm5>E(-pf19K)RBmKw5}gMawG7Q<8|BE(sT!Erqy7Ufasr%C!PXS)Of~ysI@o45uS<ba8%uI_vd%DO(}453vrIB$6qSZ<!+NOvB5HXhBxlEi)SzMWzE?l%%A36QV-!Yjy|E$)<nUL-6Of_@V^2?=~KzoM+2gcnPm<WmPRpDH5jC&gARoEojDj)<6oOQEc4?1VYUjA^pf7Bj`sZm5ggOKUi4xpU_Yy8^{RKl5%C8Wy<D&rARSQr<&z=5&FbfiS!19$%ZA@XeE+LfsWuvGMamDe`b0BX+n0xN@s$HDC0?9r5Rtt6L?@osD%XEB~?%k&YVk8>BzciWinDsQ>|rS1?CTTX9EtGT+=lpCTqSxIBYC)S}ap)GZZ2xJw#AUT}ei~Y2eoT&Z=*JQwTTX)EovfTNfPtf^f5OR^IU?ens%=PDNC%w>U&|>_G<4crd8LMfhYiJa%i4k+L#t3Y@gU=r3B~nc4^LaZB@xL5v1*Q7&kz+E#!9ni9=!b<lVE5FZHF3QV|JZsJT>p3kuol}s33K&_=+G6l4I%&~b@Aj(!=+58fHR~r2|MmP~E!>uqmEtrBVoA~Y@>=L_WNl?cS?lk&dTC&8TFm5$PWRXH#Vs`P)@!I81vh4sct5}oB1=RW`ja}8v;p3UCm<ud|yk@0O9#x7N;;EyU<H=|`8cvQ*JU_UD)nR$I=)jL70Lehc(S57_5Pw1OA;6fbK`FB>wAa8Tdaylcp%t6vs6l+#=BbvBGu&yphn9jN00n~khql#*zI*?h38K9pbwogxzrSyZ9@?zfu&6r*p-^%Q1QS7NxQ6l+v`7KP2NC>|v?}u~u>cHwT56bD{0Ep~^fUVPEuh?;#-}oVoZ4E^gSX+pxVbDM+xTgg=)msb6>BN1Whvx4s7-wr*lkS{%w1m@xMw_Nw_$^N#;bzKwryk&?oVV1+;u|KG{UY8c1jLtI<goItiIbmM5a^*eDMlfS{oSY;NAxEfctwce8>?kOUBc-AE)<40IQnL!T)d^nYX~3>|q@zIXt1WoNghva58gLXS=6nfevg>ashp&UQm7G`>-Z-$73n*V6wq)I&3cgo}FBbPlnTz>1cEu<F{R!m~mwg!T2_lf){X;;SNoI$B=zY1=EYG$<c^h4v#(!{~UE0UrtUx4`(Ct;c9$*X0&G~FP&LdoRk}A*T9_{_uqV76CU^VT79-=+=4A7H9Hb&8X*U&Col<Z1SdBc{8MRv1-dz)qrsN1ZLsBDgPFU(V4BJQ^?vZo66~S@NbOz>z>FU5Ke<3&N$B;dJ8TT`o}G?Idp7N<@p-dLfb+<2Kr1b9hu(B%aOm__xwz@ngU*;by*X6!AVlvLU8lSnlaHt4;Tf5X&WESt<I%(vLZOc|SFFQmcs08q)03;&@x@<br{3I?PqmLOK9434IGJ6X!Nb_J)TN8=UGx^9{D+&@BM&hPZMCU%DsFgljRc(|GmJ^C@fSd&YRS1stDMbXDMV9P%h|XNtM~X?@bFJw3;$#f{t4V}txLi78NBWS2=H2nb<QFy@(gXCqg0xaI({5su*xg7v6p1U3sG?$yP@>Z?N;+SwtBeVb_sl;v*J0Y6^8<4Yu@2YvYY|q)ZQM=r<rC&V5ZW-p35}X-l;;jeP@q;A4$|j`VOA>K~nG|Ssg%29=g1>dALEWtp|Js+Pab@TbZlHB^KD3_}T<d5bjaky_*?s9~<@R<e7{<PSkvGu-GN_X-14%XhuBe)D96}=SUw5fZapSj?xY{i(NNF2`tIr5g4A5N8F;C7o$rvMVC2Cza_#&i(U#U%~zkTyUhNDmV^<?JDeN-3SA_N{;sGi3X!CwgwNDGTW_?lBUIOy(e@{ZVPEs*Z}fpC9Il@fD@1<B-admFN4*~{{tNBYEWg-R?co&L!GkDGrTwb1!_PBL`#t~F3o{=aJ+Nl2Id_j;oUoBenkj}RE{`KAg$!_$==xg;xe8#vQInOZD(!UMc-w<j?DGr>U#;K^%rv6iJ+`A%h??0dHvU6ff!LN|M;>Nq^8b|-*;rN34mO03Y2d~9g_w7Fcn;W3YtWc~Yx@OrKS3H_Pa4rw(PsfMKsBnh0p9-sP)h>@6aWAK2mk;8ApmSe<W}+o008F>0018V004Alb98erbY*jNUv6)7Wnyn{YhPq#Y-x67a(OOrd8Jq1Z`(Eye%D{Y@kz=Qc`4d%2w-6CmSKGeI&9q@W*7u5(KZ`N6iCWR8svZ9ok+@}<0Q@M7n|hWcR%>u9W9$qX-ihzh_|m3=~i2AwzXV^rBvb9XQSF=tyM!hZc&8zgFj#ygiOEcv?_%$VdsnN*0L5!Sa-Kp7@IgGt5j`@R#~x9>WWpOmOG(u19F-W_`HAj*9H6K<K^XtPx-LHI$mD!jbI3D+}^kau&h)~Bdt5oU`=VkjOukMf@+>*Ns|2RoGI4w#`anyl~|KlF3rvzic*%$hfq^C*kvp$09(~N5Gp#ZMQhFCZ0U`bLUyPzD9#8iEH|ePyTDA+K~AoNXd!N-u_oBVkCO%rlrL(<E0YFkEqKMOxUp#lfdmcozu{M6ih`d~wHDn2th<&s;+&-3FI;5~ZH2i=GqP5C*e0^|E-dJfXu7<QP_=-hwFkG623z^4>@(^vg)S%ldIaZkKIAGNR*CZ(6~N~`sfuzd$}6UN+x0dbaTas~$T^`}>`vh-jKIb37w>+h%bb+g)!ad2!sefJ@9?w2SS`EMJ9I30-oYc}N*$tGc!bBpNd{^m*SxwF)d3@!G{Ao=bR%0}f-D(tE9Tj2*tN7<2%GhtBSNxg$jE78n~q^|CO(AmVDAd1GtwFaxiJE3(`}LY+y~+A0do$vbJ7WIz&E&FfC<Ks-9d!r(RQVwoatdj94*Ln^!1oIAd}TY%Cl_hut$BsF^s?iJ~*4|r@<I{D{jD`Y>>P&s8+W7ab|2^-4@*~%?h|4BzPLxht3sapq{J*FPaK;LQO{|_x1f8t-p@i`+UXpjwv2?ATg7E>gwQS?u{N*a=pgy(^G60r>d=QpYSPifzpSTYtg1P<SR`lG(&EwN<c0*wOXb0q=0!svn)fDvq9Z*y%9*P!+X$uYvDR)(2>@1gSR-5FRrz;!nIH;p=D{^cTklCx?j2F?FN3<s^qmV7wUXfHP&RH@}x0GEO^lDv$$a&AdUTX5phthV7fp@K+l2EgmzkjcStW6$)oI@YGzTS;e-Izj>C5p-E%=qkcQB!zUfRliR7?4zpc{ojy>>`MfZ5<PZywn>J(<nU%h#odSXn6VODI#O(i!FNL)i>a~W^$IOCR{=g|NdsCL-?*Id?yt;8BLR@SN&Olj5t`D<oiuyp^c<+Dfny)L)Vn#m*(zr|KRz42a$GMJN;o>6?Oc$viyOY$A?5jI6_1fj2JM!p|U&(57bjx7J&D>Rn|68aCXdvT$)(m0K5xt+VwVfuzfWA`X=qQ-c4ax%smIe#7;zt1Cu<MZlK5jVTy^O2hS?T6#wsGSKF_s`CT>nnUYLidC&{ly13PzpP$H@=`nF%zcb-9T!Hb_R$B+JImF3B%y>0w+!$H$KpdRG0Yb9eH@bKP^6s(i&zphbOq<_3-|{{|2>e#Sqamua2Aq&tFDHH&@_Hs^UFn=EFx{Y!$a0_E&y{6aP-9dwThQ7sbD_`wgeF!-70q10}f$oZM5(Z@5;5_ZfW72W!r*XmRKu^BrCRePM^x9bZJDGVVs;Jsrm_-yF`#T`+Qf*mDM((<U<^Ep;;Vfs%JG(wnHo#o+$k6|it>d+kaLYp;6$f|~uiHGC~zTCqNB`PaW3C&S<y3hqBpO9KQH000080000X0NGFY0b2$D0A&^c02u%P0CZ(@baO9sWpi|2aAk6HX>)XCZewLGaCxm*OK;;g5Wf3Y5bR;)g(G*p+a3b=5cB~QEfzWC)F2>eiMAO_q)Jjwyn+9Fhc8)@Wyfwe7e^xJ!Eff98EUCyi7IN=1z#XuR#F*s509hpf3-7AoA@N^vf3f4k*G#XD`}`EsuqTqETziuCC!YEdhWuD-;|Q-cz?x|=Griku_%#@5d7W81@pD~_6yC{Y~tQ4@I)#`3YM>!^6#p8QE<KPyi<(kgo{kE5_HglXhAEzmS!><k4B@MEs;GG(QB%5mXpkyAi7ptzGa#$!Oog&OF~6X6nkcHB$7#`>dJ5_5>r;hA~wGD9yH#fIf`p&ig2z#9d>(`Txq*}e6pl1u`NI0NlW1bEsGj_VOyql^V#^IE3ImiykTC_6~nja(|&^9a7pzB9z>P09ZSAUP;Pb=n=hm+#%{xs7R7>QcGK^)&}@HN<E*Dq3Z@gQ^UHXGyhblfwPp~xf{6qIVUmOQ>8g;61b<8oRd_reqxl?tax$7_QskH5G3($FEGk8JdV)%3Xig0UJzinDVqH)=9>(pD2N<3Z*B%MC5sqo|f{Rt^Wr#CH*grKb5^wg=WAZc^hS^(mi3M&6_Id`8!`Adgal^>MaxGclg_0X4u-{<TTwMg!=K-i*IIMaKr;2h7>N}(iDcsoUAvoK2_!9XSIYCLzmrJI=oFz4pD$;~x>soC3OyN1~<Yot_dBZy6miaDDZq+jw*19wwAjmPvCXMcKOINBOEmVHQlZ-kd!!y1^88s>1pBM9FU=}ee<kwk_;k7o@)OsEk#{)Gf38iVk?Qy%T<?kJRcG1y8wnwFDCd;Z|hUKT3(d_jH(}|;M2Ml=EmJsqC3$y*z33%r0FG`BliPN_;DOz>e7x|YA;2xM5&UW)Leb$f<<Cb)fk|D40_dHe9`Ce383_S47xd(is0%AcVGzjl-$UW$4Yi=#?t|MV=RA=0FTP{k)rp%ww=J#Br&bO&RP7DnRpCxmTg--f32}1`tt38*iM*9^h&~wCF8up|W>h}ENG{D)w6Ov09QEbc=^AVSYol<mN7~PNc|0}7$=74{zWc<g<8uRMP(&(tDLC%d9S2MXan|MvF8AK|+;sU^R6rUWPOFNu`$)pLOmwo*4DRq;juL<MHz`E@Q$rygT79zCYbya)e0+?+p!|4ulUen2>y%PNZ-+Mq<OuJ|{0x&_JMW-X`Ubf(hQ@hLG6BE6s^-;9r@?%U4{N&vH{9PLEATciNyAG0zT4(Dx#|^r5nqE-j-gm>=OzhXt%GsTDS`v)Zn%Pk~TZ3pE5;d_EQr3pROiGr#uq&lB_J`(2mFkkq;i48fs0juqP3)ovetqO~N7gE1y$hLtiCY`ea+7lvW}74kSroj=o01B?v?bmjKMV!Njuju?;x2p(J@M7lisAi-BULV}`UV;_Uxpe!tj%Ji@$K9Gq!JK0;jdqx()77nHHzcx;udeBi)Uq1?<Qp0po@GZY--_KqiPo2j2Hpf-3SF67Y)8$;!Ujv+`032gPaV}va16(2?A7zK6dL`WB(Yzetqks*dK;Z>N-M{y+T^|#_7v(v%bW`v{@*Ms?R{OVXV@YZUtdt<!Xen7fb$)4t>W6ER(wZBnq>+n4r-xI5RAB^o58TS@MinXeRb5Jp1WqS1WoXwjZPtsZ%Dl9IgWHmWsdr_3$V8_3q&gB8{g4Cbn09^^M)1FMVhZsUD6Go=+=iN`@G^NWbx_6_(c6=7&4I;Z=g~6|1OnxZTJX0dcnG(cgdEqmUBzQ$j81mhyrwikRX*pa)A0x}sdbuURt$=dWpj+()MzH7+60zJ?#OrUFr<cLE*_&rl)drmj*y56Oxj@#S#o$L-MRb!~XzkxYVM2ZN@Zz{x;Z&mVx*VNkHD=V(Z(hg|rGVNF3dt9CYemmofIHUbL{S<sp_8rzUoPM`J^I?P>ge{m@TP4n%vrQ3F5CC=)8-yX|aX&Z3r*ODnL^=6{u4bveHrh$3WOcsUo<zJ_;(60oQVp}e2JL_7`rM4EN*1+TKlm8B+c8A<N+5%#a+Nfm6@2A5!77CV?eu2gw>j6vrxQ;Y8I`viN5j$eRTl-@d>~D&n!aNTXTk(!Z@l2zC0Z>Z=1QY-O00;m803iU_uLBL71^@uM6#xJh0001VWpi|MFLY&dbYF0CWpH6~Wn?aJd97IMZ`(Ey|L(tn&@e#iXf4lmO<`c|p4qZC$R1yefS@JH=Azd`+Ob>of8X&XQMTo{YlFlI$-Cp-@19H$1b=h(hy=@7B6(G!x?zGfD<+U!Ff^lZ!b^tctSXpnHi#xoB?N-SqDoOxm9nXF5eGpq8f8sYpp;6Q<WvX{!Hc?TBx+cl(}a!eazVu+=QBetH+4t&2mH?28(J_?D+b3=E{htj7O1QZ*X{L<<GC*AycJz<BoorDWYr`Kdrd=2kyTB>n)bHuZ?11O_ia-pOo*znCa`tQ8eXtcida%XR?lN&Z_65v5PZ#+JWq+k0PDR4ZHlVo?^sG4c=p7IAhQkODIP_GSAhp=+OV{Lm*vhFmCMG7w4`nqZj{NQf5er6hpt<J|699&gOWl?0sR58v?^dJ9x%$;oF*Gm&HjNvA&SWY3sRb8DM4tQvZP9RIq&?QR82|*W9cq2D3Y*R$DAd<ZFlaNSmtt<$NqK<H#uF=JF_<$jdawZTfSzw-MrFJKy3acn2u+f$R*9S=QTpc@u2ccP*ze7@vqp0iZvBU$l+}QTmx$eCcWEZvuh|)mLWoT$t58ngg_P<J=B+4#@Z*Oc2Nhic0)02Su`62)Z~327GO50MOZp#!4*0``+yN7WKAr=g7Nu6>JyBDN1n=sA#0^YosP~nts_QSE8*!1h%B4Z?#Ee9rDSDz7R5T5>PoQi6r(ecIQq^cUYzl@5;;=hjg<)`K`4uw)Ko64@C`<5FuJT`UZx!E4a18U7(G9YB7Lqofj}b?t0<sNLpL2Hz$u{0JcQvIJwxZ{OLTVm4MrPST)G7ueT6oWTEh0(<z<A?_%w>CP$ee>5lddmZ_SP7Q>h5oI&IO{DA20JbvX|@&0;}UEY$Xx)Pw{v=DcH299v^BnpIWa(M(@lonHcAmsb}r?t$UYX30h#&`P3p1J(=xtOC-dvWBul6{BYuv7(ln$xk#_>x}043}A6<m3WgA3;K?9(4Z4$tjmnNE(EqrBRPLyq1v>7bvZoQN#C$11@~6dH&w~*vF}iQa~PgrbOFf0MwVApo}Wc~s7?k5-Uqf!IV2$X-Jc(@=V@0+CcSbIMt<o`;j~vpJA*8Hcd-VyTeV1Zdt0q76ntJ#D6v2_If+j$JS%M7)!tJdJ;hC^I0af=x8ueoBxKG*3=~v#Z>SMr&_aXJ)W(Sl#QR9cnyxMlD}g>Gpa;o#E?GmWEbICiaYF)eV?i%0-3>PI_UKCB+e23ec5A*b%7nKQ=ta2u!-K}S9>n^MHVl3|4IFgZDj=})NN6|!x^xoQ)3GzsGL@A90J!?y-ZDqill!f{52X8sfH0b#Zj+K>@f}fng{#*da=pfXAU>xKdoSmv&U|Y+ZjHsBig=&8_kKv05W^4(W-{pDZ@Wu7vLFXp@b>ku<6i<}m6{aYtQNC>p*@E^iTHp(t9d6e`ec#L#BG@gEmts!9ATyJxu+~`ozWAIkCIMyRrS2`1-he;x*Di~Y{p>rg>jxIOZWlY)VzRFo@Htt)^iqBqEl1xp`NR0$td;G%A+pBbW)Caxq>mWf+vB?MFiEP0@(%3QNjub5+hy<pF!F^#6B2hb<j^eSR&gG%>=ZrbDqG6rDf_;W$UI(p{`1SExg%Rt89f*MHXCo`h2Kf^}FU^KW;1baRo_hK2VPI+1S_(m0H=v)^cJUjorh`5)ykJ_!nC|hU%R@9Gfj)phdA^bLd}IpciyILfa4?EPA~QjYo$s^g{UK>z!Zcy85Oe!hKZmu1$XS>`B$(l+^OOB*d-a)8O^ZtM7kvCB%eE%IE4~F>%k}SS%<EtZE>o!pymv5XS6krkZ#dMe%~I4WAuR8v5uz&p_JH%WY^1gX;_xyxKAf;~HtodH>PjJ>Nn-hx9Cf_k}@pY&~lnI9|E%)@b5gP`7lC#j*$^G?}1orODM>HIb%a+gDZ#{<i1s`zzdirG_?lw{DFhGlKqKpZ9{v_r23Pda~pnRpWOqXxY0UyL<KKhqu3z9z}Gh$R44>kbR#+dz;<x@U{O%f@gPlWRiN%(?i%E*3g_NweGHe_Ki9QlRa#`#$<!c8lropjyS@F?NyH)ZdgiUd!1{ayM5`%$$58RRs+2^EUV>_Gx8BFLtpV!$9S@KA1>LpB7aH}4TtVdHE|kD15-wV`?hkMDiM9or24pQ8cL!608mQ<1QY-O00;m803iSn__T@f1poky6951d0001VWpi|MFLY&dbYF61aBp&SE^v9JSZ#0HHW2>qUqSGbovEJe&TUe_!-_6j3v3<G0s}U~5NL_Exk{u)QgO2)|9!_diLzz4Y4ahG#5>;cp68Bt!5F{K8ReR7S&kGfX|9>zi0}ewRx!RqlI|o`3cjiYF<#J`!Y{6o*rG}lv_zuOb)(f3V>}wMsuohC`d(9|N29G26`~(&Dp>_eYO0CK?w(I2wPIe{8P2whlaeV7hE!8kM6&;Af)0{tnmGlxpV>n;;B-Z&x!_y23md)HO>zIqej1HN1>K^Y)aFFvit4>65_FZIT-;Gfc66CeZxXa6rJ~DOnjqt#IVQZo8>>JhW5sfoppx>P-ka}}7b65elIlis^vS-M9~{K-i&it~R{w3Nyg$3DC^ai-^vSskAHCn*N-c#JxhP=`zkd1d_dnjg`TO=~97*gbhx3`EWCkE3s7O_pG!_z@e@w{3nXM)F7zx?f#4`3nB%ozOhxR~EPDEe5e);B|kzrhq?IQXSrITX`7ObG}hdj!OraJ+#3?PDdc#Or&Ct74)oKa&~iiQ{CAdRljqGz6PE;Ps=X)R({91*gEkcC-uuWtRp6VC8HQ+r^ep)&B+#96ie!b4e-8o~-WmK6eW%-iGzE8ub5QY<%C&=Rd$`+>~(cJ|~oj-2ln=D0`jbRHO<4vIfLCq5ez|K$f9*5`hG?VVLkslnIdOdk7VbXGDgN_gP8wKuWqL@5f{>vmaTw+ZCL8ZF`Y#_W1X?sQA10sC6<&3)$52ORq(#bFiFYHi4HKwBa0U&gyF0}KkUA7Y!pjc&h{iSs&gpa!YFkFo4iBU=+F8lpoS#0Lq}D5^5DQKD?1>_OqoAq%Vx0hyyLZ7QPkJ&-2Uf&tDPQ{@Q62=nWk4%6Nd9*yM*a^JMwY;u7KCEbzy-r#z56&=+av{xa2yZ!a$tG}>=^1$@o-%L|U7zBF!Cn+0xE2WT7|4mevK433e_MiB@a>kPo?$vC)K7WW9Wm1E0cL1C|2!|)!V<p@MJTIF9aNn2`MWDPHl<7zRwIp&!WmXEM0@51-ZZS?F0)3==&_3&>S9Cd!al?UV1)fY#{q`v|zX;wccyNJPi|dG4lU01-d01Tu8P4+Y1q8mt`16~d^@vOD#vMm{FmaJVKVgtEH&7;IcAUHbUdi%%_yF)gX~>|I<WSOE9|KOB(c-h|_?fgbdz{Oy)}u$>6ThVR>ni;@CtF8`JROaG@@WT6-G?belR$Aa?cEsPnG6JDt0yL15;U{Fx1}I@j5*;ZSpJRiEyQZl0mV3Iv!c6PZ8o_p7eLxjfQ3RH(Y&$8vbH=jUsep-HX|wsLwPiq|IXYuT015G`jWAT#v6Z2lmH6{vNDw^YGM!fis_>>Qo*GIAV59ZLvys-LmPjo{pi|Vo5gnjc6S8|+;&CR0A$h^?JPyjX{Rwe)$nr0H}QdTClv`jv`^+1&+Hve_>PXRClOoS7-8;-cAyB?POmh2Ik=@4X*PT=%jN6PTAb+JEWS<;>~+EMI_B2T?401VXXmtix;lluXk>OEvT{mKpPkbhfS_|S;*Thc$T!y^DJXfa;E|1Frcs6km~KGaA3P9fDD<mO_&omSFn?G-9xA5LTfyn0p}MH|#ZW~tbYp7<AtGhwr-xirkcCV^2x`Kla{8{bGov6>{Hlf}y+6I|SI4(~JEpRBOr56_n-t6>#Tbfd=<C|%cKEWeHGg32ou?OPh`!pqRa+cug;Bvy>dW3--<~+>4*Ol_FMJh3D=kTF6e!VZx$v_?J9@aCR`^V;0p3+QI7S!{EWvGEa5&EGzd(XS=(l?!v!T6$_on0b^Q<392F2SGp30{~^1+c1ZWS8iS-*F}&^G3h0YY~a$MX%eoK)7(%-9s{&mg_PdE)ND!P53PZs=fqmxZcxOw(axTf(#uw_i@k+!6S=u?w7<tUL7~P9(mYvkYyq{~C65YiICJ&q;S22=&C0DHe%vZ)?u~P)h>@6aWAK2mk;8ApqZWFA$Lj001Eu000#L004Alb98erbY*jNUvhPBbZKp6E^v9ZSZ$Bvwh{jBUqL8eq^nl7eY<x#7Xq5Jx7QYjT+t>43PT`hiM9#Ll0s7Z7TkaD3`tp}?e%Vg)L7UnlJjt09}X2o(GOhTw{my2mYfsOXugq5i?$*6LT`!Qaw0jaNhg|ySLE|&_Mh+m__=0#M#R3W`JOl0tk0q-noQQR-II>#Z7o)~pp}~Z6CNiM_s~6RuJrJvTUl<M6o2Tr6o{uX%@n<RpY6Hcwv`%y*1h7iZZ|@!l-N1CdZeOCNmsL8iIu2@ex!S*b`}$HWZ!6j&6w7Jjg~n$#Wg()(iJOrys1v*DqgmgXf}gbJR$J;D~|Fn9m=ZI44H+Tt|V*9Eq8-!u#i`@lYA{62Ib9+gypiLit{R+B&Ykj-(g$u`5pV-Dw&d+Z&>+A6~h?F$z)RTHAJuLwTf3vm)m@nEX+2>#o3-UJ*&-T@wbWFU-z<c^WX*5%6Pe)rR3X`T+OG`lw7CeT~Q>N<Q-?);-+Mlq)rG7t*kfj<EGylxob0=IK?pJ)&+G<*MQYat|&;;n&C{Vgo(i(A!sYbMl_%;`0k#`D&B)zYiEwAAoiVj;I$(8VDa2qGzh|k5gBzeQew|GT;*0cC1dC@Uvg_w+kj%V*zp)dz);M%l0i80A7#(2QJIb6ashst6%%W_i3@D(hsh~!R@hysf@0yY%n4-9DY0jkU4QV>MG3P(#v7A(*1p)GBwqB<|B%5MQ;JJX%eqxO_BG0esv2DQqI9U=GxAn~ggfzO1H_W&7?ezHS;v>Ng8YHZ2>FJ9|6x}!6w(uF8cK((9tkiMk_uLp#jWn3#(3qq#SO(r>3mb9WcePFI1RK2)Em$|)y`=Q8a``O2Wh;V7VzgRx|$cy;6O?I*^(d>rQCt2WWZ%oJn=XZ6{h4`a2bK#(Z?zLL?35Sk&Jb;;SIMGO~FU9t@;v+snl5DAurejgv>v{UaBJN6w9kvLLR}!9)ekQC-^;z{t|up*WW)y3kUYqOzxm+H)hs8i?yc#x&=*I9E#ImEu$~r|KsX!5x8wo><yV_*N0UWH0!$xd>PwZr)8Cg(kUq`%Ia>*@_9BjPb;ojKFw~d;>duog-5<IESt^W23|5t%>g)a$jEu^J_BmnQQ2(bBt0xR@;x4VF{MzfoKhrF)z)`BPB7wl$U)Ypgg0ze^C~~}j{{B?M+*2%j$g@P9*BVha>)n&%Ur7P(TE}V;Xpu)!Bo{cdkp`DbGq&5E}xzbcKIwj9_ITFRRPL4o!qkr+VFd73SZ^3c{m+%a5(qojm?X3G+9p5kXjZ+rrTO59S;Ye>T3;7Iy$gNFP)M&oVdYBW%(|@Wwqj73Iwh-ndzkm+pj7{o)&%|6F&`|jE54BOHcY56u@Lou&S(Nruxpb2oOC^egFy`p@KFgK6Nw!0?}>R)_L^#<EQWb^54kQV~P1yjL-ammpvlo5m2CDb>wo?=SAz?%xnS7p)jA9htw?mj=+M@4jo-l(Nxf9G*}393(<n3#5D;DqEq!yMp$sKZ3doUDBx2|6$_tcZeaNK)74Mm=!dTt7%{rNj*?d%{tV{MHG=VK=;w?jHeukrGCkM_q=yrayDfhQK<jn~2w-F_q|zxcWZ8l>q9ISBi&;um9tYvNhGAFQSoFnF^*`S*Ao>zzi(!x9i9vbhbAAPCO82kF))>U4&(|MZD~B<I7#N+*04fz5D!;yYm)Op%wW$vnmrG&l`~cMkH-!AfYMmTf-ZMmUHVPIxwFNrpm0A;>wcxZG*?U<Wf`V1?U=4%9_aY(N`%4t&C1!fe{!0tGaNy9`6ycg*b_;TMW|h%s2NoT|p9x@gXdS+W0M6E6EF>l`HsPQl9j9V(q}Ou<G?LKh#!NJreR#Tqt42P89xdRj!Rfh;=D09ms;XmD{oKRJZI7bL5K{vu!7#e*o1MGBaL7gZYNC3XJ`CP#E)8ZObep_;xifx?o$<BljE$I&V2pVML`vQp6y%m8ptG0j;Vg~NpU~R&O~<MV+8;dFogB=(#?)VJBm!H}-)h~6aF#9O*qHCur6eAsks5W!F{43h6-iDyyyxljaLuy29v&|d=%p|iZK4<NdY6K6(7|B2r|{kn1{->&ytU`KEwPnsSyeJ@%d7`VXt5M=`9C@ZfhcqbLQ@JIZ@o^!GY5x;7%EGif$X3)u1oz&Ji>YL#m}vM9r16x%uNE(Z^$i%$DKl;J-)Kq^Gd+V`teUgl9&m$)=<oN&xJAYb#LFE1$ge&jK5*X?0&#px=!)BW4`9YznHz_@L|q-rE>y`ALt5aClg6}GIUDl__+y(k}w!Ut=VAM!xyUq@8rK*ID(AGSchF@Z1>6_qZ_z{CI1(HkB+^z@5$_0|L&YeM%o`3|FyT1lVg2+brsLe3%E)@cHF(u0(&=J@DA;;oqgi69i;HJUE=k}&#d@*?n?8tw7A}Zed!`HUMtNFDm}ge-uSl=Xnf|yYi}CMIcEJ^6e;}0M`W+etD2vC&Y$o~KHLutgtZmFi^=~0P)h>@6aWAK2mk;8ApmC+{f7Pz003Gz000#L004Alb98erbY*jNUvpz=Wo~0-E^v9Z8~bnD#`$;u6$F6+WJ+^-Sa$49!<sBfGjwgyJ%CLRh%<STZlOqqq#V0h`nT_U+>u9$lBIStU|T%iefPazcXD0#ud$L@tyU^_RQ4cBC9>LCsmFR(%IbGcmOI-#k*O1JWc;DX6}-($=iLu~cU0w+^5agGG6`MRy|`GF`PNB9E#g#Em8=}KE%LI4Sw$*hd7&S-qTXDXhitbko*Yp*S;6anc>k?g_kJtZ&8p(5mQ`(@>b#6MdPn(K$Wm>AiYlx{6+PU9i4yB9uWA)HNO%_x{JhVqT4b?&^Pmz5{{tEoNfJE(jFpl}v=V70qf)L_p7Cqo28<V_gb$I*R_U%?`tK@2B%(B5tGeQ`Qf~7H*+Nu$Q}dR%5mjO<n^%$SAXI%4MU;wrnO0GrrB6IsO94_SqZJU32yIFMn8MD0TYfL@RjO(|b(aD54RpZEKgg@CtT%bm<J0|4rAZX$SzYF-CTq3h-5NSKqTJ@0`U!Bk2TW3xN&QA;s#YS6G;AK^hv@z(QVEX$BX*TyyrZqCJ{-gILCH@&EJY~=tgC@5NmPm~0YNYrd>g~t8pMF>cw`ORRiNO}YL}*wLiOSosa%WrDa!BvE#sP2Xavg~xL4(6w3a#GUp_^#04pJeUR)${<s_=AOLf1ik>Si8&%e`>1|!Ox&-8#l?)N{wf9Kw@_J+000J67OjKZ718HE%4GhQzJ5WskTU@d#~=gV(~){Sw=6#rZgt$X|K^6x{-Xs0Xs(_8jMQ-!t`b)^{9;y#rTa2eID&f^?dn4s^dm27A>+BEO$fakNwz%hXQ7CaiN!~t#At6+&)zjWqvXVFsfX5?CQz3x7ZJ5Lk)=}qU|o9?@5=jp2RG_#-HcJ_PQf%&!r^KA#_TLW`x({9^=hm`qYwJS)i!KJ=M_j6?W7^WNkBa_^3-@N<!^*`OiO8^U>t!HsLCdRU2oFtDS8~1z`58w`J86cP8e}_hqrk)1bLH3)kfBeh$-@X6Wn?JcHZRFefXnDD4t0Ppj5#VakUBopk{Up_TQ&&-z*KCXtye`4cjefF)f_e_xtxW=34G<Eic_qC@s+Jq{v^VTCeB%0k2sqisxMze{Z1gkw`x5N1oh#}*osZ?^H4pu-&>EeD)-@qS-eEdAj+P}GkQhinfh-D{iIn^~`sE6|NFrejJw-&(9$aV|?^1!B-xJFT&hjPsN#fgRc9y%eI6l1cki#62qa6`bZfAQwlS0KPaEcs$W1ylHizrL+6cXK@zZflBd4cxROH}>^)4yPvms&_pGfD*Q4HLF{bZKddmH^<972y{A@OHOtF~TwNNMrC3=_q2~xf~iF!Di-j?&Bnm9WXo&XV#7%k!IlU%y|J!aK0io<7a^U0zn{-ml`i#VoVlRLiED9BU*O_A3r+3bv}Yq+!~9F>zx1xa*C2)HA>#+I-r57s<L&+wiGp1#2W++QbDWh9pmG9i^<aY{ds<10opO?cu2jc^9PImO9ytm@zJ_Hw5~sL(hG7aKZJ!S1*Rb2B0X1cq!Mty3yM!1i74hQ{njpJEux=fnMVpT$x^Kqh=5`^hi))MPLQG`C#M;2M7EZZ?H#>lv(tSjZh1gnI{)=LylS@qA=F|%0>)MOss@epF2~0lAns&Y3fqshl-D-YqKiR-?|Mtf9`Oa4BM%0~l?2)|EZNlaU0v+z9(m0UJuPS{JCz~^e~hM%dRk8SgD46u?o!53gOrsClqB<1lxu@<1f{`Kw8AXy7Y9eE{zQ5_1Pw}?tTb>4T@-b2Zg92ZqI|5O;F71rj`0Y7LNLY4<`eu`E(0g2p9(qW>8mss_4xXPh|+>N6W+l%W22@&90$i5S;cuNdkU%fGUy;}QPSFwcEA+P+GIM}7rCmm%Fr_7stzkNLuY@C4xO}@=j52-jk_tHF2riFjy_0PKq4nNsQ`Ue6>1~uaSp{Hr4QiJ5?P*~L`=i!6w0Pc2sgL4kWY+f5aNG@9zf`WpTL=h<10uNE^orwczlZBj7x6a5(5o^dl}ee<CX&b0^JkS8G;nHV3uF{;k4)2zUpE-LDd>U3qnvY-#9n+Vr?=IE(djMe0|8B^kcE#uPf}y8e`-TaNBT`pZ$(c=b3)5adCjI;bLSlUUraaXPs<5jCH$k&&$(Fko#B8WM3fWZP#P~IZ;MyvE6Flk0z4riWH(zEvbnEzQ9BTvMrTGt71Ai!|xx{ugM|RIlp2V4i|4ITo8p6E*fmCFwhVT7<Ri?1He#}Hj2&S&~S^?1}j#g(gBuQleiTHZ@pfkaLf}9k=uauZGE?dyDvsfRa0Ko8&yKJ1s*ZR3QM=JFqK=4&XDiP?4$VGu<w|2Xo+M0zn}Ks*otFSK2!)y2lT&%GtA>t^LSv-p`~Zl(H&VC31P#w;a;Sh!CEe+%}X|l5iUw(3~3oSnhJ8qb22e>OfX?zxfuX?1$b{PsxdoU*M>Be%w3K6C9iQi*%)j)ceRVr^Q;ME2=mqoD6g>srd~1U7k1V*l(86g^LEc>6;M%yvpo$W-LZ!`b7yEe8MfHMdj%an=vi*iFu)|yh6BI6Bg269+AyQ=_Lwb<_vo<!Fg9GdC0f%kp?LAaY}-p*FiKPqI6g?PXJI;<7>HRmHQlWmfdwp8Enw3SvY4%@?{d4DXz{!4CUrS(c@G<(AxgWv{PJuXe27*9xIyWVtCo*4W-}GLL}9!|=2Vf047)nV9ryVzOGNo(EdJ232LUt39&x@cz*$69CW>m4*VTnSpeier_~P@yrWquCc<x5M9Iy{os}Ux>{+Fc|81Ond6Ut>D$|qHC^jTe43ROYg^ffUU81tqq^O7LMo4IRq!_f_JS2KfsM~_yI$P6^1-R>hVk7Tljkchrj*9{L9B|3ZDZv1PH#499`Qshf02_U6hjHwjk$_$IKj?(NMU^VJ88Yqnp&mK3$Q3&x0Tt3)6cG$;F*T@jCkMfwravM%ZSJy9ZW{0Yd<A@mWSisJ}intDQsc+Ryl`wQZzrM|IJ1}srl|zR79IX_v>gOZO1XTn6bN1`1Y}K<}24i<fXwN(Fk{`W}BnH&;A0pjJQcX+w4&wr~-q`NV>7#Pe#d3Wg>1U&8u}o9cGwi9VEKxD+=v@JJS1)}Q`p9>FLn-uRHUm^ElpW6AsjPmf8@lRI5)>G&Y7u|%76g~Nvyf$^o_+*<5s#{xkGXM6$DHKm&HK^FU!o@a_5mfGvxn$_s3Gm~b^#sy&)py2y>dIowp#CWaefb|tV<L<ct1#>$nwB<R@MP)WDZ?Di#z795E#o;QABDO6M(wsso%^29lx8RL??IC>m{uCTe;&iGM$Nc&U)>Q0$T6)Yi-z3R7|Xm6<b=mD@#&RYZf1$b#e(Lu{&wtc)5#9rPf%%gd=QnUyZa95A>xGT*R?RqqF!K39)Nj9ykYthJe^Dti_gLBQb{;Hd_xY-eZ+XC^VjUw!ltpbbQ?_TC}&u(m$LS$!!6aNsOkNmsm^s2Qd`N4%PIMR$%?8r~~>(KmtpWgK9{oM9r!p{X9{to>x>tW>6x-j15Kh<eeh^j_$TnJYy-!q`&Zaf{?9eY@+Gv?{kzFuYL_9o;BBTczu$IwkgX|UTFJ|sC79X`THWVviTEt+`qUopj5EEwt+c1PJeL2VLqP=pKdSswn&af+$I^k2W-lx|Dah(k#*9*gFMv%&az6(6ONyoRqz*&%)IGM;LN75#+yw8XL^I5rjx+AzQRw}5J+dYzI~9^UIpSCKurUN8wXLGsv@Gu89g9<m4|!T$$edltXk#eR+h~@rq|!T|AE5q4+i{2_u>&)*#4YW{<z2dvkUog$wdexz>v64Cs6RjWnM&j!iMfe&>P6kKB=VM@H9&U_yQICG5#C^^<tMoEt9RitBF2JLsVc^KWk`k(rY1ok!v8;!;E@%4O~TB7UN|=ST8WUMgefL+D&-;;q;&nU_Viq-r>c7+1+H+NfY#?ElUImUlhZtP~eqCn0@fFS4$dS8FP4PGv78_fK9QWdFj!bYOa@Z$`x_;Ud`RCfCSmK^+q>O3n-ce=OM+u+%-@`SAqwABx*818o+j-1Z@Q^Frx#TGwy>J9CtxXY0%)=Ev9>XZu{dXFY2c5O{q45r;r^^Cw}in6Nzcz^1kXPKK~H~1bZklmhIIkD^t){0#Oso2RxpI6*^JHn}e7w0QO~`>|&kkH`O@Z&X9smgGZED-=G=?&dXcOLr2TjL+z>Qm{Qe_a+-Nij){74hoCzON~FTTp&N}JtK6_<-uUe%_x}qZ=CFmYV4zG8q<w?rh++t!yIY)hF1j|2QGMY6@7_Z60AawBKGdUV(_Sg;^B1IWJy0XuO|j;icK8a~2{jjYj28XNzn)u+v1^&+Yxw7lNz7CBN>n!J(p%P`KgTiPIe^0z;&9c$p{1;MHo30H;Dq<F!tp*@02d<@?_(cHuItGGAaot101upJ2nP?R2^1EGKX(9qu`ebk9jAEwwsPx`8b${;y@B%?d3?vTe%VhqQVDgT-j@_@OwPd!fsYUA8Z;fnAVW^@Ouf6$o<&#VXIfV3?#ko;lkR|;Sx`b`HKaQEqr1-t!T`x7Z3k!e9ZV-)w60IzwyzK=a9rgsbC+l2Se0C3XOS?pGps(p3^Q70HFwer!zJKQ|Bkw6vBFR<Q(0_8R=Z#PUm&=4M_X=fL2t!|h1*;K<w&-cnYkmt?bib#0+6owK<1wR+ra`)Z;0icON@N4KknRLG>U<F(Qz-sW(W~(Vi%VBnH<VFd9(SBfi2HNUqA2`2FoQn21Kfv!)&M{l~nV^(zm|G0PK&*9)2ZIeAn$6X!z9VJV%eUq->H8eu1MkRQy>0j?EV>93yeF&}2!rd@)4Hx%>M2*WWu^1?6J4c1Y*l2C(0~GOxcKg7@vJr!5s8F}I%qHO@HrTp&@al`N4z{LPZ~4!vpT@84cB7lhhIbknuJq*X`GrMll#G0k-P*Idcr@HY1Sv*GBbe9%L>IW3q#F?H~~<r$E@*F0`e?{*{fB^u`7Ggtav?Op6Mf@lFPIP|{k-VdbRzUXhdXo2IA)Zq7oxa|-)W<06z3<?M5X1@+&nuQDZuMu|O+$CN0vAbtxp=lvFbA##aIhNGiS+s5GS5RU5RsM1k9SCwu7?ZqecmF9b6F#DWEPkEAR`%DLz`mY4qR|(?CwMrczX5aG+{o^nEuZv#siv{d#}ors<67oxhC);-^GDA*347_hKs0=FGEb@ZnHYqSa!65QD9}mT07e?LRV+;cpX({hp7bm#v8^N76WuF|#a4-P83{Q57f?$B1QY-O00;m803iVF<_1?O4gdg@EdT%$0001VWpi|MFLY&dbYF9Hb!2%iaCx;_ZExea5&rI9A*d)IrHW!F`*uNHbZ>93U37c5*gX^l41=I0+Tlf(R8dai3-aIl3`t3pEGN0`c0X)va)ulZ=jEB9<+d!U+WS=%`D9%cTd(BxRVG)SG!vij)lAg;QmAJ53;t%}C%zS`<cXL}?0CN0mV1vYFE5*+a$gHon>9Xt_}HxQVaqqdu3Yh4t&3_as&<2SA3uE3B%i7x5lR)+#1M<SQdDv)@><0;SM26>ycP9Tk+!VnD_iZEOpo%YdYA6o(_jNx<lpboji^5ho?MAE^1cv>l@LtTF0ZjhoD}(5ZW`9ViCk2?2KPHXZrRM2LuSa4h?2-#1-B!gh1zBHD36k=7I`8ZCNot3RQ~pk<UM4we;byM8NcD5&DdlzNiwdK_e*gta`~I6E+-!Ti9)fOhNA8xlo)xd-nWvc?Kf}k`F}oBfAJ&lJo09dH=XFEQnB_JOL&$s7ATReBd->Bb>#6hWvUd3<e6DS&3~zOA~X`vKe^VV-hA#6W88|;k!Uj9_}p8ZFWXU(DKScO;r+?WL>}B0RT_GK^e(0z%*~^(p3L=bu_`lJ2Vu8bCF)(34=p{AdAX}`a5WDWvn2_8cp@!{w^)YQ7nEVAl2jx`DjJ1o;>c6ivV1}(Rv7=Sm+(6Fae;p|eig6f9j3&DR@X)y)Z3Dg7f0R}du4VNYqoROK7)3BRaY&acjozNC2xcuMlC^B;4WEJ6!m;ad$V;3H?2d=hrSVP`5nu}ZF9yK4x7q(sjiBei5pSv=QC$U2vYV`Ti3b6vVt&>g_o=nxI8FEsQGL<H5B2*=#6};Idx!^V5U-Db4>J3jUiu8@;onUEjmkO1u{(y5|tE{%r~qfP-7hN_8R{OC9f#xnA<c06}Q?Xjh&Gd$`mAWEk#AP7|hNuBJah^SFS@)90Ze&hcP)iuPVL|dZwP2F~VqCY-3mg$Dvu3Z-Q>|FqU~OHV7PjKKcjUp1*t<dBJy3GPb`6BQK?RKgX&&@V)7=gp<QhT7|8e*8Y9B^Fv(b8^3>=Sn(SX5L34VsV3%SDe|;0;{AcZzD)f~M|05#1#QEgNe3d5CC=gIBQ*cn!6rfD{~%OS$(8WxE2->B{1FBnOo-uzXXn$|>jBNfK!Kg@K!x3Djcf|*Ebh&Tp`rzST0Usg!B?^wQrxoJ!6#4Hz)X9rv)7U5)A>j&Q1}lni)+r{d7k<o*Lo}K2$_{|&l=GVNmi@kjzqc8i3P&H99u|<4W)H0IZ<2k*|hxwPK>=?E{|P7>xWUJPXQD|flCCW+0H=i5{5Mxgb2h2nX8&3O9Nn-YmW(5`C)up$yzW<GXagnDNvGf76{EA3D-D49WjEAe$fm*`wE*IJ90yGBc;`0lVkoL8fs4XJOwq>p&QFvTr=-Mnl57Kq}S=Cg*bteIu6lgKjJ7b%<p|9q>7R(6RD07Zyq8qsR|@(Gk+!I=8F9M4K#Qw)A~vib>7|WNH{uRY%MdvSQz6hMTQ6%#x_L&OC08{7zpLMZw(Y+bsEM_^w}|hE=6N*kf0s17Eab<v?lSp*JwpCLA2hJIKEbjN~DRa32P?FnR{GTrX3Ka9dWncQM~Ik-<SCe(~jH+Leji)h{Dl*k0;_Wah{|ufHRB=Ry=`WksJ8}hExFn`{GU{yT*-5B1ma?#p~pXwSIiw(@;X_9V;F}`T@u~)09~6P!ndxftg*IK@RFbi7i)TP)O_Wh(ZM><A;Rsi!AJsEzU0~c3#e=OK^}XhG?|!Gs8mW5N6P-(dg*nry>_4@f~s+sN)89isdx3&I(>%xQOtOZ5qFMiuf?S8E+CBp6uBg6%1W9?(IhoR3D8~%lrm6S7?;5(CKw~gtA2ZD3nN9##bm4o}^cN%gRcmGO58Niyd+3V;#^x^oIs#y>Fq8iTh0+SPt0erXvTS47|nOF{6YqcMNUf7>1$N*+QXIA&b-zqY_l$$b8SXn3ssu_?{G5CKA0S@ofdLb;^=*N3b68R|UGPp=u_-7W<N%$y)TW;y%e_Q}z;G7mUzmYEPnLkwM{lVLjTk5P;0x_CY{^Db&f2K52mNzn_v*zn{_nKfds7$q2l)kvYzT3*MGga!@>JYj{1V3C%}Fm=Xxh)B>I3>d8O@&WLKZ8%Fue@!&ptgO@c!ICRi~Ooz*wjUC0tN+4sSz;lF;w9$ee2R0v?m<ZjbiXyvKwK;Kjoq?~mNWQ1GLiLT>qej<JSJ~MF&y^C?Fd8Tb8X5=QOP7o4dn-{$Jg?kIBOx5EpkkgJso2%1<AI=vwwWeVN)BdwD<K>Nj+$w|h5zpFZ$Est)KlQJw4LEey@URH89<YrPh4bW$TgNXM{qLQV;h@wGfvH+JyRIXQ{{I(3~XfE1hA8;xC>gjwm2|AOv<`?z2o;zP-s6(z}9d<w2~=cKybpV4KDZO<ocFAxlIf%mGIu`^I*UCRR!GTziaNK!@t{|sy$#w4;N1RZE+b$rOz|h*k)#JwgP}FgXOj2b?JOdJwEo$tky1R%&Ak+*V(idDMOkFT6rmsQMb(rXn~QuYiV?%Xlk-|=)@S@tXZh0Dc1U|#@k4asR}ek?U-%&^_Pq3k`OC({Rp8SoZDUTSx)2{@_-v^=DH2DTmXJ^nnF>9OqCR>?pyZnVR>9+cZDWIKy2KKhCNOO>(Uat7NS&4<_&{RTWb!0-yWUaNF`TUhvdF4Z?!CLdzZt9)(9_c+uNY7k5sBu<18bM<Z`#lq(TTWY!lNo?T~&~Hjo$Cc0yM{&QK8b2F`n31?w|T7lM*!k3@TuhBT(X*JDSDG5`r62s);OyQ#pi_j`q=5gy{YDj2pXqxM{(!xS4|4Mt8SC_ItKh6K(>mrEf5(=f1lr{<Q(tk~COVE#nM3Ktt+&UCanYuca6AM`C<cXvCtM;f+mseIhKvF`JI`!Sb(V>8{OsdYNG-f@4^82d1-8#b$dtQ?w%)^)+Ryt<}R8X&E%kk<l#O~41%j*HjTg#l~Ju=C|q@=cBqf%>(SrO0G1;M;ULj6gTJv+;4NP!PJYTn*%EWoP)h$%SoI9zs;)94o=(39r|@jx?=DLlk%TQT#fd6C>-lk(cr9D&>HF)Y8x;fNGjfSvEwWbbPIeKzjv}IlQN-a9D8ztA{gD(Re3R``yX(deH`(w$)gloN#ElH#K&6yg)U1(R5lOn<aItWxQyeUA{pSxtz_86|Ik#sk%gUOHywrG$g}>3T7b%i6hn7ceCZgK?#Lq?L7a@4_=l)FmE*v?NI)on|r(TNSe?oRlwgXR2pLo=|!sfV&0*MwUNNJCu1OHm2l{D`UbU|X%B48r3nSP2lB(|r_b+yXbVyg1OI&c<sUR)wTF@KfBtu?6hrdi%NH84Dk6A!!z;;i+^_L6f=JY5w0p&}7F!iK_NC(?nF+-#t5~;<v&o88u=bt9p5EI%9!^ir8lb7$kC`8a17NNF!3mud8cY>PpGY(<fEzc>`yov$_Yv_(ankA(NISSVyt1dzcmI{M2}$0(EexDOy$iNICM#JpyBINL9l+Zg=|8Nln>!<#_EHa85(m2s+aG$5#Zsq|j!H*d0mO_<|7~lG&Lp7DwWu^0()EU}HFoOT)*0Wx<3{XTC;V<%!V)4q)l%G@;#k>^j@4MV+g*1aiHfS)l~h(a0Qpt|P1uyGHK~HWDk(NXAM+<P>bp1cln&ft$AQ`_{c8*+72~Ec#DS!N?2FPggzOf2Io3;tv|tL?)Xv&KkddQ}{p_PJc#NUs+KSQ_@3t!P4I^`fZLp$Z3dkSVV=KMnisu``6Tj*ZHmm?fr1=-ewP__gG8}_^^gwI2p8I7{3P1enW%~~w{XHIg%r^&}`yVtg@VG}s)jsy!*I`Fr>CV2<GlKjYUBO@zNjGqyt>3Ao5!vj@bAPp-z4{-t<DWqtXS-Yn2B-M<reW`R2E?I&V`av3vf5nK^nAnCs93(D0Niq!n>b{j)?Ku3ihlEY*3uk#MLTLjVT?^F+y`Mes5)r#VBw#hnvLk-bro2nz;Jm5)1z{X4ojeV&<LRtLl-tCzP~u1j#4C04Rn<wzwpeEUR=ByrzPB~@%)m~!=2pjwkYrc%cyKa^iWLayj)lQWg+vr?X*2sCP3R2R-wCHm<GQ_jc%R5cXffNhcaQE6}PNdiM?_kzgp4QO020`O9kOx3I@6lp^ldb&F;Mt!M4h^6<A}ifMxQS@wA<TVL9=2b%VVXD5<R;E@_eQ2|m6{W$myy!1bHfC{D*tYx$s0Hv$fNWr^nMg4G|qpYv?*)mOsPGXVa(JmuA%%2Ft7wF2Pte!;Zq>3W?7(9hH$N>MqBApi+f<O(n;lc`stQlSnf_2ZNVJBTF86NpHnLhJ;<@(TR{#^-I|e(@M4ed2c+CnwfWg3-4mqncQz<4;X%GXMGc=$u?ztq|HXe(URFG#1<dIJ1E!L%BV=mPWX<>7fuKk8^|<>jHO;g2Ardj6C&w4qGPyG+UGCex)$13EP*9`-jOBrrMdEA2~L7+`^ro*bV%W48M<!Iv+S;mwjt>@t#d*gP3ihsqGl!!kVkO99AQ3Ow~;+6D8X|3|q^J;tyHBKjgHIIJ86?>!ZP-F_Xcl&M4J(IE)Ifq#VAh#OXAC{n+#@o<2T(F|ub}6{t1_<2}_6%%lHf(#JHH7ffVwLycb6cAbuEu2EK}puQl>$$tS*O9KQH000080000X0Iz&Gu6!2&0A)P@0384T0CaC}Y;!MSb!lv5Ut@1<VPaoyZ**m1Z*OZZaCyyJ>vG#hmj17&=z)JM=}@3#d*Un|ZY?b_wko<UNo6LZcwrD|l88V6K?9PRY^vthKEUoiVV-2abGjRBP>k)VO4Y6^vBbWfKKC0v&wC>?S%#Gq)k=y|7P(5Qyxbhl!{|=NBFif|&-1%iLgjTC(UNtTWJ)BNxZYH&JQM3Yu2ZRe&+`rr7G=H`L9nQ+x|Bg6l68@nl?XE&6jn)|se=PMRxXRMRMLLNsfCOx`|(GWXZCMZ&x<mTq*8XIk{_yvGA#6kMOdxUWNvS}#z$PU03ZdP5iSAA;6UJqSNrx+K`hhcUX~lXvBQh#=k;hh4o<EvF3x7XX3!o+rvy>HUMJOdvXlyllYz#Qy;#q84`iyO1H%6S1P2GPT!?v{q_Mul-A;rL>M|XAtEwv0px<96)vBKRSk#|{(JD;eZd893E)Pdpd^nZMwalu+(VO#;kCmQ2<*EL)dw*9y9Q4?}uD#~gGoLj~-aGN9p@Fa2k(ZA)%|q4R;^0ibday-Km?$Z}3)5PT%Q7!J-ld@dm@~GBjV+reG2WL{iBuv^77JPWURPsHrl}G`ar^FoKNC{dVR;wl4_QZVc5da>epnPTi?@%bpI=&^=*OeQLA5EQHxQmZ)9XE6xRzBIhgFDmzx=vqUW0*ID<ys`={*)cUsYvNbh_dT;r+0C!>`S!?)HUI9!s;k2QrIQ663((HzC-%2kuEFks{9mNx0SH4jK$nho1>hvkxq@d-k9OY;$nuF&3(FGjI>HKVNL2NHPbCp73~Ich3$_eu!jIiPr#S%zr|a{m$upid9o(7)es4A6eS0@ryFassLAYx*u6@F;dpx(6#*dTH^ZM`R*UUp2QIQSW|wGQBBB!Tmn8Uz05QD2_EL7zf0r;oxI4)wf8HLUNe~>)w!R~^SUbPN@3USyN`*1q3_4pyb)ruV>IwY%tljj|5E(+_x}+m*EgbomJCa_Vxlib$45s(ekf#_kS<jEl$#>W!x+XiQH_!pRW45QG@Se522XkVln(ptKmSWi>P&=b`t+$-B&Di)WP>xIK;Nhm8vw1f>Qlbzt0VE|b$gpH-e13k${TuNUFF_;vsJYq#eZ}kDLEXk6?9d{8=81?edCJ@kTO+$B1S)gs+Xz{OH!vehinsBdh!iaxRwI1Rf($bUU6DtQd_h<jb$mqsLYiD3`&um=Nq*j&|pIO1WO;+3Ypy}Wu7tW;0j1%hN4o!8Hq)jM|k4q6<^&T@=lhSOa*Y8$_050RTx#i#{6pt&l1SWvvkwOKOyR@5>X0HRD9qf`G7l$dY&ecAS;q4U{V1?oGhU7`8)#OSX6jBgj8!HwnVqSkD;6vdcJ~8e7UA4_<Y(7x5_xF!g(rbT`Fn6rjX-O5pd4&^D@k$)vJbZuyr7<QnE2vgo~wd;bQGu5Ulb85MwN;4-o`am?5N*1wO6TyowV_`BR;gGWHoxl5L@~uu!YKqG@PzoDkmQ2Cy;{UOd1*;Hy@l0?Qa8jKGI$yebllG=CU~Ip}ytbh$nfVU71Q13y1HOx6UvTIY9?hn~G?#=ru=B?kFKyf>Cn3J!s7UID!+_>Z?lc?PjrYaDi-<B2j>P<%`5k2hVrrYdt7Z2EA0ba<+%@lBG2slhw5E@*xIynp?ofBk1O470C@Sh1LI$OMKE2w#jWd-E~#I$p|(F+(z!FNrpWWnp5hpfVImR$~znFqK3qRpGHpgEO`7B1{=tXhwBkALZar!qBn~%a~yf!DU&Tukjq5$!FzrB}<|dT(-21Xom0umMgKJxC+9spnX#OREH@ud{0X_s|EcUP`2T?uuR~KYk{c2r&eG{8yHgvY=z4##|t7W43L}x&XXkDf?JK^p+__CKzBoNlo35Sgg}BIK!9PRZ)#wTwcdkFmMiA6d)5PDPRvn3E;VHcZq^m7<W!wh(kEss<&UHZHJL2D>qLQFNW<Y6TrNSu^v=o;o}oo0O33$HfQgqZm%zRVm8#be5IztV!Z?HE7A>|0V@w<lkf7z2tcaWnYBmB$6Io;MdOVq)O=siFSunY}nvujC4GDiUbkroDHyK}F1vis(xHrF82mMc_HeBTCo}XP>Z<m%h&aYiH8BfQf$;n%1mDUM%!7e~U;q{4Ifq;HYa^An#u)OzS3lFC&VnqGe^Q)85`QuGhs1EO69zNvdonFTVik7g6Y8mJ!g2rG<wYie`3;czDRh085P%6695;7aT;ZsPh?4CknSgrL--zb&7RV00@DD+=i9rRs>#V6)J{D#!R#DpnddNKNIFda{)XIGcO#o6V}Y&-=B$1udWv`7)tvK;5dD~3BHZXpFp)^jZzE=TZ`Es4$B%3Z%}JU*pEZ;yK7xF?>yGi3X>o3qI{APES*I)d3Bidk7}tv$M#T?Ny(H?z~Lzg(IL2NktiX$@_}(0Gf&c#9%j;Dy>g8ddgB5J-?q-+XfkUb+0{Y;rN0;oa`VW^OjQI-l;2pPr50TwYCQXD5%QzB#)jMj2m>&Mr^K6HIe-(D3KvyK!)Rbv+KwE<oef_R|r#Zp;i8<xz`!o`})K>+z&dJFUvpTr#vYWQ7HM!^VRzjBStM=Vi$P`h#=;7>6$!|8+K=5SSOE%d>CBQ@WD;*7hU=Xj#6Qz8KHmUTGq}+wnl32)Qn*4W_g`PBKK40POh1tA^9GN`+;s)7ZLxFylXibXF;<<`wD22>%Q+Jw5vyqwAI!rzc+Y#8>a=&Nb*=lffX((*w_!aH}e~K1vIej16lPIS$>B`n)s7V>BaGK>YrCbn@-^Qa{U+(AuSk_(;V?=xH83A5CYV+w_c=pI^N>o3&^+y*@vi1;pPNeRlNh%j2VG$2@a#_1$<vn4Ddm<B!V|(Epki`7Bt7M#xgZxXo3b)|D|c8JUj5B8Sfhi1C(tG79B7&yt_97?ghnyh-`9w7T6y2iqX_QQ$Oi1I?CPwWCaStCo4bOr;-@BX7gISPPmtu(gL{xsEpt?@9E@2-0ukJC!g~swJRSKX9C25A=6pQNGS+84tkRL7lBB$m+CL+0AtZqm?W(Gu<{?%?|I0_}2=YCB=8-+vi!^!8HN4^)@LKYTV79h;z2d<k-4FCW6oD)y;$m3pO&E+nI&BFFWmIPS@T1L)JdcWCndWTY6V#f2ymy4e?BZg4od}!1f1uPk?eLa!4JWNN;o;f1|RsFC9zGN}7f+UGUnHpS63n4macgLfTpCh-Di(kWUenobQ0}Yj=X3X`L0zw0&CEHeM?&=lWu5bFK`VcH|UWP3>e8sI8!7p2I6Apu;j+F_@$SIl;I{(nd6Z&<gIM@p&jV;sh-bCJiInr4<eDVeaRsNvoxuut~=Z93ATDk*x{6%Z7S3BJ@AXhlU_VvT}%r+Tb5b1NWQ+aL*=tF}SL$B!w7R*<(o0bXwYn|BlBriB<ElCoT*N^u*iQ#reUf<@;bOdRp9eTqN@xCo~>f7=6k_cXpht-R1uJE>6l0c|eGTHOqqOFd5``{L$cxls7pPJI6nAzU)E+80B5t;<%GJ?TOv#oosBEa;u#km$SF)U4As#I+07a(BP&<*gujSOX|3s<Ij7ojg-}a1|*m=nG}SB{VHJRrM(7gJqP2x#<kXGL%tCSndjo@_Fa$Aq2ny&M=PS6y`Vw1s+>XAkB|lMvQJcVa-wF$hcL>r>c9hR?d+PEVuV=op&cKvx+yuLU`Ot|3muN@Of_-f;}nH&QZ=O+2r`mbKt>)$hk!8)+g8zCJcJepi&^{WiS}VWbV^w%f~4cF0|ndWG?uAwF1&Tlc<1T>{R@2V3=-ET***NC7`zl{jbkjFF}c@m{WO5%vctNq<>M`?OhHSNoA$LB1OCr{Y~4swjf&Q!(&qiQ48xO<o5u3GMVg&l`RACZGk8MU;uZ-<9ZvR48zlq!bN~3rKmL-{z7BO`OwAByui#(3@Q)GWRP^aSA?XrW{LdJq*crq;xj7y2X~)kn_^^@^=88faV&FytgO^z4r*av>eVmnHrYLC(@e(im=YRH}QJf#a1Sp%q(J2YGgqDX7O)ihb7;MAZzLIO9VKs(`sp0}gUUSBfQ(_`X8=II(7+Bmu_GZoc!?1|X-DyJdIjlTC%M|@olieITPxpb0&aDUJBYZqm>x7)*!$V$pxGd@+@o;Y~*Lk_|s~k2~FCg@U)i6@`y)0kJFort~wHt*6mr1p~H=3V{6I$!DGGQC%Kdh1nQsrFS)p_wI3|T?o00F?&VC(!R&oejhs9#E4n6ZL|G#~K{sF8N`w5npjVyiJ4HxfD}s$l1WVnu-@EP)3!c$}d(mog}x3FAEm+{FD&;Ll5VP;)thAUy2mlOElIIle_79buU(v~R;Qe!yKMfcrpy3>;Y8YpV&mhxYGW&0_Az7cC)Z8~y&fpb|3erS$jXFrG#`9)0$#VQ`@*Rwvl@y~v0A0yiBB4eIXp=$-!D@p@kO_VD;!dl|3jim%1XBm6uv%oDAcjcJ~d32FPFO%Bpj9^SluMZp%MTw7GmdlZy|(;lURus&fcr36@A_}=HCjgVnb_i<pSIzkRv(2arV5#DOcUPHr%eVs+p<&S>^j*jEW^(|LxSDBnquj?l&APwL71Y#-u+b#rgYyhH%tZ{8hhKoO>Q$u*1(EOB@SA7G8V&+SAIfZI87UN7BIPrO;VKXn(-nwIbad8Fa2n%agp+HILxF&?fLR*mGp7{=e(0iQa!xH5=F4m(G-0qK)bPM1+4cnM@X8X?VThcZ8Akr%9Ih5#@0H6JC-q>t&2Rj-CpMI!OjM+V~6(>>Efiq|T?uqN~XK$}A!HI7v`7!j4uyJ<v?fCNSZ-@inm`+CL=kP|R7ejABnJ~;>T%VhS%g`GwmyZjpm*=x@1A6|De8e`|vb6BJJ@Nh3`IXMyy$MdwCVHV8Cu}peUD{TsKu)?LY<pd623;jEbc(><_R63HV!gpmPy?iK8s5xq@1m>`rEa_dXXmyr^SEAvVM>AaxFEdBX}~1ZFf=C*CQ!-GaHa-M(#ts{tOzEmH*>>MMh863Vi9y9;wkQKqQz;xOsH_YFN@`zz79VGCVCClNmf?`$ey$?&NnJE2uZmQ9;trkP%voV)~>L6C+pg47IC(^S-`2+os%4)(d(R4tT~7Ch<b7-8S9><oiw0`3jBK#vqVU;EdkLDWNY$!&LOr$6;ydDsdkB@_NV*$0+{iSoV}#mU^Bddk@6lj6X+aiU~{(K;8CX5UB(4O!-0FUwl3SiCkbp>F5sMaz;y#}puI4!ap|?E><!wn!1kzZ^~f9i!W@Hm4F~`;f+ELX9rZl;wFRw#roo6$BY6=N&fj<5V1J*46+eU)1`cmg9DfP3ALoGzshVg&2J*A}fHXDMq-l5`2KUEUr_)jxJC!Oblevi*8CSli%Rs+6;vbOG_;ike*BXPkyW8gkXF#58hgSG;y)IOzrCxeMH{T4O8Rf$nT5u~{<qrSrb`b6O;H=$PU3baDUVF?3O*4j1mf-%SAEm}c*LQn84zah8c6*rPySyI%BCq@*9yTToYJG_RdpzJIV?JSxJ_p>r#!5rWEd2X+2rHoU?W*6k2w0}!+I^I;uM)e@+7iH{0+;tk1+dqq!k2SebX#pU<z8}tbOBc@+kvhEd;pVHGTn&dm%sh}zmLD72AF)If#ynp5=B}ox4YDIAhMST@07V*`%ahV*A#5BSgTh$!A4p4c0|wFSx&8(TJIILlRQ77F0CYM&cmXG<nX%3J|k86Pp(d`^s;A1$1hDK+s%ItL`ubMUDBqoc)_2!H$m61LqykAE8R<T6vD9+$R~2o1`XQ;NB=&N4^DT5Q}F3mt1_>bD@2z9(Q@uIPAsWJ<P_z7aV<;Jf-$*#11jE0Y79YGVH$!sUDVX`i4Z^Lb5In)tS?5Uf9FSy_q+gG+vGSmm_)QNezh&eFTy0PDaPO)5s`-FQYX!A_1p9ykz=2%*R^h)iB@%XrwQtGADF~5Emah!P}N=M=|3ybXO0gz4*}SRoKW_0{6aUe!8815^kC!Ot@u9?Q|kR;o@1ISh#bRDoS!cii~*<7V3B;N7#TVi`2VCwK3{q4C=j0Z_D(gKf_;-5w>eWLbKkjW^os7FbDP|3Uyc9tDYS`|V<*QhrVk0^mGaF*0uG*<k~qccrd5ZF_`Y?B!KuF=Fb@8~N7J599IKx`_!L%tcfY><lYniT(f23oZ?d}#h}Lq^&YDcMF4I%cY+$UI)1c5ZS=`wpIpVM_Up;aB$K^$JVbJuZQIAY%3%=*$H=~p9nYM37lZz_~zfN`S&9=fpvZ_3SgZ;`DGLvjU-C#C#s*3~ZGKV{}T5~s$?Q*0RKVO`@9i2^dCst#Epo}VsEtH|B9!7JvM!X^|mZ%k{{%A3`U{xD3j7^@DaG|Z4-@Fip94b4njpJtQoH<4H8g1HZ+Qc|By{uMTALC9Wg0qHCURHPr?yT}9VlHTJqi=QdVv<E^9ZUFQ<SrJ(VNMsA;u1m;Cft!`uGiHEu9KP$1xfFsLLgOhYI4ZLYr?6ZG<FtkG|twzkgNBGkNVpDxJ1&hv1<SUz%%-q%D~aq@#doFJutv3mVq&Sv07^Wk-{V8XvL~pr=P4So@ro}IMw6)*?Lim!w_<#TN;&SyrvSP<F}I2>p96rr`hQ*(|it12TgQ)_<Zng@DX+R1d>4d>%B*{b$zDK6`L=F_ulF<?H6h39X40@DGujB?aRBhA3MRWlIrYMaUQ|({{4G8<A)zI0>thvWf<#FK)3h*kYSw*)Dl9X6YUOA<kKj>$d)aL%+gOl<kF)<ULE0r^C4z`*SVeO0N)1MWz(huW6@BdZ&vu=4uIaiw#*;s3n5%b!#qD2iqCFAJe_si8@cD7ITafFYyRtB2w|S*{|KkHgRydVAm#yLOCMqx7P50s`kFENb)To{Av{RW08DnSG;<g-Gs;fst3N^2r%#!h<~6!X-Eh~_r{2!X>;Y6<@9+)XcG`U_4=@MG0y=~3$5u{S%O~3^r)e{%w~!Vo8^5~$s6YyJbE-8uyk@uswsm3K`$Y(m9Hu%wx$J!YVKL9dbl!2WW4eZU?k}XE-naUV_`vr))0|FiVf2*<-OdMR#%&ui{V9C5F0k=8_gd|Jj%x&ejku!eEeV^>Mcchx<%KWU@|(tY>h?~ud+MTO#7DR*s|%ZWhS@COCQ<O@H&(+mO}E|dDUPRx*j%^#32x1Qpr)rgDGS^3)TF_=`oZRo^u`{YZISz^k*mE{7rET*C4bMw8|d%5xcNQjODHWa_H=cUfO#~hvsIeSxoAz_TCw%lecy=zzLzzpKoKkOx_ztcNqWWZK|5c6Y0eDbvgG_7n)ZJ3?oA5?4L<jF58ytR*2dXr8r;7O^fy(KVw2735N^iHbSh8WzH^Ro{`9iBE^%Vbi#@}XJk87P<ge!2h{Q8ii}QCuOXx)Vy-<HtE^Er>T=Pt)aY*uP7zA-11wq%@>4$L~gl0>JQ$F0T@zFK9@``tw4*#LosF401e3*8Ta;Z2$#z*omI@Goy7{kwbvw#al0Mw_{LwXNhH7Zoy9=~(GhS@GNv$Jl>!dp+}*5j{QQm*#?kgvsu4&CgVLm)xk+wthsV-5fBrrG;f&-r!~#c~AUcBCmd8-->|iTtT4T@B4wM{O@eyX|JU=ZmQwzQ(-I8UdYKWEAxok-mOqGixo(XEsIcrW>1AIeAC>h9>2B+-vGSh`|%{bw&GaNX$XoKsyb+x>_85<vDs@TP4mM9++<!llTMyhhITJ3>^d}0@K`m@V`(?0|XQR000O8001EX8YA|>h6w-wBpd($8UO$QbZ>8Lb1!ahbY)_1Z);y<Wo&77Wpa5gaCxN}+j84B_FZ3r@`=n!tn4~=;;k~{G*0W8Ch^p6AL`X|AQG}rQv?fuzSQe~?>V@TASF58U5`f=fP-_t!ci2R)BJ{BGm?w-nrqDp!tQG(`I=RltVPk3Oc8O*B+)BImi!(d>Sj^$oGhqfC9hZ-MbT)qlwwV?Y}x2WvMeKfT??rRttz3Z=Au%gk=w(IhX;POqH0z0g<o>vzo}+XOOZ3B{Ent_$xG&!fAQKZ4e6R{MC0k6!*Vowe{yk}{qW_}r?X2kBhlgh!Sd+f^>lAh&?S4jI4Tb5^fhHi-@PsN_7?~H`#F1a^xaW3y7=<zFF%}S=O>pxUBJzFMBpbn`SJAg<wd$KqOsj5M6SNQJh{kzI|4F4pS%ywG*y{wDh;aiTb*i^-;(UlhD+Al(@X^mAxc~%qtU2fOOVqTEJL9)YqeMkp~udI$+5i`MIV@Ea?LBQG+0YZS}AK1nXmXQ8<UEmT`5s47Zn)3;F>N<hAJ!#N5*c(yoeI1fG$w8hGJrOnG)qKc*QI*Sema`ev^qt*NwKy!=J0@n(JtcQr}K$DivtPu}%I#qJKZy?@!Sjw(p9Wl}odGJ-d_*GkB6zY02x@Qg^caSH#^N{5qK|g<MlT1J8oBmt{;|g$@vKV_7htpfinD1KN*IX6KIUmG{w9)U1l}n^^oY<05_k0A@Y=5<BWiLZ~82G%{z$EswwrS+n&5%)%>onqD()_f(v8p!ma=4m#zk;Bt8Ex#X48U2qSgvb&3HZo4cgSB(5d%Z8mwDP$a-cFDnVN7jwfh6)9aCA)4)DhVQlv{980X_SnH{GisZ9Px^-*&NMhWd&19nrp+5g-BA#Xc2p7gX~g~tPKLB6MLMU0e!IS1hnL`UdULcTWebXfpuWKq_eY2KI!kg{)pVRQrdgqTzA&FYhX%29L?GrXXn|6(~m!&T%LaDAW8_va`x6b82Uy5S%6n21p5wxDjSo6YHFQCLe_3YOSxuiaFn5^R7(M(1Wcd7PTZnm8Rn^3y!ZCabWHZl@6`S#$p)s>Go9&&nuT}+(Kc^^aha@WnNh7J+?tBL>Gbg6fPDWw**omP2g0X!l4C}cF(R>oZ>X)$qp{?bcP&?7QnjM{uMcCqpQI~xU+`-{>uhS&IdjItkh?{%@4MY&OC(Z|W7ae}HferROMVMFoLH}#-0r#LZ9uWwIY$Hdpdtk@J$D4C-a3RcA^^bGWD4#WWxy5lH`sTHC|kUF4X3syr?6mEE(+#Q7R)Rns&dXFhxW-u7wBILDE5w97i+Qm3pT<$EHD-zX(%W|;Gh_zwa6M-c7^J*&}<>Z4S^30hNksDiP8mDb4(#)vS8r%P?svDX$dXIH34=28DBzSbL3PyGWaf6T#;o{<>)&EzUQLmP@WbKrh}<Gt!q{>Nr8V*Hfico(1M6%55`uQQdA7_sAdV|kZelMAnnseuLQUzI4Kk$mkptkLmpIs25y``07T`wsMKIPy~k<{0{G_1Wu4sgNgybyr=)quutV&mOuf#`-+n(8?D(E4*n++)SpA_y6|RK}SCWM*P`d7|yS^;g_05=xznu-4t3F;Dh_l$oQbN9&?FuIXW}?+Ymt-YNmrYqh>CT}UiLU5mIi0+nKORh<AnS(cZx}dG6hXZOi+l!K_HrqlViJveQ92$5q|E|BO5H4sXB8w2UXWY#bf8iU4HxzDOc!o0!W`IDFD&PYSJ>C$9d*~Da9Lvh9KRQqJKsN<P*u&Ethq8u_55y&Vc!J$vt(@79fZA$LXv>5P{(uY3z$a49jGfqcQN%SR5L4WbUA|NxIKkbAnr`#SwgGjVN>WUwMxi4vIj0xkpBMFbdL6Z?3Ifxt$KU3*|N7sTUz%HW6;8Po354j8&4ae{AXsWyJ0b$Hg-%JPl2XPg^GIw9oGIHs01@js}`;@x`3{!(Z1S+pi!De2HrGxp(Zy#TX!JewxHO&nT{+@kdWWbv}mC_h>m(P+TCXb-I$32(<?bQ?mWje%-|dDD8=TX<ZZooafP1aU`-6+y7gCEIp+yxpT5Unla=DJu@)v0+(9KdWSxx>tAaeVfwkYD?Tk<aCu>^qCAic^csem4a^se#O!2?AGKE3$hP)zs)BS^2ul8TafMcAn`<$^t0aC1~1)E@PCWcnYA*4f=HfJNBa0e_MZ<|YkgwRv8F$7kl^ddX^_~|m<7+4H=zs-F2ZrEEU-k~3%y=~F`a{A-bOD3RSD@B)WS$0#*_8}+mJ~P80cgaxTL1uSXcPDq}K36@>%wNL!Mac5*jK49d^(lc+?qn30EDZiA#X&m(J3zSgX4YUvhjiOFUpt=X*l^k5yb=?SH$qd%xNRQovaLXK*KKz11l^#%6iqdFp6F5UC{sN_ISq0AGzDzi!$emVb6T4RFFQB{&lFxehI(rAGltDhZM1AG8CV0qKzDh~E62ws<yKxVrfiFcw_Qi9?KfvHw8Ac397phJJfUCRdd&4tHdQnpn8@j>eNpVxMP@K3ekD_F`2H|;x$YXDG0iYZ=<p5Gq*<8eoNWDkF?!BgfrEm_7EDuPycd~_=DMNf|NL?m{d9WrVbdj_doKeeJEGs%CiJ-;@qD18HpYi3119lXE*ebty*z-;ynEQefg@_$e+R!MMTinZ6`@lc$yD6JVhYVrwy;wOogS+!T(?7oO_O6NKhPme%$X00LFIj6Wc+{8@qC1QHA?!Cvb|Kc74l_K^lWt#*by=vOrLtAK7=XX&`z5Q$lk=@9STO=On6ssJAFlOanjMNvCmL}Ees8CZ5*)IH~5@s7s~2_7+>j-`fE^}V7{dC!Au%}0=vcx88Vd210z}{?OqR}Ew}9}Q?|g?vMs`$SEBowxjb%fj14&_UH2~MYk!c1{hM5n`cV-B{dGEVZ%#SmHTDpXFr*$6^B(1A*0Eax5kFr#8t51})Z#21j*?FyQY$9@U&91i(X49~yR$JMEr5&J{@A=NW`NC_AVD?$tHR1|pNL#y9LTa6_TxTXcMYG8iuQji#sk{<vE2f6NGb|8VmF*mFx0x4w>PlQk<|f!8;Y-v_YU1$-KNBj1&k@nne`*{=yM-ODU2aOOVc@q95!&z+%_4^dRVgg0*OD{rugC8gVFbHD)xaiFwIs_W0z#BVcaO`PDXzNP)h*<6ay3h000O8001EX)6YoXUjqODDFy%l2><{90000000000q=5hc002QpMNU*xE^TB`O928D0~7!N00;m803iU_5q!**3IG6B6aWAe00000000000001_frbMB0Az1tb1zguQ(sm$Uqeq!K|(HVWKc^10u%!j000080000X0Q%IsVMhP}08Rh^02crN00000000000HlFU4gdgUcyMKMX>Db0baO9XUukY>bYEXCaCuNm0Rj{Q6aWAK2mk;8Apni)N%*b+0080u0015U0000000000005+c$qoPjWq5F9a%pX4Zgg`mbYXK}b~i6yUukY>bYEXCaCuNm0Rj{Q6aWAK2mk;8AplImFOB39005;(000{R0000000000005+cvkw3OWq5F9a%pX4Zgg`mbYXK}b~i6!b!2IDE^v8JO928D0~7!N00;m803iU@8JnOC2mk;v6aWAj00000000000001_fzToV0A+Y^WpZh4Wo~qHFLYsZUv@VyV{B<IaCuNm0Rj{Q6aWAK2mk;8Apn9)uv@AR008AN001EX0000000000005+c6Dt4!Wq5F9a%pX4Zgg`mbYXK}b~i6%X<=t>Z*z2MV{<NWc~DCM0u%!j000080000X0I_}`M|>dw0C{Bq03ZMW00000000000HlHZIsgD=cyMKMX>Db0baO9sVRK)0H!o#&VQh6_bZKvHE^v8JO928D0~7!N00;m803iVWj!d6d2mk<t761Sq00000000000001_fwo)#0A+Y^WpZh4Wo~qHFLYsZUv@VyWq5RDa&BR4E^v8JO928D0~7!N00;m803iVR#)2p83;+OBDF6T+00000000000001_fk|Zm0A+Y^WpZh4Wo~qHFLYsZUv@VyZDn+5Z)9^WaCuNm0Rj{Q6aWAK2mk;8ApoAXkg}Bq003<g0015U0000000000005+caB=_uWq5F9a%pX4Zgg`mbYXK}b~i6@a%FH~a%E&LaCuNm0Rj{Q6aWAK2mk;8Api;4XH75-0068k000~S0000000000005+cL3scGWq5F9a%pX4Zgg`mbYXK}b~i6_WpHnDbS`jtP)h*<6ay3h000O8001EXH03a7#wGv&@tgnv8vp<R0000000000q=Bo3003opaAk67ZDnqBb1!sZb6<8hFLQKtWO*)dc~DCM0u%!j000080000X01WDgK_U?V0HZ$u04D$d00000000000HlGctpEUJcyMKMX>Db0baO9sVRK)0H!pK^b!2&8WNBe%Zf|pRX=8IPaCuNm0Rj{Q6aWAK2mk;8ApjtKw4J350037h000{R0000000000005+c5x)QcWq5F9a%pX4Zgg`mbYXK}b~i6}X=QhFE^v8JO928D0~7!N00;m803iT><wZK70002d0000S00000000000001_f$z)!0CHt<b!l>CZDnqBb1!3WY++(8VQpe%axQdubWlqH0u%!j000080000X0J4w2luH2s0Pz6;02=@R00000000000HlG+%>V#$WpQ<Ba%F90Zgg`mV{dF>Vl84}b7d}ccyv%p0Rj{Q6aWAK2mk;8Api=OHlV%$00001001EX0000000000005+cPR{@Ua%FLKX>w(4Wo~qHFJo_PVPY+CWpZO>aCB*JZZ33qbWlqH0u%!j000080000X0OeApZ(slb0BryO02crN00000000000HlFS&;S5*VRK({b#8QNZDlWCUukY>bYEXCaCuNm0Rj{Q6aWAK2mk;8Apkq}{&yA+004(I000;O0000000000005+c;Lrd7bYXK}a&>NWX>DaMVRCe7W?^G=b1rasP)h*<6ay3h000O8001EXs>wksRv7>Q)>Z%j6#xJL0000000000q=78p004Ahb6;|GZggpFWiMfKb7gdME^v8JO928D0~7!N00;m803iTEBmPWR1polB4gdfc00000000000001_fwlPn0CZt<UvhPBbZKp6FJW|aWo~q7Z*DGdc~DCM0u%!j000080000X04Mp58wwBr0KqN*02lxO00000000000HlFJ{{R4VVRK({b#8QNZDlWFZ*O#SbaG*EE^v8JO928D0~7!N00;m803iT|slQyi2LJ%k7XSbi00000000000001_fr1YK0CZt<UvhPBbZKp6FJo_RW@%?GaCuNm0Rj{Q6aWAK2mk;8Apmx>18Yba000J1000vJ0000000000005+ca2EjpbYXK}a&>NWX>DaMWMOn+E^v8JO928D0~7!N00;m803iSq7M46%3;+O~B>(^y00000000000001_f$1*+0CZt<UvhPBbZKp6FJxt7Z)9n1XD)DgP)h*<6ay3h000O8001EXi-(N^AqfBgrW^nO82|tP0000000000q=9xk0RVJib6;|GZggpFWiN1La${w1bZKvHE^v8JO928D0~7!N00;m803iUiKS&oE4gdh%H2?q_00000000000001_fzCw%0CZt<UvhPBbZKp6FK}gYb7^yQWo~0-E^v8JO928D0~7!N00;m803iTLK5x-I0{{S@2mk;W00000000000001_fgn@?0CZt<UvhPBbZKp6FK}{iW@&6`Zf7oVc~DCM0u%!j000080000X0LQX`Dq#%(0IDni02TlM00000000000HlGFR{;QYVRK({b#8QNZDlWVb#8QNZDlTSc~DCM0u%!j000080000X0RH-LcNGEv06YW$01*HH00000000000HlE^WdQ(mWpi|MFJo_RW^`q9bS`jtP)h*<6ay3h000O8001EXdi3mO{sjO4QWO9H7XSbN0000000000q=9T_0RVJmb98erbY*jNUtw}|X=Y(#baO6nc~DCM0u%!j000080000X0OP=9pvwpV0G1yB02BZK00000000000HlGRZ2<suWpi|MFLY&dbYEd}b7gdME^v8JO928D0~7!N00;m803iTnY*!xp2LJ$~761Sf00000000000001_ft+>$0CZ(@baO9sWpi|2VRd9_bS`jtP)h*<6ay3h000O8001EXsk<({<^=!%!w&!e7XSbN0000000000q=CqO0RVJmb98erbY*jNUt(`>baQlaVQ?;Rc~DCM0u%!j000080000X08PUhVPFUV0Jj|g01^NI00000000000HlHAgaH6_Wpi|MFLY&dbYEm)bYU)Vc~DCM0u%!j000080000X0GzOH{`LX@06GT%02u%P00000000000HlF)jR633Wpi|MFLY&dbYEm?VP|e{b98BAb1rasP)h*<6ay3h000O8001EXa8}Du^9KL`X&C?j7ytkO0000000000q=BH20RVJmb98erbY*jNUuAY-Y;|FDX>V>WaCuNm0Rj{Q6aWAK2mk;8Apke;9EPt3002T0000&M0000000000005+c$(R8EbY*jNb1!sdb97&BZ**m1Z*OZZaCuNm0Rj{Q6aWAK2mk;8ApmSe<W}+o008F>0018V0000000000005+cte*h@bY*jNb1!sdb97&BZ**m1Z*OZ~WMyn=c4cyTE^v8JO928D0~7!N00;m803iU`Pxt{_1^@tM761Sl00000000000001_f!m}30CZ(@baO9sWpi|2aAk6HX>)XCZewLGaCuNm0Rj{Q6aWAK2mk;8ApqE~0}Y%8006ob000&M0000000000005+cZL0wQbY*jNb1!sdb97&Da%FH~a%E&LaCuNm0Rj{Q6aWAK2mk;8Apj8gw2AQr004{=000yK0000000000005+cJ+c7+bY*jNb1!sdb97&FWpHnDbS`jtP)h*<6ay3h000O8001EX-*hh!kp}<(Ar}Au6#xJL0000000000q=8_#0RVJmb98erbY*jNUvhPBbZKp6E^v8JO928D0~7!N00;m803iTp6a9w%4*&pKH~;_@00000000000001_fh52I0CZ(@baO9sWpi|2b7N^`ZewLGaCuNm0Rj{Q6aWAK2mk;8Apq^>23IK#005LN000vJ0000000000005+cRnP$dbY*jNb1!sdb97&GbaiBTE^v8JO928D0~7!N00;m803iUcd^xUs7XScdJpce500000000000001_fv?^H0CaC}Y;!MSb!lv5Ut@1<VPaoyZ**m1Z*OZZaCuNm0Rj{Q6aWAK2mk;8Apja9_Q8e;001N$000^Q0000000000005+cX7&L9bZ>8Lb1!ahbY)_1Z);y<Wo&77Wpa5gaCuNm1qJ{B001xmFaSCY001Na0ssI2'}

def restore_source_package(package, checkout, persistent, repo_url):
    """Notebook bootstrap; stdlib only, before installing scientific packages.

    This function is copied by the notebook generator so upload of the notebook
    alone is sufficient. Its authoritative source remains this Python module.
    Existing baseline files cannot be replaced by the source package.
    """
    import base64
    import hashlib
    import io
    import json
    import os
    import re
    import stat
    import subprocess
    import zipfile
    from pathlib import Path

    if package.get("schema") != "tas-private-source-v1" or any(
        re.fullmatch(r"[a-f0-9]{40}", package.get(key, "")) is None
        for key in ("base_commit", "source_commit")
    ):
        raise ValueError("Invalid versioned source package.")
    data = base64.b85decode(package["archive_b85"])
    if hashlib.sha256(data).hexdigest() != package["sha256"]:
        raise ValueError("Source archive checksum mismatch.")
    checkout, persistent = Path(checkout), Path(persistent)
    base = package["base_commit"]

    def owned_path(name):
        relative = Path(name)
        allowed = len(relative.parts) > 1 and relative.parts[0] in {
            "tas_runtime",
            "experiments",
            "requirements",
            "tests",
            "tools",
        }
        allowed |= name in {"AGENTS.md", "docs/TAS_V7_COLAB.md"}
        return (
            allowed
            and not relative.is_absolute()
            and ".." not in relative.parts
            and (checkout / relative).resolve().is_relative_to(checkout.resolve())
        )

    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        members = archive.infolist()
        if len(members) != len(package["files"]) or {
            item.filename for item in members
        } != set(package["files"]):
            raise ValueError("Source file manifest mismatch.")
        if sum(item.file_size for item in members) > 8 * 1024**2:
            raise ValueError("Source package exceeds its code-only size limit.")
        contents = {}
        for item in members:
            if not owned_path(item.filename) or stat.S_IFMT(
                item.external_attr >> 16
            ) not in {0, stat.S_IFREG}:
                raise ValueError("Unsafe source member.")
            content = archive.read(item)
            if hashlib.sha256(content).hexdigest() != package["files"][item.filename]:
                raise ValueError("Source member checksum mismatch.")
            contents[item.filename] = content
    if not (checkout / ".git").exists():
        subprocess.run(
            ["git", "clone", "--no-checkout", str(repo_url), str(checkout)], check=True
        )
    found = subprocess.run(
        ["git", "cat-file", "-e", base + "^{commit}"], cwd=checkout, capture_output=True
    )
    if found.returncode:
        subprocess.run(["git", "fetch", "origin", base], cwd=checkout, check=True)
    for name in contents:
        baseline = subprocess.run(
            ["git", "cat-file", "-e", base + ":" + name],
            cwd=checkout,
            capture_output=True,
        )
        if baseline.returncode == 0:
            raise ValueError(
                "Source package cannot overwrite a public baseline file: " + name
            )
    subprocess.run(["git", "checkout", "--detach", base], cwd=checkout, check=True)
    actual = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=checkout, text=True
    ).strip()
    if actual != base:
        raise ValueError("Public baseline checkout mismatch.")
    previous = checkout / ".tas-source.json"
    if previous.exists():
        old = json.loads(previous.read_text())
        for name in set(old.get("files", {})) - set(contents):
            baseline = subprocess.run(
                ["git", "cat-file", "-e", base + ":" + name],
                cwd=checkout,
                capture_output=True,
            )
            if not owned_path(name) or baseline.returncode == 0:
                raise ValueError("Unsafe obsolete source path.")
            (checkout / name).unlink(missing_ok=True)

    def save(path, content):
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".installing")
        temporary.write_bytes(content)
        os.replace(temporary, path)

    for name, content in contents.items():
        save(checkout / name, content)
    receipt = {key: value for key, value in package.items() if key != "archive_b85"}
    receipt_bytes = (json.dumps(receipt, indent=2, sort_keys=True) + "\n").encode()
    save(checkout / ".tas-source.json", receipt_bytes)
    stored = (
        persistent
        / "code"
        / ("source-" + package["source_commit"] + "-" + package["sha256"][:16])
    )
    if (
        not stored.with_suffix(".zip").exists()
        or hashlib.sha256(stored.with_suffix(".zip").read_bytes()).hexdigest()
        != package["sha256"]
    ):
        save(stored.with_suffix(".zip"), data)
    save(stored.with_suffix(".json"), receipt_bytes)
    return receipt


## Connect storage and restore the exact code

Drive authorization may prompt once per runtime. No credentials are saved
in configuration or outputs. Source restoration is automatic and verifies
both the archive and every file before installation. It cannot replace any
file from the frozen public baseline. The same caches work in CPU/GPU sessions.


In [ ]:
try:
    import os, sys, json, subprocess, pathlib, shutil, time
    from IPython.display import display, Markdown, HTML
    from google.colab import drive
    drive.mount("/content/drive")
    persistent = pathlib.Path(PERSISTENT_ROOT)
    persistent.mkdir(parents=True, exist_ok=True)
    checkout = pathlib.Path("/content/tas-v7-checkout")
    if (SOURCE_PACKAGE["base_commit"], SOURCE_PACKAGE["source_commit"]) != (BASELINE_COMMIT, RESEARCH_COMMIT):
        raise ValueError("Regenerate the notebook to change its fixed code versions.")
    source_receipt = restore_source_package(SOURCE_PACKAGE, checkout, persistent, REPO_URL)
    actual_commit = source_receipt["source_commit"]
    os.chdir(checkout)
    if str(checkout) not in sys.path:
        sys.path.insert(0, str(checkout))
    # A new idea notebook may reuse this VM with different experiment modules.
    for module_name in list(sys.modules):
        if module_name == "tas_runtime" or module_name.startswith("tas_runtime."):
            del sys.modules[module_name]
    import importlib
    importlib.invalidate_caches()
    print("Public baseline:", BASELINE_COMMIT)
    print("Experiment Python source:", actual_commit, "| verified files:", len(source_receipt["files"]))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Isolated, pinned environment

Python 3.10.16 and matched PyTorch 2.4.1 / torchvision 0.19.1 CPU or CUDA
12.1 wheels run in subprocesses. The legacy Transformers 4.35.2 stack is
preserved. No notebook-kernel restart, system Torch replacement or CUDA
extension compilation is needed. GPU mode validates its actual CUDA path.


In [ ]:
try:
    query = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
    has_gpu = query is not None and query.returncode == 0 and bool(query.stdout.strip())
    effective_phase = ("run" if has_gpu else "prepare") if PHASE == "auto" else PHASE
    mode = "gpu" if effective_phase == "run" else "cpu"
    print("Hardware:", query.stdout.strip() if has_gpu else "CPU")
    print("Phase:", effective_phase, "| environment:", mode)
    if effective_phase == "run" and has_gpu and REQUIRE_A100_80GB and ("A100" not in query.stdout or int(query.stdout.strip().splitlines()[0].split(",")[-1].strip().split()[0]) < 70000):
        raise RuntimeError("This configuration requires A100 80GB; hardware check stopped before environment/model setup.")
    if effective_phase == "run" and not has_gpu:
        raise RuntimeError("Select a GPU runtime, or use PHASE=prepare/score on CPU.")
    if effective_phase == "run" and not (persistent / "preparation.json").is_file():
        raise RuntimeError("Run this notebook once in a CPU runtime first. That downloads/persists models and data before A100 allocation.")
    from tas_runtime.bootstrap import ensure_environment
    python = ensure_environment(mode, "/content/tas-v7-envs", checkout,
                                diagnostics_dir=persistent / "diagnostics/environment")
    if DATASET == "amber":
        from tas_runtime.bootstrap import ensure_amber
        ensure_amber(python, "/content/tas-v7-envs", checkout)
    env = dict(os.environ, PYTHONHASHSEED="0", TOKENIZERS_PARALLELISM="false", MPLBACKEND="Agg",
               NLTK_DATA=str(persistent / "nlp"), YOLO_CONFIG_DIR=str(pathlib.Path(LOCAL_ROOT) / "ultralytics"))
    configuration = {
        "persistent_root":PERSISTENT_ROOT, "local_root":LOCAL_ROOT, "stage":STAGE,
        "seeds":SEEDS, "methods":METHODS, "batch_sizes":BATCH_SIZES,
        "attention_backends":ATTENTION_BACKENDS, "last_position_logits":LAST_POSITION_LOGITS,
        "max_session_minutes":MAX_SESSION_MINUTES, "require_a100":REQUIRE_A100_80GB,
        "run_confirmation":RUN_CONFIRMATION, "run_controls":RUN_CONTROLS,
        "run_diagnostics":RUN_DIAGNOSTICS, "run_final":RUN_FINAL_REMAINDER,
        "reserve_pope":RESERVE_POPE_IMAGES, "split_seed":SPLIT_SEED,
        "coverage_tolerance":COVERAGE_TOLERANCE_PP / 100.0,
        "dataset":DATASET, "external_manifest":EXTERNAL_MANIFEST, "external_image_root":EXTERNAL_IMAGE_ROOT,
        "auto_continue_after_smoke":True, "source_revision":actual_commit,
        "generation":{"max_new_tokens":180,"temperature":1.0,"top_p":1.0,"top_k":None,
                      "precision":"fp16","do_sample":True,"conv_mode":"llava_v1","prompt":"Describe this image."}
    }
    config_path = pathlib.Path("/content/tas-v7-config.json")
    config_path.write_text(json.dumps(configuration, indent=2))
    def run_phase(phase):
        subprocess.run([str(python), "-m", "experiments.tas_v7.cli", "--config",str(config_path),"--phase",phase], cwd=checkout,env=env,check=True)
    if DATASET == "amber":
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp"), "--amber"], cwd=checkout, env=env, check=True)
    run_phase("plan")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Prepare or run the study

First CPU session: approximately 17–18 GB of checkpoints plus annotations,
selected images and NLP resources; existing verified files are reused.
Only 30 train images are fetched, not all COCO train2017.

GPU sessions: lightweight preflight → local asset restoration → one model
load → baseline/view audit → correctness/throughput pilot → configured
stages. Per-image RNG makes retries/resume independent of job order. A
rejected optimization is recorded; baseline failures block larger runs.
Results use immutable chunks and perception caches use shards on Drive.


In [ ]:
try:
    phase_started = time.perf_counter()
    if effective_phase == "prepare":
        # Small NLP assets enable the offline scientific fixtures first.
        subprocess.run([str(python), "-m", "tas_runtime.bootstrap", "--root", "/content/tas-v7-envs", "--nlp-root", str(persistent / "nlp")], cwd=checkout, env=env, check=True)
        run_phase("check")
        run_phase("prepare")
        run_phase("score")
        display(Markdown("**CPU preparation complete.** Files are persistent. Switch to the A100 runtime and Run all again."))
    elif effective_phase == "run":
        run_phase("check")
        run_phase("run")
    elif effective_phase == "score":
        run_phase("score")
    elif effective_phase == "check":
        run_phase("check")
    else:
        raise ValueError("Unknown phase")
    print("Session phase seconds:", round(time.perf_counter() - phase_started, 1))
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Inspect saved reports

LEGACY and HARMONIZED are separate protocols; do not infer a matched
algorithm gain across them. CHAIR and annotation category recall do not
establish caption quality or preserved relations. Inspect ambiguous images,
captions, failures, coverage, length, timings and uncertainty together.
Reports include descriptive comparisons for paired captions with exactly
equal word counts. Batch latency and amortized seconds/image are separate.
CPU scoring/report regeneration is available with PHASE="score".


In [ ]:
try:
    import csv, html
    from IPython.display import Image as DisplayImage
    runs_root = persistent / "runs"
    for run_dir in sorted(runs_root.glob("*"))[-3:]:
        display(Markdown("### Run `" + run_dir.name + "`"))
        session = run_dir / "session.json"
        if session.exists():
            display(Markdown("```json\n" + session.read_text() + "\n```"))
        gallery = run_dir / "views.png"
        if gallery.exists():
            display(DisplayImage(filename=str(gallery)))
        for summary in sorted(run_dir.glob("*/summary.csv")):
            rows = list(csv.reader(summary.open()))
            table = "<table>" + "".join("<tr>" + "".join("<td>" + html.escape(value) + "</td>" for value in row) + "</tr>" for row in rows) + "</table>"
            display(Markdown("**" + summary.parent.name + "**"))
            display(HTML(table))
    print("Persistent artifacts:", persistent)
    print("Reuse this folder for later sessions/new idea notebooks. Do not delete frozen manifests or rewrite completed runs.")
except BaseException:
    import traceback
    traceback.print_exc()
    release_runtime()
    raise


## Release the session

With `AUTO_SHUTDOWN=True`, the final cell flushes Drive writes and releases
the runtime after preparation, evaluation or a planned budget stop. Errors
also invoke this cleanup before propagating. Reports remain on Drive.
Set it to False only when keeping the runtime for inspection.


In [ ]:
release_runtime()
